# 04 · The real run on the GPU (Ollama): the whole prepaid base to train, RUN_SIZE lines scored, every customer decided by the LLM

The same notebook as `03_agents_nba_fast_ollama_gpu.ipynb` (FAST_MODE, every agent on **your laptop GPU** through
**Ollama**, only `http://127.0.0.1:11434` is called), on **the real week prepared by notebook 02**
(`data/weekly/<D>/`), with these changes:

| | What changes |
|---|---|
| **Dates** | read from the data, never typed: **D** = the most recent purchase day (`subs_btl_auto`), **T** = the campaign day (`date1` of `targeting_dataiku_auto`). Purchases (takers) are counted **from T to D**; training (`*_auto`) is anchored on **T**, scoring (`*_score`) on **D**. The run stops when there is more than one `date1`, when fewer than 95 % of the purchases fall between T and D, or when an `*_auto` window goes past T. |
| **Data** | step 0 builds `data/real25k/weekly/<D>/` from the full week of notebook 02: **training on the whole real prepaid base** (every eligible line of `base_prepaid` outside the lists, `TRAIN_LINES = "all"`), and **`RUN_SIZE` target lines scored** (`1_000`, `10_000`, `25_000`, `50_000` or `"all"`, drawn at random from the lists of `SCORE_GROUPS` and split between them). Every output goes to `data/real25k/`. No generator. |
| **LLM** | **no time budget, no line limit**: every customer gets an LLM answer, nobody gets the cell policy. Each request is retried 6 times (timeouts, connection errors, invalid JSON; back-off 2 … 60 s), Ollama is restarted when it does not answer, every answer is cached at once (a re-run resumes where it stopped), Windows is kept awake, a **pre-flight check** runs first, and a progress line with the ETA is printed every 100 answers (and written to `agents/llm_progress.log`). |
| **6 offers** | step 7b tops up the empty slots with the engine's own rules (the short list, then the same action's offers, then the favourites, then the segment's other allowed actions); every added offer is in the catalogue, has both scripts, and does not conflict with or repeat another slot. The auditor blocks the file unless every row has **6 distinct, valid offers** (A-25). Step 8b shows why lines had fewer than 6 and what was added. |
| **Report** | the final report: dates, takers and taker rate, agent 3 propensities, offers per line before / after the top-up, LLM answers (fresh / cached / retried / failed), the cell-policy share (must be 0), customers per minute. |

Run it from Jupyter (**Restart & Run All**) or headless: `python notebooks3/run_notebook04.py` (no cell timeout).
**Run order:** `02_prepare_week` (the real `data_raw/`) → **this notebook**.

## Before you start: Ollama

1. Open the **Ollama** app (or `ollama serve`). Your models: `ollama list` (`qwen2.5:7b-instruct-q8_0` is the default
   here: 8.1 GB, fits the 24 GB GPU).
2. Once, in PowerShell, then quit and restart Ollama:
   ```powershell
   setx OLLAMA_NUM_PARALLEL 8
   setx OLLAMA_FLASH_ATTENTION 1
   ```
3. Watch the GPU while it runs: `nvidia-smi -l 1`.

In [ ]:
# --- settings -------------------------------------------------------------------
RUN_DATE = None                 # set by step 0 from the data: D = the most recent purchase day (never typed)
GENERATE = False                # True: dummy data for a test only (real exports are never overwritten)
N_LINES = 3000                  # dummy lines
GENERATE_TARGETS = False        # True: dummy lists for a test only (real lists are never overwritten)

OLLAMA_URL = "http://127.0.0.1:11434"   # local only
MODEL = "qwen2.5:7b-instruct-q8_0"  # your best installed model (8.1 GB, fits the 24 GB GPU); "auto" = pick from the list in G1
PARALLEL = 8                    # customers sent to Ollama at the same time (set OLLAMA_NUM_PARALLEL to the same)
TIME_BUDGET_MIN = float("inf")  # no time budget: every customer gets an LLM answer, nobody gets the cell policy
MAX_LLM_LINES = None            # or a number: at most this many customers for agents 3 + 4 (None = the time budget)
AGENT_TARGET = "target2"        # the agents decide Target 2 (groups a = T2C and b = T2F); Target 1 is not scored
EPSILON = 0.1                   # agent 3: exploration share (the logged propensity makes the uplift measurable)
NBO_SHORTLIST = 4               # agent 4: offers per validity the LLM scores (4 x 2 validities = 8 offers for 6 slots;
                                #   8 = longer prompts and answers, slower); the other allowed offers keep their rule score
SEED = 7
FORCE = True                    # re-run every step (False: steps already done with the same inputs are skipped)
PREVIEW_ROWS = 8

RELAX_UNUSED_CONTRACT = True    # accept exports without the columns no step reads (tenure, life_in_nw_range, subscription_id,
                                #   UBSCRIPTION_ID spelling) and empty MSISDN / ACCOUNT_ID keys that no join uses
TARGETING_AUTO_OPTIONAL = True  # targeting_auto may be empty -> nobody excluded as "in another campaign" (H-05)
REV_DECREASE_OPTIONAL = False   # True = run without REV_DECREASE_204_SCORING_TD.predictions: PROBA_REV_DECR missing (flag 1) for every
                                #   line -> S-9 fragility uses FRAG_DATA / FRAG_VOICE only; the NBA tables get uniform(0, 0.02),
                                #   as the classical NBA recipe does for a missing score (it uses proba_rev_decr in its context)
SCORE_VIEW = True               # NBA logic: train on *_auto (anchor T), describe the targets with *_score + party_scoring (anchor D)
SCORE_MIN_COVERAGE = 0.5        # a *_score table that holds < 50 % of the target lines its *_auto twin holds is replaced by
                                #   the twin (said in step 4b)
MAX_FALLBACK_SHARE = 0.2        # agents 3 / 4: warn when more than 20 % of the LLM answers failed (cell policy used instead)
STOP_ON_FALLBACK = False        # True = stop the notebook in that case (the run is then not an LLM run)
TEST_BUDGET_MIN = 90            # TEST_10K: minutes for agents 3 + 4 (G2 prints how many of the 10k customers fit)
# --- which Target 2 lists to score -------------------------------------------------------------------------------------
SCORE_GROUPS = ["t2c"]          # ["t2c"] = group a only: all RUN_SIZE lines from target2_a (the default here)
                                #   | ["t2f"] = group b only | ["t2c", "t2f"] = both, RUN_SIZE split between them

# --- FAST_MODE (notebooks3): agents 3 / 4 answer once per customer profile ------------------------------------------
PROFILE_LEVEL = "minimal"       # the fastest: several customers per LLM answer (step 5b prints how many)
                                #   "coarse" = finer, slower | "fine" = about one answer per customer (very slow with no budget)
                                #   | "off" = one call per customer | "auto" = the finest that fits TIME_BUDGET_MIN (= "fine" here)
FAST_CHECK_N = 100              # agent 3: this many customers are also asked with their exact facts (0 = no check)
SHARED_LLM_CACHE = True         # keep the answers across weeks (data/runs/_llm_cache.jsonl, no MSISDN in it):
                                #   a profile already answered with the same model and prompt costs nothing
# --- notebook 04 real: the real week of notebook 02, the dates from the data, the sizes ---------------------------------
RUN_SIZE = 25_000                 # target lines SCORED: 1_000 | 10_000 | 25_000 | 50_000 | "all" (the full lists, ~50k each)
                                  #   drawn at random from the lists of SCORE_GROUPS, split between the groups
                                  #   (25_000 with ["t2c", "t2f"] = 12,500 + 12,500; with ["t2c"] = 25,000 of group a)
TRAIN_LINES = "all"               # training: "all" = every eligible line of the real base_prepaid outside the lists;
                                  #   a number (e.g. 25_000) = that many random eligible lines (a faster test)
REBUILD_COPY = False              # False = reuse data/real25k/weekly/<D> when it was built from the same week and settings
MIN_PURCHASES_IN_WINDOW = 0.95    # stop when fewer purchases of subs_btl_auto fall between T and D
STRICT_AUTO_WINDOW = False        # False: an *_auto window may end ON T (the subscription windows include their anchor
                                  #   day, BR 4.1) but never after it; True: every *_auto window must end BEFORE T
TOPUP_DROPPED = True              # the lines code 5 drops (F-04: no offer in slots 1-4) are topped up and kept in the file
LLM_RETRIES = 6                   # per answer: timeouts, connection errors, invalid JSON (back-off 2, 4, 8, 16, 32, 60 s)
LLM_TIMEOUT_S = 300               # seconds per request before a retry
PREFLIGHT_STRICT = True           # stop when the pre-flight check fails (Ollama up, model, 100 % GPU, OLLAMA_NUM_PARALLEL)
TEST_10K = True                   # internal: the base is a copy; agent 2b and the NBA training table read TRAIN_VALS only
ACTION_POLICY = "greedy_rule"     # agent 3, how the action is chosen from the LLM's scores (step 6):
                                  #   "greedy_rule" = its best action when it leads the 2nd by CLEAR_MARGIN, else the
                                  #     segment's rule action (BR-39, as the old campaign); EPSILON of the lines explore
                                  #   | "greedy" = always its best action (EPSILON explore)
                                  #   | "proportional" = the previous draw p ~ score (most lines got another action)
CLEAR_MARGIN = 10                 # points of 100: the agent's best action must lead the 2nd by this much to beat the rule
import time as _time
NB_START = _time.time()


In [ ]:
# --- the step timer: the time of every step, its estimate and the time left (do not edit) ------------------------------
import json as _json, time as _time2
from datetime import datetime as _dt, timedelta as _td
from pathlib import Path as _Path

_root = _Path.cwd()
while not (_root / "pyproject.toml").exists() and _root != _root.parent:
    _root = _root.parent
STEP_FILE = _root / "data" / "runs" / "_step_times.json"        # the times of the earlier runs (step names only, no MSISDN)
STEP_ORDER = ['Step 0', 'Part A', 'Part B', 'The LLM does not stop', 'G1', 'Pre-flight check', 'G2', 'The groups to score', 'Step 1', 'Step 2', 'Step 3', 'Step 4', 'Step 4b', 'Step 5', 'Step 5b', 'Step 6', 'Step 7', 'Step 7b', 'Step 8', 'Step 8b', 'Step 9', 'Step 10', 'Step 11', 'Step 12', 'Time per agent on your GPU', 'FAST_MODE in figures', 'G3', 'Final report', 'Proof', 'Time per step']
STEP_DEFAULT = {'Step 0': (3, 3), 'Part A': (0.3, 0.3), 'Part B': (0.2, 0.2), 'The LLM does not stop': (0.1, 0.1), 'G1': (0.5, 0.5), 'Pre-flight check': (0.1, 0.1), 'G2': (1, 1), 'The groups to score': (0.1, 0.1), 'Step 1': (0.5, 0.5), 'Step 2': (15, 15), 'Step 3': (25, 25), 'Step 4': (2, 2), 'Step 4b': (20, 20), 'Step 5': (1, 1), 'Step 5b': (2, 2), 'Step 6': (60, 60), 'Step 7': (60, 60), 'Step 7 (CPU)': (8, 8), 'Step 7b': (0.5, 0.5), 'Step 8': (8, 8), 'Step 8b': (0.1, 0.1), 'Step 9': (0.5, 0.5), 'Step 10': (0.5, 0.5), 'Step 11': (0.5, 0.5), 'Step 12': (5, 5), 'Time per agent on your GPU': (0.1, 0.1), 'FAST_MODE in figures': (0.1, 0.1), 'G3': (0.3, 0.3), 'Final report': (0.3, 0.3), 'Proof': (0.1, 0.1)}    # minutes (full week, test copy): only for the first run, then your own measured times
_NB, _SIZE = "04_agents_nba_fast_ollama_real25k", str(RUN_SIZE) + "|" + ",".join(SCORE_GROUPS) + "|train=" + str(TRAIN_LINES)
try:
    _HIST = _json.loads(STEP_FILE.read_text(encoding="utf-8")).get(_NB, {}).get(_SIZE, {})
except (OSError, ValueError):
    _HIST = {}
STEP = {"cur": None, "t0": None, "done": {}, "start": _time2.time()}


def _mm(s):
    s = int(round(s or 0))
    return f"{s // 3600}h{s % 3600 // 60:02d}m" if s >= 3600 else f"{s // 60}m{s % 60:02d}s"


def _estimate(key):
    """Seconds: agents 3 / 4 from step 5b once it ran, else the time of your last run of the same size, else the default."""
    g = globals()
    if key in ("Step 6", "Step 7") and "U" in g and "keep" in g and "s_nba" in g:
        kept = g["U"][g["keep"]]
        nba = float((kept["group"] == "t2c").sum()) * g["s_nba"]
        if key == "Step 6":
            return nba
        cpu = _HIST.get("Step 7 (CPU)", 60.0 * STEP_DEFAULT["Step 7 (CPU)"][0 if RUN_SIZE == "all" else 1])
        return float(kept["sec"].sum()) - nba + cpu
    if key in _HIST:
        return float(_HIST[key])
    d = STEP_DEFAULT.get(key)
    return 60.0 * (d[0] if RUN_SIZE == "all" else d[1]) if d else 0.0


def _save():
    try:
        allh = _json.loads(STEP_FILE.read_text(encoding="utf-8")) if STEP_FILE.exists() else {}
        keep = {k: round(v, 1) for k, v in STEP["done"].items()          # not the report cells, not an outlier
                if k not in ("Proof", "Time per step") and not (_estimate(k) > 0 and v > 10 * _estimate(k))}
        allh.setdefault(_NB, {})[_SIZE] = {**_HIST, **keep}
        STEP_FILE.parent.mkdir(parents=True, exist_ok=True)
        STEP_FILE.write_text(_json.dumps(allh, indent=1), encoding="utf-8")
    except (OSError, ValueError):
        pass


def _close():
    if STEP["cur"]:
        dur = _time2.time() - STEP["t0"]
        STEP["done"][STEP["cur"]] = dur
        print(f"[time] {STEP['cur']} done in {_mm(dur)} (estimate {_mm(_estimate(STEP['cur']))})")
        _save()
    STEP["cur"] = None


def step(key):
    """At the top of each step: closes the previous step, prints this step's estimate and the time left."""
    _close()
    STEP["cur"], STEP["t0"] = key, _time2.time()
    rest = STEP_ORDER[STEP_ORDER.index(key):] if key in STEP_ORDER else [key]
    left = sum(_estimate(k) for k in rest)
    print(f"[time] {key}: estimate {_mm(_estimate(key))} | elapsed {_mm(_time2.time() - STEP['start'])} | "
          f"left about {_mm(left)} (end about {_dt.now() + _td(seconds=left):%H:%M})")


def step_report():
    """The time of every step against its estimate (the last cell)."""
    _close()
    rows = [{"step": k, "minutes": round(STEP["done"][k] / 60, 1) if k in STEP["done"] else None,
             "estimate (min)": round(_estimate(k) / 60, 1)} for k in STEP_ORDER]
    import pandas as _pd
    t = _pd.DataFrame(rows)
    total = (_time2.time() - STEP["start"]) / 60
    print(f"total {total:.1f} min (RUN_SIZE {RUN_SIZE}); the next run of the same size uses these times as its estimates "
          f"({STEP_FILE.name})")
    return t


print(f"step timer ready: {len(STEP_ORDER)} steps, estimates from "
      + ("your last run of this size" if _HIST else "the defaults (first run of this size)"))


## Step 0 · The dates from the data, and the real copy (whole base to train, `RUN_SIZE` lines to score)

**The dates** (never typed), from the week folder notebook 02 wrote (`data/weekly/<D>/`, the latest one):
- **D** = the most recent purchase day of `subs_btl_auto` (the scoring anchor);
- **T** = the campaign day = the `date1` of `targeting_dataiku_auto` (the training anchor);
- purchases (takers) are counted **from T to D**.

The run **stops** when `targeting_dataiku_auto` holds more than one `date1`, when fewer than `MIN_PURCHASES_IN_WINDOW`
(95 %) of the purchases fall between T and D, or when the week folder was prepared for another date.

**The copy** `data/real25k/weekly/<D>/`, from the full week of notebook 02 (run 02 with `RUN_SIZE = "all"`):
- **training:** `TRAIN_LINES = "all"` = every eligible line of the real `base_prepaid` (8 digits, outside both lists);
  the tables are linked whole (no copy, no filter). A number = that many random eligible lines, the tables filtered;
- **scoring:** `RUN_SIZE` lines drawn at random (seed `SEED`) from the lists of `SCORE_GROUPS`, split between the
  groups; `"all"` = every line of the lists. Its `_manifest.json` carries D and T. It is reused when it was built from
the same week with the same settings (`REBUILD_COPY = False`). Runs, cache and outputs go to `data/real25k/`.

In [ ]:
step("Step 0")
import csv, gzip, io, json, os, shutil, time
from datetime import date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

_root = Path.cwd()
while not (_root / "pyproject.toml").exists() and _root != _root.parent:
    _root = _root.parent
REAL = _root / "data" / "real25k"
_weeks = sorted(p for p in (_root / "data" / "weekly").glob("20??-??-??") if (p / "_manifest.json").exists())
if not _weeks:
    raise SystemExit("no week prepared by notebook 02 in data/weekly/: run notebooks3/02_prepare_week.ipynb on the real data_raw first")
SRC_WEEK = _weeks[-1]
SRC_MAN = json.loads((SRC_WEEK / "_manifest.json").read_text(encoding="utf-8"))


def _rd(p, **kw):
    return pd.read_csv(p, dtype=str, keep_default_na=False, encoding="utf-8-sig", **kw)


def _dates(p, name):
    cols = pd.read_csv(p, nrows=0, encoding="utf-8-sig").columns
    real = next((c for c in cols if c.strip().lower() == name.lower()), None)
    if real is None:
        raise SystemExit(f"{p.name}: no column {name}")
    return pd.to_datetime(_rd(p, usecols=[real])[real], errors="coerce").dt.date.dropna()


_p_btl = next(SRC_WEEK.rglob("subs_btl_auto.csv.gz"), None)
_p_tg = next(SRC_WEEK.rglob("targeting_dataiku_auto.csv.gz"), None)
if _p_btl is None or _p_tg is None:
    raise SystemExit(f"{SRC_WEEK}: subs_btl_auto and targeting_dataiku_auto are both needed to read D and T")
_buy, _d1 = _dates(_p_btl, "FULFILMENTSUCCESSDATE_TIME"), _dates(_p_tg, "date1")
if _buy.empty or _d1.empty:
    raise SystemExit("subs_btl_auto or targeting_dataiku_auto has no readable date: check the exports")
D_DATE, _campaigns = _buy.max(), sorted(set(_d1))
if len(_campaigns) != 1:
    raise SystemExit(f"targeting_dataiku_auto holds {len(_campaigns)} campaign dates ({_campaigns[0]} .. {_campaigns[-1]}): "
                     "T must be ONE campaign day. Export the last campaign only (one date1), then run notebook 02 again")
T_DATE = _campaigns[0]
if T_DATE > D_DATE:
    raise SystemExit(f"the campaign day T = {T_DATE} is after the last purchase day D = {D_DATE}: check the exports")
_share = float(((_buy >= T_DATE) & (_buy <= D_DATE)).mean())
if _share < MIN_PURCHASES_IN_WINDOW:
    raise SystemExit(f"only {_share:.1%} of the {len(_buy):,} purchases fall between T = {T_DATE} and D = {D_DATE} "
                     f"(minimum {MIN_PURCHASES_IN_WINDOW:.0%}): export subs_btl for the purchases from T to D")
if SRC_WEEK.name != D_DATE.isoformat():
    raise SystemExit(f"data/weekly/{SRC_WEEK.name} was prepared for {SRC_WEEK.name}, but its last purchase day is {D_DATE}: "
                     "run notebook 02 with RUN_DATE = None")
RUN_DATE, CAMPAIGN_DATE = D_DATE.isoformat(), T_DATE.isoformat()
PURCHASE_DAYS = (D_DATE - T_DATE).days + 1
print(f"dates: D = {D_DATE} (most recent purchase, subs_btl_auto) | T = {T_DATE} (campaign date1, targeting_dataiku_auto) | "
      f"purchases counted T .. D = {PURCHASE_DAYS} days ({_share:.1%} of {len(_buy):,} purchases inside) | "
      "training *_auto at T, scoring *_score at D")

# --- the real copy: the whole base to train, RUN_SIZE lines to score --------------------------------------------------
if not (RUN_SIZE == "all" or (isinstance(RUN_SIZE, int) and RUN_SIZE > 0)):
    raise SystemExit(f'RUN_SIZE = {RUN_SIZE!r}: 1_000, 10_000, 25_000, 50_000 or "all"')
if not (TRAIN_LINES == "all" or (isinstance(TRAIN_LINES, int) and TRAIN_LINES > 0)):
    raise SystemExit(f'TRAIN_LINES = {TRAIN_LINES!r}: "all" or a number of lines')
LIST_KEYS = {"t2c": "target2/a_constraint/target2_a", "t2f": "target2/b_free/target2_b"}
SIZE_ENV = {"t2c": "FSDR_T2_CONSTRAINT_SIZE", "t2f": "FSDR_T2_FREE_SIZE"}
DST = REAL / "weekly" / RUN_DATE


def _canon_id(s):
    return s.astype(str).str.strip().str.replace(r"\.0+$", "", regex=True)


def _canon_val(s):
    s = _canon_id(s)
    return s.where(~s.str.fullmatch(r"\d{9,}"), s.str[-8:])


def _find(cols, *names):
    up = {str(c).upper(): c for c in cols}
    return [up[n] for n in names if n in up]


def _headerless(p):
    opener = gzip.open if p.name.endswith(".gz") else open
    with opener(p, "rt", encoding="utf-8-sig", errors="replace") as f:
        first = next(csv.reader(io.StringIO(f.readline())), [])
    vals = [h.strip() for h in first if h.strip()]
    return bool(vals) and sum(bool(pd.Series([v]).str.fullmatch(r"-?\d+([.,]\d+)?|\d{4}-\d{2}-\d{2}.*").iloc[0]) for v in vals) / len(vals) >= 0.5


_lists = {}
for _k in SCORE_GROUPS:
    _p = SRC_WEEK / f"{LIST_KEYS[_k]}.csv.gz"
    if not _p.exists():
        raise SystemExit(f"no list {LIST_KEYS[_k]} in {SRC_WEEK}: SCORE_GROUPS = {SCORE_GROUPS} needs it (notebook 02 writes it)")
    _df = _rd(_p)
    _c = _find(_df.columns, "ACCESS_METHOD_VAL", "MSISDN")
    if not _c:
        raise SystemExit(f"{_p}: no access_method_val column")
    _lists[_k] = _df.assign(access_method_val=_canon_val(_df[_c[0]]))
_rng_s = np.random.default_rng(SEED + 1)                           # the scored lines: RUN_SIZE, split between the groups
_full = sum(len(v) for v in _lists.values())
if RUN_SIZE != "all":
    _quota = {k: RUN_SIZE // len(_lists) + (i < RUN_SIZE % len(_lists)) for i, k in enumerate(_lists)}
    for _k in _lists:
        if _quota[_k] < len(_lists[_k]):
            _lists[_k] = _lists[_k].iloc[np.sort(_rng_s.choice(len(_lists[_k]), _quota[_k], replace=False))]
        else:
            print(f"note: RUN_SIZE asks {_quota[_k]:,} lines of {_k}, its list holds {len(_lists[_k]):,}: all of them")
print(f"scored: {sum(len(v) for v in _lists.values()):,} of the {_full:,} lines of the lists "
      f"(RUN_SIZE = {RUN_SIZE!r}: " + ", ".join(f"{k} {len(v):,}" for k, v in _lists.items()) + ")")
_want = {"source_created": SRC_MAN.get("created"), "campaign_date": CAMPAIGN_DATE, "train_lines_setting": TRAIN_LINES,
         "run_size": RUN_SIZE, "score_groups": list(SCORE_GROUPS), "seed": SEED}
_info_p = DST / "_sample_info.json"
_old = json.loads(_info_p.read_text(encoding="utf-8")) if _info_p.exists() else {}
if not REBUILD_COPY and all(_old.get(k) == v for k, v in _want.items()):
    print(f"copy reused: {DST} ({_old['train_lines']:,} training lines + {_old['score_lines']:,} target lines)")
else:
    _t0 = time.time()
    rng = np.random.default_rng(SEED)
    if DST.exists():
        shutil.rmtree(DST)
    DST.mkdir(parents=True)
    _base = _rd(next(SRC_WEEK.rglob("base_prepaid.csv.gz")))
    _VALc, _IDc, _ACCc, _PARTYc = (_find(_base.columns, n)[0] for n in ("ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "ACCOUNT_ID", "PARTY_ID"))
    _bval = _canon_val(_base[_VALc])
    _pool = pd.Index(_bval[_bval.str.fullmatch(r"\d{8}") & (_bval > "69999999")].unique())     # BR-02 eligible MSISDNs
    _listed = pd.Index([])
    for _k in LIST_KEYS:                                     # outside BOTH lists, whatever is scored
        _p = SRC_WEEK / f"{LIST_KEYS[_k]}.csv.gz"
        if _p.exists():
            _d = _rd(_p)
            _c = _find(_d.columns, "ACCESS_METHOD_VAL", "MSISDN")
            if _c:
                _listed = _listed.append(pd.Index(_canon_val(_d[_c[0]])))
    _train_pool = _pool.difference(_listed)
    if TRAIN_LINES == "all" or TRAIN_LINES >= len(_train_pool):
        train_vals = set(_train_pool)                            # the whole eligible base, outside the lists
    else:
        train_vals = set(rng.choice(_train_pool, TRAIN_LINES, replace=False))
    WHOLE = TRAIN_LINES == "all"                                 # the tables are linked whole: no filter
    score_vals = set().union(*[set(v["access_method_val"]) for v in _lists.values()])
    VALS = train_vals | score_vals
    _rows_b = _base[_bval.isin(VALS)]
    IDS, ACCOUNTS, PARTIES = (set(_canon_id(_rows_b[c])) - {""} for c in (_IDc, _ACCc, _PARTYc))

    def _keep_mask(df):
        ids, vals = _find(df.columns, "ACCESS_METHOD_ID"), _find(df.columns, "ACCESS_METHOD_VAL", "MSISDN")
        if ids or vals:
            m = pd.Series(False, index=df.index)
            for c in ids:
                m |= _canon_id(df[c]).isin(IDS)
            for c in vals:
                m |= _canon_val(df[c]).isin(VALS)
            return m, "+".join(ids + vals)
        acc, party = _find(df.columns, "ACCOUNT_ID"), _find(df.columns, "PARTY_ID")
        if acc or party:
            m = pd.Series(False, index=df.index)
            for c in acc:
                m |= _canon_id(df[c]).isin(ACCOUNTS)
            for c in party:
                m |= _canon_id(df[c]).isin(PARTIES)
            return m, "+".join(acc + party)
        return None, "copied whole"

    _files = sorted([p for p in SRC_WEEK.rglob("*.csv.gz") if p.relative_to(SRC_WEEK).parts[0] not in ("target1", "target2", "control")],
                    key=lambda p: (not p.name.startswith("base_prepaid"), not p.name.startswith("consumer_aact_account"), str(p)))
    for p in _files:
        out = DST / p.relative_to(SRC_WEEK)
        out.parent.mkdir(parents=True, exist_ok=True)
        if WHOLE:                                                # a hard link (no copy, no disk), else a copy
            try:
                os.link(p, out)
            except OSError:
                shutil.copy2(p, out)
            continue
        if _headerless(p):
            shutil.copy2(p, out)
            print(f"{p.name}: no header, copied whole")
            continue
        first, n_in, n_out, how = True, 0, 0, ""
        for chunk in _rd(p, chunksize=500_000):
            n_in += len(chunk)
            m, how = _keep_mask(chunk)
            part = chunk if m is None else chunk[m]
            n_out += len(part)
            if p.name.startswith("consumer_aact_account"):
                pc = _find(part.columns, "PARTY_ID")
                if pc:
                    PARTIES |= set(_canon_id(part[pc[0]])) - {""}
            part.to_csv(out, mode="w" if first else "a", header=first, index=False,
                        compression={"method": "gzip", "compresslevel": 6})
            first = False
        if first:
            shutil.copy2(p, out)                                     # header-only file
        print(f"{str(p.relative_to(SRC_WEEK)):72s} {n_in:>12,} -> {n_out:>9,}  [{how}]")
    for _k, rel in LIST_KEYS.items():                            # the lists of SCORE_GROUPS in full, the others empty
        dst = DST / f"{rel}.csv.gz"
        dst.parent.mkdir(parents=True, exist_ok=True)
        (_lists[_k] if _k in _lists else pd.DataFrame(columns=["access_method_val"])).to_csv(
            dst, index=False, compression={"method": "gzip", "compresslevel": 6})
    pd.DataFrame({"access_method_val": sorted(train_vals)}).to_csv(DST / "_train_lines.csv", index=False)
    _man = dict(SRC_MAN, run_date=RUN_DATE, anchor_date=CAMPAIGN_DATE, source=f"real25k copy of data/weekly/{SRC_WEEK.name}",
                train_lines=len(train_vals), score_lines=len(score_vals), score_groups=list(SCORE_GROUPS))
    (DST / "_manifest.json").write_text(json.dumps(_man, indent=2, default=str), encoding="utf-8")
    _old = dict(_want, train_lines=len(train_vals), score_lines=len(score_vals),
                env_sizes={SIZE_ENV[k]: (len(_lists[k]) if k in _lists else 0) for k in LIST_KEYS},
                score_group=SCORE_GROUPS[0], run_date=RUN_DATE, anchor_date=CAMPAIGN_DATE)
    _info_p.write_text(json.dumps(_old, indent=2, default=str), encoding="utf-8")
    print(f"copy written: {DST} -- {len(train_vals):,} training lines" + (" (the whole eligible base)" if WHOLE else "")
          + f" + {len(score_vals):,} target lines, {len(_files)} tables "
          + ("linked whole" if WHOLE else "filtered") + f" in {time.time() - _t0:,.0f} s")

_info = _old
os.environ["FSDR_WEEKLY_DIR"] = str(REAL / "weekly")
os.environ["FSDR_RUNS_DIR"] = str(REAL / "runs")
os.environ["FSDR_CACHE_DIR"] = str(REAL / "cache")
os.environ["FSDR_OUTPUT_DIR"] = str(REAL / "output")
os.environ["FSDR_CACHE_PARTITIONS"] = "8"
for _k, _v in _info["env_sizes"].items():
    os.environ[_k] = str(_v)                                       # a group that is not scored has size 0
TRAIN_VALS = set(_rd(DST / "_train_lines.csv")["access_method_val"])
GENERATE = GENERATE_TARGETS = False
print(f"RUN_SIZE {RUN_SIZE!r} | training lines {len(TRAIN_VALS):,} | scored: " + ", ".join(f"{k} {_info['env_sizes'][SIZE_ENV[k]]:,}" for k in SCORE_GROUPS)
      + f" | outputs in {REAL}")

## Part A · The engine code (40 modules, 7,762 lines)

This notebook is **standalone**: every function the steps call is defined in the cells of this part. Nothing is
imported from the project folder (`nbx/`), and the only imports are pip packages (pandas, numpy, pyarrow, pyyaml,
python-dotenv, matplotlib). **vowpalwabbit is never loaded**: it is the V1.0 bandit only, and loaded after pandas it
crashes the Jupyter kernel on Windows (two copies of `msvcp140.dll`).

Each cell starts with `%%engine <module>`: the cell runs its own code and registers it under that name, so the
cells can call each other (an `import` of `nbx.…` inside these cells resolves to the cell above, never to a file).
The code is copied from the project by `notebooks/build_step_notebooks.py`, so it is **exactly** the code of the
terminal commands. Run this part once (it takes a few seconds), then go to Part B.

T2 / T3 run in this process (`WORKERS = 1`): worker processes would have to import the code from files.

| Module | Copied from | Lines | What it does |
|---|---|---|---|
| `nbx` | `nbx/__init__.py` | 24 | NBX: agentic Next-Best-Action / Next-Best-Offer engine. |
| `nbx.core` | `nbx/core/__init__.py` | 0 |  |
| `nbx.core.config` | `nbx/core/config.py` | 107 | App-level settings (read from .env): operator, LLM backend, ports, paths, API security. |
| `nbx.fsdr` | `nbx/fsdr/__init__.py` | 4 | FSDR: production-shaped agentic NBA / NBO on top of NBX. |
| `nbx.fsdr.config` | `nbx/fsdr/config.py` | 734 | FSDR configuration: every business rule constant (BR-xx), the model-score |
| `nbx.fsdr.calendar` | `nbx/fsdr/calendar.py` | 131 | Anchor date and every aggregation window of the business rules (BR §4). |
| `nbx.fsdr.schemas` | `nbx/fsdr/schemas.py` | 447 | Data contract: the exact raw schema of every input file. |
| `nbx.fsdr.datasource` | `nbx/fsdr/datasource.py` | 280 | Where the FSDR pipelines get their tables. |
| `nbx.fsdr.features` | `nbx/fsdr/features.py` | 361 | NBA feature engineering -- port of docs/fsdr_reference/nba_lib.py, without Dataiku. |
| `nbx.fsdr.offer_rules` | `nbx/fsdr/offer_rules.py` | 270 | Offer business rules -- port of production Codes 3 (constraint arm) and 4 (free arm). |
| `nbx.fsdr.offers` | `nbx/fsdr/offers/__init__.py` | 10 | V3.0 offers of the four groups (update prompt §4): universes, anchors, the price selections and the reco tables. |
| `nbx.fsdr.offers.selection` | `nbx/fsdr/offers/selection.py` | 130 | The selections of the four groups (update §4). Input: one row per customer x candidate of ONE validity, with |
| `nbx.fsdr.panel` | `nbx/fsdr/panel.py` | 144 | Panel (update §3, master §19.2): the eligible base assigned at random to T1C / T1F / T2C / T2F / control. |
| `nbx.fsdr.audit_groups` | `nbx/fsdr/audit_groups.py` | 73 | ⑨ auditor checks of the four groups (update §7): A-12 per group, A-12b, A-17, A-30, A-31. Deterministic; any FAIL |
| `nbx.fsdr.base_features` | `nbx/fsdr/base_features.py` | 237 | T2_features (master prompt §6, B-01 … B-12): the features of the full base, partition by partition. |
| `nbx.fsdr.campaign_log` | `nbx/fsdr/campaign_log.py` | 269 | Campaign log and rewards: what was sent to whom, and what came back (prompt §6, audit M1..M4). |
| `nbx.fsdr.contract` | `nbx/fsdr/contract/__init__.py` | 9 | V2.0 input contract (plan §4, §13-§15). |
| `nbx.fsdr.steps` | `nbx/fsdr/steps.py` | 194 | Step folders (plan §14): every pipeline step saves its output before the next one starts. |
| `nbx.fsdr.contract.cache` | `nbx/fsdr/contract/cache.py` | 160 | Parquet cache of a run (plan §4, §15): each input table converted once, typed, split into partitions. |
| `nbx.fsdr.contract.completeness` | `nbx/fsdr/contract/completeness.py` | 48 | Weekly completeness (master prompt §5, §17.9): every table the pipeline about to run needs is present, has |
| `nbx.fsdr.validation` | `nbx/fsdr/validation.py` | 455 | Validation of a run folder (or any DataSource) against the data contract. |
| `nbx.fsdr.contract.data_agent` | `nbx/fsdr/contract/data_agent.py` | 117 | ① Data agent (master prompt §5): step T0_load of the training pipeline. |
| `nbx.fsdr.csvstd` | `nbx/fsdr/csvstd.py` | 181 | The FSDR CSV standard (master prompt §2, §20): the one format of every input AND output table file. |
| `nbx.fsdr.final_output` | `nbx/fsdr/final_output.py` | 196 | ⑦ Final output agent: code 5 (master §12, F-01 … F-19) -> input_flytxt, vectorised over all customers. |
| `nbx.fsdr.llm_budget` | `nbx/fsdr/llm_budget.py` | 49 | Level-2 LLM queue and the weekly budget (master §19.1a, update §6). |
| `nbx.fsdr.offers.anchors` | `nbx/fsdr/offers/anchors.py` | 65 | ARPU anchors (master §11 B-25 … B-27, update CC-03 / CF-03). |
| `nbx.fsdr.offers.universe` | `nbx/fsdr/offers/universe.py` | 77 | Offer universes (master §11 B-20b, B-21; update §4.1, §4.2 FR-01 … FR-06). |
| `nbx.fsdr.offers.recos` | `nbx/fsdr/offers/recos.py` | 211 | The reco tables of the four groups (update §5): code-2 tables + 8 reco tables in the original structure (B-32). |
| `nbx.fsdr.pipelines` | `nbx/fsdr/pipelines/__init__.py` | 71 | Deterministic FSDR pipelines (train, score, recommend). Each step returns a JSON report, |
| `nbx.fsdr.segmentation` | `nbx/fsdr/segmentation/__init__.py` | 2 | ② Segmentation (master prompt §7, §8) and ②b segment actions (§7.1): S-rules, bands, micro-segments, |
| `nbx.fsdr.segmentation.actions` | `nbx/fsdr/segmentation/actions.py` | 202 | ②b The two actions of every line (master prompt §7.1), plus the week-0 history of code 1 (B-02, B-03, B-09, B-10). |
| `nbx.fsdr.segmentation.bands` | `nbx/fsdr/segmentation/bands.py` | 56 | Bands and micro-segments (master prompt §8). Micro-segment = nationality group x age band x tenure band |
| `nbx.fsdr.segmentation.rules` | `nbx/fsdr/segmentation/rules.py` | 83 | S-rules (master prompt §7): the current segment of every line, first match wins, thresholds in config. |
| `nbx.fsdr.pipelines.training` | `nbx/fsdr/pipelines/training.py` | 367 | V2.0 training pipeline, stages T2 and T3 (plan §14, §15; master prompt §6 - §8, §18). |
| `nbx.fsdr.targets` | `nbx/fsdr/targets.py` | 186 | Target upload folders (V4.0): the lists of the four groups and the control, chosen outside the engine. They sit |
| `nbx.fsdr.pipelines.scoring_groups` | `nbx/fsdr/pipelines/scoring_groups.py` | 595 | Scoring steps of the four groups (plan §14, update §3 - §5), run after training T2 + T3: |
| `nbx.fsdr.synth` | `nbx/fsdr/synth/__init__.py` | 1 | Dummy-data world, scenarios and generator (same JSON files as the real data). |
| `nbx.fsdr.synth.scenarios` | `nbx/fsdr/synth/scenarios.py` | 124 | Use-case scenarios for the dummy data (combinable, e.g. "churn_wave,data_growth"). |
| `nbx.fsdr.synth.world` | `nbx/fsdr/synth/world.py` | 370 | Hidden customer world for the dummy data and the closed-loop simulation. |
| `nbx.fsdr.synth.generator` | `nbx/fsdr/synth/generator.py` | 722 | Dummy-data generator: writes exactly the files of the real data contract. |

In [ ]:
step("Part A")
# --- Part A loader: the cells below hold the engine code; %%engine registers each one as a module -------------
# Nothing is imported from the project folder: a cell "%%engine nbx.fsdr.csvstd" runs ITS OWN code and makes it
# available to the other cells under that name. Only pip packages are imported (pandas, numpy, pyarrow, pyyaml,
# python-dotenv, matplotlib). vowpalwabbit is never loaded.
import linecache, os, sys, types
from pathlib import Path

from IPython.core.magic import register_cell_magic

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:      # the project folder (for data/ and .env)
    ROOT = ROOT.parent
os.chdir(ROOT)
for _name in [m for m in sys.modules if m == "nbx" or m.startswith("nbx.")]:
    del sys.modules[_name]                                                  # a re-run starts clean


def _module(name, package=False):
    mod = types.ModuleType(name)
    mod.__fsdr_notebook__ = True                                            # proof: defined in this notebook
    if package:
        mod.__path__ = []          # its sub-modules are cells of this notebook, never files on disk
    mod.__file__ = str(ROOT / (name.replace(".", "/") + ("/__init__.py" if package else ".py")))  # paths only
    mod.__package__ = name if package else name.rpartition(".")[0]
    sys.modules[name] = mod
    parent, _, child = name.rpartition(".")
    if parent:
        setattr(_package(parent), child, mod)
    return mod


def _package(name):
    m = sys.modules.get(name)
    return m if getattr(m, "__fsdr_notebook__", False) else _module(name, package=True)


@register_cell_magic
def engine(line, cell):
    """%%engine <module> [package]: run this cell's code as that module."""
    name, *flags = line.split()
    mod = _package(name) if "package" in flags else _module(name)
    filename = f"<notebook: {name}>"
    linecache.cache[filename] = (len(cell), None, cell.splitlines(True), filename)   # readable tracebacks
    exec(compile(cell, filename, "exec"), mod.__dict__)


def use(module, *names):
    """The objects of an engine module (defined by a cell below), e.g. config, csvstd = use("nbx.fsdr", "config", "csvstd")."""
    mod = sys.modules[module]
    out = [getattr(mod, n) if hasattr(mod, n) else sys.modules[f"{module}.{n}"] for n in names]
    return out[0] if len(out) == 1 else tuple(out)


print("engine loader ready: the next cells define the engine modules")

In [ ]:
%%engine nbx package
"""NBX engine, standalone in this notebook (V4.0 training / scoring). vowpalwabbit is never loaded."""
import os

for _k, _v in {"GRADIO_ANALYTICS_ENABLED": "False", "HF_HUB_DISABLE_TELEMETRY": "1", "DO_NOT_TRACK": "1",
               "VLLM_NO_USAGE_STATS": "1", "ANONYMIZED_TELEMETRY": "False"}.items():
    os.environ.setdefault(_k, _v)

__version__ = "4.0.0"


In [ ]:
%%engine nbx.core package



In [ ]:
%%engine nbx.core.config
"""App-level settings (read from .env): operator, LLM backend, ports, paths, API security.

The FSDR engine's own business constants (BR-01 ... BR-86) are in nbx/fsdr/config.py.
"""
import os
from pathlib import Path

from dotenv import load_dotenv

# Project root: this file lives in nbx/core/.
BASE_DIR = Path(__file__).resolve().parents[2]

load_dotenv(BASE_DIR / ".env")

# Our operator's name in every prompt, SMS template and UI text.
OPERATOR_NAME = os.getenv("OPERATOR_NAME", "FSDR")

# ---------------------------------------------------------------------------
# LLM backend (llms/llm_client.py). Every backend runs ON THIS MACHINE (see LOCAL_ONLY below):
#   "offline"  -- pure Python, no server (default): every agent takes its rule-based path.
#   "vllm"     -- a vLLM server (GPU, WSL2 / Linux): `python run.py vllm-serve`            -> VLLM_URL
#   "ollama"   -- Ollama (Windows / macOS / Linux, CPU or GPU): `ollama serve`             -> OLLAMA_URL
#   "llamacpp" -- llama.cpp's llama-server with a GGUF file (CPU or GPU)                  -> LLAMACPP_URL
#   "lmstudio" -- LM Studio's local server (Developer tab -> Start server)                 -> LMSTUDIO_URL
# The four servers speak the same OpenAI-compatible API (/v1/models, /v1/chat/completions): one client
# (llms/vllm_client.py) serves them all. No cloud LLM API (OpenAI, Anthropic / Claude, …) is used anywhere.
# ---------------------------------------------------------------------------
LLM_BACKEND = os.getenv("LLM_BACKEND", "offline").lower()
LLM_SERVER_BACKENDS = ["vllm", "ollama", "llamacpp", "lmstudio"]
LLM_BACKENDS = ["offline", *LLM_SERVER_BACKENDS]
DEFAULT_LLM = os.getenv("DEFAULT_LLM", "qwen2.5:3b")
# 127.0.0.1, not "localhost": vLLM binds IPv4 only, and "localhost" tries ::1 first (slow on Windows).
VLLM_URL = os.getenv("VLLM_URL", "http://127.0.0.1:8001")
OLLAMA_URL = os.getenv("OLLAMA_URL", "http://127.0.0.1:11434")
LLAMACPP_URL = os.getenv("LLAMACPP_URL", "http://127.0.0.1:8080")
LMSTUDIO_URL = os.getenv("LMSTUDIO_URL", "http://127.0.0.1:1234")
VLLM_API_KEY = os.getenv("VLLM_API_KEY", "")
LLM_API_KEY = os.getenv("LLM_API_KEY", "")            # ollama / llamacpp / lmstudio, only if you set one there
# Grammar-constrained JSON (response_format json_object) per server; LM Studio only takes json_schema, so the
# agents parse its free text instead (llm_client.extract_json).
LLM_JSON_MODE = {"vllm": True, "ollama": True, "llamacpp": True, "lmstudio": False}


def llm_url(backend: str | None = None) -> str:
    """The URL of the server of a backend (the current one by default); "" for offline."""
    b = (backend or LLM_BACKEND).lower()
    return {"vllm": VLLM_URL, "ollama": OLLAMA_URL, "llamacpp": LLAMACPP_URL, "lmstudio": LMSTUDIO_URL}.get(b, "")


def llm_api_key(backend: str | None = None) -> str:
    return VLLM_API_KEY if (backend or LLM_BACKEND).lower() == "vllm" else LLM_API_KEY


# ---------------------------------------------------------------------------
# V4.0 local only. True (default): the app talks only to servers on this machine (127.0.0.1 / localhost / ::1,
# plus the hosts you list in FSDR_LOCAL_HOSTS, e.g. your own GPU box on the LAN). An LLM URL or a tracing
# endpoint elsewhere is refused (the agents fall back to their rules, tracing stays off), and the usage
# analytics of Gradio / Hugging Face / vLLM are switched off (nbx/__init__.py).
# ---------------------------------------------------------------------------
LOCAL_ONLY = os.getenv("FSDR_LOCAL_ONLY", "true").lower() == "true"
LOCAL_HOSTS = {"127.0.0.1", "localhost", "::1"} | {h.strip().lower() for h in
                                                   os.getenv("FSDR_LOCAL_HOSTS", "").split(",") if h.strip()}


def is_local_url(url: str) -> bool:
    from urllib.parse import urlparse

    host = (urlparse(url).hostname or "").lower()
    return host in LOCAL_HOSTS or host.startswith("127.")


def local_violations() -> list[str]:
    """What would leave the machine with the current settings (empty = everything local)."""
    out = []
    if LLM_BACKEND in LLM_SERVER_BACKENDS and not is_local_url(llm_url()):
        out.append(f"LLM_BACKEND={LLM_BACKEND} points to {llm_url()} (not this machine)")
    tracing = os.getenv("LANGSMITH_TRACING", os.getenv("LANGCHAIN_TRACING_V2", "")).lower() in ("true", "1")
    endpoint = os.getenv("LANGSMITH_ENDPOINT") or os.getenv("LANGCHAIN_ENDPOINT") or "https://api.smith.langchain.com"
    if tracing and not is_local_url(endpoint):
        out.append(f"LANGSMITH_TRACING=true sends traces to {endpoint} (not this machine)")
    return out
# Cap per completion; otherwise one runaway generation holds a batch slot for its whole length.
VLLM_MAX_TOKENS = int(os.getenv("VLLM_MAX_TOKENS", "512"))
LLM_TIMEOUT_S = float(os.getenv("LLM_TIMEOUT_S", "300"))

# ---------------------------------------------------------------------------
# Ports, paths, logging
# ---------------------------------------------------------------------------
FASTAPI_PORT = int(os.getenv("FASTAPI_PORT", "8000"))
GRADIO_PORT = int(os.getenv("GRADIO_PORT", "7860"))
UI_HOST = os.getenv("FSDR_UI_HOST", "127.0.0.1")            # 0.0.0.0 in the container (Dockerfile)
API_URL = os.getenv("FSDR_API_URL", "")                      # UI -> API base URL (default http://localhost:<port>)
DATA_DIR = Path(os.getenv("DATA_DIR", str(BASE_DIR / "data")))
LOG_LEVEL = os.getenv("LOG_LEVEL", "INFO")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# ---------------------------------------------------------------------------
# API security (review finding #5). With FSDR_API_KEY set, every /fsdr/* route needs the
# header `X-API-Key: <key>` (the UI sends it). Empty = open, for local development only: the API then binds
# 127.0.0.1 by default. Model promotion that bypasses the IPS rule needs FSDR_ADMIN_KEY as well.
# ---------------------------------------------------------------------------
API_KEY = os.getenv("FSDR_API_KEY", "")
ADMIN_KEY = os.getenv("FSDR_ADMIN_KEY", "")
CORS_ORIGINS = [o.strip() for o in os.getenv("FSDR_CORS_ORIGINS", f"http://localhost:{GRADIO_PORT},"
                                             f"http://127.0.0.1:{GRADIO_PORT}").split(",") if o.strip()]

# ---------------------------------------------------------------------------


In [ ]:
%%engine nbx.fsdr package
"""FSDR: production-shaped agentic NBA / NBO on top of NBX.

Plan and phase status: docs/fsdr_plan.md.
"""


In [ ]:
%%engine nbx.fsdr.config
"""FSDR configuration: every business rule constant (BR-xx), the model-score
registry, arms, paths and generator defaults.

Business rules come from docs/fsdr_reference/FSDR_NBA_business_rules.md;
each constant names its BR number. Logic modules import from here and never
hard-code these values.
"""
from __future__ import annotations

import json
import os
from dataclasses import dataclass
from datetime import date
from pathlib import Path

from nbx.core import config as nbx_config

OPERATOR_NAME = os.getenv("OPERATOR_NAME", "FSDR")
CURRENCY = os.getenv("FSDR_CURRENCY", "OMR")
SCHEMA_VERSION = "2.0"

# ---------------------------------------------------------------------------
# Paths
# ---------------------------------------------------------------------------
FSDR_DATA_DIR = Path(os.getenv("FSDR_DATA_DIR", str(nbx_config.DATA_DIR / "fsdr")))
FSDR_STATE_DIR = FSDR_DATA_DIR / "state"          # registry, campaign log, rewards
CAMPAIGN_DIR = "campaign"                         # <data root>/state/campaign: nba_campaign_log.jsonl, nba_rewards.jsonl
JSON_SCHEMA_DIR = nbx_config.BASE_DIR / "docs" / "fsdr_json_schemas"
MANIFEST_FILE = "_manifest.json"
VALIDATION_JSON = "_validation.json"
VALIDATION_MD = "_validation.md"

# V2.0 data layout (plan §13-§15)
FSDR_INPUT_DIR = Path(os.getenv("FSDR_INPUT_DIR", str(nbx_config.DATA_DIR / "input")))    # <D>/<folder>/<table>.*
FSDR_CACHE_DIR = Path(os.getenv("FSDR_CACHE_DIR", str(nbx_config.DATA_DIR / "cache")))    # <D>/<table>/part-NN.parquet
FSDR_RUNS_DIR = Path(os.getenv("FSDR_RUNS_DIR", str(nbx_config.DATA_DIR / "runs")))       # <D>/{train,score,final}/<step>/
# V4.0 data layout: the inputs of a run date are split in two twin folders with the same 5 sub-folders, and the
# results of each target go to data/output/<D>/<target>/ (docs/SETUP_NEW_LAPTOP.md §5)
#   data/training/<D>/{usage,models,subscription,catalog,others}/     the base, history and model scores (T0 - T3)
#   data/scoring/<D>/{usage,models,subscription,catalog,others}/      this week's scoring tables (P1 - P7)
#   data/scoring/<D>/target1/{a_constraint,b_free}/  target2/{a_constraint,b_free}/  control/   the target lists
#   data/output/<D>/target1/  target2/  control/  final/                one folder per target, scored separately
TRAINING_FOLDER, SCORING_FOLDER = "training", "scoring"
FSDR_TRAINING_DIR = Path(os.getenv("FSDR_TRAINING_DIR", str(nbx_config.DATA_DIR / TRAINING_FOLDER)))
FSDR_SCORING_DIR = Path(os.getenv("FSDR_SCORING_DIR", str(nbx_config.DATA_DIR / SCORING_FOLDER)))
FSDR_WEEKLY_DIR = Path(os.getenv("FSDR_WEEKLY_DIR", str(nbx_config.DATA_DIR / "weekly")))   # notebooks 03 / 04: one folder
FSDR_OUTPUT_DIR = Path(os.getenv("FSDR_OUTPUT_DIR", str(nbx_config.DATA_DIR / "output")))
INPUT_FOLDERS = ("usage", "models", "subscription", "catalog", "others")
INPUT_FORMATS = (".csv.gz", ".json", ".jsonl", ".csv", ".parquet")   # V3.0: .csv.gz = production (master §2)
PRODUCTION_FORMAT = ".csv.gz"
# master §2: plain .csv / .json / .jsonl / .parquet only for tests and the dummy generator (its manifest says so)
ALLOW_TEST_FORMATS = os.getenv("FSDR_ALLOW_TEST_FORMATS", "false").lower() == "true"
CSV_GZIP_LEVEL = 6                                # FSDR CSV standard
ID_TEXT_COLUMNS = {"ACCESS_METHOD_VAL", "MSISDN", "ACCESS_METHOD_ID", "ACCOUNT_ID", "PARTY_ID", "SUBSCRIPTION_ID",
                   "UBSCRIPTION_ID"}              # always read and written as text
OUTPUT_COMPRESSION = os.getenv("FSDR_OUTPUT_COMPRESSION", os.getenv("FSDR_PUSH_COMPRESSION", "gzip"))    # "gzip" | "none" (also writes input_flytxt.csv)
CACHE_PARTITIONS = int(os.getenv("FSDR_CACHE_PARTITIONS", "16"))     # plan §15: hash of ACCESS_METHOD_ID
TABLE_FOLDERS = {                                 # plan §13; the 11 model-score tables go to "models"
    "usage": ["subs_amt_auto", "subs_amt_score", "activities_w1_auto", "activities_w2_auto", "activities_w3_auto",
              "activities_w4_auto", "activities_m1_auto", "activities_auto_m2", "activities_m3_auto",
              "activities_w1_score", "activities_w2_score", "activities_w3_score", "activities_w4_score",
              "activities_m1_score", "activities_m2_score", "activities_m3_score", "paygo_activity_auto",
              "usage_data_voice_payg_tg"],
    "subscription": ["base_prepaid", "consumer_aact_account", "access_method_id_auto", "subscriptions_tg",
                     "subs_btl_auto", "sub_btl_3months"],
    "catalog": ["btl_catalog", "BTL_script_english_arabic_ok", "conflicting_offers_h"],
    "others": ["party_auto", "party_scoring", "customer_profile", "location_auto", "consent_contact",
               "DEVICEBEHAVIORSEGMENTATION.dv_segment", "targeting_dataiku_auto", "targeting_auto"],
}                                                 # any other table (e.g. the test sample) -> "others"
# V4.0: the tables filed under data/scoring/<D>/ (everything else under data/training/<D>/). A table found in both
# twins is read from its own side (btl_catalog is written to both: training history and this week's offers).
SCORING_TABLES = {"subscriptions_tg", "usage_data_voice_payg_tg", "sub_btl_3months", "targeting_auto",
                  "btl_catalog", "BTL_script_english_arabic_ok", "conflicting_offers_h", "consent_contact",
                  "subs_amt_score", "activities_w1_score", "activities_w2_score", "activities_w3_score",
                  "activities_w4_score", "activities_m1_score", "activities_m2_score", "activities_m3_score",
                  "party_scoring"}
BOTH_TABLES = {"btl_catalog"}                     # the generator writes these to both twins


def table_side(table: str) -> str:
    return SCORING_FOLDER if table in SCORING_TABLES or table == TARGET_SAMPLE_TABLE else TRAINING_FOLDER


def run_twins(folder: Path) -> dict[str, Path]:
    """{"training": ..., "scoring": ...} of a run folder in the V4.0 layout (data/training/<D> <-> data/scoring/<D>),
    only the twins that exist; {} for a single folder (V3.0 layout: every table in data/input/<D>/)."""
    folder = Path(folder)
    side = folder.parent.name
    if side not in (TRAINING_FOLDER, SCORING_FOLDER):
        return {}
    other = SCORING_FOLDER if side == TRAINING_FOLDER else TRAINING_FOLDER
    twins = {side: folder, other: folder.parent.parent / other / folder.name}
    return {k: v for k, v in twins.items() if v.is_dir()}


def training_folder(run_date, root: Path | None = None) -> Path:
    return Path(root or FSDR_TRAINING_DIR) / str(run_date)


def scoring_folder(run_date, root: Path | None = None) -> Path:
    return Path(root or FSDR_SCORING_DIR) / str(run_date)


def output_folder(run_date, root: Path | None = None) -> Path:
    return Path(root or FSDR_OUTPUT_DIR) / str(run_date)


STEP_FILE = "_step.json"
STEP_PREVIEW_ROWS = 1000                          # training outputs (1M rows): Parquet + the first rows as preview.csv
STEP_FULL_CSV_MAX_ROWS = 100_000                  # scoring outputs (<= 50k rows): full CSV next to the Parquet

# customer_profile (master prompt §2.1, §5): age and tenure of every line, provided weekly
PROFILE_TABLE = "customer_profile"
PROFILE_TENURE_UNIT = os.getenv("FSDR_PROFILE_TENURE_UNIT", "days")          # open item 5
PROFILE_AGE_RANGE = (15, 100)                     # outside -> warning
PROFILE_AGE_TOLERANCE_YEARS = 2                   # vs party_*.AGE
PROFILE_TENURE_TOLERANCE_DAYS = 30                # vs run date - access_method_subs_start_dt
PROFILE_MIN_COVERAGE = 0.95                       # share of the base with a profile row (below -> warning)

# ---------------------------------------------------------------------------
# Calendar (BR notation: D = run date, T = campaign anchor)
# ---------------------------------------------------------------------------
REFERENCE_THURSDAY = date(2026, 3, 12)            # T = D - (((D - 2026-03-12) MOD 7) + 7) MOD 7
SNAPSHOT_LAG_DAYS = 2                             # BR-01 ACTIVITY_DATE = D-2
HISTORY_DAYS_USAGE = 97                           # daily history D-96..D: deepest window is paygo M3 = T-90, T >= D-6
HISTORY_DAYS_FULFILMENT = 270                     # BR-24 interactions D-270..D-1

# ---------------------------------------------------------------------------
# Eligibility (BR-01 .. BR-12)
# ---------------------------------------------------------------------------
MSISDN_MIN_EXCLUSIVE = "69999999"                 # BR-02 string comparison
# BR-12 (SUBSTR(MSISDN, 4)) is applied upstream in the SQL exports: every MSISDN in the JSON files,
# including access_method_id_auto.MSISDN, is already the 8-digit ACCESS_METHOD_VAL. No prefix handling here.
EXCLUDED_PLANS = [                                # BR-10 internal / employee / business plans (13)
    "FSDR Internal",
    "Employee Data with Voice",
    "Shahry Business",
    "FSDR Shahry",
    "Employee superfast 100 Mbps plan",
    "Employee superfast 200 Mbps plan",
    "Employee superfast 400 Mbps plan",
    "5G Employee Plan 100 Mbps",
    "5G Employee Plan 50 Mbps",
    "Employee Fast HBB 150 GB Plan",
    "Employee Fast HBB 300 GB Plan",
    "Employee Fast HBB 600 GB Plan",
    "Fast  Home Broadband 150GB (Employee)",      # double space as in production
]

# ---------------------------------------------------------------------------
# Demographics (BR-13 .. BR-15)
# ---------------------------------------------------------------------------
NATIONALITY_KEEP = ["Oman", "India", "Bangladesh", "Pakistan"]   # BR-15; the rest -> "others"
NATIONALITY_OTHERS = "others"

# ---------------------------------------------------------------------------
# Segments and actions (BR-36 .. BR-45)
# ---------------------------------------------------------------------------
# BR-38. Fixed order: the bandit's probability vector is returned in this order. Never shuffle it.
ACTIONS = ["CROSS SELL DATA", "UPSELL DATA", "RETAIN", "MAINTAIN", "UPSELL", "CROSS SELL VOICE", "UPSELL VOICE"]
NO_CONTACT = "NO_CONTACT"                         # llm-arm only, never a VW action (plan Q10)

EXCLUDED_SEGMENTS = {"nursering_early_churn_at_risk", "nursering_early_churn_not_at_risk"}   # BR-36

# BR-39, lower-case keys so "RGS_..." and "Rgs_..." resolve to the same entry.
SEGMENT_TO_ACTION = {
    "retention_low_probability_to_reconnect": "RETAIN",
    "retention_med_probability_to_reconnect": "RETAIN",
    "retention_high_probability_to_reconnect": "RETAIN",                 # CHECK (plan Q1): was UPSELL then RETAIN
    "retention_rgs_at_risk": "RETAIN",
    "retention_rgs_fragile": "UPSELL",
    "rgs_not_at_risk_dualplay_upgrade": "MAINTAIN",
    "rgs_not_at_risk_data_only_voice_appetency": "UPSELL VOICE",          # CHECK (plan Q1b): or CROSS SELL VOICE
    "rgs_not_at_risk_data_only_no_voice_appetency": "CROSS SELL VOICE",
    "rgs_not_at_risk_voice_only_data_appetency": "UPSELL DATA",
    "rgs_not_at_risk_voice_only_no_data_appetency": "CROSS SELL DATA",
    "onboarding_early_churn_at_risk": "UPSELL",
    "onboarding_early_churn_not_at_risk": "UPSELL",
}

# Canonical segment spellings, and the mixed-case variants production also emits.
SEGMENTS = [
    "Retention_Low_Probability_To_Reconnect", "Retention_Med_Probability_To_Reconnect",
    "Retention_High_Probability_To_Reconnect", "Retention_RGS_At_Risk", "Retention_RGS_Fragile",
    "RGS_Not_At_Risk_DualPlay_Upgrade", "RGS_Not_At_Risk_Data_Only_Voice_Appetency",
    "RGS_Not_At_Risk_Data_Only_No_Voice_Appetency", "RGS_Not_At_Risk_Voice_Only_Data_Appetency",
    "RGS_Not_At_Risk_Voice_Only_No_Data_Appetency", "Onboarding_Early_Churn_At_Risk",
    "Onboarding_Early_Churn_Not_At_Risk",
    "Nursering_Early_Churn_At_Risk", "Nursering_Early_Churn_Not_At_Risk",
]
NURSERING_MAX_TENURE_DAYS = 45                    # BR-36
ONBOARDING_MAX_TENURE_DAYS = 90

ACTION_PRIOR = {                                  # BR-45 share of historical targeting volume
    "RETAIN": 0.5086577,
    "UPSELL": 0.2624428,
    "MAINTAIN": 0.1015183,
    "CROSS SELL VOICE": 0.0528180,
    "UPSELL DATA": 0.0422645,
    "UPSELL VOICE": 0.0239859,
    "CROSS SELL DATA": 0.0083128,
}
MISSING_SCORE_RANGE = (0.0, 0.02)                 # BR-47, drawn per row

# ---------------------------------------------------------------------------
# Model-score registry (BR-46). Adding / removing a model = editing this list.
# ---------------------------------------------------------------------------


@dataclass(frozen=True)
class ModelScore:
    table: str                      # exact file / table name
    feature: str                    # feature name in the NBA context
    combine: str = "max"            # how rows mapping to the same feature are merged
    signal: str = ""                # hidden-world signal the generator drives it with
    tenure_range: tuple[int, int] | None = None   # generator: only lines in this tenure band
    coverage: tuple[float, float] = (0.70, 0.90)  # generator: share of the base scored


MODEL_SCORE_REGISTRY: list[ModelScore] = [
    ModelScore("DAIM_CHURN_GLOBAL_SCORING_TD.prediction", "PROBA_CHURN", "max", "churn"),
    ModelScore("DAIM_EARLY_CHURN_0_30_SCORING_TD.Early_Churn_0_30_Prediction", "PROBA_CHURN", "max", "churn",
               tenure_range=(0, 30), coverage=(0.90, 0.98)),
    ModelScore("DAIM_EARLY_CHURN_30_60_SCORING_TD.scoring_data_scored", "PROBA_CHURN", "max", "churn",
               tenure_range=(30, 60), coverage=(0.90, 0.98)),
    ModelScore("DAIM_PROPENSITY_REACTIVATION_30D_404_SCORING_TD.Scored", "PROBA_REACT_30", "max", "reactivation_30"),
    ModelScore("PROPENSITY_REACTIVATION_15D_401_SCORING.Scored", "PROBA_REACT_15", "max", "reactivation_15"),
    ModelScore("APPETENCY_DATA_204_SCORING_TD.prediction", "PROBA_APP_DATA", "max", "data_appetite"),
    ModelScore("APPETENCY_VOICE_204_SCORING_TD.Scored", "PROBA_APP_VOICE", "max", "voice_appetite"),
    ModelScore("CHURN_ONE_DAY_ACTIVE_107_SCORING.Churn_One_Day_Scoring", "PROBA_CHURN_1D", "max", "churn_1d"),
    ModelScore("REV_DECREASE_204_SCORING_TD.predictions", "PROBA_REV_DECR", "max", "revenue_decrease"),
    ModelScore("DAIM_FRAGILITY_DATA_SCORING_TD.prediction", "PROBA_FRAG_DATA", "max", "fragility_data"),
    ModelScore("DAIM_FRAGILITY_VOICE_SCORING_TD.prediction", "PROBA_FRAG_VOICE", "max", "fragility_voice"),
]


def proba_features(registry: list[ModelScore] | None = None) -> list[str]:
    """Distinct feature names, in registry order (PROBA_CHURN first)."""
    out: list[str] = []
    for m in registry or MODEL_SCORE_REGISTRY:
        if m.feature not in out:
            out.append(m.feature)
    return out


# ---------------------------------------------------------------------------
# Experiment arms (BR-59 .. BR-62)
# ---------------------------------------------------------------------------
ARMS = ["cg", "free", "cstr", "llm"]              # cg = control (no contact); llm = new arm
ARM_SHARES = {"cg": 0.25, "free": 0.25, "cstr": 0.25, "llm": 0.25}
TARGET_SAMPLE_TABLE = os.getenv("FSDR_TARGET_SAMPLE_TABLE", "sample_cg_tg_new_7oct2026")   # plan Q16
TARGET_SAMPLE_SHARE = 0.40                        # generator: share of the base in the test sample
STRATA_FIELDS = ["nationality_group", "revenue_band", "voice_group", "data_group", "plan"]   # BR-62

# ---------------------------------------------------------------------------
# Offers, ARPU anchor and price bands (BR-63 .. BR-82); used from Phase 4
# ---------------------------------------------------------------------------
SERVICES = ["DATA", "VOICE", "COMBO"]
GENEROSITY_LEVELS = ["L0", "L1", "L2"]            # BR-65 L0 < L1 < L2
VALIDITIES = [7, 30]                              # BR-64
GENEROSITY_MAPPING = os.getenv("FSDR_GENEROSITY_MAPPING", "all_L2")   # BR-66 "all_L2" (active) | "normal" (plan Q3)
ACTION_FAMILIES = {                               # BR-66: (service A, service B) per action
    "MAINTAIN": ("COMBO", "DATA"),
    "UPSELL": ("COMBO", "DATA"),
    "RETAIN": ("COMBO", "DATA"),
    "UPSELL VOICE": ("DATA", "COMBO"),
    "CROSS SELL VOICE": ("DATA", "COMBO"),
    "CROSS SELL DATA": ("VOICE", "COMBO"),
    "UPSELL DATA": ("VOICE", "COMBO"),
}
ACTION_GENEROSITY_NORMAL = {                      # BR-66 "normal" (commented out in production): (gen A, gen B)
    "MAINTAIN": ("L0", "L0"),
    "UPSELL": ("L1", "L1"),
    "RETAIN": ("L2", "L2"),
    "UPSELL VOICE": ("L0", "L1"),
    "CROSS SELL VOICE": ("L0", "L2"),
    "CROSS SELL DATA": ("L0", "L2"),
    "UPSELL DATA": ("L0", "L2"),
}
FREE_ARM_GENEROSITY = "L2"                        # BR-67
FREE_ARM_SERVICES = ("COMBO", "DATA")             # BR-67 VOICE excluded
ARPU_DATA_SHARE = 0.9                             # BR-74
ARPU_VOICE_SHARE = 0.1                            # BR-75
PRICE_BAND_K = {                                  # BR-78 pass-1 upper margin
    "UPSELL": 3.0, "MAINTAIN": 2.0, "RETAIN": 2.0,
    "UPSELL VOICE": 3.0, "CROSS SELL DATA": 3.0, "CROSS SELL VOICE": 3.0, "UPSELL DATA": 3.0,
    "free": 3.0,
}
MAX_OFFERS_PER_VALIDITY = 2                       # BR-82
MAX_OFFERS_TOTAL = 4                              # BR-82

# ---------------------------------------------------------------------------
# Campaign windows (BR-21 .. BR-25), in days relative to D
# ---------------------------------------------------------------------------
TARGETING_WINDOW = (-8, -2)                       # BR-21 targeting_dataiku_auto.date1
ARBITRATION_WINDOW = (-7, 0)                      # BR-22 targeting_auto.start_date
REWARD_WINDOW = (-7, -1)                          # BR-23 fulfilment success date
INTERACTION_WINDOW = (-270, -1)                   # BR-24 sub_btl_3months

# ---------------------------------------------------------------------------
# Dummy-data generator defaults
# ---------------------------------------------------------------------------
GENERATOR_DEFAULT_N = 5000
GENERATOR_MAX_N = 2_000_000                       # V2.0: 1M designed, 2M tested on the P16 (plan §16)
GENERATOR_MODES = {"small": 2_000, "medium": 100_000, "full": 1_000_000}
GENERATOR_SEED = 42
GENERATOR_CHUNK = 20_000                          # customers per daily-history chunk (memory bound)
SCENARIOS = ["baseline", "churn_wave", "data_growth", "price_sensitive", "new_offers",
             "campaign_fatigue", "data_quality_issues"]
SCENARIO_RAMP_DAYS = 28                           # scenario drifts build up over the last 4 weeks
TARGETED_SHARE = 0.55                             # share of the base targeted in last week's campaign

# ---------------------------------------------------------------------------
# Features (BR-46 .. BR-53)
# ---------------------------------------------------------------------------
USAGE_COLS = ["LOCAL_VOICE_MIN", "DATA_VOLUME_MB", "PG_DATA_AMOUNT"]   # columns the model reads from activities_*
# BR-53 trend pairs as (recent, older): change FROM older TO recent (W1 = most recent week).
TREND_PAIRS = {
    "REV": [("W1", "W2"), ("W2", "W3"), ("W3", "W4"), ("M1", "M2"), ("M2", "M3")],
    "VOICE": [("W1", "W2"), ("W2", "W3"), ("W3", "W4"), ("M1", "M2")],
    "DATA": [("W1", "W2"), ("W2", "W3"), ("W3", "W4"), ("M1", "M2")],
}
TREND_SOURCE = {"REV": "REV_{p}", "VOICE": "LOCAL_VOICE_MIN_{p}", "DATA": "DATA_VOLUME_MB_{p}"}   # BR-49 REV = SUB_AMT + PG_DATA
PCT_CLIP = (-100.0, 500.0)                        # BR-50 % change, capped when the older period is tiny
# M3 last-campaign context, from the campaign log (campaign_log.last_campaign_features). Part of the model's
# feature list, so the compatibility check refuses a model trained without them (and the reverse).
LAST_CAMPAIGN_FEATURES = os.getenv("FSDR_LAST_CAMPAIGN_FEATURES", "true").lower() == "true"
LAST_CAMPAIGN_NUMERIC = ["PREV_OFFER_TAKEN", "DAYS_SINCE_LAST_BTL", "NB_CONTACTS_LAST_4W"]
PREV_ACTION_NONE = "NONE"                          # PREV_ACTION of a line never contacted
# V1.0 review #7: nationality as a targeting feature is a legal question (review F4); false drops it from the
# model (the fairness report per nationality group is produced either way).
USE_NATIONALITY = os.getenv("FSDR_USE_NATIONALITY", "true").lower() == "true"
CATEGORICAL_FEATURES = (["NATIONALITY_GROUP"] if USE_NATIONALITY else []) + (["PREV_ACTION"] if LAST_CAMPAIGN_FEATURES else [])
# V1.0 review #11 / #12: one PROBA_<x>_MISSING flag per score (BR-47's random fill stays, the model now sees it)
# and LOG_TENURE (tenure of the model-score files). false = the production feature list (parity).
EXTENDED_FEATURES = os.getenv("FSDR_EXTENDED_FEATURES", "true").lower() == "true"

# ---------------------------------------------------------------------------
# Bandit, training and scoring (BR-54 .. BR-58)
# ---------------------------------------------------------------------------
VW_EPSILON = 0.2                                  # BR-54 (allowed range for the training agent below)
VW_EPSILON_RANGE = (0.05, 0.3)
VW_ARGS_TEMPLATE = "--cb_explore_adf -q UA --quiet --epsilon {epsilon}"   # BR-54
TRAIN_MODE = "logged"                             # "logged" (all targeted, recommended) | "simulation" (BR-43/55/56)
TRAIN_PASSES = 2
TRAIN_PASSES_RANGE = (1, 5)
SIM_ITERATIONS = 20_000                           # BR-55 simulation mode
HOLDOUT_SHARE = 0.2
REWARD_DEFINITIONS = ["any_btl", "same_service", "exact_offer"]
REWARD_DEFINITION = "any_btl"                     # BR-40 production (plan Q4)
# BR-40: a taker is a targeted line with ANY catalogue BTL fulfilled in D-7..D-1 (production). Set true to also
# require the fulfilment on or after the line's targeting date1 (plan Q2 option; the two windows overlap).
REWARD_AFTER_TARGETING = os.getenv("FSDR_REWARD_AFTER_TARGETING", "false").lower() == "true"
PROPENSITY_CLIP = (0.01, 1.0)
SCORING_POLICY = "sample"                         # BR-57 sample (production) | greedy
BASE_SCORING_POLICY = "greedy"
MAX_MODEL_AGE_DAYS = 14                           # plan Q18
PROMOTION_TOLERANCE = 0.0                         # V0 rule: challenger must reach champion IPS - tolerance
# V1.0 review #2 / #3: the holdout is evaluated for the policy that is deployed (sampling from the PMF, BR-57), and
# the challenger is compared with the champion on the SAME holdout (out-of-time for the champion) by a paired
# bootstrap of the SNIPS gain. "paired_ci": refused only when the challenger is significantly worse (upper bound of
# the gain < -PROMOTION_MARGIN); significantly better -> promoted; inconclusive -> promoted (the fresher model wins
# when the data cannot separate them; scoring refuses a champion older than 14 days anyway). "point" = V0 (point
# estimates, which promote and refuse on noise: 0.215 vs 0.133 on 221 rows).
PROMOTION_RULE = os.getenv("FSDR_PROMOTION_RULE", "paired_ci").lower()     # paired_ci | point (V0)
PROMOTION_MARGIN = float(os.getenv("FSDR_PROMOTION_MARGIN", "0.0"))
BOOTSTRAP_SAMPLES = 500
CONFIDENCE = 0.95
# V1.0 review #1: the legacy campaign assigns the action from the segment (BR-39), so P(action | context) = 1 for
# the action sent. "deterministic" logs 1.0 (and flags the week as not usable for off-policy estimates);
# "share" is V0 (this week's action share, which the review found biased).
TARGETING_PROPENSITY = os.getenv("FSDR_TARGETING_PROPENSITY", "deterministic").lower()
# V1.0 review #4: uplift reward. The control group's organic take rate is modelled from the features (logistic
# regression on the cg rows of the campaign log); the bandit's cost becomes -(reward - baseline), so it learns the
# lift of a contact instead of "who buys anyway". false = V0 (cost = -reward).
UPLIFT_REWARD = os.getenv("FSDR_UPLIFT_REWARD", "true").lower() == "true"
PSI_BINS = 10
PSI_ALERT = 0.25                                  # PSI above this on a feature = drift alert
GROUP_TO_FLAG = {"free": "free", "cstr": "with_constraint", "llm": "llm"}   # BR-59; cg is not scored (control)

# ---------------------------------------------------------------------------
# Offer ranker (BR-83 .. BR-86)
# ---------------------------------------------------------------------------
RANKER = os.getenv("FSDR_RANKER", "bpr")          # bpr (implicit, default) | lightfm (if installed) (plan Q12)
RANKER_TOP_N = 100                                # BR-85
# Which offers the ranker may propose (BR-83/85). "history" = production Codes 3/4: only offers bought in
# the 270-day history after the action's family + validity filter (item_id = interaction1.BTL_NAME.unique()).
# "catalogue" = every catalogue offer of the family (the commented-out production line); needed for brand-new
# offers, which have no history yet (scenario new_offers).
RANKER_ITEMS = os.getenv("FSDR_RANKER_ITEMS", "history").lower()
BPR_PARAMS = {"factors": 32, "iterations": 60, "learning_rate": 0.05, "regularization": 0.01, "random_state": 42,
              "num_threads": 1}               # single thread: same data -> same ranking (offline llm arm = cstr arm)
CONTACT_MIN_GAP_DAYS = 2                          # see CONTACT_* below
# V1.0 review #10 / #14: offer arbitration. "value" ranks each family's offers on expected revenue
# P(accept | customer, offer) x price x OFFER_MARGIN, P from an acceptance model trained on last week's
# campaign (offers sent -> fulfilments), on offer ATTRIBUTES, so new offers get a score too (cold start); a share
# OFFER_EXPLORATION of the picks is a random family offer, so new offers get logged data. Families (BR-66/67),
# max offers (BR-82) and conflicts stay hard rules. "price_band" = production's 3 passes (BR-78..81), used as
# the fallback while the acceptance model has too little data.
OFFER_POLICY = os.getenv("FSDR_OFFER_POLICY", "value").lower()
OFFER_MARGIN = float(os.getenv("FSDR_OFFER_MARGIN", "1.0"))       # no cost data in the contract yet (review Q2)
OFFER_EXPLORATION = float(os.getenv("FSDR_OFFER_EXPLORATION", "0.05"))
ACCEPTANCE_MIN_POSITIVES = 20
# Price floor of the value policy. "anchor" keeps production's floor: every pick is priced above the line's ARPU
# anchor (as in the 3 passes, BR-78..80), so a line whose anchor is above every family price gets no offer (BR-82)
# and only the choice among the allowed offers changes. "none" lets the value policy also offer cheaper than the
# anchor (more lines contacted, more revenue in the simulation) -- a change of BR-78..82 that needs business sign-off.
OFFER_PRICE_FLOOR = os.getenv("FSDR_OFFER_PRICE_FLOOR", "anchor").lower()
LIGHTFM_PARAMS = {"no_components": 400, "loss": "warp", "k": 15, "epochs": 30}   # BR-84, used only by LightFMRanker

# ---------------------------------------------------------------------------
# Recommender agent: freedom, hard and soft rules (prompt §5.2)
# ---------------------------------------------------------------------------
EPSILON_LLM = float(os.getenv("FSDR_EPSILON_LLM", "0.1"))   # share of llm-arm decisions replaced by a bandit sample
CHURN_BLOCK_THRESHOLD = 0.60                      # plan Q9: PROBA_CHURN >= this blocks upsell-family actions
UPSELL_FAMILY = {"UPSELL", "UPSELL DATA", "UPSELL VOICE"}
# Contact cap (plan Q8, adjusted): campaigns are weekly and last week's targeting is D-8..D-2,
# so "1 contact per 7 days" would block most of the base. Hard rule = no new contact within
# CONTACT_MIN_GAP_DAYS of the last one, and at most CONTACT_CAP_28D in 28 days (campaign log, Phase 6).
CONTACT_CAP_28D = 3                               # L4 hard rule: at most 3 contacts logged in the last 28 days
CONTACT_CAP_DAYS = 28
# Training labels (H5): "auto" uses the campaign log when it holds last week's campaign (action actually sent,
# logged propensity), else the targeting tables with this week's action shares; "targeting" / "campaign_log" force one.
LABEL_SOURCE = os.getenv("FSDR_LABEL_SOURCE", "auto").lower()
# M4 price-escalation guard: cap the ARPU anchor at (1 + g) x the anchor logged for the line last time
# (campaign purchases raise next week's spend, so anchors could climb week after week).
# V1.0 review #9: on by default (10 % a week); "off" disables it (V0).
_GROWTH = os.getenv("FSDR_ANCHOR_MAX_GROWTH", "0.10").strip().lower()
ANCHOR_MAX_WEEKLY_GROWTH = None if _GROWTH in ("", "off", "none") else float(_GROWTH)
# V1.0 review #6: consent / do-not-contact (consent_contact table). A line with dnd = 1 or marketing_opt_in = 0 is
# never contacted, in any arm: hard rule, like eligibility.
CONSENT_TABLE = "consent_contact"

# ---------------------------------------------------------------------------
# Batch add-on offers: roaming and international (not in the BR catalogue; added for the batch flow)
# They are recommended BESIDE the main offers: BR-82 (max 4) applies to the main offers only,
# and an add-on may never conflict with a main offer.
# ---------------------------------------------------------------------------
ROAMING_SERVICES = [s.strip().upper() for s in os.getenv("FSDR_ROAMING_SERVICES", "ROAMING").split(",") if s.strip()]
INTERNATIONAL_SERVICES = [s.strip().upper() for s in os.getenv("FSDR_INTERNATIONAL_SERVICES", "INTERNATIONAL").split(",") if s.strip()]
ADDON_TOP_ROAMING = 2
ADDON_TOP_INTERNATIONAL = 2
INTL_MIN_CALLS_M1 = 3          # international calls in M1 (Call_Count - Local_Call_Count) to offer international packs
BATCH_SIZE = 25                # customers per batch in the batch flow (UI / API)
BATCH_MAX = 2000
# Customers decided in parallel by the agents. vLLM batches concurrent requests on the GPU, so it gets more.
# The code runs each agent's required evidence tools before the first LLM call and hands the results
# over as prior evidence: same tools, same trace, but the LLM only reasons and decides (fewer, faster
# calls, and small models stop wasting steps on refused or missing tool calls). 0 = the LLM calls them.
AGENT_PREFETCH = os.getenv("FSDR_AGENT_PREFETCH", "1") not in ("0", "false", "no")
AGENT_CONCURRENCY = int(os.getenv("FSDR_AGENT_CONCURRENCY", "8" if nbx_config.LLM_BACKEND == "vllm" else "4"))

# ---------------------------------------------------------------------------
# Validation thresholds
# ---------------------------------------------------------------------------
BLOCKING_KEY_ISSUE_SHARE = 0.01                   # null / duplicate / orphan keys above 1 % block the run
MIN_SCORE_COVERAGE = 0.70                         # below -> warning

# ---------------------------------------------------------------------------
# Weekly completeness per pipeline (master prompt §17.9, "which table is used where")
# required: missing = the run is blocked; optional: missing = warning (a fallback exists)
# ---------------------------------------------------------------------------
_ACT_AUTO = ["activities_w1_auto", "activities_w2_auto", "activities_w3_auto", "activities_w4_auto",
             "activities_m1_auto", "activities_auto_m2", "activities_m3_auto"]
_ACT_SCORE = ["activities_w1_score", "activities_w2_score", "activities_w3_score", "activities_w4_score",
              "activities_m1_score", "activities_m2_score", "activities_m3_score"]
_SCORES = [m.table for m in MODEL_SCORE_REGISTRY]
PIPELINE_TABLES = {
    "train_week0": {
        "required": ["base_prepaid", "consumer_aact_account", "access_method_id_auto", "party_auto",
                     "subs_amt_auto", *_ACT_AUTO, *_SCORES, "subscriptions_tg",
                     "usage_data_voice_payg_tg", "targeting_dataiku_auto", "targeting_auto", "subs_btl_auto",
                     "sub_btl_3months", "btl_catalog"],
        # customer_profile missing -> AGE from the party table, tenure from access_method_subs_start_dt (§2.1).
        # V4.0: week 0 reads only the *_auto history, so the D-anchored *_score tables are optional here: a weekly
        # export of fresh *_auto tables (one folder, no *_score) is a complete week-0 run.
        "optional": [PROFILE_TABLE, "location_auto", "DEVICEBEHAVIORSEGMENTATION.dv_segment", "paygo_activity_auto",
                     CONSENT_TABLE, "party_scoring", "subs_amt_score", *_ACT_SCORE],
    },
    "train_weekly": {
        "required": ["base_prepaid", "consumer_aact_account", "party_scoring", "subs_amt_score", *_ACT_SCORE, *_SCORES,
                     "subscriptions_tg", "usage_data_voice_payg_tg", "targeting_auto", "subs_btl_auto",
                     "sub_btl_3months", "btl_catalog"],
        "optional": [PROFILE_TABLE, "location_auto", "DEVICEBEHAVIORSEGMENTATION.dv_segment", "targeting_dataiku_auto",
                     CONSENT_TABLE],
    },
    "score": {   # + the validated training bundle (checked by fsdr-score itself)
        "required": ["subscriptions_tg", "usage_data_voice_payg_tg", "targeting_auto", "sub_btl_3months", "btl_catalog",
                     "BTL_script_english_arabic_ok", "conflicting_offers_h"],
        "optional": [TARGET_SAMPLE_TABLE, CONSENT_TABLE],
    },
}

# ---------------------------------------------------------------------------
# V2.0 Phase 3: features (B-01 .. B-12), S-rules, bands, micro-segments, the two actions (master prompt §6 - §8)
# ---------------------------------------------------------------------------
FEATURE_WORKERS = int(os.getenv("FSDR_FEATURE_WORKERS", "2"))     # partitions in parallel (8 on the P16, plan §15)
# B-04: the 3 churn tables are stacked in registry order (global, 0-30, 30-60); the first value per line is kept
CHURN_FEATURE = "PROBA_CHURN"
# B-05: a missing score is flagged and imputed with the median of the line's nationality group (full base)
MISSING_FLAG_SUFFIX = "_MISSING"
# B-11 / §9: the context the LLM NBA agent reads besides the scores and trends (customer_360), fixed order
CUSTOMER_360_CONTEXT = ["NATIONALITY_GROUP", "AGE_BAND", "TENURE_BAND", "MICRO_SEGMENT", "VALUE_BAND", "USAGE_TYPE",
                        "REGION", "DV_SEGMENT", "SEGMENT", "RULE_ACTION", "ACTION_ORIGINAL", "ACTION_AGENT",
                        "ACTION_AGENT_RANK2", "ARPU_M1", "TENURE_DAYS", "AGE"]

# S-rules (§7): current segment of every line, first match wins
S_TENURE_NURSERING = 45                           # S-1/2: tenure < 45 days -> excluded
S_TENURE_ONBOARDING = 90                          # S-3/4
S_CHURN_AT_RISK = 0.5                             # S-1..4 and S-8 (PROBA_CHURN)
S_CHURN_1D_AT_RISK = 0.5                          # S-8 (PROBA_CHURN_1D)
S_RECONNECT_HIGH = 0.6                            # S-5
S_RECONNECT_MED = 0.3                             # S-6 (below -> S-7)
S_FRAGILITY = 0.5                                 # S-9
S_REV_DROP_PCT = -30.0                            # S-9 revenue M1 vs M2
S_VOICE_USER_MIN = 1.0                            # Local_Voice_min_M1 >= 1
S_DATA_USER_MB = 10.0                             # Data_Volume_MB_M1 >= 10
S_APPETENCY = 0.5                                 # S-11/12 (PROBA_APP_VOICE), S-13/14 (PROBA_APP_DATA)
# (rule id, segment, rule action); None = excluded (B-02). The order is the evaluation order.
S_RULES = [
    ("S-1", "Nursering_Early_Churn_At_Risk", None),
    ("S-2", "Nursering_Early_Churn_Not_At_Risk", None),
    ("S-3", "Onboarding_Early_Churn_At_Risk", "UPSELL"),
    ("S-4", "Onboarding_Early_Churn_Not_At_Risk", "UPSELL"),
    ("S-5", "Retention_High_Probability_To_Reconnect", "RETAIN"),
    ("S-6", "Retention_Med_Probability_To_Reconnect", "RETAIN"),
    ("S-7", "Retention_Low_Probability_To_Reconnect", "RETAIN"),
    ("S-8", "Retention_RGS_At_Risk", "RETAIN"),
    ("S-9", "Retention_RGS_Fragile", "UPSELL"),
    ("S-10", "RGS_Not_At_Risk_DualPlay_Upgrade", "MAINTAIN"),
    ("S-11", "Data_Only_Voice_Appetency", "UPSELL VOICE"),
    ("S-12", "Data_Only_No_Voice_Appetency", "CROSS SELL VOICE"),
    ("S-13", "Voice_Only_Data_Appetency", "UPSELL DATA"),
    ("S-14", "Voice_Only_No_Data_Appetency", "CROSS SELL DATA"),
]
S_SEGMENTS = [s for _, s, _ in S_RULES]

# B-03: the original dict_segm_actions of code 1, entry by entry in source order. It is a Python dict literal
# with repeated keys, so the LAST value wins (open item 1) and keys are case-sensitive (RGS_ vs Rgs_).
DICT_SEGM_ACTIONS_SOURCE = [
    ("Retention_Low_Probability_To_Reconnect", "RETAIN"), ("RGS_Not_At_Risk_DualPlay_Upgrade", "MAINTAIN"),
    ("Retention_RGS_At_Risk", "RETAIN"), ("Retention_RGS_Fragile", "UPSELL"),
    ("RGS_Not_At_Risk_Data_Only_Voice_Appetency", "UPSELL VOICE"), ("Retention_High_Probability_To_Reconnect", "UPSELL"),
    ("RGS_Not_At_Risk_Data_Only_No_Voice_Appetency", "CROSS SELL VOICE"), ("Onboarding_Early_Churn_At_Risk", "UPSELL"),
    ("RGS_Not_At_Risk_Voice_Only_Data_Appetency", "UPSELL DATA"), ("Onboarding_Early_Churn_Not_At_Risk", "UPSELL"),
    ("RGS_Not_At_Risk_Voice_Only_No_Data_Appetency", "CROSS SELL DATA"), ("Retention_Rgs_Fragile", "UPSELL"),
    ("Retention_Rgs_At_Risk", "RETAIN"), ("Rgs_Not_At_Risk_Data_Only_No_Voice_Appetency", "CROSS SELL VOICE"),
    ("Rgs_Not_At_Risk_Dualplay_Upgrade", "MAINTAIN"), ("Rgs_Not_At_Risk_Voice_Only_No_Data_Appetency", "CROSS SELL DATA"),
    ("Rgs_Not_At_Risk_Voice_Only_Data_Appetency", "UPSELL DATA"), ("Rgs_Not_At_Risk_Data_Only_Voice_Appetency", "UPSELL VOICE"),
    ("Retention_Med_Probability_To_Reconnect", "RETAIN"), ("Retention_Low_Probability_To_Reconnect", "RETAIN"),
    ("Retention_High_Probability_To_Reconnect", "RETAIN"), ("Rgs_Not_At_Risk_Data_Only_No_Voice_Appetency", "CROSS SELL VOICE"),
    ("Rgs_Not_At_Risk_Dualplay_Upgrade", "MAINTAIN"), ("Retention_Rgs_Fragile", "UPSELL"),
    ("Onboarding_Early_Churn_Not_At_Risk", "UPSELL"), ("Rgs_Not_At_Risk_Data_Only_Voice_Appetency", "CROSS SELL VOICE"),
    ("Onboarding_Early_Churn_At_Risk", "UPSELL"), ("Retention_Rgs_At_Risk", "RETAIN"),
    ("Rgs_Not_At_Risk_Voice_Only_Data_Appetency", "UPSELL DATA"), ("Rgs_Not_At_Risk_Voice_Only_No_Data_Appetency", "CROSS SELL DATA"),
]
DICT_SEGM_ACTIONS = dict(DICT_SEGM_ACTIONS_SOURCE)                 # last value wins, as in code 1
# §7.1: S-rule segment -> original segment name, for lines without a segment-engine segment this week.
# Every line is a CHECK item (plan §11): the S-rules re-create the segment engine, they are not the engine.
SEGMENT_CROSSWALK = {
    "Nursering_Early_Churn_At_Risk": ("Nursering_Early_Churn_At_Risk", "same name; excluded (B-02)"),
    "Nursering_Early_Churn_Not_At_Risk": ("Nursering_Early_Churn_Not_At_Risk", "same name; excluded (B-02)"),
    "Onboarding_Early_Churn_At_Risk": ("Onboarding_Early_Churn_At_Risk", "same name"),
    "Onboarding_Early_Churn_Not_At_Risk": ("Onboarding_Early_Churn_Not_At_Risk", "same name"),
    "Retention_High_Probability_To_Reconnect": ("Retention_High_Probability_To_Reconnect",
                                                "same name; original action RETAIN by last-value-wins (was UPSELL)"),
    "Retention_Med_Probability_To_Reconnect": ("Retention_Med_Probability_To_Reconnect", "same name"),
    "Retention_Low_Probability_To_Reconnect": ("Retention_Low_Probability_To_Reconnect", "same name"),
    "Retention_RGS_At_Risk": ("Retention_RGS_At_Risk", "same name (RGS_ spelling)"),
    "Retention_RGS_Fragile": ("Retention_RGS_Fragile", "same name (RGS_ spelling)"),
    "RGS_Not_At_Risk_DualPlay_Upgrade": ("RGS_Not_At_Risk_DualPlay_Upgrade", "same name (RGS_ spelling)"),
    "Data_Only_Voice_Appetency": ("RGS_Not_At_Risk_Data_Only_Voice_Appetency",
                                  "RGS_ spelling -> UPSELL VOICE (the Rgs_ spelling gives CROSS SELL VOICE)"),
    "Data_Only_No_Voice_Appetency": ("RGS_Not_At_Risk_Data_Only_No_Voice_Appetency", "RGS_ spelling"),
    "Voice_Only_Data_Appetency": ("RGS_Not_At_Risk_Voice_Only_Data_Appetency",
                                  "S-13 also takes lines with no usage; the engine may not"),
    "Voice_Only_No_Data_Appetency": ("RGS_Not_At_Risk_Voice_Only_No_Data_Appetency",
                                     "S-14 also takes lines with no usage; the engine may not"),
}

# §8 bands. Micro-segment = nationality group x age band x tenure band (5 x 4 x 6 = 120 cells)
NATIONALITY_GROUPS = ["Oman", "Pakistan", "India", "Bangladesh", "others"]
AGE_MIN, AGE_MAX = 15, 100                        # outside -> unknown
AGE_BANDS = [("young", 0, 40), ("middle", 40, 60), ("senior", 60, 200)]   # [lo, hi)
AGE_BAND_NAMES = [b for b, _, _ in AGE_BANDS] + ["unknown"]
YOUNG_SPLIT_AGE = 25                              # open item 5: under 25 / 25-39, tested in Phase 4
TENURE_BANDS = [("t0_lt3m", 0, 90), ("t1_3_6m", 90, 183), ("t2_6_12m", 183, 366), ("t3_1_2y", 366, 731),
                ("t4_gt2y", 731, 10**9)]          # days, [lo, hi)
TENURE_BAND_NAMES = [b for b, _, _ in TENURE_BANDS] + ["unknown"]
VALUE_BANDS = ["zero", "low", "mid", "high"]      # ARPU_M1 = 0, then tertiles of the paying lines (context only)
USAGE_TYPES = ["dual", "data_only", "voice_only", "none"]
AGENT_DICTIONARY_PREFIX = "dict_segm_actions_agent_v"

# ---------------------------------------------------------------------------
# V3.0: 100k contacted in four groups of 25k + control (update prompt PROMPT_FSDR_update_100k_targets.md)
# 2 x 2 design: rules (production / challenger) x offer universe (constraint = code 3 / free = code 4)
# ---------------------------------------------------------------------------
T1_CONSTRAINT_SIZE = int(os.getenv("FSDR_T1_CONSTRAINT_SIZE", "25000"))
T1_FREE_SIZE = int(os.getenv("FSDR_T1_FREE_SIZE", "25000"))
# V4.0: Target 2 only by default (the LLM agents' target), 50k a (T2C) + 50k b (T2F); Target 1 is off (FSDR_TARGETS)
T2_CONSTRAINT_SIZE = int(os.getenv("FSDR_T2_CONSTRAINT_SIZE", "50000"))
T2_FREE_SIZE = int(os.getenv("FSDR_T2_FREE_SIZE", "50000"))
CONTROL_SIZE = int(os.getenv("FSDR_CONTROL_SIZE", "0"))            # V4.0: no control group by default; 20000 = a baseline
T2_SPLIT = os.getenv("FSDR_T2_SPLIT", "true").lower() == "true"     # false -> one 50k T2 with the CC rules
GROUP_SIZE_TOLERANCE = 0.02                       # A-30: sizes within +-2 %
BALANCE_MAX_POINTS = 2.0                          # A-31: stratum shares differ by at most 2 points between groups

# One entry per contacted group. universe: "constraint" (code 3 families by action) | "free" (code 4: L2,
# DATA / COMBO, action not used). rules: "code3" | "code4" | "CC" | "CF". rule_action: the NBA hint and fallback.
GROUPS = {
    "t1c": {"panel_group": "t1_constraint", "target": "target1", "universe": "constraint", "rules": "code3",
            "agent": "6A-C", "tg_flag": "constraint", "rule_action": "ACTION_ORIGINAL", "nbo_sees_action": True,
            "consistency_loop": True, "rules_version": "production_constraint",
            "scored_table": "TG_40K_constraint_scored_nba", "recos_monthly": "recos_monthly_40k_constraint",
            "recos_weekly": "recos_weekly_40k_constraint", "action_column": True},
    "t1f": {"panel_group": "t1_free", "target": "target1", "universe": "free", "rules": "code4",
            "agent": "6A-F", "tg_flag": "free", "rule_action": "ACTION_ORIGINAL",
            "nbo_sees_action": os.getenv("FSDR_T1F_NBO_SEES_ACTION", "false").lower() == "true",
            "consistency_loop": False, "rules_version": "production_free",
            "scored_table": "TG_40k_free_scored_nba", "recos_monthly": "recos_monthly_tg_free",
            "recos_weekly": "recos_weekly_tg_free", "action_column": False},
    "t2c": {"panel_group": "t2_constraint", "target": "target2", "universe": "constraint", "rules": "CC",
            "agent": "6B-C", "tg_flag": "challenger_constraint", "rule_action": "ACTION_AGENT", "nbo_sees_action": True,
            "consistency_loop": True, "rules_version": "CC-v1",
            "scored_table": "TG_challenger_constraint_scored_nba", "recos_monthly": "recos_monthly_challenger_constraint",
            "recos_weekly": "recos_weekly_challenger_constraint", "action_column": True},
    "t2f": {"panel_group": "t2_free", "target": "target2", "universe": "free", "rules": "CF",
            "agent": "6B-F", "tg_flag": "challenger_free", "rule_action": "ACTION_AGENT",
            "nbo_sees_action": os.getenv("FSDR_T2F_NBO_SEES_ACTION", "false").lower() == "true",
            "consistency_loop": False, "rules_version": "CF-v1",
            "scored_table": "TG_challenger_free_scored_nba", "recos_monthly": "recos_monthly_challenger_free",
            "recos_weekly": "recos_weekly_challenger_free", "action_column": False},
}
CONTROL_GROUP = "control"
NOT_SELECTED = "not_selected"
PANEL_GROUPS = [g["panel_group"] for g in GROUPS.values()] + [CONTROL_GROUP, NOT_SELECTED]
# the provided test sample (master §2.4): its "group" column -> our groups; T2C / T2F come from the rest of the base
SAMPLE_GROUP_MAP = {"cstr": "t1c", "free": "t1f", "cg": "control"}
# Tables of the earlier update prompts, replaced by the 4-group names above (kept out of every new output)
SUPERSEDED_TABLES = ["TG_40k_challenger_scored_nba", "recos_monthly_challenger", "recos_weekly_challenger",
                     "TG_40k_challenger_constraint_scored_nba", "TG_40k_challenger_free_scored_nba"]


def group_sizes() -> dict[str, int]:
    """Configured size of every panel group of the ACTIVE targets (t2_split false: one T2 of both sizes, CC rules)."""
    sizes = {}
    if "target1" in TARGETS:
        sizes.update(t1c=T1_CONSTRAINT_SIZE, t1f=T1_FREE_SIZE)
    if "target2" in TARGETS:
        if T2_SPLIT:
            sizes.update(t2c=T2_CONSTRAINT_SIZE, t2f=T2_FREE_SIZE)
        else:
            sizes["t2c"] = T2_CONSTRAINT_SIZE + T2_FREE_SIZE
    if CONTROL_SIZE > 0:                                # V4.0 default 0: no control list, no control file
        sizes[CONTROL_GROUP] = CONTROL_SIZE
    return sizes


# Code 4 / FR rules (T1F) and the free universe of T2F
FREE_UNIVERSE_GENEROSITY = "L2"                   # FR-02
FREE_UNIVERSE_SERVICES = ("COMBO", "DATA")        # FR-03 no VOICE; order = (type A, type B) as code 4
FREE_PRICE_K = 3.0                                # FR-09
RECOMMENDABLE_HISTORY_DAYS = 270                  # B-20 / B-20b (sub_btl_3months)
ONLY_OWN_PAST_OFFERS = os.getenv("FSDR_ONLY_OWN_PAST_OFFERS", "false").lower() == "true"   # B-20b switch
MAX_CANDIDATES_PER_VALIDITY = 25                  # §10: candidates per validity sent to the NBO agent
TARGET1_FREE_SHARE = 0.0                          # B-23: the code-4 part now lives in its own group (T1F)

# Challenger CC-v1 (T2C) and CF-v1 (T2F): open item 4, approved before the first campaign
CHALLENGER_BAND = (1.0, 1.5)                      # CC-02 / CF-02: price between 1.0x and 1.5x the anchor
CHALLENGER_BAND_WIDENED = (1.0, 2.0)              # CC-04 / CF-04: widened once, upwards only
CHALLENGER_UPSELL_STEP = 0.5                      # minimum +0.5 OMR above the anchor for the UPSELL actions
UPSELL_ACTIONS = ["UPSELL", "UPSELL DATA", "UPSELL VOICE"]
DEFAULT_DATA_SHARE = 0.9                          # CC-03 / CF-03: no spend history -> 90 / 10
DO_NOT_DISTURB_IGNORED = 3                        # CC-05 / CF-05
GUARD_MODE_TARGET1 = os.getenv("FSDR_GUARD_MODE_TARGET1", "repair")   # open item 3: "repair" | "report"

# LLM budget at 100k (update §6)
LLM_WEEKLY_BUDGET_H = float(os.getenv("FSDR_LLM_WEEKLY_BUDGET_H", "10"))
LEVEL2_MAX_OUTPUT_TOKENS = 120
LEVEL2_MAX_INPUT_TOKENS = 1500

# ---------------------------------------------------------------------------
# V3.0 target upload folders and the final output (code 5, F-01 .. F-19)
# ---------------------------------------------------------------------------
# V4.0: the lists sit directly in the scoring folder, one sub-folder per target and per a / b group:
#   data/scoring/<D>/target1/a_constraint/target1_a.csv.gz   T1C   target1/b_free/target1_b.csv.gz   T1F
#   data/scoring/<D>/target2/a_constraint/target2_a.csv.gz   T2C   target2/b_free/target2_b.csv.gz   T2F
#   data/scoring/<D>/control/control.csv.gz                   never contacted
TARGETS_FOLDER = os.getenv("FSDR_TARGETS_FOLDER", ".")
ALL_TARGET_FILES = {"t1c": "target1/a_constraint/target1_a", "t1f": "target1/b_free/target1_b",
                    "t2c": "target2/a_constraint/target2_a", "t2f": "target2/b_free/target2_b", "control": "control/control"}
ALL_TARGETS = {"target1": ["t1c", "t1f"], "target2": ["t2c", "t2f"]}   # scored separately (V4.0)
GROUP_SUBFOLDER = {k: v.split("/")[1] for k, v in ALL_TARGET_FILES.items() if k != "control"}   # t1c -> a_constraint
# V4.0: the targets this deployment scores. Default: Target 2 only (where the LLM agents decide); Target 1 (the
# production rules) is not scored, its lists are not read and its folders are not written. FSDR_TARGETS=target1,target2
# brings it back.
TARGETS: dict[str, list[str]] = {}
TARGET_FILES: dict[str, str] = {}


def set_targets(names) -> None:
    """Activates these targets (in place, so every module sees it): TARGETS and the list files they read."""
    names = [n for n in names if n]
    unknown = [n for n in names if n not in ALL_TARGETS]
    if unknown or not names:
        raise ValueError(f"FSDR_TARGETS must name one or more of {list(ALL_TARGETS)}, got {names}")
    TARGETS.clear()
    TARGETS.update({t: ALL_TARGETS[t] for t in ALL_TARGETS if t in names})
    TARGET_FILES.clear()
    TARGET_FILES.update({k: v for k, v in ALL_TARGET_FILES.items()
                         if (k == CONTROL_GROUP and CONTROL_SIZE > 0) or any(k in g for g in TARGETS.values())})


set_targets([t.strip() for t in os.getenv("FSDR_TARGETS", "target2").split(",")])
# V4.0: recommended offers per customer. 6 = 3 monthly + 3 weekly (slots 1-2 monthly, 3-4 weekly, 5 = monthly
# rank 3, 6 = weekly rank 3; a favourite / forced offer only fills a slot 5 / 6 left empty). 4 = the V3.0 file.
RECO_OFFERS = int(os.getenv("FSDR_RECO_OFFERS", "6"))
if RECO_OFFERS not in (4, 6):
    raise ValueError("FSDR_RECO_OFFERS must be 4 or 6")
RECOS_PER_VALIDITY = RECO_OFFERS // 2
TARGETS_MANIFEST = "_targets_manifest.json"
# F-08 forced bundles in slot 6, active version (code 5 "Change the forced offers above to L2 offers"), in order:
# (anchor column, low, high, offer, popular list that blocks the forcing)
_POP_CM = ["CM_3R_8GB_240MIN", "CM_4R_10GB_300MIN", "CM_5R_14GB_360MIN"]
_POP_DM = _POP_CM + ["DM_3R_10GB", "DM_4R_14GB", "DM_5R_18GB"]
FORCED_BUNDLES = [("F05_MONTHLY", 0, 3, "CM_3R_8GB_240MIN", _POP_CM), ("F05_MONTHLY", 3, 4, "CM_4R_10GB_300MIN", _POP_CM),
                  ("F05_MONTHLY", 4, 5, "CM_5R_14GB_360MIN", _POP_CM), ("F05_DATA_MONTHLY", 0, 3, "DM_3R_10GB", _POP_DM),
                  ("F05_DATA_MONTHLY", 3, 4, "DM_4R_14GB", _POP_DM), ("F05_DATA_MONTHLY", 4, 5, "DM_5R_18GB", _POP_DM)]
FAVOURITE_ARPU_FILTER = False                     # F-07 (code 5 commented it out)
OPPORTUNITY_PREFIX = "nbo_LFM_"                   # F-18, expected by the campaign platform
INPUT_FLYTXT_COLUMNS = (["date", "start_date", "end_date", "msisdn"] + [f"channel_{i}" for i in range(1, 7)]
                        + [f"opportunity_name_{i}" for i in range(1, 7)] + [f"destination_offer_name_{i}" for i in range(1, 7)]
                        + ["flag_splio"] + [f"offer_script_en_{i}" for i in range(1, 7)]
                        + [f"offer_script_ar_{i}" for i in range(1, 7)] + ["segment", "segment_btl_taker", "segment_btl_taker_upsell"])


In [ ]:
%%engine nbx.fsdr.calendar
"""Anchor date and every aggregation window of the business rules (BR §4).

D = run date, T = campaign anchor = most recent Thursday on or before D.
Windows are inclusive [start, end] date ranges. The generator aggregates its
daily history with exactly these windows, and the validation / features use
them to know what each column covers, so the two can never disagree.
"""
from __future__ import annotations

from dataclasses import dataclass
from datetime import date, timedelta

from nbx.fsdr import config

WEEKS = ["W1", "W2", "W3", "W4"]      # W1 = most recent week
MONTHS = ["M1", "M2", "M3"]           # M1 = most recent month
PERIODS = WEEKS + MONTHS


@dataclass(frozen=True)
class Window:
    start: date
    end: date

    @property
    def days(self) -> int:
        return (self.end - self.start).days + 1

    def contains(self, d: date) -> bool:
        return self.start <= d <= self.end

    def as_dict(self) -> dict:
        return {"start": self.start.isoformat(), "end": self.end.isoformat(), "days": self.days}


def anchor_date(run_date: date) -> date:
    """T = D - (((D - 2026-03-12) MOD 7) + 7) MOD 7 -- the last Thursday on or before D."""
    return run_date - timedelta(days=(((run_date - config.REFERENCE_THURSDAY).days % 7) + 7) % 7)


def _w(anchor: date, start_off: int, end_off: int) -> Window:
    return Window(anchor + timedelta(days=start_off), anchor + timedelta(days=end_off))


# Offsets (start, end) relative to the anchor, per BR §4.
_SUBS_OFFSETS = {                     # BR 4.1 -- window INCLUDES the anchor day
    "W1": (-6, 0), "W2": (-13, -7), "W3": (-20, -14), "W4": (-27, -21),
    "M1": (-29, 0), "M2": (-59, -30), "M3": (-89, -60),
}
_ACTIVITY_OFFSETS = {                 # BR 4.2 -- window EXCLUDES the anchor day
    "W1": (-7, -1), "W2": (-14, -8), "W3": (-21, -15), "W4": (-28, -22),
    "M1": (-28, -1), "M2": (-56, -29), "M3": (-85, -57),
}
_SUBSCRIPTIONS_TG_OFFSETS = {         # BR 4.3 subscriptions_tg, anchored on D (2-day lag)
    "W1": (-8, -2), "W2": (-15, -9), "W3": (-22, -16), "W4": (-29, -23),
    "M1": (-31, -2), "M2": (-61, -32), "M3": (-91, -62),
}
_USAGE_TG_VOLUME_OFFSETS = {          # BR 4.3 usage_data_voice_payg_tg voice & data p1..p3
    "p1": (-31, -2), "p2": (-61, -32), "p3": (-91, -62),
}
_PAYG_OFFSETS = {                     # BR 4.3 PAYG windows (usage_data_voice_payg_tg on D, paygo_activity_auto on T)
    "M1": (-30, -1), "M2": (-60, -31), "M3": (-90, -61),
    "W1": (-7, -1), "W2": (-14, -8), "W3": (-21, -15), "W4": (-28, -22),
}


def subs_windows(anchor: date) -> dict[str, Window]:
    return {p: _w(anchor, *o) for p, o in _SUBS_OFFSETS.items()}


def activity_windows(anchor: date) -> dict[str, Window]:
    return {p: _w(anchor, *o) for p, o in _ACTIVITY_OFFSETS.items()}


def subscriptions_tg_windows(run_date: date) -> dict[str, Window]:
    return {p: _w(run_date, *o) for p, o in _SUBSCRIPTIONS_TG_OFFSETS.items()}


def usage_tg_volume_windows(run_date: date) -> dict[str, Window]:
    return {p: _w(run_date, *o) for p, o in _USAGE_TG_VOLUME_OFFSETS.items()}


def payg_windows(anchor: date) -> dict[str, Window]:
    return {p: _w(anchor, *o) for p, o in _PAYG_OFFSETS.items()}


def campaign_windows(run_date: date) -> dict[str, Window]:
    """BR-21 .. BR-24, relative to D."""
    return {
        "targeting": _w(run_date, *config.TARGETING_WINDOW),      # BR-21
        "arbitration": _w(run_date, *config.ARBITRATION_WINDOW),  # BR-22
        "reward": _w(run_date, *config.REWARD_WINDOW),            # BR-23
        "interactions": _w(run_date, *config.INTERACTION_WINDOW),  # BR-24
    }


def snapshot_date(run_date: date) -> date:
    """BR-01: base snapshot is ACTIVITY_DATE = D-2."""
    return run_date - timedelta(days=config.SNAPSHOT_LAG_DAYS)


def location_month(run_date: date) -> tuple[int, int]:
    """BR-13: previous calendar month of D-2 -> (month, year); January rolls back to December."""
    snap = snapshot_date(run_date)
    return (12, snap.year - 1) if snap.month == 1 else (snap.month - 1, snap.year)


def usage_history_start(run_date: date) -> date:
    """First day any usage / subscription window can need (D-96)."""
    return run_date - timedelta(days=config.HISTORY_DAYS_USAGE - 1)


def all_windows(run_date: date) -> dict:
    """Every window for D, for the manifest, the UI and the docs."""
    t = anchor_date(run_date)
    d = run_date
    return {
        "run_date": d.isoformat(),
        "anchor_date": t.isoformat(),
        "snapshot_date": snapshot_date(d).isoformat(),
        "location_month": dict(zip(("month", "year"), location_month(d))),
        "subs_amt_auto": {p: w.as_dict() for p, w in subs_windows(t).items()},
        "subs_amt_score": {p: w.as_dict() for p, w in subs_windows(d).items()},
        "activities_auto": {p: w.as_dict() for p, w in activity_windows(t).items()},
        "activities_score": {p: w.as_dict() for p, w in activity_windows(d).items()},
        "subscriptions_tg": {p: w.as_dict() for p, w in subscriptions_tg_windows(d).items()},
        "usage_tg_volume": {p: w.as_dict() for p, w in usage_tg_volume_windows(d).items()},
        "usage_tg_payg": {p: w.as_dict() for p, w in payg_windows(d).items()},
        "paygo_activity_auto": {p: w.as_dict() for p, w in payg_windows(t).items()},
        "campaign": {k: w.as_dict() for k, w in campaign_windows(d).items()},
    }


In [ ]:
%%engine nbx.fsdr.schemas
"""Data contract: the exact raw schema of every input file.

One file per table, file name = table name + ".json" (array), ".jsonl", ".csv" or ".parquet", filed in one of the
5 input folders (usage / models / subscription / catalog / others, plan §13): TableSchema.folder.
Field names, case and production quirks are kept on purpose
(`UBSCRIPTION_ID`, `activities_auto_m2`, `Access_Method_id`, "Price data",
numbers stored as strings in `btl_catalog` and in the test sample), so the
same code runs on the dummy data and on real exports.

Raw types follow the production tables: string, int, bigint, tinyint,
double, date ("YYYY-MM-DD") and timestamp (ISO 8601). In JSON, every value
may also be null unless the field is a key.
"""
from __future__ import annotations

import json
import re
from dataclasses import dataclass, field
from datetime import date, datetime
from pathlib import Path

import numpy as np
import pandas as pd

from nbx.fsdr import config
from nbx.fsdr.calendar import MONTHS, WEEKS

INT_TYPES = {"int", "bigint", "tinyint"}
RAW_TYPES = INT_TYPES | {"string", "double", "date", "timestamp"}
_DATE_RE = re.compile(r"^\d{4}-\d{2}-\d{2}$")


@dataclass(frozen=True)
class Field:
    name: str
    raw_type: str
    key: bool = False           # join key: must be present and non-null


@dataclass
class TableSchema:
    name: str
    fields: list[Field]
    description: str
    group: str
    anchor: str | None = None               # "T", "D" or None
    primary_key: tuple[str, ...] = ()       # duplicates on it are reported
    duplicates_blocking: bool = True        # False: duplicates are expected / resolved downstream
    used_by_model: bool = True              # BR §12: some tables are loaded but not used
    family: str | None = None               # e.g. "model_score", "activities"
    required: bool = True                  # False: a missing file is a warning (a fallback exists)

    def __post_init__(self):
        for f in self.fields:
            if f.raw_type not in RAW_TYPES:
                raise ValueError(f"{self.name}.{f.name}: unknown raw type {f.raw_type}")

    @property
    def field_names(self) -> list[str]:
        return [f.name for f in self.fields]

    @property
    def key_fields(self) -> list[str]:
        return [f.name for f in self.fields if f.key]

    def types(self) -> dict[str, str]:
        return {f.name: f.raw_type for f in self.fields}

    @property
    def folder(self) -> str:
        """Input sub-folder of the table (plan §13)."""
        return folder_of(self.name, self.family)


def folder_of(table: str, family: str | None = None) -> str:
    if family == "model_score":
        return "models"
    for folder, names in config.TABLE_FOLDERS.items():
        if table in names:
            return folder
    return "others"


def _f(name: str, raw_type: str, key: bool = False) -> Field:
    return Field(name, raw_type, key)


# ---------------------------------------------------------------------------
# Field builders for the repetitive table families
# ---------------------------------------------------------------------------
ACTIVITY_METRICS = [  # (name stem, raw type) -- SQL order of activities_*
    ("Call_Count", "int"), ("Data_Session_Count", "int"), ("Voice_Traffic_min", "double"),
    ("Data_Volume_MB", "double"), ("PG_Total_Amount", "double"), ("PG_Voice_Amount", "double"),
    ("PG_Data_Amount", "double"), ("Onnet_Call_Count", "int"), ("Onnet_Voice_min", "double"),
    ("Offnet_Call_Count", "int"), ("Offnet_Voice_min", "double"), ("Local_Call_Count", "int"),
    ("Local_Voice_min", "double"), ("Active_Days_Count", "int"), ("Active_Days_Voice_Count", "int"),
    ("Active_Days_Data_Count", "int"),
]


def activity_table_name(period: str, anchor: str) -> str:
    """activities_w1_auto ... activities_m3_auto, except M2 auto = activities_auto_m2 (production name)."""
    p = period.lower()
    if anchor == "T":
        return "activities_auto_m2" if p == "m2" else f"activities_{p}_auto"
    return f"activities_{p}_score"


def _activity_fields(period: str) -> list[Field]:
    return [_f("ACCESS_METHOD_VAL", "string", True), _f("ACCESS_METHOD_ID", "bigint", True),
            _f("ACCOUNT_ID", "int", True)] + [_f(f"{stem}_{period}", t) for stem, t in ACTIVITY_METRICS]


def _subs_amt_fields() -> list[Field]:
    return ([_f("ACCESS_METHOD_VAL", "string", True), _f("ACCESS_METHOD_ID", "int", True),
             _f("ACCOUNT_ID", "double", True)]
            + [_f(f"Sub_Amt_{p.lower()}", "double") for p in WEEKS + MONTHS])


def _subscriptions_tg_fields() -> list[Field]:
    out = [_f("ACCESS_METHOD_VAL", "string", True), _f("ACCESS_METHOD_ID", "int", True), _f("ACCOUNT_ID", "double", True)]
    for w in WEEKS:
        out += [_f(f"{k}_Sub_Amt_{w.lower()}", "double") for k in ("Total", "Data", "Voice")]
    for m in MONTHS:
        out += [_f(f"Sub_Count_{m}", "int")] + [_f(f"{k}_Sub_Amt_{m}", "double") for k in ("Total", "Data", "Voice")]
    return out


def _payg_fields(prefix_cols: list[Field]) -> list[Field]:
    return prefix_cols + [_f(f"PG_Total_Amount_{p.lower()}", "double") for p in MONTHS + WEEKS]


def _targeting_auto_fields() -> list[Field]:
    out = [_f("DATE1", "date"), _f("START_DATE", "date"), _f("END_DATE", "date"), _f("MSISDN", "string", True)]
    first, last = range(1, 5), range(5, 7)
    out += [_f(f"CHANNEL_{i}", "string") for i in first]
    out += [_f(f"OPPORTUNITY_NAME_{i}", "string") for i in first]
    out += [_f(f"DESTINATION_OFFER_NAME_{i}", "string") for i in first]
    out += [_f("FLAG_SPLIO", "string")]
    out += [_f(f"OFFER_SCRIPT_EN_{i}", "string") for i in first]
    out += [_f(f"OFFER_SCRIPT_AR_{i}", "string") for i in first]
    out += [_f("SEGMENT", "string"), _f("FILENAME", "string"), _f("ETL_CREATE_DT", "date"), _f("JOB_ID", "int")]
    for i in last:
        out += [_f(f"CHANNEL_{i}", "string"), _f(f"OPPORTUNITY_NAME_{i}", "string"),
                _f(f"DESTINATION_OFFER_NAME_{i}", "string"), _f(f"OFFER_SCRIPT_EN_{i}", "string"),
                _f(f"OFFER_SCRIPT_AR_{i}", "string")]
    return out + [_f("SEGMENT_BTL_TAKER", "string"), _f("SEGMENT_BTL_TAKER_UPSELL", "string")]


_CONFLICT_SIDE = ["Offer", "Gen_Level", "Price", "Data", "Min", "Validity", "Offer_Type"]

# The 11 model-score files share ONE identical structure.
MODEL_SCORE_FIELDS = [
    _f("subscription_id", "string"), _f("access_method_val", "bigint", True), _f("access_method_id", "bigint", True),
    _f("tenure", "bigint"), _f("life_in_nw_range", "string"), _f("proba_0", "double"), _f("proba_1", "double"),
    _f("prediction", "string"),
]

SAMPLE_FIELDS = [  # every column is a string in production
    "access_method_val", "access_method_id", "account_id", "tenure", "activation_date", "plan_name", "payg_revenue",
    "subscription_revenue", "total_revenue", "total_revenue_3m", "voice_traffic", "data_volume", "nationality",
    "region", "ARPU", "ARPU_3m", "voice_trf_group", "data_trf_group", "Final", "channel", "group",
    "strata", "counts",
]
_SAMPLE_KEYS = {"access_method_val", "access_method_id", "account_id", "group"}


def model_score_schema(table: str) -> TableSchema:
    return TableSchema(table, list(MODEL_SCORE_FIELDS), "Model score (one shared structure; proba_1 is the NBA score).",
                       "model_scores", primary_key=("access_method_val", "access_method_id"),
                       duplicates_blocking=False, family="model_score")


def _build_tables() -> dict[str, TableSchema]:
    t: list[TableSchema] = [
        # --- Base & reference --------------------------------------------------
        TableSchema("base_prepaid", [
            _f("UBSCRIPTION_ID", "string"), _f("ACCESS_METHOD_VAL", "string", True), _f("ACCESS_METHOD_ID", "int", True),
            _f("ACCOUNT_ID", "int", True), _f("PARTY_ID", "int", True), _f("access_method_subs_start_dt", "string"),
            _f("CURRENT_PLAN", "string")],
            "Active prepaid consumer lines, snapshot D-2 (BR-01..BR-10).", "base", primary_key=("ACCESS_METHOD_VAL",)),
        TableSchema("consumer_aact_account", [_f("ACCOUNT_ID", "bigint", True), _f("PARTY_ID", "bigint", True)],
                    "Active consumer account -> party (BR-11).", "base", primary_key=("ACCOUNT_ID",)),
        TableSchema("access_method_id_auto", [
            _f("ACCESS_METHOD_ID", "int", True), _f("MSISDN", "string", True), _f("ACTIVATION_DATE", "string")],
            "Line id -> MSISDN with 3-character country prefix (BR-12).", "base",
            primary_key=("ACCESS_METHOD_ID", "MSISDN")),
        *[TableSchema(name, [_f("party_id", "int", True), _f("AGE", "int"), _f("GENDER", "string"),
                             _f("NATIONALITY", "string"), _f("nationality_identifier", "string")],
                      "Customer demographics (BR-15).", "base", primary_key=("party_id",))
          for name in ("party_auto", "party_scoring")],
        TableSchema("location_auto", [
            _f("Access_Method_id", "int", True), _f("Calling_Month", "int"), _f("Calling_Year", "int"),
            _f("REGION", "string"), _f("WILAYAT", "string"), _f("AREA", "string")],
            "Home location, previous calendar month of D-2 (BR-13/14). Not used by the model.", "base",
            primary_key=("Access_Method_id", "REGION", "WILAYAT", "AREA"), used_by_model=False),
        TableSchema("btl_catalog", [_f("btl_name", "string", True)] + [_f(n, "string") for n in (
            "type", "generosity", "gb", "min", "price", "Price data", "Price voice", "service", "validity")],
            "BTL offer catalogue; every column is a string (BR-63..65).", "base", primary_key=("btl_name",)),
        TableSchema("BTL_script_english_arabic_ok", [
            _f("Sl No.", "string"), _f("destination_offer_name", "string", True), _f("English Script", "string"),
            _f("Arabic Script", "string")],
            "English / Arabic script per offer.", "base", primary_key=("destination_offer_name",), used_by_model=False),
        TableSchema("conflicting_offers_h",
                    [_f(f"{c}_A", "string", c == "Offer") for c in _CONFLICT_SIDE]
                    + [_f(f"{c}_B", "string", c == "Offer") for c in _CONFLICT_SIDE]
                    + [_f(n, "string") for n in ("Price_Difference", "Data_Difference", "SAME_VALIDITY", "SAME_OFFER_TYPE")],
                    "Pairs of offers that must never be sent together.", "base", primary_key=("Offer_A", "Offer_B")),
        TableSchema(config.CONSENT_TABLE, [
            _f("access_method_val", "string", True), _f("dnd", "tinyint"), _f("marketing_opt_in", "tinyint"),
            _f("updated_at", "date")],
            "V1.0 consent / do-not-contact per line (review #6): dnd = 1 or marketing_opt_in = 0 -> never contacted. "
            "Optional: without it no consent check runs (warning).", "base", primary_key=("access_method_val",),
            used_by_model=False),
        TableSchema(config.PROFILE_TABLE, [_f("msisdn", "string", True), _f("age", "int"), _f("tenure", "int")],
                    f"V2.0 age and tenure per line ({config.PROFILE_TENURE_UNIT}), provided weekly: age and tenure bands "
                    "of the micro-segments. Overrides party_*.AGE and the activation-date tenure. Optional: without it "
                    "AGE comes from party_scoring and tenure from access_method_subs_start_dt.", "base",
                    primary_key=("msisdn",), required=False),
        TableSchema("DEVICEBEHAVIORSEGMENTATION.dv_segment", [_f("ACCESS_METHOD_ID", "int", True), _f("DV_SEGMENT", "string")],
                    "Device behaviour segment. Not used by the model.", "base", primary_key=("ACCESS_METHOD_ID",),
                    used_by_model=False),
        # --- Revenue & usage ---------------------------------------------------
        TableSchema("subs_amt_auto", _subs_amt_fields(), "Subscription amounts, windows on T (BR 4.1).", "usage", "T",
                    ("ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "ACCOUNT_ID"), family="subs_amt"),
        TableSchema("subs_amt_score", _subs_amt_fields(), "Subscription amounts, windows on D (BR 4.1).", "usage", "D",
                    ("ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "ACCOUNT_ID"), family="subs_amt"),
        *[TableSchema(activity_table_name(p, a), _activity_fields(p),
                      f"Usage activity {p}, windows on {a} (BR 4.2).", "usage", a,
                      ("ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "ACCOUNT_ID"), family="activities")
          for a in ("T", "D") for p in WEEKS + MONTHS],
        TableSchema("paygo_activity_auto", _payg_fields([
            _f("ACCESS_METHOD_ID", "bigint", True), _f("ACCOUNT_ID", "int", True), _f("ACCESS_METHOD_VAL", "string", True)]),
            "PAYG amounts on T (BR 4.3). Loaded but not used.", "usage", "T",
            ("ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "ACCOUNT_ID"), used_by_model=False),
        TableSchema("subscriptions_tg", _subscriptions_tg_fields(),
                    "Offer-pricing subscriptions on D, 2-day lag (BR 4.3, BR-19 SUBSCRIPTION_AMT).", "pricing", "D",
                    ("ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "ACCOUNT_ID")),
        TableSchema("usage_data_voice_payg_tg", _payg_fields([
            _f("ACCESS_METHOD_ID", "bigint", True), _f("ACCOUNT_ID", "int", True), _f("ACCESS_METHOD_VAL", "string", True)]
            + [_f(f"local_Voice_min_p{i}", "double") for i in (1, 2, 3)]
            + [_f(f"data_volume_mb_p{i}", "double") for i in (1, 2, 3)]),
            "Offer-pricing usage + positive PAYG on D (BR 4.3, BR-33/34).", "pricing", "D",
            ("ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "ACCOUNT_ID")),
        # --- Campaign ----------------------------------------------------------
        TableSchema("targeting_dataiku_auto", [
            _f("date1", "string"), _f("access_method_id", "int", True), _f("account_id", "int"), _f("segment", "string")]
            + [_f(f"destination_offer_name_{i}", "string") for i in range(1, 7)],
            "Last campaign targeting, date1 in D-8..D-2 (BR-21, BR-36..39).", "campaign", "D",
            ("access_method_id", "date1"), duplicates_blocking=False),
        TableSchema("targeting_auto", _targeting_auto_fields(),
                    "Arbitration targeting, start_date in D-7..D (BR-22). Loaded but not used.", "campaign", "D",
                    duplicates_blocking=False, used_by_model=False),
        TableSchema("subs_btl_auto", [
            _f("access_method_val", "string", True), _f("PRODUCTNAME", "string", True),
            _f("FULFILMENTSUCCESSDATE_TIME", "timestamp"), _f("sub_flag", "tinyint"), _f("nb_sub", "int")],
            "Fulfilments D-7..D-1, MSISDN without prefix (BR-23/25).", "campaign", "D",
            ("access_method_val", "PRODUCTNAME", "FULFILMENTSUCCESSDATE_TIME")),
        TableSchema("sub_btl_3months", [
            _f("access_method_val", "string", True), _f("PRODUCTNAME", "string", True), _f("sub_flag", "tinyint"),
            _f("nb_sub", "int")],
            "Fulfilments D-270..D-1 despite the name (BR-24).", "campaign", "D", ("access_method_val", "PRODUCTNAME")),
        # --- Test sample -------------------------------------------------------
        TableSchema(config.TARGET_SAMPLE_TABLE, [_f(n, "string", n in _SAMPLE_KEYS) for n in SAMPLE_FIELDS],
                    "Campaign test sample: control cg + arms free / cstr / llm, stratified (BR-59..62). All strings.",
                    "sample", "D", ("access_method_val",)),
    ]
    tables = {s.name: s for s in t}
    for m in config.MODEL_SCORE_REGISTRY:
        tables.setdefault(m.table, model_score_schema(m.table))
    return tables


TABLES: dict[str, TableSchema] = _build_tables()

# Accepted alternative file names (production typos) -> canonical table.
TABLE_ALIASES = {"activities_w1_score_hfds": "activities_w1_score"}


def refresh() -> None:
    """Rebuilds TABLES after config.MODEL_SCORE_REGISTRY was changed (tests, plug-in models)."""
    global TABLES
    TABLES = _build_tables()


def model_score_tables() -> list[str]:
    return [m.table for m in config.MODEL_SCORE_REGISTRY]


# ---------------------------------------------------------------------------
# Cross-table key relations (validated: child keys must exist in the parent)
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class Relation:
    child: str
    child_col: str
    parent: str
    parent_col: str
    blocking: bool = True
    note: str = ""


def relations() -> list[Relation]:
    rels = [
        Relation("base_prepaid", "ACCESS_METHOD_ID", "access_method_id_auto", "ACCESS_METHOD_ID"),
        Relation("base_prepaid", "PARTY_ID", "party_auto", "party_id"),
        Relation("base_prepaid", "ACCOUNT_ID", "consumer_aact_account", "ACCOUNT_ID", blocking=False),
        Relation("targeting_dataiku_auto", "access_method_id", "access_method_id_auto", "ACCESS_METHOD_ID"),
        Relation(config.TARGET_SAMPLE_TABLE, "access_method_id", "base_prepaid", "ACCESS_METHOD_ID"),
        Relation(config.TARGET_SAMPLE_TABLE, "account_id", "consumer_aact_account", "ACCOUNT_ID"),
        Relation("subs_amt_auto", "ACCESS_METHOD_VAL", "base_prepaid", "ACCESS_METHOD_VAL", blocking=False),
        Relation("activities_w1_auto", "ACCESS_METHOD_VAL", "access_method_id_auto", "MSISDN", blocking=False),
        # B6: every fulfilment line must belong to a known line (reported, not blocking: churned lines buy too)
        Relation("subs_btl_auto", "access_method_val", "base_prepaid", "ACCESS_METHOD_VAL", blocking=False),
        Relation("sub_btl_3months", "access_method_val", "base_prepaid", "ACCESS_METHOD_VAL", blocking=False),
        *[Relation("targeting_dataiku_auto", f"destination_offer_name_{i}", "btl_catalog", "btl_name", blocking=False)
          for i in range(1, 7)],
        Relation("conflicting_offers_h", "Offer_A", "btl_catalog", "btl_name", blocking=False),
        Relation("conflicting_offers_h", "Offer_B", "btl_catalog", "btl_name", blocking=False),
        Relation(config.PROFILE_TABLE, "msisdn", "base_prepaid", "ACCESS_METHOD_VAL", blocking=False,
                 note="profile of a line outside the base"),
    ]
    rels += [Relation(t, "access_method_val", "base_prepaid", "ACCESS_METHOD_VAL", blocking=False,
                      note="model score for a line outside the base") for t in model_score_tables()]
    return rels


# ---------------------------------------------------------------------------
# Raw-type checks and coercion
# ---------------------------------------------------------------------------
def value_matches(raw_type: str, v) -> bool:
    """Does one non-null JSON value match the raw type?"""
    if raw_type == "string":
        return isinstance(v, str)
    if raw_type in INT_TYPES:
        return isinstance(v, (int, np.integer)) and not isinstance(v, bool)
    if raw_type == "double":
        return isinstance(v, (int, float, np.integer, np.floating)) and not isinstance(v, bool)
    if raw_type == "date":
        return isinstance(v, str) and bool(_DATE_RE.match(v))
    if raw_type == "timestamp":
        if not isinstance(v, str):
            return False
        try:
            datetime.fromisoformat(v)
            return True
        except ValueError:
            return False
    return False


def type_errors(series: pd.Series, raw_type: str, sample: int = 3) -> tuple[int, list]:
    """(number of non-null values with the wrong raw type, a few examples).

    value_matches depends only on the Python type of the value, except for dates and timestamps (content), so
    the check runs once per type present in the column, and the content checks on strings only (review #15:
    value_matches was called once per value, 45 M times at 100 k lines)."""
    s = series[series.notna()]
    if s.empty:
        return 0, []
    kind = pd.api.types.infer_dtype(s, skipna=True)
    if kind in ("string", "integer", "floating", "boolean"):
        groups = [(s.iloc[0].__class__, np.ones(len(s), bool))]
    else:
        types = s.map(type)
        groups = [(t, (types == t).to_numpy()) for t in types.unique()]
    ok = np.zeros(len(s), bool)
    for t, m in groups:
        sub = s[m]
        if raw_type == "date" and issubclass(t, str):
            ok[m] = sub.str.match(_DATE_RE.pattern).fillna(False).to_numpy(dtype=bool)
        elif raw_type == "timestamp" and issubclass(t, str):
            res = {u: value_matches(raw_type, u) for u in pd.unique(sub)}
            ok[m] = sub.map(res).to_numpy(dtype=bool)
        else:
            ok[m] = value_matches(raw_type, sub.iloc[0])
    bad = s[~ok]
    return int(len(bad)), [repr(v) for v in bad.head(sample).tolist()]


def coerce(df: pd.DataFrame, table: str) -> pd.DataFrame:
    """Raw (object) frame -> typed frame for the pipelines: ints -> Int64,
    doubles -> float64, strings -> pandas string, dates / timestamps kept as
    strings. Values that cannot be converted become missing (the validation
    report already lists them)."""
    schema = TABLES[table]
    out = df.copy()
    for f in schema.fields:
        if f.name not in out.columns:
            continue
        col = out[f.name]
        if f.raw_type in INT_TYPES:
            out[f.name] = pd.to_numeric(col, errors="coerce").round().astype("Int64")
        elif f.raw_type == "double":
            out[f.name] = pd.to_numeric(col, errors="coerce").astype("float64")
        else:
            out[f.name] = col.astype("string")
    return out


# ---------------------------------------------------------------------------
# JSON Schema export (docs/fsdr_json_schemas/<table>.json)
# ---------------------------------------------------------------------------
_JSON_TYPE = {"string": "string", "int": "integer", "bigint": "integer", "tinyint": "integer",
              "double": "number", "date": "string", "timestamp": "string"}


def json_schema(table: str) -> dict:
    s = TABLES[table]
    props = {}
    for f in s.fields:
        p: dict = {"type": _JSON_TYPE[f.raw_type] if f.key else [_JSON_TYPE[f.raw_type], "null"],
                   "x-raw-type": f.raw_type}
        if f.raw_type == "date":
            p["pattern"] = _DATE_RE.pattern
        if f.raw_type == "timestamp":
            p["format"] = "date-time"
        props[f.name] = p
    return {
        "$schema": "https://json-schema.org/draft/2020-12/schema",
        "$id": f"fsdr/{table}.json",
        "title": table,
        "description": s.description,
        "type": "array",
        "items": {"type": "object", "properties": props, "required": s.field_names, "additionalProperties": True},
        "x-schema-version": config.SCHEMA_VERSION,
        "x-group": s.group,
        "x-folder": s.folder,
        "x-anchor": s.anchor,
        "x-primary-key": list(s.primary_key),
        "x-used-by-model": s.used_by_model,
    }


def export_json_schemas(out_dir: Path | None = None) -> list[Path]:
    out_dir = Path(out_dir or config.JSON_SCHEMA_DIR)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for name in TABLES:
        p = out_dir / f"{name}.json"
        p.write_text(json.dumps(json_schema(name), indent=2, ensure_ascii=False), encoding="utf-8")
        paths.append(p)
    index = {"schema_version": config.SCHEMA_VERSION, "generated": date.today().isoformat(),
             "tables": {n: {"group": s.group, "folder": s.folder, "anchor": s.anchor, "fields": len(s.fields), "used_by_model": s.used_by_model}
                        for n, s in TABLES.items()}}
    (out_dir / "_index.json").write_text(json.dumps(index, indent=2), encoding="utf-8")
    return paths


In [ ]:
%%engine nbx.fsdr.datasource
"""Where the FSDR pipelines get their tables.

    JsonSource("data/input/2026-10-22")   a run folder, one file per table (real exports or generator output)
    SyntheticSource(n=5000, ...)          the same tables generated in memory

Both return the same thing for a table: a raw frame (python values, None =
null, schema columns first) exactly as a JSON file holds it, so swapping
dummy data for real data is a path change, not a code change. Use
schemas.coerce() to get typed columns.

Layout (plan §13): the tables sit in the 5 input folders (usage/, models/,
subscription/, catalog/, others/) or, as in V1.0, all in the run folder itself.

Formats: "<table>.json" (array of records; JSON Lines is detected from the
content), "<table>.jsonl", "<table>.csv" (text: numbers are parsed with the
raw type of the contract) or "<table>.parquet".
"""
from __future__ import annotations

import json
import re
from datetime import date
from pathlib import Path
from typing import Protocol

import pandas as pd

from nbx.fsdr import config, schemas


class DataSource(Protocol):
    def tables(self) -> list[str]: ...
    def has(self, table: str) -> bool: ...
    def read(self, table: str) -> pd.DataFrame: ...
    def manifest(self) -> dict: ...


def _raw_frame(records: list[dict], table: str) -> pd.DataFrame:
    """Records -> object frame: schema columns first (in schema order), then any extra fields."""
    schema = schemas.TABLES.get(table)
    # dtype=object: no type inference, so an int column with nulls stays ints (from_records would make 7 -> 7.0)
    df = pd.DataFrame(records, dtype=object) if records else pd.DataFrame()
    df = df.where(pd.notna(df), None) if len(df) else df
    if schema is None:
        return df
    extra = [c for c in df.columns if c not in schema.field_names]
    present = [c for c in schema.field_names if c in df.columns]
    return df[present + extra] if len(df.columns) else pd.DataFrame(columns=schema.field_names)


def compact_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Raw frame -> numeric columns as numpy int64 (no null) / float64 (nulls, or ints mixed with floats); text
    stays object. For the pipelines only, once validation has seen the raw values (review #15: a column of Python
    objects costs ~4x the memory of a numpy column, ~30 GB for a 1.7 M-line run)."""
    out = {}
    for c in df.columns:
        s = df[c]
        kind = pd.api.types.infer_dtype(s, skipna=True) if s.dtype == object else ""
        try:
            if kind == "integer":
                out[c] = s.astype("int64") if s.notna().all() else s.astype("float64")
            elif kind in ("floating", "mixed-integer-float"):
                out[c] = s.astype("float64")
            else:
                out[c] = s
        except (OverflowError, TypeError, ValueError):     # e.g. an int beyond int64: keep the raw values
            out[c] = s
    return pd.DataFrame(out, index=df.index)


def read_json_records(path: Path) -> list[dict]:
    """Reads a JSON array or JSON Lines file (auto-detected from the first non-blank character)."""
    text = Path(path).read_text(encoding="utf-8-sig")
    stripped = text.lstrip()
    if not stripped:
        return []
    if stripped[0] == "[":
        data = json.loads(stripped)
        if not isinstance(data, list):
            raise ValueError(f"{path}: expected a JSON array of records")
        return data
    return [json.loads(line) for line in text.splitlines() if line.strip()]


_INT_TEXT = re.compile(r"^-?\d+(\.0+)?$")


def _conform(df: pd.DataFrame, table: str, from_text: bool) -> pd.DataFrame:
    """CSV / Parquet frame -> the raw values a JSON file would hold: None for nulls, python ints for integer
    fields (a Parquet int column with nulls arrives as float), floats for doubles, ISO strings for dates. CSV
    text that does not parse keeps its string, so validation reports it as a type error."""
    schema = schemas.TABLES.get(table)
    types = schema.types() if schema else {}
    out = {}
    for c in df.columns:
        s = df[c]
        rt = types.get(c)
        if pd.api.types.is_datetime64_any_dtype(s):
            fmt = "%Y-%m-%d" if rt == "date" else "%Y-%m-%dT%H:%M:%S"
            vals = s.dt.strftime(fmt).astype(object)
        elif from_text:
            vals = s.astype(object).where(s.astype(str) != "", None)
            if rt in schemas.INT_TYPES or rt == "double":
                num = pd.to_numeric(vals, errors="coerce")
                ok = num.notna()
                if rt in schemas.INT_TYPES:
                    ok &= vals.astype(str).str.match(_INT_TEXT.pattern)
                    conv = [int(v) for v in num[ok]]
                else:
                    conv = [float(v) for v in num[ok]]
                vals = vals.copy()
                vals[ok] = pd.Series(conv, index=num[ok].index, dtype=object)
        else:
            vals = s.astype(object)
            if rt in schemas.INT_TYPES and pd.api.types.is_float_dtype(s):
                whole = s.notna() & (s % 1 == 0)
                vals = vals.copy()
                vals[whole] = pd.Series([int(v) for v in s[whole]], index=s[whole].index, dtype=object)
            elif pd.api.types.is_numeric_dtype(s):
                vals = pd.Series(s.tolist(), index=s.index, dtype=object)       # numpy scalars -> python
        out[c] = vals.where(pd.notna(vals), None)
    return pd.DataFrame(out, index=df.index, columns=df.columns)


def _frame_raw(df: pd.DataFrame, table: str) -> pd.DataFrame:
    """Conformed frame -> schema columns first (in schema order), then any extra fields."""
    schema = schemas.TABLES.get(table)
    if schema is None:
        return df
    extra = [c for c in df.columns if c not in schema.field_names]
    present = [c for c in schema.field_names if c in df.columns]
    return df[present + extra] if len(df.columns) else pd.DataFrame(columns=schema.field_names)


def read_table_file(path: Path, table: str) -> pd.DataFrame:
    """One table file, any supported format -> raw frame."""
    from nbx.fsdr import csvstd

    suffix = csvstd.file_format(path)
    if suffix in (csvstd.SUFFIX, ".csv"):                  # V3.0: .csv.gz (FSDR CSV standard), text in, typed here
        return _frame_raw(_conform(csvstd.read_csv(path), table, from_text=True), table)
    if suffix == ".parquet":
        return _frame_raw(_conform(pd.read_parquet(path), table, from_text=False), table)
    return _raw_frame(read_json_records(path), table)


class JsonSource:
    """A run folder: one file per table (file name = exact table name), in the 5 input folders or flat."""

    def __init__(self, folder: str | Path, compact: bool = False):
        """compact=True: numeric columns typed (compact_frame), for the pipelines; False: raw values, for validation."""
        self.folder = Path(folder)
        if not self.folder.is_dir():
            raise FileNotFoundError(f"No such data folder: {self.folder}")
        self.compact = compact
        self._cache: dict[str, pd.DataFrame] = {}
        self._files = self._index()

    def _index(self) -> dict[str, Path]:
        """Table -> file. Fills found_in (table -> input folder, "" = the run folder itself) and duplicate_files
        (a table found more than once: validation blocks, since nobody can tell which file is the right one)."""
        from nbx.fsdr import csvstd

        # V4.0: data/training/<D> and data/scoring/<D> are read as one run (config.run_twins); single folder otherwise
        self.twins = config.run_twins(self.folder) or {"": self.folder}
        found: dict[str, dict[str, list[tuple[str, Path]]]] = {}
        for side, root in self.twins.items():
            dirs = [(root, "")] + [(root / f, f) for f in config.INPUT_FOLDERS if (root / f).is_dir()]
            for d, sub in dirs:
                for p in sorted(d.iterdir()):
                    if (not p.is_file() or not csvstd.file_format(p) or p.name.startswith("_")
                            or p.name.endswith(csvstd.SCHEMA_SUFFIX)):
                        continue
                    name = csvstd.table_name(p)
                    found.setdefault(schemas.TABLE_ALIASES.get(name, name), {}).setdefault(side, []).append((sub, p))
        files: dict[str, Path] = {}
        self.found_in: dict[str, str] = {}
        self.side_of: dict[str, str] = {}
        self.duplicate_files = {}
        for t, by_side in found.items():
            side = config.table_side(t) if config.table_side(t) in by_side else next(iter(by_side))
            cands = by_side[side]                                                  # a table in both twins: its own side
            if len(cands) > 1:
                self.duplicate_files[t] = [str(p) for _, p in cands]
            home = schemas.TABLES[t].folder if t in schemas.TABLES else None
            sub, p = next(((s, p) for s, p in cands if s == home), cands[0])     # its own folder first
            files[t], self.found_in[t], self.side_of[t] = p, sub, side
        return files

    def misfiled(self) -> dict[str, tuple[str, str]]:
        """V4.0: tables in the wrong twin (a scoring table under training/ or the reverse): table -> (found, expected).
        A warning only: the run still reads them."""
        if set(self.twins) == {""}:
            return {}
        return {t: (s, config.table_side(t)) for t, s in self.side_of.items()
                if t in schemas.TABLES and s != config.table_side(t)}

    @property
    def layout(self) -> str:
        """"folders" (plan §13), "flat" (V1.0: every file in the run folder) or "mixed"."""
        subs = set(self.found_in.values())
        return "flat" if subs <= {""} else "folders" if "" not in subs else "mixed"

    def misplaced(self) -> dict[str, tuple[str, str]]:
        """Contract tables filed in a folder that is not theirs: table -> (found in, expected)."""
        return {t: (sub, schemas.TABLES[t].folder) for t, sub in self.found_in.items()
                if sub and t in schemas.TABLES and sub != schemas.TABLES[t].folder}

    def tables(self) -> list[str]:
        return list(self._files)

    def unknown_files(self) -> list[str]:
        return [t for t in self._files if t not in schemas.TABLES]

    def path(self, table: str) -> Path | None:
        return self._files.get(table)

    def has(self, table: str) -> bool:
        return table in self._files

    def read(self, table: str) -> pd.DataFrame:
        if table not in self._cache:
            if table not in self._files:
                raise FileNotFoundError(f"{table}: no {table}.csv.gz (or a test format) in {self.folder}")
            df = read_table_file(self._files[table], table)
            self._cache[table] = compact_frame(df) if self.compact else df
        return self._cache[table]

    def manifest(self) -> dict:
        for root in [self.folder, *self.twins.values()]:
            p = root / config.MANIFEST_FILE
            if p.exists():
                return json.loads(p.read_text(encoding="utf-8"))
        return {}

    def manifest_files(self) -> list[Path]:
        return [r / config.MANIFEST_FILE for r in dict.fromkeys([self.folder, *self.twins.values()])]

    def compact_table(self, table: str) -> None:
        """Replaces a cached raw table by its typed version (validation, once its raw checks are done)."""
        if table in self._cache:
            self._cache[table] = compact_frame(self._cache[table])


class SyntheticSource:
    """Generates the same tables in memory (no files) -- tests, demos, simulation."""

    def __init__(self, n: int = config.GENERATOR_DEFAULT_N, run_date: date | None = None, scenarios=None,
                 seed: int = config.GENERATOR_SEED, world=None, as_of: date | None = None):
        from nbx.fsdr.synth import generator
        from nbx.fsdr.synth import scenarios as scen
        from nbx.fsdr.synth.world import World

        run_date = run_date or date.today()
        world = world or World.create(n, seed=seed, scenarios=scen.parse(scenarios), as_of=as_of or run_date)
        self.run = generator.generate_run(world, run_date)
        self._cache: dict[str, pd.DataFrame] = {}

    def tables(self) -> list[str]:
        return list(self.run.tables)

    def has(self, table: str) -> bool:
        return table in self.run.tables

    def read(self, table: str) -> pd.DataFrame:
        if table not in self._cache:
            # same path as JsonSource: records -> raw frame
            self._cache[table] = _raw_frame(self.run.tables[table].to_dict("records"), table)
        return self._cache[table]

    def manifest(self) -> dict:
        return dict(self.run.manifest)

    def write(self, folder: str | Path, fmt: str = "json", layout: str = "folders") -> Path:
        from nbx.fsdr.synth import generator

        return generator.write_run(self.run, Path(folder), fmt, layout)


FolderSource = JsonSource          # V2.0 name: the folder holds JSON, JSONL, CSV or Parquet files


In [ ]:
%%engine nbx.fsdr.features
"""NBA feature engineering -- port of docs/fsdr_reference/nba_lib.py, without Dataiku.

One function, build_features(), is used for BOTH training (T-anchored *_auto
tables) and scoring (D-anchored *_score tables), so the two can never drift.

Fixes kept from nba_lib (README_NBA_pipelines.md):
  - one key type everywhere: ACCESS_METHOD_VAL is a canonical 8-digit string,
    ids are Int64 (the original merged strings with integers);
  - missing model score -> uniform(0, 0.02) drawn PER ROW (BR-47), seeded;
  - trends go from the older period to the recent one, W1 = most recent week,
    % change clipped to [-100, 500] (BR-50..53).

V1.0 (review #11 / #12, FSDR_EXTENDED_FEATURES): PROBA_<x>_MISSING flags (the BR-47 fill hides missingness) and
LOG_TENURE (tenure carried by the model-score files). Off = production's feature list.
"""
from __future__ import annotations

from datetime import date

import numpy as np
import pandas as pd

from nbx.fsdr import config, schemas
from nbx.fsdr.calendar import PERIODS
from nbx.fsdr.datasource import DataSource

KEY_ID_COLS = ["ACCESS_METHOD_ID", "ACCOUNT_ID", "PARTY_ID"]


# ---------------------------------------------------------------------------
# Feature names (derived from the registry / config, never hard-coded)
# ---------------------------------------------------------------------------
def trend_feature_names() -> list[str]:
    names = []
    for k, pairs in config.TREND_PAIRS.items():
        for recent, older in pairs:
            names += [f"VAR_{k}_{recent}_{older}", f"LOG_{k}_{recent}_{older}", f"{k}_{older}_ZERO"]
    return names


def numeric_features(registry: list[config.ModelScore] | None = None, extra: list[str] | tuple = ()) -> list[str]:
    """PROBA_* (one per registry feature) + trends (+ extra): the 48 numeric features of production."""
    return config.proba_features(registry) + trend_feature_names() + list(extra)


def extended_feature_names(registry: list[config.ModelScore] | None = None) -> list[str]:
    return [f"{p}_MISSING" for p in config.proba_features(registry)] + ["LOG_TENURE"]


def model_features(registry: list[config.ModelScore] | None = None) -> list[str]:
    """What the bandit reads: the production features + the last-campaign features (M3) when enabled
    + the V1.0 extended features when enabled."""
    extra = list(config.LAST_CAMPAIGN_NUMERIC if config.LAST_CAMPAIGN_FEATURES else ())
    if config.EXTENDED_FEATURES:
        extra += extended_feature_names(registry)
    return numeric_features(registry, extra)


# ---------------------------------------------------------------------------
# Key normalisation
# ---------------------------------------------------------------------------
def canonical_keys(s: pd.Series, strip_zero: bool, empty_none: bool) -> pd.Series:
    """Key values -> canonical strings, vectorised per Python type (a column holds one or two types):
    ints -> str(int); floats -> str(int) when integral, else str; NaN / None -> None; strings stripped
    (strip_zero: '91234567.0' -> '91234567'; empty_none: '' -> None). Same rules as the per-value functions it
    replaces (review #15: they ran once per value, 15-45 M calls at 100 k lines)."""
    out = np.full(len(s), None, dtype=object)       # numpy, not a Series: masked Series assignment turns None into NaN
    if not len(s):
        return pd.Series(out, index=s.index, dtype=object)
    if s.dtype.kind in "iu":
        return s.astype(str).astype(object)
    kind = pd.api.types.infer_dtype(s, skipna=True)
    if kind == "string" or (kind == "empty" and s.dtype == object):
        groups = [(str, s.notna().to_numpy())]
    elif s.dtype.kind == "f" or kind == "floating":
        groups = [(float, s.notna().to_numpy())]
    elif kind == "integer":
        groups = [(int, s.notna().to_numpy())]
    else:
        types = s.map(type)
        groups = [(t, (types == t).to_numpy()) for t in types.unique() if t is not type(None)]
    for t, m in groups:
        if not m.any():
            continue
        sub = s[m]
        if issubclass(t, str):
            x = sub.str.strip()
            if strip_zero:
                drop = x.str.endswith(".0") & x.str[:-2].str.isdigit()
                x = x.where(~drop, x.str[:-2])
            if empty_none:
                x = x.where(x != "", None)
            out[m] = x.to_numpy(dtype=object)
        elif issubclass(t, (bool, int, np.integer)):
            out[m] = [str(int(v)) for v in sub]
        elif issubclass(t, (float, np.floating)):
            arr = sub.to_numpy(dtype=float)
            nan = np.isnan(arr)
            integral = ~nan & np.isfinite(arr) & (np.mod(arr, 1) == 0)
            vals = np.empty(len(arr), dtype=object)
            vals[nan] = None
            vals[integral] = [str(int(v)) for v in arr[integral]]
            rest = ~nan & ~integral
            vals[rest] = [str(v) for v in sub.to_numpy(dtype=object)[rest]]
            out[m] = vals
        else:
            out[m] = [_canonical_one(v, strip_zero, empty_none) for v in sub]
    return pd.Series(out, index=s.index, dtype=object)


def _canonical_one(v, strip_zero: bool, empty_none: bool):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if isinstance(v, (int, np.integer)):
        return str(int(v))
    if isinstance(v, (float, np.floating)):
        return str(int(v)) if float(v).is_integer() else str(v)
    out = str(v).strip()
    if strip_zero and out.endswith(".0") and out[:-2].isdigit():
        out = out[:-2]
    return (out or None) if empty_none else out


def norm_val(s: pd.Series) -> pd.Series:
    """MSISDN -> canonical string: 91234567, 91234567.0 and '91234567' all become '91234567'."""
    return canonical_keys(s, strip_zero=True, empty_none=True)


def _norm_val_per_value(s: pd.Series) -> pd.Series:
    """Reference implementation of norm_val (one value at a time), kept for the equivalence test."""
    def one(v):
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return None
        if isinstance(v, (int, np.integer)):
            out = str(int(v))
        elif isinstance(v, (float, np.floating)):
            out = str(int(v)) if float(v).is_integer() else str(v)
        else:
            out = str(v).strip()
            if out.endswith(".0") and out[:-2].isdigit():
                out = out[:-2]
        return out or None
    return s.map(one).astype(object)


def norm_id(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").round().astype("Int64")


def upper(df: pd.DataFrame) -> pd.DataFrame:
    """Upper-case column names (production tables mix cases) and normalise the join keys."""
    out = df.copy()
    out.columns = [str(c).upper() for c in out.columns]
    if "ACCESS_METHOD_VAL" in out.columns:
        out["ACCESS_METHOD_VAL"] = norm_val(out["ACCESS_METHOD_VAL"])
    for c in KEY_ID_COLS:
        if c in out.columns:
            out[c] = norm_id(out[c])
    return out


def _numeric(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("float64") if c in df.columns else 0.0
    return df


# ---------------------------------------------------------------------------
# Loaders (each returns one row per ACCESS_METHOD_VAL)
# ---------------------------------------------------------------------------
def load_party(src: DataSource, table: str) -> pd.DataFrame:
    df = upper(src.read(table))
    return df[["PARTY_ID", "NATIONALITY"]].dropna(subset=["PARTY_ID"]).drop_duplicates("PARTY_ID")


def load_model_scores(src: DataSource, registry: list[config.ModelScore] | None = None) -> tuple[pd.DataFrame, dict]:
    """One row per line with one column per registry feature. Rows of the same
    feature (e.g. the 3 churn files) and stacked duplicates are merged by max."""
    registry = registry or config.MODEL_SCORE_REGISTRY
    per_feature: dict[str, list[pd.DataFrame]] = {}
    coverage: dict[str, int] = {}
    for m in registry:
        if not src.has(m.table):
            coverage[m.table] = 0
            continue
        df = upper(src.read(m.table))
        df = df[["ACCESS_METHOD_VAL", "PROBA_1"]].dropna(subset=["ACCESS_METHOD_VAL"])
        df["PROBA_1"] = pd.to_numeric(df["PROBA_1"], errors="coerce")
        coverage[m.table] = int(df["ACCESS_METHOD_VAL"].nunique())
        per_feature.setdefault(m.feature, []).append(df.rename(columns={"PROBA_1": m.feature}))
    out: pd.DataFrame | None = None
    for feature, frames in per_feature.items():
        agg = "max" if all(m.combine == "max" for m in registry if m.feature == feature) else "mean"
        f = pd.concat(frames, ignore_index=True).groupby("ACCESS_METHOD_VAL", as_index=False)[feature].agg(agg)
        out = f if out is None else out.merge(f, on="ACCESS_METHOD_VAL", how="outer")
    if out is None:
        out = pd.DataFrame(columns=["ACCESS_METHOD_VAL"])
    return out, coverage


def load_tenure(src: DataSource, registry: list[config.ModelScore] | None = None) -> pd.DataFrame:
    """Tenure (days) per line from the model-score files (they share one structure with a tenure field), max."""
    frames = []
    for m in registry or config.MODEL_SCORE_REGISTRY:
        if src.has(m.table):
            df = upper(src.read(m.table))
            if "TENURE" in df.columns:
                frames.append(pd.DataFrame({"ACCESS_METHOD_VAL": df["ACCESS_METHOD_VAL"],
                                            "TENURE": pd.to_numeric(df["TENURE"], errors="coerce")}))
    if not frames:
        return pd.DataFrame(columns=["ACCESS_METHOD_VAL", "TENURE"])
    return pd.concat(frames).dropna(subset=["ACCESS_METHOD_VAL"]).groupby("ACCESS_METHOD_VAL", as_index=False)["TENURE"].max()


def load_usage(src: DataSource, anchor: str) -> pd.DataFrame:
    """anchor 'T' -> activities_*_auto (training), 'D' -> activities_*_score (scoring)."""
    out: pd.DataFrame | None = None
    for p in PERIODS:
        table = schemas.activity_table_name(p, anchor)
        if not src.has(table):
            continue
        df = upper(src.read(table))
        cols = [f"{u}_{p}" for u in config.USAGE_COLS if f"{u}_{p}" in df.columns]
        df = _numeric(df[["ACCESS_METHOD_VAL"] + cols].dropna(subset=["ACCESS_METHOD_VAL"]).copy(), cols)
        df = df.groupby("ACCESS_METHOD_VAL", as_index=False)[cols].sum()
        out = df if out is None else out.merge(df, on="ACCESS_METHOD_VAL", how="outer")
    return out if out is not None else pd.DataFrame(columns=["ACCESS_METHOD_VAL"])


def load_subs(src: DataSource, anchor: str) -> pd.DataFrame:
    table = "subs_amt_auto" if anchor == "T" else "subs_amt_score"
    if not src.has(table):
        return pd.DataFrame(columns=["ACCESS_METHOD_VAL"])
    df = upper(src.read(table))
    cols = [f"SUB_AMT_{p}" for p in PERIODS if f"SUB_AMT_{p}" in df.columns]
    df = _numeric(df[["ACCESS_METHOD_VAL"] + cols].dropna(subset=["ACCESS_METHOD_VAL"]).copy(), cols)
    return df.groupby("ACCESS_METHOD_VAL", as_index=False)[cols].sum()


# ---------------------------------------------------------------------------
# Eligible base (BR-02, BR-08, BR-10)
# ---------------------------------------------------------------------------
def eligible_base(src: DataSource) -> tuple[pd.DataFrame, dict]:
    return eligible_frame(upper(src.read("base_prepaid")))


def eligible_frame(df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    """BR-02 / BR-08 / BR-10 on an upper-cased base_prepaid frame (the whole base, or one cache partition)."""
    n0 = len(df)
    df = df.dropna(subset=["ACCESS_METHOD_VAL", "ACCESS_METHOD_ID"])
    null_keys = n0 - len(df)
    val = df["ACCESS_METHOD_VAL"].astype(str)
    ok_msisdn = val.str.fullmatch(r"\d{8}") & (val > config.MSISDN_MIN_EXCLUSIVE)        # BR-02
    excluded_plan = df["CURRENT_PLAN"].isin(config.EXCLUDED_PLANS)                       # BR-10
    df = df[ok_msisdn & ~excluded_plan]
    before = len(df)
    df = df.sort_values("ACCESS_METHOD_SUBS_START_DT").drop_duplicates("ACCESS_METHOD_VAL", keep="last")   # BR-08
    report = {"base_rows": n0, "eligible": len(df), "null_keys": null_keys, "excluded_plan": int(excluded_plan.sum()),
              "bad_msisdn": int((~ok_msisdn).sum()), "duplicates_removed": before - len(df)}
    return df.reset_index(drop=True), report


def nationality_groups(src: DataSource) -> pd.Series:
    """ACCESS_METHOD_VAL -> nationality group (BR-15), for the fairness report (review #7)."""
    base = upper(src.read("base_prepaid"))[["ACCESS_METHOD_VAL", "PARTY_ID"]].dropna()
    party = load_party(src, "party_scoring")
    df = base.merge(party, on="PARTY_ID", how="left").drop_duplicates("ACCESS_METHOD_VAL")
    grp = np.where(df["NATIONALITY"].isin(config.NATIONALITY_KEEP), df["NATIONALITY"], config.NATIONALITY_OTHERS)
    return pd.Series(grp, index=df["ACCESS_METHOD_VAL"].astype(str).to_numpy())


def consent_blocked(src: DataSource) -> set[str]:
    """V1.0 review #6: lines that must never be contacted (dnd = 1 or marketing_opt_in = 0 in consent_contact).
    No table = nothing blocked (validation warns that the consent check could not run)."""
    if not src.has(config.CONSENT_TABLE):
        return set()
    df = upper(src.read(config.CONSENT_TABLE))
    dnd = pd.to_numeric(df.get("DND"), errors="coerce").fillna(0) == 1
    opt_out = pd.to_numeric(df.get("MARKETING_OPT_IN"), errors="coerce").fillna(1) == 0
    return set(df.loc[dnd | opt_out, "ACCESS_METHOD_VAL"].dropna().astype(str))


# ---------------------------------------------------------------------------
# Feature engineering (identical for training and scoring)
# ---------------------------------------------------------------------------
def build_features(pop: pd.DataFrame, party: pd.DataFrame, scores: pd.DataFrame, usage: pd.DataFrame, subs: pd.DataFrame,
                   seed: int = config.GENERATOR_SEED, registry: list[config.ModelScore] | None = None) -> pd.DataFrame:
    """pop: one row per line with at least ACCESS_METHOD_VAL and PARTY_ID."""
    proba_cols = config.proba_features(registry)
    df = pop.drop(columns=[c for c in ["NATIONALITY"] if c in pop.columns])
    df = df.dropna(subset=["ACCESS_METHOD_VAL"]).drop_duplicates("ACCESS_METHOD_VAL").copy()
    df = (df.merge(party, on="PARTY_ID", how="left")
            .merge(scores, on="ACCESS_METHOD_VAL", how="left")
            .merge(usage, on="ACCESS_METHOD_VAL", how="left")
            .merge(subs, on="ACCESS_METHOD_VAL", how="left"))

    # BR-47: missing model scores -> small random value, drawn PER ROW
    rng = np.random.default_rng(seed)
    lo, hi = config.MISSING_SCORE_RANGE
    for c in proba_cols:
        if c not in df.columns:
            df[c] = np.nan
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("float64")
        miss = df[c].isna()
        if config.EXTENDED_FEATURES:
            df[f"{c}_MISSING"] = miss.astype(int)
        df.loc[miss, c] = rng.uniform(lo, hi, int(miss.sum()))

    # BR-48 / BR-49: missing usage = 0; NBA revenue = subscriptions + PAYG data
    for p in PERIODS:
        for c in [f"SUB_AMT_{p}", f"PG_DATA_AMOUNT_{p}", f"LOCAL_VOICE_MIN_{p}", f"DATA_VOLUME_MB_{p}"]:
            df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0).astype("float64") if c in df.columns else 0.0
        df[f"REV_{p}"] = df[f"SUB_AMT_{p}"] + df[f"PG_DATA_AMOUNT_{p}"]

    add_trends(df)

    # BR-15 nationality group
    df["NATIONALITY_GROUP"] = np.where(df["NATIONALITY"].isin(config.NATIONALITY_KEEP), df["NATIONALITY"], config.NATIONALITY_OTHERS)
    return df.reset_index(drop=True)


def add_trends(df: pd.DataFrame) -> pd.DataFrame:
    """BR-50..53 (V2.0 B-07): % change, log change and "older period was zero" flag, from older to recent, in place."""
    for k, pairs in config.TREND_PAIRS.items():
        for recent, older in pairs:
            cur = df[config.TREND_SOURCE[k].format(p=recent)]
            prev = df[config.TREND_SOURCE[k].format(p=older)]
            zero = prev == 0
            df[f"{k}_{older}_ZERO"] = zero.astype(int)
            pct = np.where(zero, 0.0, (cur - prev) / prev.where(~zero, 1.0) * 100.0)
            df[f"VAR_{k}_{recent}_{older}"] = np.clip(pct, *config.PCT_CLIP)
            df[f"LOG_{k}_{recent}_{older}"] = np.log1p(cur.clip(lower=0)) - np.log1p(prev.clip(lower=0))
    return df


def build_feature_frame(src: DataSource, pop: pd.DataFrame, anchor: str, seed: int = config.GENERATOR_SEED,
                        registry: list[config.ModelScore] | None = None, log_root=None,
                        context_date=None) -> tuple[pd.DataFrame, dict]:
    """Loads every input for `anchor` ('T' training, 'D' scoring) and builds the features of `pop`.
    log_root / context_date: campaign log and the decision date of the last-campaign features (M3);
    without a log every line gets the "never contacted" values."""
    party = load_party(src, "party_auto" if anchor == "T" else "party_scoring")
    scores, coverage = load_model_scores(src, registry)
    feat = build_features(pop, party, scores, load_usage(src, anchor), load_subs(src, anchor), seed, registry)
    if config.EXTENDED_FEATURES:
        ten = load_tenure(src, registry)
        feat = feat.drop(columns=[c for c in ("TENURE", "LOG_TENURE") if c in feat.columns]).merge(ten, on="ACCESS_METHOD_VAL", how="left")
        feat["LOG_TENURE"] = np.log1p(feat["TENURE"].clip(lower=0)).fillna(0.0)
    if config.LAST_CAMPAIGN_FEATURES:
        from nbx.fsdr import campaign_log
        run_date = src.manifest().get("run_date")
        when = context_date or (date.fromisoformat(run_date) if run_date else date.today())
        lc = campaign_log.last_campaign_features(log_root, feat["ACCESS_METHOD_VAL"], when)
        feat = feat.drop(columns=[c for c in lc.columns if c != "ACCESS_METHOD_VAL" and c in feat.columns])
        feat = feat.merge(lc, on="ACCESS_METHOD_VAL", how="left")
    pop_vals = set(pop["ACCESS_METHOD_VAL"].dropna())
    scored = set(scores["ACCESS_METHOD_VAL"].dropna()) if len(scores) else set()
    report = {"lines": len(feat), "anchor": anchor, "numeric_features": len(numeric_features(registry)), "model_features": len(model_features(registry)),
              "score_coverage": round(len(pop_vals & scored) / len(pop_vals), 4) if pop_vals else 0.0,
              "model_rows": coverage}
    return feat, report


In [ ]:
%%engine nbx.fsdr.offer_rules
"""Offer business rules -- port of production Codes 3 (constraint arm) and 4 (free arm).

  - catalogue, conflicts, eligibility                       BR-63 .. BR-65, conflicting_offers_h
  - ARPU anchor = mean + std of periods, data 90 % / voice 10 %     BR-68 .. BR-77
  - action -> offer families and generosity (normal | all_L2)      BR-66, free arm BR-67
  - 3-pass price-band selection, at most 2 offers per validity      BR-78 .. BR-82
  - NEW: two conflicting offers are never sent together

Fixes vs the original: CROSS SELL VOICE gets its own family filter (the
original repeated "CROSS SELL DATA"), and conflicts are enforced.
"""
from __future__ import annotations

import hashlib
from dataclasses import asdict, dataclass

import numpy as np
import pandas as pd

from nbx.fsdr import config
from nbx.fsdr.calendar import MONTHS, WEEKS
from nbx.fsdr.datasource import DataSource
from nbx.fsdr.features import norm_val, upper

ANCHOR_BY_SERVICE = {"COMBO": "TOTAL", "DATA": "DATA", "VOICE": "VOICE"}   # BR-77


# ---------------------------------------------------------------------------
# Catalogue and conflicts
# ---------------------------------------------------------------------------
def load_catalog(src: DataSource) -> pd.DataFrame:
    """btl_catalog with numeric price / validity / gb / min (production stores them as strings)."""
    df = src.read("btl_catalog").copy()
    df.columns = [str(c).upper().replace(" ", "_") for c in df.columns]
    for c in ("PRICE", "GB", "MIN", "PRICE_DATA", "PRICE_VOICE"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    df["VALIDITY"] = pd.to_numeric(df["VALIDITY"], errors="coerce").astype("Int64")
    for c in ("SERVICE", "GENEROSITY"):
        df[c] = df[c].astype(str).str.strip().str.upper()
    df["BTL_NAME"] = df["BTL_NAME"].astype(str).str.strip()
    return df.drop_duplicates("BTL_NAME").reset_index(drop=True)


def catalog_version(catalog: pd.DataFrame) -> str:
    key = "|".join(f"{r.BTL_NAME}:{r.PRICE}:{r.VALIDITY}" for r in catalog.sort_values("BTL_NAME").itertuples())
    return hashlib.sha1(key.encode()).hexdigest()[:10]


def load_conflicts(src: DataSource) -> set[frozenset]:
    if not src.has("conflicting_offers_h"):
        return set()
    df = src.read("conflicting_offers_h")
    return {frozenset((str(a).strip(), str(b).strip())) for a, b in zip(df["Offer_A"], df["Offer_B"]) if a and b}


def conflicts_with(name: str, chosen: list[str], conflicts: set[frozenset]) -> bool:
    return any(frozenset((name, c)) in conflicts for c in chosen)


# ---------------------------------------------------------------------------
# ARPU anchors (BR-68 .. BR-77)
# ---------------------------------------------------------------------------
def arpu_anchors(src: DataSource) -> pd.DataFrame:
    """One row per line: revenue per period = subscriptions + total PAYG (BR-68),
    anchor = mean + std (BR-71, pandas ddof=1), data = 0.9 x anchor, voice = 0.1 x anchor."""
    sub = upper(src.read("subscriptions_tg")) if src.has("subscriptions_tg") else pd.DataFrame(columns=["ACCESS_METHOD_VAL"])
    use = upper(src.read("usage_data_voice_payg_tg")) if src.has("usage_data_voice_payg_tg") else pd.DataFrame(columns=["ACCESS_METHOD_VAL"])
    m_cols = [f"TOTAL_SUB_AMT_{m}" for m in MONTHS] + [f"TOTAL_SUB_AMT_{w}" for w in WEEKS]
    p_cols = [f"PG_TOTAL_AMOUNT_{p}" for p in MONTHS + WEEKS]
    sub = sub[["ACCESS_METHOD_VAL"] + [c for c in m_cols if c in sub.columns]].copy()
    use = use[["ACCESS_METHOD_VAL"] + [c for c in p_cols if c in use.columns]].copy()
    for df in (sub, use):
        for c in df.columns[1:]:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    sub = sub.dropna(subset=["ACCESS_METHOD_VAL"]).groupby("ACCESS_METHOD_VAL").sum()
    use = use.dropna(subset=["ACCESS_METHOD_VAL"]).groupby("ACCESS_METHOD_VAL").sum()
    out = sub.join(use, how="outer").fillna(0.0)                                       # BR-76 missing = 0
    for c in m_cols + p_cols:
        if c not in out.columns:
            out[c] = 0.0
    for p in MONTHS + WEEKS:
        out[f"REV_{p}"] = out[f"TOTAL_SUB_AMT_{p}"] + out[f"PG_TOTAL_AMOUNT_{p}"]
    for name, periods in (("MONTHLY", MONTHS), ("WEEKLY", WEEKS)):
        cols = [f"REV_{p}" for p in periods]
        out[f"MAX_REV_{name}"] = out[cols].max(axis=1)                                  # BR-72 (not used)
        out[f"STD_REV_{name}"] = out[cols].std(axis=1, ddof=1).fillna(0.0)
        out[f"AVG_REV_{name}"] = out[cols].mean(axis=1) + out[f"STD_REV_{name}"]        # BR-71
        out[f"AVG_REV_DATA_{name}"] = config.ARPU_DATA_SHARE * out[f"AVG_REV_{name}"]   # BR-74
        out[f"AVG_REV_VOICE_{name}"] = config.ARPU_VOICE_SHARE * out[f"AVG_REV_{name}"]  # BR-75
    keep = [f"REV_{p}" for p in MONTHS + WEEKS] + [c for c in out.columns if c.startswith(("AVG_REV", "STD_REV", "MAX_REV"))]
    return out[keep]


def cap_anchor_growth(anchors: pd.DataFrame, previous: dict[str, dict], growth: float) -> pd.DataFrame:
    """M4 price-escalation guard: AVG_REV_* at most (1 + growth) x the anchor logged for the line last time
    (campaign purchases raise next week's spend, so anchors -- and prices -- could climb week after week).
    The data / voice anchors follow (BR-74/75). Lines never logged keep their anchor."""
    out = anchors.copy()
    out["ANCHOR_CAPPED"] = False
    for period, key in (("MONTHLY", "monthly"), ("WEEKLY", "weekly")):
        prev = pd.Series({v: a.get(key) for v, a in previous.items() if a and a.get(key) is not None}, dtype=float)
        prev = prev.reindex(out.index)
        cap = prev * (1.0 + growth)
        over = prev.notna() & (out[f"AVG_REV_{period}"] > cap)
        out.loc[over, f"AVG_REV_{period}"] = cap[over]
        out["ANCHOR_CAPPED"] |= over
        out[f"AVG_REV_DATA_{period}"] = config.ARPU_DATA_SHARE * out[f"AVG_REV_{period}"]
        out[f"AVG_REV_VOICE_{period}"] = config.ARPU_VOICE_SHARE * out[f"AVG_REV_{period}"]
    return out


def anchor_column(service: str, validity: int) -> str:
    """BR-77: COMBO vs total ARPU, DATA vs data ARPU, VOICE vs voice ARPU; weekly offers use weekly anchors.
    Services outside BR-77 (the roaming / international add-on families) use the total anchor, as the add-on
    engine does -- an unknown service must never crash an agent tool."""
    kind = ANCHOR_BY_SERVICE.get(str(service).upper(), "TOTAL")
    period = "WEEKLY" if int(validity) == 7 else "MONTHLY"
    return f"AVG_REV_{period}" if kind == "TOTAL" else f"AVG_REV_{kind}_{period}"


def anchor_value(anchors: pd.DataFrame, val: str, service: str, validity: int) -> float:
    if val not in anchors.index:
        return 0.0
    return float(anchors.at[val, anchor_column(service, validity)])


# ---------------------------------------------------------------------------
# Families (BR-66, BR-67) and price margins (BR-78)
# ---------------------------------------------------------------------------
def families(action: str | None, arm: str, mapping: str | None = None) -> list[tuple[str, str]]:
    """[(service, generosity) for type A, type B]. cstr: from the action; free: L2 COMBO + DATA."""
    if arm == "free":
        return [(s, config.FREE_ARM_GENEROSITY) for s in config.FREE_ARM_SERVICES]
    if action not in config.ACTION_FAMILIES:
        raise ValueError(f"Unknown action {action!r}")
    svc_a, svc_b = config.ACTION_FAMILIES[action]                 # CROSS SELL VOICE has its own entry (typo fixed)
    mapping = mapping or config.GENEROSITY_MAPPING
    if mapping == "all_L2":
        return [(svc_a, "L2"), (svc_b, "L2")]
    gen_a, gen_b = config.ACTION_GENEROSITY_NORMAL[action]
    return [(svc_a, gen_a), (svc_b, gen_b)]


def k_margin(action: str | None, arm: str) -> float:
    return config.PRICE_BAND_K["free" if arm == "free" else action]


def family_items(catalog: pd.DataFrame, fams: list[tuple[str, str]], validity: int) -> pd.DataFrame:
    mask = np.zeros(len(catalog), bool)
    for svc, gen in fams:
        mask |= (catalog["SERVICE"] == svc) & (catalog["GENEROSITY"] == gen)
    return catalog[mask & (catalog["VALIDITY"] == validity)]


# ---------------------------------------------------------------------------
# 3-pass selection (BR-78 .. BR-82)
# ---------------------------------------------------------------------------
@dataclass
class Pick:
    btl_name: str
    service: str
    generosity: str
    validity: int
    price: float
    score: float
    rank: int
    pass_no: int
    anchor: float
    anchor_type: str
    p_accept: float | None = None        # V1.0 value policy: acceptance model's P(accept); None for the price bands

    def as_dict(self) -> dict:
        return asdict(self)


def select_offers(cands: pd.DataFrame, anchors: pd.DataFrame, val: str, fams: list[tuple[str, str]], k: float,
                  validity: int, conflicts: set[frozenset], already: list[str] | None = None) -> list[Pick]:
    """At most one offer per family (so <= 2) for one customer and one validity.

    cands: ranker candidates for this customer (BTL_NAME, SERVICE, GENEROSITY, VALIDITY, PRICE, SCORE, RANK).
    pass 1: anchor < price < anchor + k, sorted score desc, price desc        (BR-78, BR-81 since 9 Oct)
    pass 2: the missing type only, price > anchor, no cap, score desc, price asc (BR-79)
    pass 3: no offer yet: both types, price > anchor, no cap, score desc, price asc (BR-80)
    A customer whose anchor is above every price gets nothing (BR-82)."""
    chosen: list[Pick] = []
    taken = list(already or [])
    c = cands[cands["VALIDITY"] == validity]
    # numpy arrays once per call: pick() runs up to 4 times per customer (review #15: pandas filtering per pick was
    # ~4 ms, i.e. hours for a 1.7 M base); same filters, same order (score desc, then price; ties keep input order)
    names, svcs, gens = c["BTL_NAME"].to_numpy(), c["SERVICE"].to_numpy(), c["GENEROSITY"].to_numpy()
    price, score, rank_ = c["PRICE"].to_numpy(float), c["SCORE"].to_numpy(float), c["RANK"].to_numpy()

    def pick(svc: str, gen: str, lo: float, hi: float | None, asc_price: bool, pass_no: int) -> Pick | None:
        m = (svcs == svc) & (gens == gen) & (price > lo)
        if hi is not None:
            m &= price < hi
        idx = np.flatnonzero(m)
        for j in idx[np.lexsort((price[idx] if asc_price else -price[idx], -score[idx]))]:
            if not conflicts_with(names[j], taken, conflicts):
                return Pick(names[j], svc, gen, int(validity), float(price[j]), float(score[j]), int(rank_[j]), pass_no,
                            round(lo, 6), anchor_column(svc, validity))   # 6 dp: 12.9995 must not print as 13.0
        return None

    anchors_f = [anchor_value(anchors, val, svc, validity) for svc, _ in fams]
    first = []
    for (svc, gen), a in zip(fams, anchors_f):          # sequential: the 2nd pick must see the 1st (conflicts)
        p = pick(svc, gen, a, a + k, False, 1)
        first.append(p)
        if p:
            chosen.append(p)
            taken.append(p.btl_name)
    found = [p is not None for p in first]
    if any(found) and not all(found):                       # pass 2: missing type only
        i = found.index(False)
        svc, gen = fams[i]
        p = pick(svc, gen, anchors_f[i], None, True, 2)
        if p:
            chosen.append(p)
            taken.append(p.btl_name)
    elif not any(found):                                    # pass 3: nothing yet
        for (svc, gen), a in zip(fams, anchors_f):
            p = pick(svc, gen, a, None, True, 3)
            if p:
                chosen.append(p)
                taken.append(p.btl_name)
    return chosen[: config.MAX_OFFERS_PER_VALIDITY]


# ---------------------------------------------------------------------------
# Hard rules (code-enforced; the LLM cannot override them)
# ---------------------------------------------------------------------------
def check_offers(names: list[str], catalog: pd.DataFrame, conflicts: set[frozenset]) -> list[str]:
    """Violations for a proposed list of offers (empty list = all hard rules pass)."""
    v: list[str] = []
    cat = catalog.set_index("BTL_NAME")
    if len(names) != len(set(names)):
        v.append("duplicate offer in the list")
    known = [n for n in names if n in cat.index]
    for n in names:
        if n not in cat.index:
            v.append(f"{n}: not in the catalogue (BR-63)")
        elif not cat.at[n, "PRICE"] > 0:
            v.append(f"{n}: price must be > 0")
    if len(names) > config.MAX_OFFERS_TOTAL:
        v.append(f"{len(names)} offers: at most {config.MAX_OFFERS_TOTAL} in total (BR-82)")
    for validity in config.VALIDITIES:
        k = sum(1 for n in known if int(cat.at[n, "VALIDITY"]) == validity)
        if k > config.MAX_OFFERS_PER_VALIDITY:
            v.append(f"{k} offers with validity {validity}: at most {config.MAX_OFFERS_PER_VALIDITY} (BR-82)")
    for i, a in enumerate(known):
        for b in known[i + 1:]:
            if frozenset((a, b)) in conflicts:
                v.append(f"{a} and {b} conflict (conflicting_offers_h)")
    return v


def customer_eligibility(base_row: dict | None, segment: str | None = None) -> list[str]:
    """Reasons a line may not be contacted (BR-02, BR-10, BR-36); empty = eligible."""
    if base_row is None:
        return ["not in the eligible prepaid base"]
    reasons = []
    val = str(base_row.get("ACCESS_METHOD_VAL") or "")
    if not (len(val) == 8 and val.isdigit() and val > config.MSISDN_MIN_EXCLUSIVE):
        reasons.append("MSISDN outside the prepaid range (BR-02)")
    if base_row.get("CURRENT_PLAN") in config.EXCLUDED_PLANS:
        reasons.append("internal / employee plan (BR-10)")
    if segment and segment.strip().lower() in config.EXCLUDED_SEGMENTS:
        reasons.append("Nursering segment, tenure < 45 days (BR-36)")
    return reasons


In [ ]:
%%engine nbx.fsdr.offers package
"""V3.0 offers of the four groups (update prompt §4): universes, anchors, the price selections and the reco tables.

    universe.py   B-20b recommendable offers; constraint_candidates (T1C, T2C) and free_candidates (T1F, T2F)
    anchors.py    B-26 anchors per offer type, CC-03 / CF-03 data share from the customer's own spend
    selection.py  select_code3 (T1C, fallback of T2C), select_code4 (T1F, fallback of T2F), select_challenger (T2C, T2F)
    replay.py     literal replays of reco_btl3 / reco_btl2 selection, for the parity test only
    recos.py      the code-2 tables and the 8 reco tables of a panel (fallback scores until the LLM agents)

Every function works on whole frames (one row per customer x candidate): no per-customer Python loop.
"""


In [ ]:
%%engine nbx.fsdr.offers.selection
"""The selections of the four groups (update §4). Input: one row per customer x candidate of ONE validity, with
ACCESS_METHOD_VAL, BTL_NAME, SERVICE, GENEROSITY, VALIDITY, PRICE, SCORE (the LLM NBO score, or its fallback),
ACTION, and ANCHOR (B-27, anchors.anchor_for). Output: the selected rows, at most 1 per type (service) per customer,
with PASS and SELECTED_BY. Sorting is stable (mergesort), so a tie keeps the candidate order.

select_code3      T1C, code 3 reco_btl3: k = 2 (MAINTAIN, RETAIN) / 3 (UPSELL and the voice / data actions)
select_code4      T1F, code 4 reco_btl2: k = 3, the action is not used
select_challenger T2C (CC-v1) / T2F (CF-v1): relative band on the challenger anchor (CC-03), score x price,
                  widened once upwards, then the matching T1 passes as fallback (logged)

price passes (B-28 … B-30 / FR-09 … FR-11):
  pass 1  anchor < price < anchor + k, per type: score desc, price desc
  pass 2  customers with one type: the missing type, price > anchor (no cap): score desc, price asc
  pass 3  customers with nothing: both types, price > anchor (no cap): score desc, price asc
  EXT-6   V4.0 (FSDR_RECO_OFFERS=6): a third offer per validity, extend() below
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from nbx.fsdr import config

VAL = "ACCESS_METHOD_VAL"


def _best(df: pd.DataFrame, price_asc: bool, by: list[str] | None = None) -> pd.DataFrame:
    """Best row per (customer, type): score desc, then price desc (pass 1) or asc (passes 2 and 3)."""
    s = df.sort_values([VAL, "SCORE", "PRICE"], ascending=[True, False, price_asc], kind="mergesort")
    return s.drop_duplicates(by or [VAL, "SERVICE"])


def price_passes(c: pd.DataFrame, k: pd.Series | float, rule_ids: tuple[str, str, str]) -> pd.DataFrame:
    """The 3 passes on all customers of c at once. k: per row (code 3, by action) or one value (code 4)."""
    c = c.assign(_K=k)
    p1 = _best(c[(c["PRICE"] > c["ANCHOR"]) & (c["PRICE"] < c["ANCHOR"] + c["_K"])], price_asc=False)
    types = p1.groupby(VAL)["SERVICE"].nunique()
    one = types[types == 1].index
    found = p1.drop_duplicates(VAL).set_index(VAL)["SERVICE"]
    miss = c[c[VAL].isin(one) & (c["SERVICE"] != c[VAL].map(found)) & (c["PRICE"] > c["ANCHOR"])]
    p2 = _best(miss, price_asc=True, by=[VAL])
    p3 = _best(c[~c[VAL].isin(p1[VAL]) & (c["PRICE"] > c["ANCHOR"])], price_asc=True)
    out = pd.concat([p1.assign(PASS=1, SELECTED_BY=rule_ids[0]), p2.assign(PASS=2, SELECTED_BY=rule_ids[1]),
                     p3.assign(PASS=3, SELECTED_BY=rule_ids[2])], ignore_index=True)
    return out.drop(columns="_K").sort_values([VAL, "PASS"], kind="mergesort").reset_index(drop=True)


def select_code3(c: pd.DataFrame) -> pd.DataFrame:
    """T1C: code 3 passes, k by the customer's action (B-28)."""
    k = c["ACTION"].map(config.PRICE_BAND_K).astype(float)
    return price_passes(c, k, ("B-28", "B-29", "B-30"))


def select_code4(c: pd.DataFrame) -> pd.DataFrame:
    """T1F: code 4 passes, k = 3 whatever the action (FR-09 … FR-11)."""
    return price_passes(c, config.FREE_PRICE_K, ("FR-09", "FR-10", "FR-11"))


def challenger_band(c: pd.DataFrame, band: tuple[float, float]) -> tuple[pd.Series, pd.Series]:
    """CC-02 / CF-02: [max(lo x anchor, anchor + 0.5 for UPSELL actions), hi x anchor] on the challenger anchor;
    never below the anchor (no downgrade); MAINTAIN, RETAIN and the cross-sell actions may sit at the anchor."""
    a = c["CH_ANCHOR"]
    step = np.where(c["ACTION"].isin(config.UPSELL_ACTIONS), config.CHALLENGER_UPSELL_STEP, 0.0) \
        if "ACTION" in c.columns else 0.0
    lo = np.maximum(band[0] * a, a + step)
    return pd.Series(lo, index=c.index), band[1] * a


def select_challenger(c: pd.DataFrame, fallback: str, prefix: str = "CC") -> pd.DataFrame:
    """T2C (prefix CC, fallback "code3") or T2F (prefix CF, fallback "code4"). c also needs CH_ANCHOR."""
    c = c.assign(EV=c["SCORE"] * c["PRICE"])                                   # C-04: expected value
    lo, hi = challenger_band(c, config.CHALLENGER_BAND)

    def best_ev(df: pd.DataFrame) -> pd.DataFrame:
        s = df.sort_values([VAL, "EV", "SCORE"], ascending=[True, False, False], kind="mergesort")
        return s.drop_duplicates([VAL, "SERVICE"])

    p1 = best_ev(c[(c["PRICE"] >= lo) & (c["PRICE"] <= hi)]).assign(PASS=1, SELECTED_BY=f"{prefix}-04")
    have = set(zip(p1[VAL], p1["SERVICE"]))
    _, hi_w = challenger_band(c, config.CHALLENGER_BAND_WIDENED)
    missing = ~pd.Series([k in have for k in zip(c[VAL], c["SERVICE"])], index=c.index)
    p2 = best_ev(c[missing & (c["PRICE"] >= lo) & (c["PRICE"] <= hi_w)]).assign(PASS=2, SELECTED_BY=f"{prefix}-04 widened")
    sel = pd.concat([p1, p2], ignore_index=True)
    rest = c[~c[VAL].isin(sel[VAL])]                                           # nothing even widened: T1 passes
    fb = (select_code3 if fallback == "code3" else select_code4)(rest.drop(columns=["EV"])) if len(rest) else rest.iloc[:0]
    if len(fb):
        fb = fb.assign(PASS=fb["PASS"] + 2, SELECTED_BY="fallback " + fb["SELECTED_BY"])
    out = pd.concat([sel.drop(columns=["EV"]), fb], ignore_index=True)
    return out.sort_values([VAL, "PASS"], kind="mergesort").reset_index(drop=True)


EXTENSION_PASS, EXTENSION_RULE = 6, "EXT-6"


def extend(c: pd.DataFrame, sel: pd.DataFrame, per_validity: int) -> pd.DataFrame:
    """V4.0, 6 offers: after the group's passes, a customer with fewer than per_validity offers of this validity gets
    the next candidates of his universe: score desc, price desc, priced above his anchor (the challenger anchor for
    Target 2: never a downgrade), not already selected, and not the same service + price as a selected offer (F-09
    would clear it). PASS = 6, SELECTED_BY = EXT-6. Customers with nothing selected stay with nothing."""
    if not len(sel) or not len(c) or per_validity <= 0:
        return sel
    have = sel.groupby(VAL).size()
    need = (per_validity - have)[lambda x: x > 0]
    if not len(need):
        return sel
    pool = c[c[VAL].isin(need.index)]
    taken = pd.MultiIndex.from_frame(sel[[VAL, "BTL_NAME"]])
    same = pd.MultiIndex.from_frame(sel[[VAL, "SERVICE", "PRICE"]])
    pool = pool[~pd.MultiIndex.from_frame(pool[[VAL, "BTL_NAME"]]).isin(taken)
                & ~pd.MultiIndex.from_frame(pool[[VAL, "SERVICE", "PRICE"]]).isin(same)]
    anchor = pool["CH_ANCHOR"].fillna(pool["ANCHOR"]) if "CH_ANCHOR" in pool.columns else pool["ANCHOR"]
    pool = pool[pool["PRICE"] > anchor + 1e-9]
    pool = pool.sort_values([VAL, "SCORE", "PRICE"], ascending=[True, False, False], kind="mergesort")
    pool = pool.drop_duplicates([VAL, "SERVICE", "PRICE"])
    pool = pool[pool.groupby(VAL).cumcount() < pool[VAL].map(need)]
    if not len(pool):
        return sel
    ext = pool.assign(PASS=EXTENSION_PASS, SELECTED_BY=EXTENSION_RULE)
    out = pd.concat([sel, ext[[col for col in sel.columns if col in ext.columns]]], ignore_index=True)
    return out.sort_values([VAL, "PASS"], kind="mergesort").reset_index(drop=True)


def select(group: str, c: pd.DataFrame) -> pd.DataFrame:
    """The selection of a group key (t1c / t1f / t2c / t2f)."""
    rules = config.GROUPS[group]["rules"]
    if rules == "code3":
        return select_code3(c)
    if rules == "code4":
        return select_code4(c)
    return select_challenger(c, "code3" if rules == "CC" else "code4", prefix=rules)


In [ ]:
%%engine nbx.fsdr.panel
"""Panel (update §3, master §19.2): the eligible base assigned at random to T1C / T1F / T2C / T2F / control.

    panel, report = assign_panel(pool, sizes=config.group_sizes(), seed=7)

Stratified random, never by score:
  1. the pool (eligible, not Nursering, not in another campaign this week, not cooling down) is sorted by the
     `Final` stratum (nationality group x value band x usage type x plan), then age band, tenure band, segment,
     then a random key: neighbours in that order are alike on every balancing variable
  2. systematic sample of the 120k: one customer every pool / 120k positions, random start
  3. the sample is cut into blocks of 24 (= 5 : 5 : 5 : 5 : 4 for 25k x 4 + 20k); every block gets a random
     permutation of the 24 labels. So every stratum is split in the target proportions up to one block.
The provided test sample (sample_cg_tg_new_7oct2026) is used when given: cstr -> T1C, free -> T1F, cg -> control
(topped up from the same strata); T2C / T2F come from the rest of the pool the same way.
A pool smaller than the sizes is scaled down proportionally (reported).
"""
from __future__ import annotations

import math
from functools import reduce

import numpy as np
import pandas as pd

from nbx.fsdr import config

VAL = "ACCESS_METHOD_VAL"
STRATUM_FIELDS = ["NATIONALITY_GROUP", "VALUE_BAND", "USAGE_TYPE", "CURRENT_PLAN"]     # the `Final` stratum
BALANCE_FIELDS = ["AGE_BAND", "TENURE_BAND", "SEGMENT"]
REPORT_FIELDS = ["NATIONALITY_GROUP", "VALUE_BAND", "USAGE_TYPE", "AGE_BAND", "TENURE_BAND", "SEGMENT"]


def panel_group(key: str) -> str:
    return config.CONTROL_GROUP if key == config.CONTROL_GROUP else config.GROUPS[key]["panel_group"]


def eligible_pool(segments: pd.DataFrame, exclude: set[str] | None = None) -> pd.DataFrame:
    """H-01 / B-02 / H-05: not excluded by the S-rules (Nursering), not in `exclude` (other campaigns, cool-down)."""
    pool = segments[segments["EXCLUDED"] == 0]
    if exclude:
        pool = pool[~pool[VAL].isin(exclude)]
    return pool.drop_duplicates(VAL).reset_index(drop=True)


def stratum(df: pd.DataFrame) -> pd.Series:
    cols = [c for c in STRATUM_FIELDS if c in df.columns]
    return reduce(lambda a, b: a + "|" + b, [df[c].astype("string").fillna("-") for c in cols])


def _ordered(df: pd.DataFrame, rng: np.random.Generator) -> pd.DataFrame:
    keys = ["STRATUM"] + [c for c in BALANCE_FIELDS if c in df.columns]
    return df.assign(_R=rng.random(len(df))).sort_values(keys + ["_R"], kind="mergesort").drop(columns="_R")


def _labels(sizes: dict[str, int], n: int, rng: np.random.Generator) -> np.ndarray:
    """n labels in the proportions of sizes: blocks of the smallest pattern, each block randomly permuted."""
    g = reduce(math.gcd, [v for v in sizes.values() if v > 0])
    pattern = np.repeat(list(sizes), [v // g for v in sizes.values()])
    blocks = math.ceil(n / len(pattern))
    out = np.concatenate([rng.permutation(pattern) for _ in range(blocks)])[:n]
    return out


def _draw(pool: pd.DataFrame, sizes: dict[str, int], rng: np.random.Generator) -> pd.DataFrame:
    """Systematic stratified sample of sum(sizes) from the ordered pool, labelled in proportion."""
    need = sum(sizes.values())
    ordered = _ordered(pool, rng)
    if need >= len(ordered):
        sample = ordered
    else:
        step = len(ordered) / need
        idx = (rng.random() * step + step * np.arange(need)).astype(int)
        sample = ordered.iloc[np.minimum(idx, len(ordered) - 1)]
    return sample.assign(GROUP_KEY=_labels(sizes, len(sample), rng))


def scale_sizes(sizes: dict[str, int], available: int) -> tuple[dict[str, int], float]:
    """A pool smaller than the panel: every group scaled by the same factor (multiples of the label pattern)."""
    total = sum(sizes.values())
    if available >= total:
        return dict(sizes), 1.0
    f = available / total
    return {k: int(v * f) for k, v in sizes.items()}, round(f, 6)


def assign_panel(pool: pd.DataFrame, sizes: dict[str, int] | None = None, seed: int = config.GENERATOR_SEED,
                 sample: pd.DataFrame | None = None) -> tuple[pd.DataFrame, dict]:
    """pool: eligible lines with ACCESS_METHOD_VAL and the stratum / balance columns.
    sample: optional (ACCESS_METHOD_VAL, group) of the provided test sample."""
    rng = np.random.default_rng(seed)
    sizes, factor = scale_sizes(dict(sizes or config.group_sizes()), len(pool))
    pool = pool.assign(STRATUM=stratum(pool))
    fixed = pd.DataFrame(columns=[VAL, "GROUP_KEY"])
    if sample is not None and len(sample):
        s = sample.assign(GROUP_KEY=sample["group"].astype(str).str.lower().map(config.SAMPLE_GROUP_MAP))
        s = s.dropna(subset=["GROUP_KEY"])
        s = s[s[VAL].isin(pool[VAL]) & s["GROUP_KEY"].isin(list(sizes))].drop_duplicates(VAL)
        # never more than the configured size: a random subset of an over-full sample group
        s = s.sample(frac=1.0, random_state=int(rng.integers(1 << 31)))
        s = s[s.groupby("GROUP_KEY").cumcount() < s["GROUP_KEY"].map(sizes)]
        fixed = s[[VAL, "GROUP_KEY"]]
    left = {k: v - int((fixed["GROUP_KEY"] == k).sum()) for k, v in sizes.items()}
    left = {k: v for k, v in left.items() if v > 0}
    drawn = _draw(pool[~pool[VAL].isin(fixed[VAL])], left, rng) if left else pool.iloc[:0].assign(GROUP_KEY=None)
    chosen = pd.concat([fixed.merge(pool, on=VAL, how="left"), drawn], ignore_index=True)
    chosen["PANEL_GROUP"] = chosen["GROUP_KEY"].map(panel_group)
    chosen["TARGET"] = chosen["GROUP_KEY"].map(lambda k: "control" if k == config.CONTROL_GROUP else config.GROUPS[k]["target"])
    chosen["SOURCE"] = np.where(chosen[VAL].isin(fixed[VAL]), "sample", "drawn")
    report = {"pool": len(pool), "requested": dict(sizes), "scale_factor": factor,
              "sizes": chosen["GROUP_KEY"].value_counts().reindex(list(sizes), fill_value=0).astype(int).to_dict(),
              "from_sample": int((chosen["SOURCE"] == "sample").sum()), "seed": seed,
              "t2_split": config.T2_SPLIT}
    return chosen.reset_index(drop=True), report


def balance(panel: pd.DataFrame, fields: list[str] | None = None) -> pd.DataFrame:
    """A-31: per field and value, the share (%) of each group, the max gap in points and a chi-square p-value."""
    from scipy.stats import chi2_contingency

    rows = []
    for f in fields or [c for c in REPORT_FIELDS if c in panel.columns]:
        tab = pd.crosstab(panel[f].astype("string").fillna("-"), panel["GROUP_KEY"])
        share = tab / tab.sum() * 100
        try:
            p = float(chi2_contingency(tab.to_numpy())[1]) if tab.shape[0] > 1 and tab.shape[1] > 1 else 1.0
        except ValueError:
            p = 1.0
        for value, r in share.iterrows():
            rows.append({"field": f, "value": value, **{f"share_{g}": round(v, 2) for g, v in r.items()},
                         "max_gap_points": round(float(r.max() - r.min()), 2), "chi2_p": round(p, 4)})
    return pd.DataFrame(rows)


def check_sizes(panel: pd.DataFrame, sizes: dict[str, int], tol: float = config.GROUP_SIZE_TOLERANCE) -> list[str]:
    """A-30: every group within +-tol of its size, no MSISDN in two groups."""
    problems = []
    got = panel["GROUP_KEY"].value_counts()
    for k, v in sizes.items():
        n = int(got.get(k, 0))
        if v and abs(n - v) > tol * v:
            problems.append(f"{k}: {n:,} lines, expected {v:,} (+-{tol:.0%})")
    dup = int(panel[VAL].duplicated().sum())
    if dup:
        problems.append(f"{dup} MSISDN(s) in two groups")
    return problems


In [ ]:
%%engine nbx.fsdr.audit_groups
"""⑨ auditor checks of the four groups (update §7): A-12 per group, A-12b, A-17, A-30, A-31. Deterministic; any FAIL
blocks the final file (H-08: it is not written). The other A / T / R checks come with Phase 7.

    rep = audit_groups(panel, selected, recommendable, sizes)
    rep["status"]  "PASS" | "WARN" | "FAIL"      rep["blocked"]  True on any FAIL
"""
from __future__ import annotations

import pandas as pd

from nbx.fsdr import config
from nbx.fsdr.offer_rules import families
from nbx.fsdr.offers.selection import challenger_band
from nbx.fsdr.panel import balance, check_sizes

VAL = "ACCESS_METHOD_VAL"
EPS = 1e-9


def _check(cid: str, rule: str, bad: pd.DataFrame | list, what: str, warn: bool = False) -> dict:
    n = len(bad)
    examples = (bad[[c for c in (VAL, "GROUP_KEY", "BTL_NAME", "PRICE") if c in bad.columns]].head(5).to_dict("records")
                if isinstance(bad, pd.DataFrame) else list(bad)[:5])
    status = "PASS" if n == 0 else ("WARN" if warn else "FAIL")
    return {"id": cid, "rule": rule, "status": status, "count": n, "what": what, "examples": examples}


def check_universe(sel: pd.DataFrame) -> list[dict]:
    """A-12: every offer inside its group's universe."""
    free = sel[sel["GROUP_KEY"].map(lambda k: config.GROUPS[k]["universe"]) == "free"]
    bad_free = free[(free["SERVICE"] == "VOICE") | (free["GENEROSITY"] != config.FREE_UNIVERSE_GENEROSITY)]
    cons = sel[sel["GROUP_KEY"].map(lambda k: config.GROUPS[k]["universe"]) == "constraint"]
    ok = [(s, g) in families(a, "cstr") for a, s, g in zip(cons["ACTION"], cons["SERVICE"], cons["GENEROSITY"])]
    bad_cons = cons[[not x for x in ok]]
    return [_check("A-12", "FR-02 / FR-03", bad_free, "T1F / T2F offers are L2 and never VOICE"),
            _check("A-12", "B-21 / CC-01", bad_cons, "T1C / T2C offers in the action's families and generosity")]


def check_passes(sel: pd.DataFrame) -> list[dict]:
    """A-11 / A-12: pass-1 offers of T1C / T1F inside anchor < price < anchor + k; T2 band offers inside C-02."""
    t1 = sel[sel["GROUP_KEY"].isin(["t1c", "t1f"]) & (sel["PASS"] == 1)]
    k = t1.apply(lambda r: config.FREE_PRICE_K if r["GROUP_KEY"] == "t1f" else config.PRICE_BAND_K[r["ACTION"]], axis=1) \
        if len(t1) else pd.Series(dtype=float)
    bad1 = t1[~((t1["PRICE"] > t1["ANCHOR"] + EPS) & (t1["PRICE"] < t1["ANCHOR"] + k - EPS))] if len(t1) else t1
    t2 = sel[sel["GROUP_KEY"].isin(["t2c", "t2f"]) & sel["SELECTED_BY"].str.match(r"C[CF]-04")]
    bad2 = t2.iloc[:0]
    if len(t2):
        lo, hi = challenger_band(t2, config.CHALLENGER_BAND)
        _, hi_w = challenger_band(t2, config.CHALLENGER_BAND_WIDENED)
        top = hi.where(~t2["SELECTED_BY"].str.endswith("widened"), hi_w)
        bad2 = t2[(t2["PRICE"] < lo - EPS) | (t2["PRICE"] > top + EPS)]
    return [_check("A-12", "B-28 / FR-09", bad1, "Target 1 pass-1 offers inside anchor < price < anchor + k"),
            _check("A-12", "CC-02 / CF-02", bad2, "Target 2 offers inside the relative ARPU band")]


def audit_groups(panel: pd.DataFrame, selected: pd.DataFrame, recommendable: pd.DataFrame,
                 sizes: dict[str, int]) -> dict:
    checks = [_check("A-30", "sizes", check_sizes(panel, sizes), "group sizes within +-2 %, no MSISDN in two groups")]
    bal = balance(panel)
    imbalanced = bal[(bal["max_gap_points"] > config.BALANCE_MAX_POINTS) & (bal["chi2_p"] < 0.01)]
    checks.append(_check("A-31", "balance", [f"{r.field}={r.value}: gap {r.max_gap_points} pts (p={r.chi2_p})"
                                             for r in imbalanced.itertuples()],
                         "stratum shares within 2 points between groups (significant gaps only)"))
    ctrl = set(panel.loc[panel["GROUP_KEY"] == config.CONTROL_GROUP, VAL])
    if len(selected):
        checks += check_universe(selected) + check_passes(selected)
        checks.append(_check("A-12b", "B-20b", selected[~selected["BTL_NAME"].isin(set(recommendable["BTL_NAME"]))],
                             "every offer is in the catalogue and bought in the base in 270 days"))
        wrong = selected[selected["RULES_VERSION"] != selected["GROUP_KEY"].map(lambda k: config.GROUPS[k]["rules_version"])]
        checks.append(_check("A-17", "rules version", wrong, "each group uses only its own rules / approved version"))
        checks.append(_check("A-21", "H-01", selected[selected[VAL].isin(ctrl)], "no control line gets an offer"))
    status = "FAIL" if any(c["status"] == "FAIL" for c in checks) else ("WARN" if any(c["status"] == "WARN" for c in checks) else "PASS")
    return {"status": status, "blocked": status == "FAIL", "checks": checks}


In [ ]:
%%engine nbx.fsdr.base_features
"""T2_features (master prompt §6, B-01 … B-12): the features of the full base, partition by partition.

    df, stats = partition_features(CacheSource(D), part=3, run_date=D, anchor="T")   # raw: scores may be NaN
    medians = score_medians(paths)                                                   # B-05, on the whole base
    impute(df, medians)

Every line of one cache partition is in the same partition of every table (contract/cache.py), so a partition is
built with joins inside it only, and 8 partitions run in parallel on the P16 (plan §15). Two passes, because B-05
imputes a missing score with the median of the nationality group over the WHOLE base:
  pass 1  per partition: population, joins, revenue, trends, profile        -> base_features/part-NN.parquet
  between the score medians per nationality group and the ARPU tertiles are computed from those files
  pass 2  per partition: impute the missing scores, keep the *_MISSING flags -> the same file

What changes from V1.0's features.py (the rules of code 1 are the same):
  B-04  the 3 churn tables are stacked in registry order and the FIRST value per line is kept (V1.0: max)
  B-05  missing score -> flag + nationality-group median (V1.0: uniform(0, 0.02) draw, documented as replaced)
        A missing score never triggers "at risk": the S-rules read the flags (segmentation/rules.py)
  §8    AGE and TENURE_DAYS come from customer_profile when present, else party AGE and run date - activation
"""
from __future__ import annotations

from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

from nbx.fsdr import config, schemas
from nbx.fsdr.calendar import PERIODS
from nbx.fsdr.features import add_trends, eligible_frame, norm_id, norm_val, upper

USAGE_V2 = ["LOCAL_VOICE_MIN", "DATA_VOLUME_MB", "PG_DATA_AMOUNT", "PG_TOTAL_AMOUNT", "ACTIVE_DAYS_COUNT"]
KEYS = ["ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "ACCOUNT_ID", "PARTY_ID"]
_REFERENCE: dict = {}                              # reference tables, read once per worker process


def anchor_of(pipeline: str) -> str:
    """B-01: week 0 trains on the T-anchored *_auto tables, every later week on the D-anchored *_score tables."""
    return "T" if pipeline == "train_week0" else "D"


def party_table(anchor: str, cache) -> str:
    first, second = ("party_auto", "party_scoring") if anchor == "T" else ("party_scoring", "party_auto")
    return first if cache.has(first) else second


def _part(cache, table: str, part: int) -> pd.DataFrame | None:
    return upper(cache.read(table, part)) if cache.has(table) else None


def _reference(cache, table: str) -> pd.DataFrame | None:
    key = (str(cache.dir), table)
    if key not in _REFERENCE:
        _REFERENCE[key] = upper(cache.read(table)) if cache.has(table) else None
    return _REFERENCE[key]


def _num(df: pd.DataFrame, cols: list[str]) -> pd.DataFrame:
    for c in cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("float64")
    return df


def model_scores(cache, part: int, registry: list[config.ModelScore] | None = None) -> pd.DataFrame:
    """One row per line, one column per score feature (NaN = not scored). B-04: churn = first row of the stack."""
    per_feature: dict[str, list[pd.DataFrame]] = {}
    for m in registry or config.MODEL_SCORE_REGISTRY:
        df = _part(cache, m.table, part)
        if df is None or "PROBA_1" not in df.columns:
            continue
        d = pd.DataFrame({"ACCESS_METHOD_VAL": df["ACCESS_METHOD_VAL"],
                          m.feature: pd.to_numeric(df["PROBA_1"], errors="coerce")}).dropna()
        per_feature.setdefault(m.feature, []).append(d)
    out: pd.DataFrame | None = None
    for feature, frames in per_feature.items():
        st = pd.concat(frames, ignore_index=True)
        if feature == config.CHURN_FEATURE:
            st = st.drop_duplicates("ACCESS_METHOD_VAL", keep="first")
        else:
            st = st.groupby("ACCESS_METHOD_VAL", as_index=False)[feature].max()
        out = st if out is None else out.merge(st, on="ACCESS_METHOD_VAL", how="outer")
    return out if out is not None else pd.DataFrame(columns=["ACCESS_METHOD_VAL"])


def usage(cache, part: int, anchor: str) -> pd.DataFrame:
    out: pd.DataFrame | None = None
    for p in PERIODS:
        df = _part(cache, schemas.activity_table_name(p, anchor), part)
        if df is None:
            continue
        cols = [f"{u}_{p}" for u in USAGE_V2 if f"{u}_{p}" in df.columns]
        d = _num(df[["ACCESS_METHOD_VAL"] + cols].dropna(subset=["ACCESS_METHOD_VAL"]).copy(), cols)
        d = d.groupby("ACCESS_METHOD_VAL", as_index=False)[cols].sum()
        out = d if out is None else out.merge(d, on="ACCESS_METHOD_VAL", how="outer")
    return out if out is not None else pd.DataFrame(columns=["ACCESS_METHOD_VAL"])


def subscriptions(cache, part: int, anchor: str) -> pd.DataFrame:
    df = _part(cache, "subs_amt_auto" if anchor == "T" else "subs_amt_score", part)
    if df is None:
        return pd.DataFrame(columns=["ACCESS_METHOD_VAL"])
    cols = [f"SUB_AMT_{p}" for p in PERIODS if f"SUB_AMT_{p}" in df.columns]
    d = _num(df[["ACCESS_METHOD_VAL"] + cols].dropna(subset=["ACCESS_METHOD_VAL"]).copy(), cols)
    return d.groupby("ACCESS_METHOD_VAL", as_index=False)[cols].sum()


def profile(cache, part: int) -> pd.DataFrame:
    df = _part(cache, config.PROFILE_TABLE, part)
    if df is None or "MSISDN" not in df.columns:
        return pd.DataFrame(columns=["ACCESS_METHOD_VAL", "AGE_PROFILE", "TENURE_PROFILE"])
    d = pd.DataFrame({"ACCESS_METHOD_VAL": norm_val(df["MSISDN"]),
                      "AGE_PROFILE": pd.to_numeric(df.get("AGE"), errors="coerce"),
                      "TENURE_PROFILE": pd.to_numeric(df.get("TENURE"), errors="coerce")})
    return d.dropna(subset=["ACCESS_METHOD_VAL"]).drop_duplicates("ACCESS_METHOD_VAL", keep="last")


def region(cache, part: int) -> pd.DataFrame:
    df = _part(cache, "location_auto", part)
    if df is None or "REGION" not in df.columns:
        return pd.DataFrame(columns=["ACCESS_METHOD_ID", "REGION"])
    d = df.assign(_Y=pd.to_numeric(df.get("CALLING_YEAR"), errors="coerce"),
                  _M=pd.to_numeric(df.get("CALLING_MONTH"), errors="coerce"))
    d = d.dropna(subset=["ACCESS_METHOD_ID"]).sort_values(["_Y", "_M"]).drop_duplicates("ACCESS_METHOD_ID", keep="last")
    return d[["ACCESS_METHOD_ID", "REGION"]]


def device(cache, part: int) -> pd.DataFrame:
    df = _part(cache, "DEVICEBEHAVIORSEGMENTATION.dv_segment", part)
    if df is None or "DV_SEGMENT" not in df.columns:
        return pd.DataFrame(columns=["ACCESS_METHOD_ID", "DV_SEGMENT"])
    return df[["ACCESS_METHOD_ID", "DV_SEGMENT"]].dropna(subset=["ACCESS_METHOD_ID"]).drop_duplicates("ACCESS_METHOD_ID", keep="last")


def partition_features(cache, part: int, run_date: date, anchor: str,
                       registry: list[config.ModelScore] | None = None) -> tuple[pd.DataFrame, dict]:
    """Pass 1 of one partition: B-01 population and left joins in code-1 order, B-05 (usage), B-06, B-07, B-08,
    profile. Scores stay NaN where missing (pass 2 imputes them)."""
    proba = config.proba_features(registry)
    base = _part(cache, "base_prepaid", part)
    if base is None:
        raise FileNotFoundError("base_prepaid is not in the cache")
    base, elig = eligible_frame(base)
    df = base[[c for c in KEYS + ["ACCESS_METHOD_SUBS_START_DT", "CURRENT_PLAN"] if c in base.columns]].copy()

    accounts = _reference(cache, "consumer_aact_account")             # B-01: party through the account if missing
    if accounts is not None and {"ACCOUNT_ID", "PARTY_ID"} <= set(accounts.columns) and df["PARTY_ID"].isna().any():
        acc = accounts.dropna(subset=["ACCOUNT_ID"]).drop_duplicates("ACCOUNT_ID").set_index("ACCOUNT_ID")["PARTY_ID"]
        df["PARTY_ID"] = df["PARTY_ID"].fillna(df["ACCOUNT_ID"].map(acc))
    party = _reference(cache, party_table(anchor, cache))
    if party is not None:
        pty = party[[c for c in ("PARTY_ID", "NATIONALITY", "AGE") if c in party.columns]]
        pty = pty.dropna(subset=["PARTY_ID"]).drop_duplicates("PARTY_ID").rename(columns={"AGE": "AGE_PARTY"})
        df = df.merge(pty, on="PARTY_ID", how="left")
    for c in ("NATIONALITY", "AGE_PARTY"):
        if c not in df.columns:
            df[c] = np.nan

    df = (df.merge(model_scores(cache, part, registry), on="ACCESS_METHOD_VAL", how="left")
            .merge(usage(cache, part, anchor), on="ACCESS_METHOD_VAL", how="left")
            .merge(subscriptions(cache, part, anchor), on="ACCESS_METHOD_VAL", how="left")
            .merge(profile(cache, part), on="ACCESS_METHOD_VAL", how="left")
            .merge(region(cache, part), on="ACCESS_METHOD_ID", how="left")
            .merge(device(cache, part), on="ACCESS_METHOD_ID", how="left"))

    cols: dict[str, pd.Series] = {}
    for f in proba:                                                    # B-05 flags (imputed in pass 2)
        s = pd.to_numeric(df[f], errors="coerce").astype("float64") if f in df.columns else pd.Series(np.nan, index=df.index)
        cols[f] = s
        cols[f"{f}{config.MISSING_FLAG_SUFFIX}"] = s.isna().astype("int8")
    for p in PERIODS:                                                  # B-05 missing usage / spend = 0
        for c in [f"SUB_AMT_{p}"] + [f"{u}_{p}" for u in USAGE_V2]:
            cols[c] = pd.to_numeric(df[c], errors="coerce").fillna(0.0).astype("float64") if c in df.columns \
                else pd.Series(0.0, index=df.index)
        cols[f"REV_{p}"] = cols[f"SUB_AMT_{p}"] + cols[f"PG_DATA_AMOUNT_{p}"]       # B-06
    df = pd.concat([df.drop(columns=[c for c in cols if c in df.columns]), pd.DataFrame(cols, index=df.index)], axis=1)
    trends = add_trends(pd.DataFrame(index=df.index).assign(**{c: df[c] for c in df.columns
                                                                  if c.startswith(("REV_", "LOCAL_VOICE_MIN_", "DATA_VOLUME_MB_"))}))
    df = pd.concat([df, trends.drop(columns=[c for c in trends.columns if c in df.columns])], axis=1)   # B-07

    nat = df["NATIONALITY"].astype("string").str.strip()
    df["NATIONALITY_GROUP"] = np.where(nat.isin(config.NATIONALITY_KEEP), nat, config.NATIONALITY_OTHERS)   # B-08
    df["ARPU_M1"] = df["SUB_AMT_M1"] + df["PG_TOTAL_AMOUNT_M1"]          # §7 REV_M1 of the S-rules (and value band)

    age_p = pd.to_numeric(df["AGE_PROFILE"], errors="coerce")           # §8 sources
    df["AGE"] = age_p.fillna(pd.to_numeric(df["AGE_PARTY"], errors="coerce"))
    df["AGE_SOURCE"] = np.where(age_p.notna(), "profile", np.where(df["AGE"].notna(), "party", "missing"))
    start = pd.to_datetime(df["ACCESS_METHOD_SUBS_START_DT"], errors="coerce")
    by_activation = (pd.Timestamp(run_date) - start).dt.days.astype("float64")
    ten_p = pd.to_numeric(df["TENURE_PROFILE"], errors="coerce")
    df["TENURE_DAYS"] = ten_p.fillna(by_activation)
    df["TENURE_SOURCE"] = np.where(ten_p.notna(), "profile", np.where(by_activation.notna(), "activation", "missing"))
    df = df.drop(columns=["AGE_PROFILE", "TENURE_PROFILE"])

    stats = {"part": part, **elig, "lines": len(df),
             "missing_scores": {f: int(df[f"{f}{config.MISSING_FLAG_SUFFIX}"].sum()) for f in proba},
             "age_source": df["AGE_SOURCE"].value_counts().to_dict(),
             "tenure_source": df["TENURE_SOURCE"].value_counts().to_dict(),
             "no_nationality": int(nat.isna().sum()), "no_region": int(df["REGION"].isna().sum()),
             "no_device": int(df["DV_SEGMENT"].isna().sum())}
    return df.reset_index(drop=True), stats


def score_medians(paths: list[Path], registry: list[config.ModelScore] | None = None) -> dict:
    """B-05 medians of the observed scores per nationality group, over every partition (+ "_global")."""
    proba = config.proba_features(registry)
    frames = [pd.read_parquet(p, columns=["NATIONALITY_GROUP"] + proba) for p in paths]
    df = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=["NATIONALITY_GROUP"] + proba)
    med = {g: {f: (None if pd.isna(v) else round(float(v), 6)) for f, v in row.items()}
           for g, row in df.groupby("NATIONALITY_GROUP")[proba].median().iterrows()}
    med["_global"] = {f: (None if pd.isna(df[f].median()) else round(float(df[f].median()), 6)) for f in proba}
    return med


def arpu_cutoffs(paths: list[Path]) -> list[float]:
    """Tertile cut points of ARPU_M1 among the paying lines (value band, context only: §8)."""
    arpu = pd.concat([pd.read_parquet(p, columns=["ARPU_M1"])["ARPU_M1"] for p in paths], ignore_index=True)
    paying = arpu[arpu > 0]
    return [round(float(q), 4) for q in paying.quantile([1 / 3, 2 / 3])] if len(paying) else [0.0, 0.0]


def impute(df: pd.DataFrame, medians: dict, registry: list[config.ModelScore] | None = None) -> pd.DataFrame:
    """Pass 2 (B-05): every missing score = the median of its nationality group (else global, else 0). In place."""
    glob = medians.get("_global", {})
    for f in config.proba_features(registry):
        miss = df[f].isna()
        if miss.any():
            by_group = {g: m.get(f) for g, m in medians.items() if g != "_global"}
            fill = df.loc[miss, "NATIONALITY_GROUP"].map(by_group).astype("float64")
            df.loc[miss, f] = fill.fillna(glob.get(f) if glob.get(f) is not None else 0.0).fillna(0.0)
    return df


def column_stats(df: pd.DataFrame) -> pd.DataFrame:
    """Per numeric column: count, nulls, sum, min, max (summed over partitions into feature_summary.csv)."""
    num = df.select_dtypes("number")
    return pd.DataFrame({"count": num.count(), "nulls": num.isna().sum(), "sum": num.sum(), "min": num.min(),
                         "max": num.max()})


In [ ]:
%%engine nbx.fsdr.campaign_log
"""Campaign log and rewards: what was sent to whom, and what came back (prompt §6, audit M1..M4).

    nba_campaign_log.jsonl   one row per (run_date, arm, line): action, offers (code, price, validity),
                             policy_source, propensity, bandit PMF, model version, LLM model,
                             justification id, the line's ARPU anchors. Append-only.
    nba_rewards.jsonl        one row per logged line once the reward window D+7 is known: the three reward
                             definitions (any_btl / same_service / exact_offer), revenue, last BTL date.
                             Append-only.

Both files live in <data root>/state/campaign/, next to the dated run folders, so one log follows the
weekly cycle. Appending never rewrites a row: a key (run_date, arm, msisdn) already present is skipped,
so re-running a step is safe.

Uses: training labels with the logged propensity (H5), last-campaign features (M3), the 28-day contact
cap (L4), the anchor growth guard (M4), and the weekly report per arm.

V1.0 (review #15): the parsed rows and the key set of each file are cached on (size, mtime), and an append only
parses the bytes added since, so appending is no longer a full re-read of a growing file (quadratic over time).
"""
from __future__ import annotations

import hashlib
import json
from datetime import date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

from nbx.fsdr import config

LOG_FILE = "nba_campaign_log.jsonl"
REWARDS_FILE = "nba_rewards.jsonl"
KEY = ("run_date", "arm", "msisdn")
LOG_FIELDS = ["run_date", "arm", "msisdn", "action", "offers", "policy_source", "propensity", "bandit_pmf",
              "model_version", "llm_model", "justification_id", "anchors", "logged_at"]


def root_of(data_root: str | Path | None = None) -> Path:
    """<data root>/state/campaign (default data root: config.FSDR_DATA_DIR)."""
    return Path(data_root or config.FSDR_DATA_DIR).resolve() / config.FSDR_STATE_DIR.name / config.CAMPAIGN_DIR


def root_for(data_dir: str | Path) -> Path:
    """The log of the run folder <data root>/<D>."""
    return root_of(Path(data_dir).resolve().parent)


_CACHE: dict[str, dict] = {}          # resolved path -> {"size", "mtime", "rows", "keys"}


def _key(r: dict) -> tuple:
    return tuple(str(r.get(k)) for k in KEY)


def _load(path: Path) -> dict:
    """Rows + keys of a JSONL file; re-parses only the bytes appended since the cached state."""
    p = str(path.resolve())
    if not path.exists():
        _CACHE.pop(p, None)
        return {"size": 0, "mtime": 0, "rows": [], "keys": set()}
    st = path.stat()
    c = _CACHE.get(p)
    if c and c["size"] == st.st_size and c["mtime"] == st.st_mtime_ns:
        return c
    start = c["size"] if c and st.st_size > c["size"] else 0       # grown: read the tail; else (rewritten) all
    rows, keys = (list(c["rows"]), set(c["keys"])) if start else ([], set())
    with path.open("rb") as fh:
        fh.seek(start)
        for line in fh.read().decode("utf-8").splitlines():
            if line.strip():
                r = json.loads(line)
                rows.append(r)
                keys.add(_key(r))
    c = _CACHE[p] = {"size": st.st_size, "mtime": st.st_mtime_ns, "rows": rows, "keys": keys}
    return c


def _read_jsonl(path: Path) -> list[dict]:
    return list(_load(path)["rows"])


def _append(path: Path, rows: list[dict]) -> int:
    path.parent.mkdir(parents=True, exist_ok=True)
    seen = set(_load(path)["keys"])
    new = []
    for r in rows:
        k = _key(r)
        if k not in seen:
            seen.add(k)
            new.append(r)
    with path.open("a", encoding="utf-8") as fh:
        for r in new:
            fh.write(json.dumps(r, ensure_ascii=False, default=_json_default) + "\n")
    return len(new)


def _json_default(v):
    if isinstance(v, (np.integer,)):
        return int(v)
    if isinstance(v, (np.floating,)):
        return round(float(v), 6)
    if isinstance(v, (date, datetime)):
        return v.isoformat()
    return str(v)


def append_log(root: Path, rows: list[dict]) -> int:
    """Adds campaign rows (missing fields become null); returns how many were new."""
    stamp = datetime.now().isoformat(timespec="seconds")
    return _append(Path(root) / LOG_FILE, [{**{f: None for f in LOG_FIELDS}, "logged_at": stamp, **r} for r in rows])


def append_rewards(root: Path, rows: list[dict]) -> int:
    return _append(Path(root) / REWARDS_FILE, rows)


def read_log(root: Path, start: date | None = None, end: date | None = None, arm: str | None = None) -> pd.DataFrame:
    df = pd.DataFrame(_read_jsonl(Path(root) / LOG_FILE), columns=LOG_FIELDS)
    return _filter(df, start, end, arm)


def read_rewards(root: Path, start: date | None = None, end: date | None = None, arm: str | None = None) -> pd.DataFrame:
    df = pd.DataFrame(_read_jsonl(Path(root) / REWARDS_FILE))
    return _filter(df, start, end, arm) if len(df) else df


def _filter(df: pd.DataFrame, start, end, arm) -> pd.DataFrame:
    if not len(df):
        return df
    d = pd.to_datetime(df["run_date"]).dt.date
    keep = pd.Series(True, index=df.index)
    if start:
        keep &= d >= start
    if end:
        keep &= d <= end
    if arm:
        keep &= df["arm"] == arm
    return df[keep].reset_index(drop=True)


def justification_id(*parts) -> str:
    return hashlib.sha1("|".join(str(p) for p in parts).encode()).hexdigest()[:12]


# ---------------------------------------------------------------------------
# Rows from each arm
# ---------------------------------------------------------------------------
def _pmf(row: pd.Series) -> dict:
    from nbx.core.vw_format import action_token
    return {a: round(float(row.get(f"P_{action_token(a)}", np.nan)), 6) for a in config.ACTIONS
            if pd.notna(row.get(f"P_{action_token(a)}", np.nan))}


def _anchors(anchors: pd.DataFrame | None, val: str) -> dict | None:
    if anchors is None or val not in anchors.index:
        return None
    return {"monthly": round(float(anchors.at[val, "AVG_REV_MONTHLY"]), 6), "weekly": round(float(anchors.at[val, "AVG_REV_WEEKLY"]), 6)}


def rows_from_recos(recos: pd.DataFrame, scored: pd.DataFrame, arm: str, run_date: date,
                    anchors: pd.DataFrame | None = None) -> list[dict]:
    """cstr: the bandit's sampled action and its PMF probability; free: the ranker picks the offers (BR-60),
    the scored action is logged with its probability so the row stays usable for off-policy learning."""
    flag = {"free": "free", "cstr": "with_constraint"}[arm]
    pop = scored[scored["TG_FLAG"] == flag].copy()
    pop["ACCESS_METHOD_VAL"] = pop["ACCESS_METHOD_VAL"].astype(str)
    by_val = {v: g for v, g in recos.groupby("ACCESS_METHOD_VAL")} if len(recos) else {}
    rows = []
    for r in pop.itertuples(index=False):
        s = pd.Series(r._asdict())
        val = s["ACCESS_METHOD_VAL"]
        g = by_val.get(val)
        offers = [] if g is None else [{"code": o.BTL_NAME, "price": float(o.PRICE), "validity": int(o.VALIDITY)}
                                       for o in g.itertuples(index=False)]
        rows.append({"run_date": run_date.isoformat(), "arm": arm, "msisdn": val, "action": s["ACTION"], "offers": offers,
                     "policy_source": "bandit" if arm == "cstr" else "ranker", "propensity": round(float(s["PROBA"]), 6),
                     "bandit_pmf": _pmf(s), "model_version": s.get("MODEL_VERSION"), "llm_model": None,
                     "justification_id": None, "anchors": _anchors(anchors, val)})
    return rows


def rows_from_decisions(decisions: list[dict], anchors: pd.DataFrame | None = None) -> list[dict]:
    """llm arm: one row per agent decision (recommender_agent.decide)."""
    rows = []
    for d in decisions:
        just = d.get("justification") or {}
        rows.append({"run_date": d["run_date"], "arm": "llm", "msisdn": str(d["msisdn"]), "action": d["action"],
                     "offers": [{"code": o["offer"], "price": float(o["price"]), "validity": int(o["validity"])} for o in d.get("offers", [])],
                     "policy_source": d["policy_source"], "propensity": d["propensity"], "bandit_pmf": d.get("bandit_pmf"),
                     "model_version": d.get("model_version"), "llm_model": d.get("model"),
                     "justification_id": justification_id(d["msisdn"], d["run_date"], json.dumps(just, sort_keys=True)),
                     "anchors": _anchors(anchors, str(d["msisdn"]))})
    return rows


def rows_control(sample_vals: list[str], run_date: date) -> list[dict]:
    """cg: logged as not contacted, so the report can compare every arm with the control."""
    return [{"run_date": run_date.isoformat(), "arm": "cg", "msisdn": str(v), "action": config.NO_CONTACT, "offers": [],
             "policy_source": "control", "propensity": 1.0} for v in sample_vals]


# ---------------------------------------------------------------------------
# Rewards (feedback at D+7) and last-campaign features
# ---------------------------------------------------------------------------
def compute_rewards(log: pd.DataFrame, fulfilments: pd.DataFrame, catalog: pd.DataFrame) -> list[dict]:
    """Rewards of logged rows from the fulfilments of the reward window that follows them (BR-23, BR-40).
    fulfilments: ACCESS_METHOD_VAL, PRODUCTNAME, DATE, SERVICE (labels.load_fulfilments)."""
    price = dict(zip(catalog["BTL_NAME"], catalog["PRICE"]))
    service = dict(zip(catalog["BTL_NAME"], catalog["SERVICE"]))
    by_val = {v: g for v, g in fulfilments.groupby("ACCESS_METHOD_VAL")} if len(fulfilments) else {}
    rows = []
    for r in log.itertuples(index=False):
        val, sent = str(r.msisdn), [o["code"] for o in (r.offers or [])]
        f = by_val.get(val)
        bought = [] if f is None else list(f["PRODUCTNAME"])
        sent_services = {service.get(c) for c in sent}
        rows.append({"run_date": r.run_date, "arm": r.arm, "msisdn": val, "action": r.action,
                     "propensity": r.propensity, "policy_source": r.policy_source,
                     "reward_any_btl": int(bool(bought)),
                     "reward_same_service": int(any(service.get(b) in sent_services for b in bought)),
                     "reward_exact_offer": int(any(b in sent for b in bought)),
                     "revenue": round(float(sum(price.get(b, 0.0) for b in bought)), 3),
                     "last_btl_date": None if f is None else max(f["DATE"]).isoformat()})
    return rows


def last_campaign_features(root: Path | None, vals: pd.Series, run_date: date) -> pd.DataFrame:
    """M3, for every line: PREV_ACTION (last logged action, 'NONE' if never contacted), PREV_OFFER_TAKEN
    (exact offer taken after that contact), DAYS_SINCE_LAST_BTL (capped at the 270-day history),
    NB_CONTACTS_LAST_4W (contacts logged in D-28..D-1, control rows excluded)."""
    cap = -config.INTERACTION_WINDOW[0]
    out = pd.DataFrame({"ACCESS_METHOD_VAL": vals.astype(str).to_numpy(), "PREV_ACTION": config.PREV_ACTION_NONE,
                        "PREV_OFFER_TAKEN": 0, "DAYS_SINCE_LAST_BTL": float(cap), "NB_CONTACTS_LAST_4W": 0.0})
    if root is None:
        return out
    log = read_log(root, end=run_date - timedelta(days=1))
    if not len(log):
        return out
    contacts = log[(log["arm"] != "cg") & (log["action"] != config.NO_CONTACT)].copy()
    contacts["d"] = pd.to_datetime(contacts["run_date"]).dt.date
    last = contacts.sort_values("d").drop_duplicates("msisdn", keep="last").set_index("msisdn")
    window = contacts[contacts["d"] >= run_date - timedelta(days=config.CONTACT_CAP_DAYS)]
    n4w = window.groupby("msisdn").size()
    rew = read_rewards(root, end=run_date - timedelta(days=1))
    taken, last_btl = {}, {}
    if len(rew):
        rew = rew.assign(d=pd.to_datetime(rew["run_date"]).dt.date).sort_values("d")
        taken = rew[rew["arm"] != "cg"].drop_duplicates("msisdn", keep="last").set_index("msisdn")["reward_exact_offer"].to_dict()
        lb = rew.dropna(subset=["last_btl_date"])
        last_btl = lb.drop_duplicates("msisdn", keep="last").set_index("msisdn")["last_btl_date"].to_dict()
    v = out["ACCESS_METHOD_VAL"]
    out["PREV_ACTION"] = v.map(last["action"]).fillna(config.PREV_ACTION_NONE).str.replace(" ", "_")
    out["PREV_OFFER_TAKEN"] = v.map(taken).fillna(0).astype(int)
    days = v.map(last_btl).map(lambda s: (run_date - date.fromisoformat(s)).days if isinstance(s, str) else cap)
    out["DAYS_SINCE_LAST_BTL"] = days.clip(upper=cap).astype(float)
    out["NB_CONTACTS_LAST_4W"] = v.map(n4w).fillna(0).astype(float)
    return out


def previous_anchors(root: Path | None, run_date: date) -> dict[str, dict]:
    """Latest anchors logged before run_date, per line (for the M4 growth guard)."""
    if root is None:
        return {}
    log = read_log(root, end=run_date - timedelta(days=1)).dropna(subset=["anchors"])
    if not len(log):
        return {}
    log = log.assign(d=pd.to_datetime(log["run_date"]).dt.date).sort_values("d").drop_duplicates("msisdn", keep="last")
    return dict(zip(log["msisdn"].astype(str), log["anchors"]))


In [ ]:
%%engine nbx.fsdr.contract package
"""V2.0 input contract (plan §4, §13-§15).

    schemas.py (nbx/fsdr/schemas.py)   the 47 tables: fields, raw types, keys, input folder
    datasource.py                      reads a run folder: 5 input folders or flat, JSON / JSONL / CSV / Parquet
    validation.py                      per-table, cross-table, layout and customer_profile checks
    contract/completeness.py           which tables each pipeline needs this week (master prompt §17.9)
    contract/cache.py                  Parquet cache, hash-partitioned on the line id (plan §15)
    contract/data_agent.py             ① data agent: step T0_load = validate + completeness + cache, or block
"""


In [ ]:
%%engine nbx.fsdr.steps
"""Step folders (plan §14): every pipeline step saves its output before the next one starts.

    data/runs/<D>/train/T0_load/  ...  data/runs/<D>/score/P10_report/  ...  data/runs/<D>/final/

    with steps.Step(runs_dir(D) / "train", "T0_load", inputs=fingerprint(files)) as st:
        if st.done:                       # same inputs, finished before: skipped (resume)
            return st.read_json("summary.json")
        st.write_table(df, "base_features")         # Parquet + CSV (full or preview)
        st.write_json("summary.json", {...})
        st.rows(rows_in=..., rows_out=...)

Each step folder holds its files and _step.json: status (running / done / failed), start, duration, rows in / out,
the fingerprint of its inputs and the list of its outputs. A step is skipped when its _step.json says done with
the same inputs; force=True re-runs it (fsdr-* --from <step>).
"""
from __future__ import annotations

import hashlib
import json
import time
from datetime import date, datetime
from pathlib import Path

import pandas as pd

from nbx.fsdr import config

STATUSES = ("running", "done", "failed", "blocked")


def runs_dir(run_date: date | str, root: Path | None = None) -> Path:
    d = run_date.isoformat() if isinstance(run_date, date) else str(run_date)
    return Path(root or config.FSDR_RUNS_DIR) / d


def fingerprint(paths: list[Path] | dict, extra: dict | None = None) -> str:
    """Cheap, stable id of a step's inputs: name, size and modification time of each file (hashing 12 GB of JSON
    every Thursday would cost minutes), plus any parameters. A dict is hashed as it is."""
    if isinstance(paths, dict):
        payload = paths
    else:
        payload = {str(p): [p.stat().st_size, int(p.stat().st_mtime)] for p in sorted(map(Path, paths)) if p.exists()}
    blob = json.dumps({"inputs": payload, "extra": extra or {}}, sort_keys=True, default=str)
    return hashlib.sha256(blob.encode()).hexdigest()[:16]


def sha256_file(path: Path, chunk: int = 1 << 20) -> str:
    h = hashlib.sha256()
    with Path(path).open("rb") as fh:
        for block in iter(lambda: fh.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


class Step:
    """One step folder. Context manager: status running -> done, or failed (with the error) on an exception."""

    def __init__(self, parent: Path, name: str, inputs: str = "", force: bool = False):
        self.dir = Path(parent) / name
        self.name = name
        self.inputs = inputs
        self.force = force
        self.outputs: list[str] = []
        self._rows: dict = {}
        self._extra: dict = {}
        self._t0 = 0.0
        prev = read_step(self.dir)
        self.done = (not force and prev.get("status") == "done" and prev.get("inputs") == inputs
                     and all((self.dir / f).exists() for f in prev.get("outputs", [])))
        self.previous = prev

    # -- lifecycle -------------------------------------------------------------
    def __enter__(self) -> "Step":
        self._t0 = time.time()
        if not self.done:
            self.dir.mkdir(parents=True, exist_ok=True)
            self._write_meta("running")
        return self

    def __exit__(self, exc_type, exc, tb) -> bool:
        if self.done:
            return False
        if exc is not None:
            self._write_meta("failed", error=f"{exc_type.__name__}: {exc}")
        else:
            self._write_meta(self._extra.pop("_status", "done"))
        return False

    def block(self, reason: str) -> None:
        """Ends the step as blocked (the run stops here; the outputs explain why)."""
        self._extra["_status"] = "blocked"
        self._extra["reason"] = reason

    def rows(self, **counts) -> None:
        self._rows.update({k: int(v) for k, v in counts.items()})

    def note(self, **info) -> None:
        self._extra.update(info)

    def _write_meta(self, status: str, error: str | None = None) -> None:
        meta = {"step": self.name, "status": status, "inputs": self.inputs,
                "started": datetime.fromtimestamp(self._t0).isoformat(timespec="seconds"),
                "seconds": round(time.time() - self._t0, 2), "rows": self._rows, "outputs": self.outputs,
                **{k: v for k, v in self._extra.items() if not k.startswith("_")}}
        if error:
            meta["error"] = error
        tmp = self.dir / (config.STEP_FILE + ".tmp")
        tmp.write_text(json.dumps(meta, indent=2, default=str), encoding="utf-8")
        tmp.replace(self.dir / config.STEP_FILE)

    # -- outputs ---------------------------------------------------------------
    def _add(self, name: str) -> Path:
        if name not in self.outputs:
            self.outputs.append(name)
        return self.dir / name

    def output(self, name: str) -> Path:
        """Registers an output written by other code (e.g. one partition file "base_features/part-03.parquet")."""
        p = self._add(name)
        p.parent.mkdir(parents=True, exist_ok=True)
        return p

    def write_json(self, name: str, obj) -> Path:
        p = self._add(name)
        p.write_text(json.dumps(obj, indent=2, ensure_ascii=False, default=str), encoding="utf-8")
        return p

    def write_text(self, name: str, text: str) -> Path:
        p = self._add(name)
        p.write_text(text, encoding="utf-8")
        return p

    def write_csv(self, df: pd.DataFrame, name: str) -> Path:
        p = self._add(name)
        df.to_csv(p, index=False, encoding="utf-8")
        return p

    def write_table(self, df: pd.DataFrame, stem: str, csv: str = "auto") -> list[Path]:
        """<stem>.parquet always, plus <stem>.csv in full (small tables, e.g. the 50k of scoring) or a
        preview.csv of the first rows (1M-row training tables). csv: "auto" | "full" | "preview" | "none"."""
        out = [self._add(f"{stem}.parquet")]
        _parquet_safe(df).to_parquet(out[0], index=False)
        mode = csv if csv != "auto" else ("full" if len(df) <= config.STEP_FULL_CSV_MAX_ROWS else "preview")
        if mode == "full":
            out.append(self._add(f"{stem}.csv"))
            df.to_csv(out[-1], index=False, encoding="utf-8")
        elif mode == "preview":
            out.append(self._add(f"{stem}_preview.csv"))
            df.head(config.STEP_PREVIEW_ROWS).to_csv(out[-1], index=False, encoding="utf-8")
        return out

    def read_json(self, name: str):
        return json.loads((self.dir / name).read_text(encoding="utf-8"))


def _parquet_safe(df: pd.DataFrame) -> pd.DataFrame:
    """Object columns mixing types (raw input values) cannot go to Parquet as they are: written as text."""
    out = df.copy()
    for c in out.columns:
        if out[c].dtype == object:
            kinds = {type(v) for v in out[c].dropna().head(10_000)}
            if len(kinds) > 1:
                out[c] = out[c].map(lambda v: None if v is None else str(v)).astype("string")
    return out


def read_step(step_dir: Path) -> dict:
    p = Path(step_dir) / config.STEP_FILE
    try:
        return json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}
    except (OSError, json.JSONDecodeError):
        return {}


def list_steps(run_dir: Path) -> list[dict]:
    """Every step of a run (train / score / final), in order, with its status and files: for the UI and the API."""
    out = []
    for phase in ("train", "score", "final"):
        d = Path(run_dir) / phase
        if not d.is_dir():
            continue
        for sd in sorted((p for p in d.iterdir() if p.is_dir()), key=lambda p: _step_order(p.name)):
            meta = read_step(sd)
            out.append({"phase": phase, "step": sd.name, "status": meta.get("status", "unknown"),
                        "seconds": meta.get("seconds"), "rows": meta.get("rows", {}),
                        "files": sorted(f.name for f in sd.iterdir() if f.is_file() and f.name != config.STEP_FILE)})
    return out


def _step_order(name: str) -> tuple:
    """T0 < T1 < ... < T8; P0 < P1 < ... < P6 < P6b < P7 < P7b < ... < P10."""
    head = name.split("_", 1)[0]
    digits = "".join(ch for ch in head[1:] if ch.isdigit())
    return (head[:1], int(digits) if digits else 0, head)


In [ ]:
%%engine nbx.fsdr.contract.cache
"""Parquet cache of a run (plan §4, §15): each input table converted once, typed, split into partitions.

    data/cache/<D>/<table>/part-00.parquet ... part-15.parquet    tables with a line (MSISDN or ACCESS_METHOD_ID)
    data/cache/<D>/<table>/part-all.parquet                        reference tables (catalogue, party, accounts, ...)
    data/cache/<D>/_cache_manifest.json                            rows and SHA-256 of every file

Every per-line table is partitioned on the SAME key: the line's ACCESS_METHOD_ID (taken from base_prepaid when the
table only has the MSISDN), canonicalised, then hashed. So all rows of one line, whatever the table, land in the
same partition number: features and segments run partition by partition (8 in parallel on the P16) and every join
stays inside a partition. A line outside the base is hashed on its own key (it joins nothing anyway).

    cache = build_cache(src, date(2026, 10, 21))
    CacheSource(date(2026, 10, 21)).read("activities_w1_score", part=3)
"""
from __future__ import annotations

import json
import time
from datetime import date, datetime
from pathlib import Path

import numpy as np
import pandas as pd

from nbx.fsdr import config, schemas
from nbx.fsdr.datasource import DataSource, compact_frame
from nbx.fsdr.steps import _parquet_safe, sha256_file

CACHE_MANIFEST = "_cache_manifest.json"
LINE_ID_COLS = ("ACCESS_METHOD_ID", "access_method_id", "Access_Method_id")
LINE_VAL_COLS = ("ACCESS_METHOD_VAL", "access_method_val", "MSISDN", "msisdn")
ALL = "all"


def cache_dir(run_date: date | str, root: Path | None = None) -> Path:
    d = run_date.isoformat() if isinstance(run_date, date) else str(run_date)
    return Path(root or config.FSDR_CACHE_DIR) / d


def _canon(s: pd.Series) -> pd.Series:
    from nbx.fsdr.validation import norm_keys

    return norm_keys(s)


def line_map(src: DataSource) -> pd.Series:
    """Canonical MSISDN -> canonical ACCESS_METHOD_ID, from base_prepaid (first row wins on a duplicate MSISDN)."""
    if not src.has("base_prepaid"):
        return pd.Series(dtype=object)
    base = src.read("base_prepaid")
    if not {"ACCESS_METHOD_VAL", "ACCESS_METHOD_ID"} <= set(base.columns):
        return pd.Series(dtype=object)
    m = pd.Series(_canon(base["ACCESS_METHOD_ID"]).to_numpy(), index=_canon(base["ACCESS_METHOD_VAL"]).to_numpy())
    m = m[m.index.notna() & m.notna()]
    return m[~m.index.duplicated(keep="first")]


def line_key(df: pd.DataFrame, lines: pd.Series) -> pd.Series | None:
    """The partition key of every row: canonical ACCESS_METHOD_ID, or None for a table without a line column."""
    for c in LINE_ID_COLS:
        if c in df.columns:
            return _canon(df[c])
    for c in LINE_VAL_COLS:
        if c in df.columns:
            val = _canon(df[c])
            return val.map(lines).fillna("v:" + val.fillna(""))        # MSISDN outside the base: its own hash
    return None


def partition_of(keys: pd.Series, parts: int) -> np.ndarray:
    """Stable partition number of canonical keys (pandas' fixed-key hash: same key -> same partition, every run)."""
    h = pd.util.hash_pandas_object(keys.fillna("").astype(str), index=False).to_numpy()
    return (h % np.uint64(parts)).astype(np.int16)


def build_cache(src: DataSource, run_date: date, root: Path | None = None, parts: int | None = None,
                tables: list[str] | None = None) -> dict:
    """Writes every table of the source (or `tables`) to the cache; returns and writes the manifest."""
    parts = int(parts or config.CACHE_PARTITIONS)
    out = cache_dir(run_date, root)
    out.mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    lines = line_map(src)
    man = {"run_date": run_date.isoformat(), "partitions": parts, "key": "ACCESS_METHOD_ID (via base_prepaid for MSISDN)",
           "built_at": datetime.now().isoformat(timespec="seconds"), "tables": {}}
    for t in tables or [t for t in src.tables() if t in schemas.TABLES]:
        df = compact_frame(src.read(t))
        tdir = out / t
        tdir.mkdir(exist_ok=True)
        for old in tdir.glob("part-*.parquet"):
            old.unlink()
        key = line_key(df, lines)
        files = {}
        if key is None:
            p = tdir / f"part-{ALL}.parquet"
            _parquet_safe(df).to_parquet(p, index=False)
            files[ALL] = {"rows": len(df), "sha256": sha256_file(p)}
        else:
            part = partition_of(key, parts)
            safe = _parquet_safe(df)
            for k in range(parts):
                p = tdir / f"part-{k:02d}.parquet"
                sub = safe[part == k]
                sub.to_parquet(p, index=False)
                files[f"{k:02d}"] = {"rows": len(sub), "sha256": sha256_file(p)}
        man["tables"][t] = {"rows": len(df), "partitioned": key is not None, "folder": schemas.TABLES[t].folder,
                            "files": files}
    man["seconds"] = round(time.time() - t0, 2)
    man["bytes"] = sum(p.stat().st_size for p in out.rglob("*.parquet"))
    (out / CACHE_MANIFEST).write_text(json.dumps(man, indent=2), encoding="utf-8")
    return man


def verify_cache(run_date: date, root: Path | None = None) -> list[str]:
    """Files whose SHA-256 no longer matches the manifest (empty list = intact)."""
    d = cache_dir(run_date, root)
    man = json.loads((d / CACHE_MANIFEST).read_text(encoding="utf-8"))
    bad = []
    for t, info in man["tables"].items():
        for k, f in info["files"].items():
            p = d / t / f"part-{k}.parquet"
            if not p.exists() or sha256_file(p) != f["sha256"]:
                bad.append(str(p.relative_to(d)))
    return bad


class CacheSource:
    """Reads the cache: a whole table, or one partition of a per-line table (reference tables have no partitions:
    any part returns the whole table)."""

    def __init__(self, run_date: date | str, root: Path | None = None):
        self.dir = cache_dir(run_date, root)
        p = self.dir / CACHE_MANIFEST
        if not p.exists():
            raise FileNotFoundError(f"No cache for {run_date}: run fsdr-load first ({p})")
        self._man = json.loads(p.read_text(encoding="utf-8"))
        self.partitions = int(self._man["partitions"])

    def tables(self) -> list[str]:
        return list(self._man["tables"])

    def has(self, table: str) -> bool:
        return table in self._man["tables"]

    def partitioned(self, table: str) -> bool:
        return bool(self._man["tables"][table]["partitioned"])

    def read(self, table: str, part: int | None = None) -> pd.DataFrame:
        if not self.has(table):
            raise FileNotFoundError(f"{table} is not in the cache {self.dir}")
        tdir = self.dir / table
        if not self.partitioned(table):
            return pd.read_parquet(tdir / f"part-{ALL}.parquet")
        if part is not None:
            return pd.read_parquet(tdir / f"part-{int(part):02d}.parquet")
        return pd.concat([pd.read_parquet(tdir / f"part-{k:02d}.parquet") for k in range(self.partitions)],
                         ignore_index=True)

    def manifest(self) -> dict:
        return dict(self._man)


In [ ]:
%%engine nbx.fsdr.contract.completeness
"""Weekly completeness (master prompt §5, §17.9): every table the pipeline about to run needs is present, has
rows and belongs to this week's run.

    rep = completeness.check(src, "train_week0", date(2026, 10, 21))
    rep["complete"]        False -> the run is blocked (a required table is missing, empty or stale)

Pipelines: train_week0 (_auto + _score + history), train_weekly (_score + purchases), score (the tables scoring reads
directly; the rest comes from the validated bundle). The lists live in config.PIPELINE_TABLES.
"""
from __future__ import annotations

from datetime import date

from nbx.fsdr import config, schemas
from nbx.fsdr.datasource import DataSource

PIPELINES = tuple(config.PIPELINE_TABLES)


def detect_pipeline(src: DataSource) -> str:
    """Week 0 when the history tables of the segment engine are there (_auto + targeting_dataiku_auto)."""
    week0 = ["activities_w1_auto", "subs_amt_auto", "targeting_dataiku_auto"]
    return "train_week0" if all(src.has(t) for t in week0) else "train_weekly"


def check(src: DataSource, pipeline: str, run_date: date | None = None) -> dict:
    if pipeline not in config.PIPELINE_TABLES:
        raise ValueError(f"pipeline must be one of {PIPELINES}")
    spec = config.PIPELINE_TABLES[pipeline]
    man = src.manifest()
    man_date = man.get("run_date")
    fresh = run_date is None or man_date is None or man_date == run_date.isoformat()
    rows = []
    for need in ("required", "optional"):
        for t in spec[need]:
            present = src.has(t)
            n = len(src.read(t)) if present else 0
            status = "ok" if present and n else "empty" if present else "missing"
            level = "ok" if status == "ok" else "blocking" if need == "required" else "warning"
            rows.append({"table": t, "folder": schemas.folder_of(t, schemas.TABLES[t].family if t in schemas.TABLES else None),
                         "need": need, "present": present, "rows": n, "status": status, "level": level})
    blocking = [r["table"] for r in rows if r["level"] == "blocking"]
    reasons = [f"required table {r['table']} is {r['status']}" for r in rows if r["level"] == "blocking"]
    if not fresh:
        reasons.append(f"stale data: the manifest is for {man_date}, the run is for {run_date}")
    return {"pipeline": pipeline, "run_date": run_date.isoformat() if run_date else None, "manifest_run_date": man_date,
            "fresh": fresh, "complete": not blocking and fresh, "blocking": blocking,
            "warnings": [r["table"] for r in rows if r["level"] == "warning"], "reasons": reasons, "tables": rows}


In [ ]:
%%engine nbx.fsdr.validation
"""Validation of a run folder (or any DataSource) against the data contract.

Checks, each graded blocking / warning / info:
  - files: every table present (missing model-used table = blocking)
  - fields: missing fields (blocking for model-used tables), unknown extra fields (warning)
  - raw types: wrong type in a key field = blocking, elsewhere = warning
  - keys: null keys and duplicates on the primary key (blocking above 1 % unless
    duplicates are expected, e.g. stacked model scores resolved by max)
  - cross-table: child keys missing from the parent table (BR joins)
  - coverage: share of the base scored by each model (< 70 % = warning)
  - consistency: activities M1 = W1+..+W4, Sub_Amt_m1 >= its weeks, dates inside
    their BR windows, manifest anchor date = calendar anchor

The report is JSON-serialisable and also rendered as Markdown; both are
written next to the data as _validation.json / _validation.md.
"""
from __future__ import annotations

import json
from dataclasses import asdict, dataclass, field
from datetime import date, datetime
from pathlib import Path

import numpy as np
import pandas as pd

from nbx.fsdr import calendar as cal
from nbx.fsdr import config, schemas
from nbx.fsdr.datasource import DataSource, JsonSource

LEVELS = ("blocking", "warning", "info")


@dataclass
class Issue:
    level: str
    check: str
    table: str
    message: str
    count: int = 0
    column: str | None = None
    examples: list = field(default_factory=list)


@dataclass
class TableReport:
    table: str
    present: bool
    rows: int = 0
    missing_fields: list[str] = field(default_factory=list)
    unknown_fields: list[str] = field(default_factory=list)
    type_errors: dict[str, int] = field(default_factory=dict)
    null_keys: dict[str, int] = field(default_factory=dict)
    duplicate_rows: int = 0


@dataclass
class ValidationReport:
    run_date: str | None
    anchor_date: str | None
    source: str
    tables: dict[str, TableReport] = field(default_factory=dict)
    issues: list[Issue] = field(default_factory=list)
    coverage: dict[str, float] = field(default_factory=dict)
    relations: list[dict] = field(default_factory=list)
    layout: str | None = None                       # "folders" / "flat" / "mixed" (plan §13)
    profile: dict = field(default_factory=dict)     # customer_profile coverage and disagreement counts
    checked_at: str = field(default_factory=lambda: datetime.now().isoformat(timespec="seconds"))

    def add(self, level: str, check: str, table: str, message: str, count: int = 0, column: str | None = None,
            examples: list | None = None) -> None:
        self.issues.append(Issue(level, check, table, message, int(count), column, list(examples or [])))

    @property
    def status(self) -> str:
        if any(i.level == "blocking" for i in self.issues):
            return "blocked"
        return "warnings" if any(i.level == "warning" for i in self.issues) else "ok"

    @property
    def usable(self) -> bool:
        return self.status != "blocked"

    def counts(self) -> dict[str, int]:
        return {lv: sum(1 for i in self.issues if i.level == lv) for lv in LEVELS}

    def to_dict(self) -> dict:
        return {"status": self.status, "usable": self.usable, "run_date": self.run_date, "anchor_date": self.anchor_date,
                "source": self.source, "checked_at": self.checked_at, "issue_counts": self.counts(),
                "issues": [asdict(i) for i in self.issues], "coverage": self.coverage, "relations": self.relations,
                "layout": self.layout, "profile": self.profile,
                "tables": {k: asdict(v) for k, v in self.tables.items()}}

    def to_markdown(self) -> str:
        c = self.counts()
        icon = {"ok": "✅", "warnings": "⚠️", "blocked": "⛔"}[self.status]
        lines = [f"# FSDR validation report — {self.run_date or '?'}", "",
                 f"**Status:** {icon} {self.status}  ·  blocking {c['blocking']} · warnings {c['warning']} · info {c['info']}",
                 f"**Anchor date (T):** {self.anchor_date or '?'}  ·  **Source:** {self.source}  ·  checked {self.checked_at}", ""]
        for lv in LEVELS:
            items = [i for i in self.issues if i.level == lv]
            if not items:
                continue
            lines += [f"## {lv.capitalize()} ({len(items)})", "", "| Check | Table | Column | Count | Message |", "|---|---|---|---|---|"]
            for i in items:
                ex = f" e.g. {', '.join(map(str, i.examples[:3]))}" if i.examples else ""
                lines.append(f"| {i.check} | `{i.table}` | {i.column or ''} | {i.count} | {i.message}{ex} |")
            lines.append("")
        if self.coverage:
            lines += ["## Model-score coverage of the base", "", "| Model file | Coverage |", "|---|---|"]
            lines += [f"| `{k}` | {v:.1%} |" for k, v in self.coverage.items()]
            lines.append("")
        if self.profile:
            lines += [f"## {config.PROFILE_TABLE}", "", "| Check | Value |", "|---|---|"]
            lines += [f"| {k} | {v:.1%} |" if k == "coverage" else f"| {k} | {v} |" for k, v in self.profile.items()]
            lines.append("")
        lines += ["## Tables", "", "| Table | Present | Rows | Missing fields | Unknown fields | Type errors | Null keys | Duplicates |",
                  "|---|---|---|---|---|---|---|---|"]
        for t in self.tables.values():
            lines.append(f"| `{t.table}` | {'yes' if t.present else '**no**'} | {t.rows} | {len(t.missing_fields)} | "
                         f"{len(t.unknown_fields)} | {sum(t.type_errors.values())} | {sum(t.null_keys.values())} | {t.duplicate_rows} |")
        return "\n".join(lines) + "\n"


# ---------------------------------------------------------------------------
# Helpers
# ---------------------------------------------------------------------------
def norm_keys(s: pd.Series) -> pd.Series:
    """_norm_key on a whole column, vectorised (review #15)."""
    from nbx.fsdr.features import canonical_keys

    return canonical_keys(s, strip_zero=False, empty_none=False)


def _key_set(s: pd.Series, drop_empty: bool = False) -> set:
    k = norm_keys(s).dropna()
    return set(k[k != ""]) if drop_empty else set(k)


def _norm_key(v) -> str | None:
    """Key value -> canonical string ('91234567', 1001, 1001.0 and '1001' compare equal)."""
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return None
    if isinstance(v, (int, np.integer)):
        s = str(int(v))
    elif isinstance(v, (float, np.floating)):
        s = str(int(v)) if float(v).is_integer() else str(v)
    else:
        s = str(v).strip()
    return s


def _share(count: int, rows: int) -> float:
    return count / rows if rows else 0.0


def _over(count: int, rows: int) -> bool:
    return _share(count, rows) > config.BLOCKING_KEY_ISSUE_SHARE


def _num(df: pd.DataFrame, col: str) -> pd.Series:
    return pd.to_numeric(df[col], errors="coerce") if col in df.columns else pd.Series(dtype=float)


# ---------------------------------------------------------------------------
# Checks
# ---------------------------------------------------------------------------
def _check_table(src: DataSource, schema: schemas.TableSchema, rep: ValidationReport,
                 pipeline: str | None = None) -> pd.DataFrame | None:
    name = schema.name
    if not src.has(name):
        level = "blocking" if schema.used_by_model and schema.required and schema.family != "model_score" else "warning"
        # V4.0: a weekly run does not need the week-0 history (*_auto): missing, it is a warning, not a block
        not_needed = (pipeline in config.PIPELINE_TABLES and level == "blocking"
                      and name not in config.PIPELINE_TABLES[pipeline]["required"])
        if not_needed:
            level = "warning"
        why = ("missing model file: its score falls back to random 0-0.02 (BR-47)" if schema.family == "model_score"
               else f"not needed by the {pipeline} pipeline (week-0 history)" if not_needed
               else "required table missing" if level == "blocking"
               else "optional table missing (fallback used)" if schema.used_by_model else "table missing (not used by the model)")
        rep.add(level, "file_present", name, why)
        rep.tables[name] = TableReport(name, present=False)
        return None
    df = src.read(name)
    tr = TableReport(name, present=True, rows=len(df))
    rep.tables[name] = tr
    if not len(df):
        rep.add("warning" if not schema.used_by_model else "blocking", "rows", name, "file has no rows")
        return df

    tr.missing_fields = [f for f in schema.field_names if f not in df.columns]
    tr.unknown_fields = [c for c in df.columns if c not in schema.field_names]
    if tr.missing_fields:
        rep.add("blocking" if schema.used_by_model else "warning", "fields_missing", name,
                f"{len(tr.missing_fields)} field(s) missing", len(tr.missing_fields), examples=tr.missing_fields)
    if tr.unknown_fields:
        rep.add("warning", "fields_unknown", name, "extra field(s) not in the contract (ignored)",
                len(tr.unknown_fields), examples=tr.unknown_fields)

    for f in schema.fields:
        if f.name not in df.columns:
            continue
        bad, examples = schemas.type_errors(df[f.name], f.raw_type)
        if bad:
            tr.type_errors[f.name] = bad
            rep.add("blocking" if f.key else "warning", "raw_type", name,
                    f"{bad} value(s) are not {f.raw_type}", bad, f.name, examples)
        if f.key:
            nulls = int(df[f.name].isna().sum())
            if nulls:
                tr.null_keys[f.name] = nulls
                rep.add("blocking" if _over(nulls, len(df)) else "warning", "null_key", name,
                        f"{nulls} null key value(s) ({_share(nulls, len(df)):.1%})", nulls, f.name)

    pk = [c for c in schema.primary_key if c in df.columns]
    if pk:
        keyed = df[pk].apply(norm_keys)
        dups = int(keyed.duplicated().sum())
        tr.duplicate_rows = dups
        if dups:
            blocking = schema.duplicates_blocking and _over(dups, len(df))
            note = "" if schema.duplicates_blocking else " (expected: resolved downstream, e.g. max of stacked scores)"
            rep.add("blocking" if blocking else "warning", "duplicates", name,
                    f"{dups} duplicate row(s) on ({', '.join(pk)}){note}", dups, ",".join(pk))
    return df


def _check_relations(src: DataSource, rep: ValidationReport) -> None:
    for r in schemas.relations():
        if not (src.has(r.child) and src.has(r.parent)):
            continue
        child, parent = src.read(r.child), src.read(r.parent)
        if r.child_col not in child.columns or r.parent_col not in parent.columns:
            continue
        parent_keys = _key_set(parent[r.parent_col])
        ck = norm_keys(child[r.child_col]).dropna().tolist()
        if not ck:
            continue
        orphans = sum(1 for k in ck if k not in parent_keys)
        rep.relations.append({"child": f"{r.child}.{r.child_col}", "parent": f"{r.parent}.{r.parent_col}",
                              "child_keys": len(ck), "orphans": orphans, "orphan_share": round(_share(orphans, len(ck)), 4)})
        if orphans:
            level = "blocking" if r.blocking and _over(orphans, len(ck)) else "warning"
            rep.add(level, "cross_table", r.child, f"{orphans} {r.child_col} value(s) not found in {r.parent}.{r.parent_col}"
                    + (f" -- {r.note}" if r.note else ""), orphans, r.child_col)


def _check_coverage(src: DataSource, rep: ValidationReport) -> None:
    if not src.has("base_prepaid"):
        return
    base = _key_set(src.read("base_prepaid")["ACCESS_METHOD_VAL"], drop_empty=True)
    for m in config.MODEL_SCORE_REGISTRY:
        if not src.has(m.table) or "access_method_val" not in src.read(m.table).columns:
            continue
        scored = _key_set(src.read(m.table)["access_method_val"], drop_empty=True)
        cov = _share(len(scored & base), len(base))
        rep.coverage[m.table] = round(cov, 4)
        if m.tenure_range is None and cov < config.MIN_SCORE_COVERAGE:
            rep.add("warning", "coverage", m.table, f"scores only {cov:.1%} of the base (missing -> random 0-0.02, BR-47)")


def _check_consistency(src: DataSource, rep: ValidationReport, run_date: date | None) -> None:
    # activities: M1 window = W1..W4 exactly (BR 4.2)
    for anchor in ("T", "D"):
        names = {p: schemas.activity_table_name(p, anchor) for p in ("W1", "W2", "W3", "W4", "M1")}
        if not all(src.has(t) for t in names.values()):
            continue
        m1 = src.read(names["M1"])
        if "ACCESS_METHOD_VAL" not in m1.columns or "Local_Voice_min_M1" not in m1.columns:
            continue
        tot = pd.Series(0.0, index=norm_keys(m1["ACCESS_METHOD_VAL"]))
        weeks = pd.Series(dtype=float)
        for w in ("W1", "W2", "W3", "W4"):
            df = src.read(names[w])
            col = f"Local_Voice_min_{w}"
            if col in df.columns:
                s = pd.Series(_num(df, col).to_numpy(), index=norm_keys(df["ACCESS_METHOD_VAL"]))
                weeks = weeks.add(s.groupby(level=0).sum(), fill_value=0)
        m1v = pd.Series(_num(m1, "Local_Voice_min_M1").to_numpy(), index=tot.index).groupby(level=0).sum()
        diff = (m1v - weeks.reindex(m1v.index).fillna(0)).abs()
        off = int((diff > 0.05 + 0.001 * m1v.abs()).sum())
        if off:
            rep.add("warning", "window_consistency", names["M1"], f"{off} line(s) where Local_Voice_min_M1 != W1+W2+W3+W4", off)

    # subscriptions: M1 (30 days incl. anchor) contains W1..W4 (28 days)
    for t in ("subs_amt_auto", "subs_amt_score"):
        if not src.has(t):
            continue
        df = src.read(t)
        cols = [f"Sub_Amt_w{i}" for i in range(1, 5)]
        if not set(cols + ["Sub_Amt_m1"]) <= set(df.columns):
            continue
        off = int((_num(df, "Sub_Amt_m1") + 0.01 < sum(_num(df, c).fillna(0) for c in cols)).sum())
        if off:
            rep.add("warning", "window_consistency", t, f"{off} line(s) where Sub_Amt_m1 < Sub_Amt_w1..w4", off)

    if run_date is None:
        return
    win = cal.campaign_windows(run_date)
    if src.has("targeting_dataiku_auto") and "date1" in src.read("targeting_dataiku_auto").columns:
        d1 = pd.to_datetime(src.read("targeting_dataiku_auto")["date1"], errors="coerce").dt.date
        out = int(((d1 < win["targeting"].start) | (d1 > win["targeting"].end)).sum())
        if out:
            rep.add("warning", "date_window", "targeting_dataiku_auto",
                    f"{out} date1 value(s) outside {win['targeting'].start}..{win['targeting'].end} (BR-21)", out, "date1")
    if src.has("subs_btl_auto") and "FULFILMENTSUCCESSDATE_TIME" in src.read("subs_btl_auto").columns:
        ts = pd.to_datetime(src.read("subs_btl_auto")["FULFILMENTSUCCESSDATE_TIME"], errors="coerce").dt.date
        out = int(((ts < win["reward"].start) | (ts > win["reward"].end)).sum())
        if out:
            rep.add("warning", "date_window", "subs_btl_auto",
                    f"{out} fulfilment(s) outside {win['reward'].start}..{win['reward'].end} (BR-23)", out,
                    "FULFILMENTSUCCESSDATE_TIME")
    if src.has("base_prepaid") and "ACCESS_METHOD_VAL" in src.read("base_prepaid").columns:
        vals = norm_keys(src.read("base_prepaid")["ACCESS_METHOD_VAL"]).dropna()
        bad = int((~vals.str.fullmatch(r"\d{8}") | (vals <= config.MSISDN_MIN_EXCLUSIVE)).sum())
        if bad:
            rep.add("warning", "msisdn_format", "base_prepaid",
                    f"{bad} MSISDN(s) not 8 digits > '{config.MSISDN_MIN_EXCLUSIVE}' (BR-02: filtered out downstream)", bad)


def _check_layout(src: JsonSource, rep: ValidationReport) -> None:
    """Plan §13: the 5 input folders. A misplaced table still loads (warning); a table found twice blocks."""
    rep.layout = src.layout
    if src.layout != "folders":
        rep.add("warning", "layout", "-", f"{src.layout} layout: tables are expected in the 5 folders "
                f"{', '.join(config.INPUT_FOLDERS)} (plan §13); files loaded as they are")
    for t, (found, expected) in sorted(src.misplaced().items()):
        rep.add("warning", "folder", t, f"found in {found}/, expected in {expected}/ (loaded anyway)")
    for t, (found, expected) in sorted(src.misfiled().items()):
        rep.add("warning", "twin", t, f"found under {found}/, expected under {expected}/ (V4.0 layout; loaded anyway)")
    root = src.folder.parent.parent if src.twins.keys() - {""} else src.folder
    for t, paths in sorted(src.duplicate_files.items()):
        rep.add("blocking", "file_duplicate", t, f"{len(paths)} files for one table: keep only one",
                len(paths), examples=[str(Path(p).relative_to(root)) for p in paths])
    _check_formats(src, rep)


def _check_formats(src: JsonSource, rep: ValidationReport) -> None:
    """Master §2 (V3.0): production tables are .csv.gz. Another format is accepted only for tests
    (FSDR_ALLOW_TEST_FORMATS=true) and for the dummy generator's data (its manifest says source=generator)."""
    from nbx.fsdr import csvstd

    other = {t: csvstd.file_format(src.path(t)) for t in src.tables()
             if src.path(t) is not None and csvstd.file_format(src.path(t)) != config.PRODUCTION_FORMAT}
    if not other:
        return
    if src.manifest().get("source") == "generator":            # the dummy generator's data: accepted as it is
        return
    fmts = sorted(set(other.values()))
    if config.ALLOW_TEST_FORMATS:
        rep.add("info", "format", "-", f"{len(other)} table(s) in a test format ({', '.join(fmts)}): accepted "
                f"(FSDR_ALLOW_TEST_FORMATS=true); production inputs must be {config.PRODUCTION_FORMAT}", len(other))
    else:
        rep.add("blocking", "format", "-", f"{len(other)} table(s) are not {config.PRODUCTION_FORMAT} "
                f"({', '.join(fmts)}): production inputs must be the Dataiku compressed CSV export (master §2)",
                len(other), examples=sorted(other)[:5])


def _check_profile(src: DataSource, rep: ValidationReport, run_date: date | None) -> None:
    """customer_profile (master prompt §5): coverage, ages outside 15-100, negative tenure, disagreement with
    party_*.AGE (> 2 years) and with the activation-date tenure (> 30 days). Warnings with counts: the profile
    wins downstream, so a disagreement is reported, never repaired."""
    t = config.PROFILE_TABLE
    if not src.has(t) or not src.has("base_prepaid"):
        return
    prof, base = src.read(t), src.read("base_prepaid")
    if not {"msisdn", "age", "tenure"} <= set(prof.columns) or "ACCESS_METHOD_VAL" not in base.columns:
        return
    p = pd.DataFrame({"msisdn": norm_keys(prof["msisdn"]), "age": _num(prof, "age"), "tenure": _num(prof, "tenure")})
    p = p.dropna(subset=["msisdn"]).drop_duplicates("msisdn", keep="last")
    b = pd.DataFrame({"msisdn": norm_keys(base["ACCESS_METHOD_VAL"]),
                      "party": norm_keys(base["PARTY_ID"]) if "PARTY_ID" in base.columns else None,
                      "start": base["access_method_subs_start_dt"] if "access_method_subs_start_dt" in base.columns else None})
    b = b.dropna(subset=["msisdn"]).drop_duplicates("msisdn")
    m = b.merge(p, on="msisdn", how="left")
    covered = int(m["msisdn"].isin(p["msisdn"]).sum())
    cov = _share(covered, len(b))
    rep.profile = {"rows": len(prof), "base_lines": len(b), "covered": covered, "coverage": round(cov, 4)}
    if cov < config.PROFILE_MIN_COVERAGE:
        rep.add("warning", "profile_coverage", t, f"profile for {cov:.1%} of the base (< {config.PROFILE_MIN_COVERAGE:.0%}): "
                "the others use party_scoring.AGE and the activation-date tenure", len(b) - covered)
    lo, hi = config.PROFILE_AGE_RANGE
    checks = {"age_out_of_range": (((p["age"] < lo) | (p["age"] > hi)).sum(), f"age outside {lo}-{hi}"),
              "negative_tenure": ((p["tenure"] < 0).sum(), "negative tenure")}
    party_t = "party_scoring" if src.has("party_scoring") else "party_auto" if src.has("party_auto") else None
    if party_t and "party" in m and {"party_id", "AGE"} <= set(src.read(party_t).columns):
        party = src.read(party_t)
        ages = pd.Series(_num(party, "AGE").to_numpy(), index=norm_keys(party["party_id"]))
        ages = ages[~ages.index.duplicated(keep="last")]
        diff = (m["age"] - m["party"].map(ages)).abs()
        tol = config.PROFILE_AGE_TOLERANCE_YEARS
        checks["age_vs_party"] = ((diff > tol).sum(), f"age differs from {party_t}.AGE by more than {tol} years")
    if run_date is not None and m["start"].notna().any():
        start = pd.to_datetime(m["start"], errors="coerce")
        days = (pd.Timestamp(run_date) - start).dt.days
        if config.PROFILE_TENURE_UNIT == "months":
            days = days / 30
        tol = config.PROFILE_TENURE_TOLERANCE_DAYS / (30 if config.PROFILE_TENURE_UNIT == "months" else 1)
        checks["tenure_vs_activation"] = (((m["tenure"] - days).abs() > tol).sum(),
                                          f"tenure differs from run date - access_method_subs_start_dt by more than "
                                          f"{config.PROFILE_TENURE_TOLERANCE_DAYS} days")
    for name, (count, what) in checks.items():
        count = int(count)
        rep.profile[name] = count
        if count:
            rep.add("warning", f"profile_{name}", t, f"{count} line(s): {what}", count)


def validate(src: DataSource, run_date: date | None = None, pipeline: str | None = None) -> ValidationReport:
    """pipeline (train_week0 / train_weekly / score): a table that pipeline does not need may be missing (V4.0)."""
    manifest = src.manifest()
    if run_date is None and manifest.get("run_date"):
        run_date = date.fromisoformat(manifest["run_date"])
    anchor = cal.anchor_date(run_date) if run_date else None
    rep = ValidationReport(run_date.isoformat() if run_date else None, anchor.isoformat() if anchor else None,
                           manifest.get("source", "unknown"))
    if not manifest:
        rep.add("warning", "manifest", config.MANIFEST_FILE, "no _manifest.json: run date taken from the folder name or argument")
    elif manifest.get("anchor_date") and anchor and manifest["anchor_date"] != anchor.isoformat():
        rep.add("blocking", "manifest", config.MANIFEST_FILE,
                f"manifest anchor_date {manifest['anchor_date']} != calendar anchor {anchor} for run date {run_date}")
    if run_date is None:
        rep.add("warning", "run_date", "-", "run date unknown: date-window checks skipped")

    for schema in schemas.TABLES.values():
        _check_table(src, schema, rep, pipeline)
        if isinstance(src, JsonSource):      # raw values are only needed by the per-table checks above (review #15)
            src.compact_table(schema.name)
    if isinstance(src, JsonSource):
        for t in src.unknown_files():
            rep.add("info", "file_unknown", t, "file not in the data contract (ignored)")
        _check_layout(src, rep)
    _check_relations(src, rep)
    _check_coverage(src, rep)
    _check_consistency(src, rep, run_date)
    _check_profile(src, rep, run_date)
    return rep


def validate_folder(folder: str | Path, run_date: date | None = None, write: bool = True) -> ValidationReport:
    """Validates a run folder; the run date comes from the manifest, the argument or the folder name."""
    folder = Path(folder)
    src = JsonSource(folder)
    if run_date is None and not src.manifest():
        try:
            run_date = date.fromisoformat(folder.name)
        except ValueError:
            pass
    rep = validate(src, run_date)
    if write:
        (folder / config.VALIDATION_JSON).write_text(json.dumps(rep.to_dict(), indent=2, ensure_ascii=False, default=str),
                                                     encoding="utf-8")
        (folder / config.VALIDATION_MD).write_text(rep.to_markdown(), encoding="utf-8")
    return rep


In [ ]:
%%engine nbx.fsdr.contract.data_agent
"""① Data agent (master prompt §5): step T0_load of the training pipeline.

    rep = run_t0(Path("data/input/2026-10-21"))            # pipeline detected: train_week0 or train_weekly
    rep["status"]     "done" (the cache is built) or "blocked" (the run stops: rep["explanation"] says why)

What it does, in order, each result saved in data/runs/<D>/train/T0_load/ (plan §14):
  1. validate every table against the contract (fields, raw types, keys, duplicates, cross-table orphans, model
     coverage, date windows, the 5-folder layout, customer_profile)    -> validation_report.json / .md
  2. weekly completeness for the pipeline about to run (§17.9)          -> completeness.csv / .json
  3. if nothing blocks: convert every table to the partitioned Parquet cache (plan §15) -> cache_manifest.json
  4. a plain-language summary of what blocks or warns                   -> summary.json

The checks are code; the agent only reports. Blocking issues stop the run here, before any feature is computed.
A re-run with the same input files is skipped (resume), unless force=True.
"""
from __future__ import annotations

import json
from datetime import date
from pathlib import Path

import pandas as pd

from nbx.fsdr import config
from nbx.fsdr.contract import cache as cache_mod
from nbx.fsdr.contract import completeness
from nbx.fsdr.datasource import JsonSource
from nbx.fsdr.steps import Step, fingerprint, runs_dir
from nbx.fsdr.validation import ValidationReport, validate

STEP = "T0_load"


def run_date_of(folder: Path, src: JsonSource, run_date: date | None = None) -> date | None:
    if run_date:
        return run_date
    man = src.manifest()
    if man.get("run_date"):
        return date.fromisoformat(man["run_date"])
    try:
        return date.fromisoformat(Path(folder).name)
    except ValueError:
        return None


def explain(rep: ValidationReport, comp: dict) -> str:
    """Plain-language summary of the run's data: what blocks first, then the main warnings (deterministic)."""
    c = rep.counts()
    lines = []
    if rep.status == "blocked" or not comp["complete"]:
        lines.append("The run is BLOCKED: no feature is computed until these are fixed.")
        for i in [i for i in rep.issues if i.level == "blocking"][:10]:
            lines.append(f"- {i.table}{'.' + i.column if i.column else ''}: {i.message}")
        for r in comp["reasons"]:
            lines.append(f"- {r}")
    else:
        lines.append(f"The data of {rep.run_date} is usable for {comp['pipeline']}: "
                     f"{len(rep.tables)} tables checked, {c['warning']} warning(s), {c['info']} note(s).")
    warns = [i for i in rep.issues if i.level == "warning"]
    if warns:
        lines.append("Warnings (the run continues):")
        lines += [f"- {i.table}{'.' + i.column if i.column else ''}: {i.message}" for i in warns[:10]]
        if len(warns) > 10:
            lines.append(f"- ... and {len(warns) - 10} more in validation_report.md")
    if comp["warnings"]:
        lines.append(f"Optional tables missing (fallback used): {', '.join(comp['warnings'])}.")
    return "\n".join(lines)


def run_t0(folder: Path | str, pipeline: str | None = None, run_date: date | None = None,
           runs_root: Path | None = None, cache_root: Path | None = None, force: bool = False) -> dict:
    folder = Path(folder)
    src = JsonSource(folder)
    run_date = run_date_of(folder, src, run_date)
    if run_date is None:
        raise ValueError(f"{folder}: no run date (no _manifest.json, folder name is not YYYY-MM-DD, none given)")
    pipeline = pipeline or completeness.detect_pipeline(src)
    files = [src.path(t) for t in src.tables()] + src.manifest_files()
    inputs = fingerprint(files, {"pipeline": pipeline, "parts": config.CACHE_PARTITIONS, "cache": str(cache_root or "")})
    with Step(runs_dir(run_date, runs_root) / "train", STEP, inputs=inputs, force=force) as st:
        if st.done:
            return {**st.read_json("summary.json"), "skipped": True}
        rep = validate(src, run_date, pipeline)
        st.write_json("validation_report.json", rep.to_dict())
        st.write_text("validation_report.md", rep.to_markdown())
        comp = completeness.check(src, pipeline, run_date)
        st.write_json("completeness.json", comp)
        st.write_csv(pd.DataFrame(comp["tables"]), "completeness.csv")
        rows_in = sum(t.rows for t in rep.tables.values())
        blocked = rep.status == "blocked" or not comp["complete"]
        twins = config.run_twins(folder)
        if twins and src.misfiled():
            st.write_json("misfiled_tables.json", {t: {"found": f, "expected": e} for t, (f, e) in src.misfiled().items()})
        summary = {"run_date": run_date.isoformat(), "pipeline": pipeline, "source": str(folder),
                   "training_source": str(twins.get(config.TRAINING_FOLDER, folder)),
                   "scoring_source": str(twins.get(config.SCORING_FOLDER, folder)),
                   "misfiled_tables": len(src.misfiled()),
                   "layout": rep.layout, "status": "blocked" if blocked else "done",
                   "validation": rep.status, "issue_counts": rep.counts(), "complete": comp["complete"],
                   "tables": len(src.tables()), "rows_in": rows_in, "profile": rep.profile,
                   "explanation": explain(rep, comp), "step_dir": str(st.dir)}
        if blocked:
            st.block("; ".join(comp["reasons"] + [i.message for i in rep.issues if i.level == "blocking"][:5]))
        else:
            man = cache_mod.build_cache(src, run_date, cache_root)
            st.write_json("cache_manifest.json", man)
            summary["cache"] = {"dir": str(cache_mod.cache_dir(run_date, cache_root)), "bytes": man["bytes"],
                                "partitions": man["partitions"], "seconds": man["seconds"]}
        st.rows(rows_in=rows_in, tables=len(src.tables()))
        st.note(pipeline=pipeline, validation=rep.status)
        st.write_json("summary.json", summary)
        return summary


def load_summary(run_date: date, runs_root: Path | None = None) -> dict:
    p = runs_dir(run_date, runs_root) / "train" / STEP / "summary.json"
    return json.loads(p.read_text(encoding="utf-8")) if p.exists() else {}


In [ ]:
%%engine nbx.fsdr.csvstd
"""The FSDR CSV standard (master prompt §2, §20): the one format of every input AND output table file.

    write_csv(df, path)                 <name>.csv.gz (gzip level 6), + <name>.schema.json next to it
    read_csv(path)                      every column as text (IDs keep their leading zeros), "" for missing
    write_outputs_manifest(dir, files)  _outputs_manifest.json: file list, row counts, SHA-256, run id, versions

| Compression | gzip, level 6 (mtime 0: the same table gives the same bytes)  |
| Encoding    | UTF-8, no BOM                                                 |
| Separator   | ,   quoting only when needed (comma, quote, line break), "" inside |
| Line ending | \\n                                                            |
| Decimal     | dot, no thousands separator, never scientific notation        |
| Dates       | YYYY-MM-DD / YYYY-MM-DD HH:MM:SS                              |
| Missing     | empty field (never nan / None / NULL)                         |
| Index       | none                                                          |
"""
from __future__ import annotations

import csv
import gzip
import io
import json
import math
from datetime import date, datetime
from decimal import Decimal
from pathlib import Path

import numpy as np
import pandas as pd

from nbx.fsdr import config

SUFFIX = ".csv.gz"
SCHEMA_SUFFIX = ".schema.json"
OUTPUTS_MANIFEST = "_outputs_manifest.json"


def table_name(path: Path) -> str:
    """activities_w1_score.csv.gz -> activities_w1_score (also .csv / .json / .jsonl / .parquet)."""
    name = Path(path).name
    for suffix in (SUFFIX, *config.INPUT_FORMATS):
        if name.lower().endswith(suffix):
            return name[: -len(suffix)]
    return Path(path).stem


def file_format(path: Path) -> str:
    """".csv.gz", ".csv", ".json", ".jsonl" or ".parquet" ("" for anything else)."""
    name = Path(path).name.lower()
    return next((f for f in (SUFFIX, *config.INPUT_FORMATS) if name.endswith(f)), "")


def format_value(v) -> str:
    """One cell -> its text in the FSDR CSV standard."""
    if v is None or v is pd.NA or v is pd.NaT:
        return ""
    if isinstance(v, (bool, np.bool_)):
        return "1" if v else "0"
    if isinstance(v, (int, np.integer)):
        return str(int(v))
    if isinstance(v, (float, np.floating)):
        f = float(v)
        if math.isnan(f) or math.isinf(f):
            return ""
        return np.format_float_positional(f, trim="0")              # 7.0, 12.5, 0.0001 -- never 1e-04
    if isinstance(v, Decimal):
        return format(v, "f")
    if isinstance(v, datetime):                                     # pd.Timestamp included
        return v.strftime("%Y-%m-%d %H:%M:%S")
    if isinstance(v, date):
        return v.isoformat()
    return str(v)


def to_text_frame(df: pd.DataFrame) -> pd.DataFrame:
    """Every cell formatted as the standard writes it (also what read_csv gives back)."""
    out = {}
    for c in df.columns:
        out[c] = _text_column(df[c])
    return pd.DataFrame(out, columns=df.columns, index=df.index)


def _float_text(s: pd.Series) -> pd.Series:
    """Vectorised repr of floats ("7.0", "12.5"); the rare exponent forms are rewritten positionally."""
    f = s.astype("float64")
    t = f.astype(str)
    exp = t.str.contains("e", regex=False) & f.notna() & np.isfinite(f)
    if exp.any():
        t[exp] = [np.format_float_positional(v, trim="0") for v in f[exp]]
    return t.where(f.notna() & np.isfinite(f), "")


def _text_column(s: pd.Series) -> pd.Series:
    """One column -> text, by dtype first (fast paths for 1M-row tables), value by value only for mixed columns."""
    if pd.api.types.is_datetime64_any_dtype(s):
        d = s.dropna()
        fmt = "%Y-%m-%d" if bool((d.dt.normalize() == d).all()) else "%Y-%m-%d %H:%M:%S"
        return s.dt.strftime(fmt).fillna("")
    if pd.api.types.is_bool_dtype(s):
        return s.map(format_value)
    if pd.api.types.is_integer_dtype(s):
        return s.astype("string").fillna("").astype(object)
    if pd.api.types.is_float_dtype(s):
        return _float_text(s)
    kind = pd.api.types.infer_dtype(s, skipna=True)
    if kind in ("string", "empty"):
        return s.where(s.notna(), "").astype(str)
    if kind == "integer":
        return s.astype("Int64").astype("string").fillna("").astype(object)
    if kind == "floating":
        return _float_text(s)
    return s.map(format_value)


def column_types(df: pd.DataFrame) -> list[dict]:
    """Column names and types for <name>.schema.json (identifier columns are always text)."""
    cols = []
    for c in df.columns:
        s = df[c]
        if str(c).upper() in config.ID_TEXT_COLUMNS:
            t = "string"
        elif pd.api.types.is_bool_dtype(s):
            t = "boolean"
        elif pd.api.types.is_integer_dtype(s):
            t = "bigint"
        elif pd.api.types.is_float_dtype(s):
            t = "double"
        elif pd.api.types.is_datetime64_any_dtype(s):
            t = "timestamp"
        else:
            t = "string"
        cols.append({"name": str(c), "type": t})
    return cols


def write_csv(df: pd.DataFrame, path: Path, schema: bool = True, compress: bool = True) -> Path:
    """Writes df in the FSDR CSV standard. path may end in .csv.gz (default) or .csv (compress=False)."""
    path = Path(path)
    if compress and not path.name.endswith(SUFFIX):
        path = path.with_name(table_name(path) + SUFFIX)
    path.parent.mkdir(parents=True, exist_ok=True)
    buf = io.StringIO()
    w = csv.writer(buf, delimiter=",", quotechar='"', quoting=csv.QUOTE_MINIMAL, lineterminator="\n")
    w.writerow([str(c) for c in df.columns])
    w.writerows(to_text_frame(df).itertuples(index=False, name=None))
    data = buf.getvalue().encode("utf-8")
    if compress:
        with path.open("wb") as fh, gzip.GzipFile(fileobj=fh, mode="wb", compresslevel=config.CSV_GZIP_LEVEL, mtime=0,
                                                  filename="") as gz:
            gz.write(data)
    else:
        path.write_bytes(data)
    if schema:
        schema_path(path).write_text(json.dumps({"table": table_name(path), "columns": column_types(df)}, indent=2),
                                     encoding="utf-8")
    return path


def schema_path(path: Path) -> Path:
    return Path(path).with_name(table_name(path) + SCHEMA_SUFFIX)


def read_csv(path: Path, columns: list[str] | None = None) -> pd.DataFrame:
    """A .csv.gz / .csv of the standard -> text frame ("" stays "", so a missing value is never "nan")."""
    path = Path(path)
    opener = gzip.open if path.name.lower().endswith(".gz") else open
    with opener(path, "rt", encoding="utf-8-sig", newline="") as fh:
        return pd.read_csv(fh, dtype=str, keep_default_na=False, na_filter=False, usecols=columns)


def write_outputs_manifest(folder: Path, files: list[Path], run_id: str = "", versions: dict | None = None) -> Path:
    from nbx.fsdr.steps import sha256_file

    entries = []
    for f in files:
        f = Path(f)
        rows = len(read_csv(f)) if file_format(f) in (SUFFIX, ".csv") else None
        entries.append({"file": f.name, "rows": rows, "sha256": sha256_file(f),
                        "schema": schema_path(f).name if schema_path(f).exists() else None})
    p = Path(folder) / OUTPUTS_MANIFEST
    p.write_text(json.dumps({"run_id": run_id, "versions": versions or {}, "files": entries}, indent=2), encoding="utf-8")
    return p


In [ ]:
%%engine nbx.fsdr.final_output
"""⑦ Final output agent: code 5 (master §12, F-01 … F-19) -> input_flytxt, vectorised over all customers.

    flytxt, slots = build_input_flytxt(monthly, weekly, scored, src, run_date)

monthly / weekly  the reco tables of the groups (B-32 structure: access_method_val, btl_name, price, …)
scored            results_scoring_nba (access_method_val, action, proba, tg_flag): the population (F-01)
flytxt            the 38 lower-case columns of F-19, one row per MSISDN
slots             per customer and slot 1-6: offer, reason (recommended / favourite / forced) and the rule that
                  cleared it (F-09 / F-11 / F-12): the detail behind every row of the file

Code-5 issues, fixed (master §12): (a) a cleared slot also loses its script and opportunity name, (b) an empty slot
is an empty field (never nbo_LFM_nan), (c) favourites are one row per offer, by price then purchase count,
(d) the F-05 anchor is separate from B-26. The bug-reproduction replay for the parity test A-25 comes with Phase 7.
"""
from __future__ import annotations

from datetime import date, timedelta
from itertools import combinations

import numpy as np
import pandas as pd

from nbx.fsdr import config
from nbx.fsdr.calendar import MONTHS, WEEKS
from nbx.fsdr.features import norm_val, upper
from nbx.fsdr.offer_rules import load_catalog

VAL = "ACCESS_METHOD_VAL"
SLOTS = range(1, 7)


def f05_anchors(src, vals: pd.Series | None = None) -> pd.DataFrame:
    """F-05: revenue = Total_Sub_Amt + PG_Total_Amount, zero periods treated as missing, anchor = min(mean + std,
    max); data 0.9 x. No non-zero period -> no anchor (NaN). Index: ACCESS_METHOD_VAL."""
    sub = upper(src.read("subscriptions_tg")).drop_duplicates(VAL).set_index(VAL)
    use = upper(src.read("usage_data_voice_payg_tg")).drop_duplicates(VAL).set_index(VAL)
    idx = sub.index.union(use.index) if vals is None else pd.Index(vals.unique())
    out = pd.DataFrame(index=idx)
    for name, periods in (("MONTHLY", MONTHS), ("WEEKLY", WEEKS)):
        rev = pd.DataFrame({p: pd.to_numeric(sub.get(f"TOTAL_SUB_AMT_{p}"), errors="coerce").reindex(idx).fillna(0.0)
                            + pd.to_numeric(use.get(f"PG_TOTAL_AMOUNT_{p}"), errors="coerce").reindex(idx).fillna(0.0)
                            for p in periods}).replace(0.0, np.nan)
        avg = rev.mean(axis=1) + rev.std(axis=1, ddof=1)
        out[f"F05_{name}"] = pd.concat([avg, rev.max(axis=1)], axis=1).min(axis=1)
        out[f"F05_DATA_{name}"] = config.ARPU_DATA_SHARE * out[f"F05_{name}"]
    return out


# F-02 / F-03 slots of the recommended offers by price rank. V4.0 with 6 offers: the third monthly offer goes to
# slot 5, the third weekly to slot 6 (slots 1-4 keep their V3.0 meaning).
MONTHLY_SLOTS = {1: 1, 2: 2, 3: 5}
WEEKLY_SLOTS = {1: 3, 2: 4, 3: 6}


def recommended_slots() -> list[int]:
    n = config.RECOS_PER_VALIDITY
    return sorted([MONTHLY_SLOTS[k] for k in range(1, n + 1)] + [WEEKLY_SLOTS[k] for k in range(1, n + 1)])


def _slots_from_recos(recos: pd.DataFrame, slots: dict[int, int]) -> pd.DataFrame:
    """F-02 / F-03: price rank (highest first, ties by first occurrence) -> slot of that rank."""
    slots = {k: s for k, s in slots.items() if k <= config.RECOS_PER_VALIDITY}
    if not len(recos):
        return pd.DataFrame(columns=[VAL] + [f"S{s}" for s in slots.values()]).set_index(VAL)
    r = recos.assign(_RANK=recos.groupby("access_method_val")["price"].rank(method="first", ascending=False))
    out = {}
    for k, s in slots.items():
        top = r[r["_RANK"] == k].drop_duplicates("access_method_val")
        out[f"S{s}"] = top.set_index("access_method_val")["btl_name"]
    return pd.DataFrame(out).rename_axis(VAL)


def favourites(src, vals: pd.Index, catalog: pd.DataFrame) -> pd.DataFrame:
    """F-06 (issue c fixed): the customer's 270-day catalogue purchases, one row per offer, price desc then count."""
    b = src.read("sub_btl_3months")
    df = pd.DataFrame({VAL: norm_val(b["access_method_val"]), "BTL_NAME": b["PRODUCTNAME"].astype(str).str.strip(),
                       "NB": pd.to_numeric(b.get("nb_sub"), errors="coerce").fillna(1)})
    df = df[df[VAL].isin(vals) & df["BTL_NAME"].isin(set(catalog["BTL_NAME"]))]
    df = df.groupby([VAL, "BTL_NAME"], as_index=False)["NB"].sum()
    df = df.merge(catalog[["BTL_NAME", "PRICE", "VALIDITY"]], on="BTL_NAME", how="left")
    return df.sort_values([VAL, "PRICE", "NB"], ascending=[True, False, False], kind="mergesort")


def build_input_flytxt(monthly: pd.DataFrame, weekly: pd.DataFrame, scored: pd.DataFrame, src, run_date: date
                       ) -> tuple[pd.DataFrame, pd.DataFrame]:
    catalog = load_catalog(src)
    pop = pd.Index(scored["access_method_val"].astype(str).unique(), name=VAL)                       # F-01
    s = pd.DataFrame(index=pop).join(_slots_from_recos(monthly, MONTHLY_SLOTS)).join(_slots_from_recos(weekly, WEEKLY_SLOTS))
    monthly_slots = set(MONTHLY_SLOTS.values())
    for i in SLOTS:
        s[f"S{i}"] = s.get(f"S{i}")
        s[f"REASON_{i}"] = np.where(s[f"S{i}"].notna(), "recommended monthly" if i in monthly_slots
                                    else "recommended weekly", None)
    keep = s[[f"S{i}" for i in recommended_slots()]].notna().any(axis=1)
    dropped = s.index[~keep]                                                                           # F-04
    s = s[keep].copy()

    anchors = f05_anchors(src, pd.Series(s.index))
    fav = favourites(src, s.index, catalog)                                                             # F-06
    if config.FAVOURITE_ARPU_FILTER:                                                                    # F-07 (off)
        a = fav[VAL].map(anchors["F05_MONTHLY"]).where(fav["VALIDITY"] == 30, fav[VAL].map(anchors["F05_WEEKLY"]))
        fav = fav[fav["PRICE"] >= a]
    fav = fav.assign(_K=fav.groupby(VAL).cumcount() + 1)
    f1, f2 = (fav[fav["_K"] == k].set_index(VAL)["BTL_NAME"].reindex(s.index) for k in (1, 2))
    empty5, empty6 = s["S5"].isna(), s["S6"].isna()           # V4.0: a favourite only fills an EMPTY slot 5 / 6
    fill5 = empty5 & f1.notna()
    s.loc[fill5, "S5"], s.loc[fill5, "REASON_5"] = f1[fill5], "favourite"
    six = f2.where(fill5, f1)                                 # slot 5 took the first favourite -> the second here
    fill6 = empty6 & six.notna()
    s.loc[fill6, "S6"], s.loc[fill6, "REASON_6"] = six[fill6], "favourite"

    in_catalog = set(catalog["BTL_NAME"])
    s = s.join(anchors)
    for col, lo, hi, offer, popular in config.FORCED_BUNDLES:                                            # F-08
        if offer not in in_catalog:
            continue
        holds = s[[f"S{i}" for i in SLOTS]].isin(popular).any(axis=1)
        m = s[col].ge(lo) & s[col].lt(hi) & s["S6"].isna() & ~holds
        s.loc[m, "S6"] = offer
        s.loc[m, "REASON_6"] = f"forced F-08 {col} [{lo}, {hi})"

    for i in SLOTS:
        s[f"REMOVED_{i}"] = None
    info = catalog.set_index("BTL_NAME")[["TYPE", "PRICE"]]
    key = {i: s[f"S{i}"].map(info["TYPE"]).astype(str) + "|" + s[f"S{i}"].map(info["PRICE"]).astype(str) for i in SLOTS}
    has_type = {i: s[f"S{i}"].map(info["TYPE"]).notna() for i in SLOTS}
    for i in SLOTS:                                                                                    # F-09
        dup = pd.Series(False, index=s.index)
        for j in range(1, i):
            dup |= has_type[j] & has_type[i] & (key[j] == key[i]) & s[f"S{j}"].notna()
        _clear(s, i, dup & s[f"S{i}"].notna(), "F-09")
    for i in range(2, 7):                                                                              # F-11
        dup = pd.Series(False, index=s.index)
        for j in range(1, i):
            dup |= s[f"S{i}"].notna() & (s[f"S{i}"] == s[f"S{j}"])
        _clear(s, i, dup, "F-11")
    pairs = _conflict_pairs(src)                                                                       # F-12
    for i, j in combinations(SLOTS, 2):
        a, b = s[f"S{i}"], s[f"S{j}"]
        offer_a = (a.astype(str) + "|" + b.astype(str)).map(pairs)
        both = a.notna() & b.notna() & offer_a.notna()
        _clear(s, i, both & (offer_a == a), "F-12")
        _clear(s, j, both & (offer_a != a) & (offer_a == b), "F-12")

    flytxt = _format(s, scored, src, catalog, run_date)
    slots = s.reset_index().assign(IN_INPUT_FLYTXT=True)
    if len(dropped):
        slots = pd.concat([slots, pd.DataFrame({VAL: dropped, "IN_INPUT_FLYTXT": False, "DROP_RULE": "F-04"})],
                          ignore_index=True)
    return flytxt, slots


def _clear(s: pd.DataFrame, i: int, mask: pd.Series, rule: str) -> None:
    if mask.any():
        s.loc[mask, f"REMOVED_{i}"] = rule + ": " + s.loc[mask, f"S{i}"].astype(str)
        s.loc[mask, f"S{i}"] = None


def _conflict_pairs(src) -> pd.Series:
    """F-12: "a|b" and "b|a" -> Offer_A of the row; rows with Offer_Type_A = VOICE ignored (as code 5)."""
    if not src.has("conflicting_offers_h"):
        return pd.Series(dtype=object)
    c = src.read("conflicting_offers_h")
    c.columns = [str(x).lower() for x in c.columns]
    c = c[c["offer_type_a"].astype(str) != "VOICE"]
    a, b = c["offer_a"].astype(str), c["offer_b"].astype(str)
    pairs = pd.concat([pd.Series(a.to_numpy(), index=(a + "|" + b).to_numpy()),
                       pd.Series(a.to_numpy(), index=(b + "|" + a).to_numpy())])
    return pairs[~pairs.index.duplicated(keep="last")]           # later rows win, as the code-5 dict does


def _format(s: pd.DataFrame, scored: pd.DataFrame, src, catalog: pd.DataFrame, run_date: date) -> pd.DataFrame:
    """F-10, F-13 … F-19: scripts, segment, BTL taker flags, dates, opportunity names, the 38 columns."""
    scripts = src.read("BTL_script_english_arabic_ok")
    scripts.columns = [str(c).upper() for c in scripts.columns]
    scripts = scripts.drop_duplicates("DESTINATION_OFFER_NAME").set_index("DESTINATION_OFFER_NAME")
    action = scored.drop_duplicates("access_method_val").set_index("access_method_val")["action"]
    bought = src.read("sub_btl_3months")
    takers = set(norm_val(bought.loc[bought["PRODUCTNAME"].astype(str).str.strip().isin(set(catalog["BTL_NAME"])),
                                     "access_method_val"]).dropna())
    out = pd.DataFrame({"date": run_date.isoformat(), "start_date": (run_date + timedelta(days=1)).isoformat(),
                        "end_date": "", "msisdn": s.index.astype(str)}, index=s.index)
    for i in SLOTS:
        out[f"channel_{i}"] = ""
    for i in SLOTS:
        out[f"opportunity_name_{i}"] = (config.OPPORTUNITY_PREFIX + s[f"S{i}"].astype(str)).where(s[f"S{i}"].notna(), "")
    for i in SLOTS:
        out[f"destination_offer_name_{i}"] = s[f"S{i}"].fillna("")
    out["flag_splio"] = 1
    for lang, col in (("en", "ENGLISH SCRIPT"), ("ar", "ARABIC SCRIPT")):
        for i in SLOTS:
            out[f"offer_script_{lang}_{i}"] = s[f"S{i}"].map(scripts[col]).fillna("") if col in scripts.columns else ""
    out["segment"] = action.reindex(s.index).fillna("").to_numpy()                                   # F-14
    out["segment_btl_taker"] = np.where(s.index.isin(takers), "BTL_Taker", "No_BTL_Taker")            # F-15
    out["segment_btl_taker_upsell"] = np.where(out["segment"] == "UPSELL", "BTL_upsell", "")          # F-16
    return out[config.INPUT_FLYTXT_COLUMNS].drop_duplicates("msisdn").reset_index(drop=True)          # F-13, F-19


In [ ]:
%%engine nbx.fsdr.llm_budget
"""Level-2 LLM queue and the weekly budget (master §19.1a, update §6).

    q = llm_queue(decisions)                          # priority tier first, the 4 groups interleaved inside a tier
    q = apply_budget(q, seconds_per_customer=0.4)     # LEVEL = level2_llm | cell_policy_timebox

Each contacted customer costs one NBA + one NBO call (for T1F / T2F both run in the same wave: the offers do not
depend on the action). Past the budget (FSDR_LLM_WEEKLY_BUDGET_H, default 10 h) the rest get their cell policy
applied by code, logged per group. Interleaving keeps the coverage of the 4 groups equal (+-2 %).
"""
from __future__ import annotations

import pandas as pd

from nbx.fsdr import config

PRIORITY_TIERS = [   # master §19.1a
    ["Retention_RGS_At_Risk", "Retention_High_Probability_To_Reconnect", "Retention_Med_Probability_To_Reconnect"],
    ["Retention_RGS_Fragile", "Onboarding_Early_Churn_At_Risk", "Onboarding_Early_Churn_Not_At_Risk"],
    ["Data_Only_Voice_Appetency", "Data_Only_No_Voice_Appetency", "Voice_Only_Data_Appetency", "Voice_Only_No_Data_Appetency"],
    ["RGS_Not_At_Risk_DualPlay_Upgrade", "Retention_Low_Probability_To_Reconnect"],
]
TIER = {s: i + 1 for i, tier in enumerate(PRIORITY_TIERS) for s in tier}


def llm_queue(decisions: pd.DataFrame) -> pd.DataFrame:
    """Order of the level-2 calls: inside each group, priority tier first; across groups, round-robin (the k-th
    customer of every group, then the k+1-th), so a budget cut leaves the 4 groups with the same coverage."""
    q = decisions[decisions["GROUP_KEY"] != config.CONTROL_GROUP].copy()
    q["TIER"] = q["SEGMENT"].map(TIER).fillna(len(PRIORITY_TIERS) + 1).astype(int)
    q = q.sort_values(["TIER", "SEGMENT", "ACCESS_METHOD_VAL"], kind="mergesort")
    q["_K"] = q.groupby("GROUP_KEY").cumcount()
    order = {k: i for i, k in enumerate(config.GROUPS)}
    q = q.sort_values(["_K", "GROUP_KEY"], key=lambda s: s.map(order) if s.name == "GROUP_KEY" else s,
                      kind="mergesort")
    return q.drop(columns="_K").reset_index(drop=True)


def apply_budget(q: pd.DataFrame, seconds_per_customer: float, budget_h: float | None = None) -> pd.DataFrame:
    budget_s = 3600.0 * (config.LLM_WEEKLY_BUDGET_H if budget_h is None else budget_h)
    n = int(budget_s // max(seconds_per_customer, 1e-9))
    q = q.copy()
    q["LEVEL"] = "cell_policy_timebox"
    q.loc[q.index[:n], "LEVEL"] = "level2_llm"
    return q


def coverage(q: pd.DataFrame) -> pd.Series:
    """Share of each group decided by the level-2 LLM."""
    return q.groupby("GROUP_KEY")["LEVEL"].apply(lambda s: round(float((s == "level2_llm").mean()), 4))


In [ ]:
%%engine nbx.fsdr.offers.anchors
"""ARPU anchors (master §11 B-25 … B-27, update CC-03 / CF-03).

B-26   anchor = mean + std (ddof 1) of (Total_Sub_Amt + PG_Total_Amount), monthly M1-M3 / weekly W1-W4;
       data = 0.9 x, voice = 0.1 x  (offer_rules.arpu_anchors, shared with V1.0)
B-27   COMBO -> total anchor, DATA -> data anchor, VOICE -> voice anchor; weekly offers use the weekly anchors
CC-03  challenger split: data / voice from the customer's own data vs voice subscription spend (no history: 90 / 10)
C-02   a customer with no spend gets the median anchor of his micro-segment (challenger bands only)
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from nbx.fsdr import config
from nbx.fsdr.calendar import MONTHS
from nbx.fsdr.features import upper
from nbx.fsdr.offer_rules import arpu_anchors

b26_anchors = arpu_anchors


def data_share(src) -> pd.Series:
    """ACCESS_METHOD_VAL -> share of data in the customer's data + voice subscription spend (M1-M3)."""
    if not src.has("subscriptions_tg"):
        return pd.Series(dtype=float)
    df = upper(src.read("subscriptions_tg"))
    d = sum(pd.to_numeric(df.get(f"DATA_SUB_AMT_{m}"), errors="coerce").fillna(0.0) for m in MONTHS)
    v = sum(pd.to_numeric(df.get(f"VOICE_SUB_AMT_{m}"), errors="coerce").fillna(0.0) for m in MONTHS)
    share = pd.Series(np.where((d + v) > 0, d / (d + v).where((d + v) > 0, 1.0), np.nan), index=df["ACCESS_METHOD_VAL"])
    return share[~share.index.duplicated(keep="first")]


def _period(validity) -> np.ndarray:
    return np.where(pd.to_numeric(validity) == 7, "WEEKLY", "MONTHLY")


def anchor_for(cands: pd.DataFrame, anchors: pd.DataFrame) -> pd.Series:
    """B-27 anchor of every candidate row (by its service and validity)."""
    period = _period(cands["VALIDITY"])
    svc = cands["SERVICE"].astype(str).str.upper().to_numpy()
    kind = np.select([svc == "DATA", svc == "VOICE"], ["DATA_", "VOICE_"], "")
    col = np.array([f"AVG_REV_{k}{p}" for k, p in zip(kind, period)], dtype=object)
    out = np.zeros(len(cands))
    vals = cands["ACCESS_METHOD_VAL"].to_numpy()
    for c in set(col):
        m = col == c
        if c in anchors.columns:
            out[m] = anchors[c].reindex(vals[m]).fillna(0.0).to_numpy()
    return pd.Series(out, index=cands.index)


def challenger_anchor(cands: pd.DataFrame, anchors: pd.DataFrame, share: pd.Series,
                      micro_median: pd.Series | None = None) -> pd.Series:
    """CC-03 / CF-03: total anchor for COMBO, own data share x total for DATA, (1 - share) x total for VOICE.
    C-02: a zero anchor is replaced by the micro-segment median (micro_median: ACCESS_METHOD_VAL -> median total)."""
    period = _period(cands["VALIDITY"])
    vals = cands["ACCESS_METHOD_VAL"].to_numpy()
    total = np.where(period == "WEEKLY", anchors["AVG_REV_WEEKLY"].reindex(vals).fillna(0.0).to_numpy(),
                     anchors["AVG_REV_MONTHLY"].reindex(vals).fillna(0.0).to_numpy())
    if micro_median is not None:
        med = micro_median.reindex(vals).to_numpy(dtype=float)
        total = np.where((total <= 0) & ~np.isnan(med), med, total)
    s = share.reindex(vals).fillna(config.DEFAULT_DATA_SHARE).to_numpy(dtype=float)
    svc = cands["SERVICE"].astype(str).str.upper().to_numpy()
    return pd.Series(np.select([svc == "DATA", svc == "VOICE"], [s * total, (1 - s) * total], total), index=cands.index)


In [ ]:
%%engine nbx.fsdr.offers.universe
"""Offer universes (master §11 B-20b, B-21; update §4.1, §4.2 FR-01 … FR-06).

recommendable   in the current btl_catalog AND bought by at least one line of the base in 270 days (sub_btl_3months)
constraint      code 3: the action's two families (service, generosity), active mapping = L2 for all; CROSS SELL
                VOICE has its own filter (the code-3 typo that left it unfiltered is fixed)
free            code 4: L2 only, DATA and COMBO (no VOICE), the action is not used
"""
from __future__ import annotations

import pandas as pd

from nbx.fsdr import config
from nbx.fsdr.features import norm_val
from nbx.fsdr.offer_rules import families

CAND_COLS = ["BTL_NAME", "TYPE", "SERVICE", "GENEROSITY", "VALIDITY", "PRICE", "GB", "MIN"]


def purchases(src) -> pd.DataFrame:
    """sub_btl_3months (270 days, no date): ACCESS_METHOD_VAL, BTL_NAME, NB_SUB."""
    if not src.has("sub_btl_3months"):
        return pd.DataFrame(columns=["ACCESS_METHOD_VAL", "BTL_NAME", "NB_SUB"])
    df = src.read("sub_btl_3months")
    out = pd.DataFrame({"ACCESS_METHOD_VAL": norm_val(df["access_method_val"]),
                        "BTL_NAME": df["PRODUCTNAME"].astype(str).str.strip(),
                        "NB_SUB": pd.to_numeric(df.get("nb_sub"), errors="coerce").fillna(1)})
    return out.dropna(subset=["ACCESS_METHOD_VAL"])


def recommendable(catalog: pd.DataFrame, bought: pd.DataFrame) -> pd.DataFrame:
    """B-20b / FR-05: catalogue offers bought by someone in the base in the last 270 days."""
    sold = set(bought.loc[bought["NB_SUB"] > 0, "BTL_NAME"])
    cols = [c for c in CAND_COLS if c in catalog.columns]
    return catalog.loc[catalog["BTL_NAME"].isin(sold), cols].reset_index(drop=True)


def constraint_universe(rec: pd.DataFrame, action: str, validity: int) -> pd.DataFrame:
    """T1C / T2C (CC-01): the action's families and generosity, one validity."""
    mask = pd.Series(False, index=rec.index)
    for svc, gen in families(action, "cstr"):
        mask |= (rec["SERVICE"] == svc) & (rec["GENEROSITY"] == gen)
    return rec[mask & (rec["VALIDITY"] == validity)]


def free_universe(rec: pd.DataFrame, validity: int) -> pd.DataFrame:
    """T1F / T2F (FR-01 … FR-06, CF-01): L2, DATA / COMBO, whatever the action."""
    m = ((rec["GENEROSITY"] == config.FREE_UNIVERSE_GENEROSITY) & rec["SERVICE"].isin(config.FREE_UNIVERSE_SERVICES)
         & (rec["VALIDITY"] == validity))
    return rec[m]


def _own_filter(cands: pd.DataFrame, own: pd.DataFrame | None) -> pd.DataFrame:
    """B-20b switch only_own_past_offers: slots 1-4 limited to the customer's own past offers."""
    if not config.ONLY_OWN_PAST_OFFERS or own is None:
        return cands
    keys = own[["ACCESS_METHOD_VAL", "BTL_NAME"]].drop_duplicates()
    return cands.merge(keys, on=["ACCESS_METHOD_VAL", "BTL_NAME"], how="inner")


def constraint_candidates(customers: pd.DataFrame, rec: pd.DataFrame, validity: int,
                          own: pd.DataFrame | None = None) -> pd.DataFrame:
    """customers: ACCESS_METHOD_VAL, ACTION -> one row per customer x offer of the action's universe."""
    parts = []
    for action, grp in customers.groupby("ACTION", sort=False):
        uni = constraint_universe(rec, action, validity)
        if len(uni):
            parts.append(grp[["ACCESS_METHOD_VAL", "ACTION"]].merge(uni, how="cross"))
    out = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=["ACCESS_METHOD_VAL", "ACTION", *CAND_COLS])
    return _own_filter(out, own)


def free_candidates(customers: pd.DataFrame, rec: pd.DataFrame, validity: int,
                    own: pd.DataFrame | None = None) -> pd.DataFrame:
    """customers: ACCESS_METHOD_VAL (+ ACTION, carried but not used) -> customer x free-universe offer."""
    cols = ["ACCESS_METHOD_VAL"] + (["ACTION"] if "ACTION" in customers.columns else [])
    out = customers[cols].merge(free_universe(rec, validity), how="cross")
    return _own_filter(out, own)


In [ ]:
%%engine nbx.fsdr.offers.recos
"""The reco tables of the four groups (update §5): code-2 tables + 8 reco tables in the original structure (B-32).

    out = build_group_recos(panel, cache)
    out["results_scoring_nba"], out["TG_40K_constraint_scored_nba"], out["recos_monthly_40k_constraint"], ...

Until the LLM agents exist (master Phases 5-6), the decisions are their documented deterministic fallbacks, and
every row says so:
  action  the group's rule action: action_original (Target 1) or action_agent (Target 2); DECIDED_BY = fallback_rule_action
  score   the NBO fallback order without helpers yet: E-O3 own past purchase, then E-O1 buyer share of the offer
          in the 270-day base; SCORE_SOURCE = fallback:E-O3+E-O1
  proba   the logged propensity: 1.0 for a deterministic fallback
V4.0 agents (optional, `agents=` of build_group_recos; the GPU notebook passes them):
  actions  ③ NBA: a table ACCESS_METHOD_VAL, ACTION, PROBA, DECIDED_BY that replaces the rule action of those lines
  nbo      ④ NBO: a function (group key, candidates) -> score in [0, 1] per candidate row (NaN = keep the fallback);
           it scores the candidates the RULES allowed, and the passes (price bands, families, EXT-6) still choose
  groups   the group keys the agents act on (Target 2 by default: Target 1 stays the production baseline)
10.1 consistency (constraint groups only): a customer with no offer at all for his action is sent back once with
his second action (action_agent rank 2, else the other dictionary's action, else the next B-10 prior); final.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from nbx.fsdr import config
from nbx.fsdr.offer_rules import load_catalog
from nbx.fsdr.offers import anchors as anc
from nbx.fsdr.offers import selection, universe

VAL = "ACCESS_METHOD_VAL"
VALIDITY_TABLE = {30: "recos_monthly", 7: "recos_weekly"}
RECO_COLUMNS = ["access_method_val", "item_id", "rank", "score", "btl_name", "type", "generosity", "service",
                "validity", "price", "action", "anchor", "pass", "selected_by", "score_source"]


def group_actions(panel: pd.DataFrame) -> pd.DataFrame:
    """ACTION of every contacted line = its group's rule action (fallback until ③ decides)."""
    out = panel[panel["GROUP_KEY"] != config.CONTROL_GROUP].copy()
    rule_col = out["GROUP_KEY"].map(lambda k: config.GROUPS[k]["rule_action"])
    picked = np.where(rule_col == "ACTION_AGENT", out["ACTION_AGENT"], out["ACTION_ORIGINAL"])
    out["ACTION"] = pd.Series(picked, index=out.index).where(pd.notna(picked), out["RULE_ACTION"])
    out["DECIDED_BY"] = "fallback_rule_action"
    return out


def second_action(row: pd.Series) -> str | None:
    for a in (row.get("ACTION_AGENT_RANK2"), row.get("ACTION_AGENT"), row.get("ACTION_ORIGINAL")):
        if isinstance(a, str) and a != row["ACTION"]:
            return a
    return next(a for a in sorted(config.ACTION_PRIOR, key=config.ACTION_PRIOR.get, reverse=True) if a != row["ACTION"])


def fallback_scores(cands: pd.DataFrame, bought: pd.DataFrame, base_lines: int) -> pd.Series:
    """0.5 x (the customer bought it in 270 days) + 0.5 x (buyer share of the offer / the best buyer share)."""
    share = bought.groupby("BTL_NAME")[VAL].nunique() / max(base_lines, 1)
    share = share / share.max() if len(share) else share
    own = pd.MultiIndex.from_frame(cands[[VAL, "BTL_NAME"]]).isin(
        pd.MultiIndex.from_frame(bought[[VAL, "BTL_NAME"]].drop_duplicates()))
    return (0.5 * own + 0.5 * cands["BTL_NAME"].map(share).fillna(0.0)).round(6)


def _raw_candidates(key: str, cust: pd.DataFrame, rec: pd.DataFrame, validity: int, ctx: dict) -> pd.DataFrame:
    """The offers the rules allow for each line of one validity, with the fallback score (E-O3 + E-O1)."""
    g = config.GROUPS[key]
    if g["universe"] == "constraint":
        c = universe.constraint_candidates(cust, rec, validity, ctx["bought"])
    else:
        c = universe.free_candidates(cust, rec, validity, ctx["bought"])
    if not len(c):
        return c
    return c.assign(SCORE=fallback_scores(c, ctx["bought"], ctx["base_lines"]), SCORE_SOURCE="fallback:E-O3+E-O1")


def _llm_scores(key: str, raw: dict[int, pd.DataFrame], ctx: dict) -> dict[int, pd.DataFrame]:
    """④ NBO: one call of the agent for the candidates of both validities (one prompt per line inside it)."""
    agents = ctx.get("agents") or {}
    if not agents.get("nbo") or key not in agents.get("groups", ()) or not any(len(c) for c in raw.values()):
        return raw
    both = pd.concat([c.assign(_V=v) for v, c in raw.items() if len(c)], ignore_index=True)
    both = both.assign(ANCHOR=anc.anchor_for(both, ctx["anchors"]))         # the price reference the agent sees
    s = pd.to_numeric(pd.Series(agents["nbo"](key, both), index=both.index), errors="coerce").clip(0.0, 1.0)
    llm = s.notna()
    both.loc[llm, "SCORE"] = s[llm].round(6)
    both.loc[llm, "SCORE_SOURCE"] = f"llm:{agents.get('label', 'agent')}"
    return {v: both[both["_V"] == v].drop(columns="_V").reset_index(drop=True) for v in raw}


def _finish(key: str, c: pd.DataFrame, ctx: dict) -> pd.DataFrame:
    """Rank, keep at most 25 per validity, and the anchors (B-27, challenger band)."""
    g = config.GROUPS[key]
    if not len(c):
        return c
    c = c.sort_values([VAL, "SCORE", "PRICE"], ascending=[True, False, False], kind="mergesort")
    c = c.assign(RANK=c.groupby(VAL).cumcount() + 1)
    c = c[c["RANK"] <= config.MAX_CANDIDATES_PER_VALIDITY]                 # §10: at most 25 per validity
    c = c.assign(ANCHOR=anc.anchor_for(c, ctx["anchors"]))
    if g["rules"] in ("CC", "CF"):
        c = c.assign(CH_ANCHOR=anc.challenger_anchor(c, ctx["anchors"], ctx["share"], ctx["micro_median"]))
    return c.reset_index(drop=True)


def _candidates(key: str, cust: pd.DataFrame, rec: pd.DataFrame, validity: int, ctx: dict) -> pd.DataFrame:
    return _finish(key, _llm_scores(key, {validity: _raw_candidates(key, cust, rec, validity, ctx)}, ctx)[validity], ctx)


def _select_group(key: str, cust: pd.DataFrame, rec: pd.DataFrame, ctx: dict) -> pd.DataFrame:
    parts = []
    raw = _llm_scores(key, {v: _raw_candidates(key, cust, rec, v, ctx) for v in (30, 7)}, ctx)
    for validity in (30, 7):
        c = _finish(key, raw[validity], ctx)
        if len(c):
            sel = selection.select(key, c)
            if config.RECOS_PER_VALIDITY > 2:                                  # V4.0: the third offer (EXT-6)
                sel = selection.extend(c, sel, config.RECOS_PER_VALIDITY)
            parts.append(sel.assign(N_CANDIDATES=sel[VAL].map(c.groupby(VAL).size())))
    return pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=[VAL])


def _micro_median(panel: pd.DataFrame, anchors: pd.DataFrame) -> pd.Series:
    """C-02: a customer with no spend gets the median anchor of his micro-segment (positive anchors only)."""
    a = anchors["AVG_REV_MONTHLY"].reindex(panel[VAL]).to_numpy()
    df = pd.DataFrame({VAL: panel[VAL].to_numpy(), "MS": panel["MICRO_SEGMENT"].to_numpy(), "A": a})
    med = df[df["A"] > 0].groupby("MS")["A"].median()
    return pd.Series(df["MS"].map(med).to_numpy(), index=df[VAL])


def apply_agent_actions(contacted: pd.DataFrame, agents: dict | None) -> pd.DataFrame:
    """③ NBA: the agent's action (and its logged propensity) replaces the rule action of the lines it decided."""
    contacted["PROBA"] = 1.0
    acts = (agents or {}).get("actions")
    if acts is None or not len(acts):
        return contacted
    a = acts.drop_duplicates(VAL).set_index(VAL)
    m = contacted[VAL].isin(a.index) & contacted["GROUP_KEY"].isin(agents.get("groups", ()))
    m &= contacted[VAL].map(a["ACTION"]).isin(config.ACTIONS)                  # an invented action is never used
    contacted.loc[m, "ACTION"] = contacted.loc[m, VAL].map(a["ACTION"])
    contacted.loc[m, "PROBA"] = contacted.loc[m, VAL].map(a["PROBA"]).astype(float)
    contacted.loc[m, "DECIDED_BY"] = contacted.loc[m, VAL].map(a["DECIDED_BY"]).fillna("agent")
    return contacted


def build_group_recos(panel: pd.DataFrame, src, no_contact: set[str] | None = None,
                      agents: dict | None = None) -> dict[str, pd.DataFrame]:
    """panel: assign_panel output joined with the T3 segment columns. src: the run's cache (CacheSource).
    agents: the V4.0 LLM agents ③ / ④ (see the module docstring); None = the deterministic fallbacks."""
    catalog = load_catalog(src)
    bought = universe.purchases(src)
    bought = bought[bought["BTL_NAME"].isin(set(catalog["BTL_NAME"]))]          # B-20: catalogue offers only
    rec = universe.recommendable(catalog, bought)
    item_id = {b: i for i, b in enumerate(rec["BTL_NAME"])}
    a = anc.b26_anchors(src)
    ctx = {"bought": bought, "base_lines": int(src.read("base_prepaid").shape[0]) if src.has("base_prepaid") else len(panel),
           "anchors": a, "share": anc.data_share(src), "micro_median": _micro_median(panel, a), "agents": agents}
    contacted = apply_agent_actions(group_actions(panel), agents)
    if no_contact:                                                             # C-05: do-not-disturb (Target 2)
        dnd = contacted[VAL].isin(no_contact) & contacted["TARGET"].eq("target2")
        contacted.loc[dnd, "DECIDED_BY"] = "no_contact_C-05"
    contacted["CONSISTENCY_RETRY"] = False
    out: dict[str, pd.DataFrame] = {}
    selected = []
    for key in config.GROUPS:
        cust = contacted[(contacted["GROUP_KEY"] == key) & (contacted["DECIDED_BY"] != "no_contact_C-05")]
        if not len(cust):
            continue
        sel = _select_group(key, cust, rec, ctx)
        if config.GROUPS[key]["consistency_loop"]:                              # 10.1: once, the second choice is final
            lost = cust[~cust[VAL].isin(sel[VAL] if len(sel) else [])]
            if len(lost):
                retry = lost.assign(ACTION=lost.apply(second_action, axis=1))
                sel = pd.concat([sel, _select_group(key, retry, rec, ctx)], ignore_index=True)
                second = retry.set_index(VAL)["ACTION"]
                m = contacted[VAL].isin(second.index)
                contacted.loc[m, "ACTION"] = contacted.loc[m, VAL].map(second)
                contacted.loc[m, "CONSISTENCY_RETRY"] = True
        if len(sel):
            selected.append(sel.assign(GROUP_KEY=key, RULES_VERSION=config.GROUPS[key]["rules_version"]))
        g = config.GROUPS[key]
        for validity, prefix in VALIDITY_TABLE.items():
            part = sel[sel["VALIDITY"] == validity] if len(sel) else sel
            out[g[prefix]] = reco_table(part, g, item_id)
    out["results_scoring_nba"] = results_scoring(contacted)
    for key, g in config.GROUPS.items():
        out[g["scored_table"]] = out["results_scoring_nba"][out["results_scoring_nba"]["tg_flag"] == g["tg_flag"]].reset_index(drop=True)
    out["_selected"] = pd.concat(selected, ignore_index=True) if selected else pd.DataFrame()
    out["_decisions"] = contacted
    out["_recommendable"] = rec
    return out


def reco_table(sel: pd.DataFrame, group: dict, item_id: dict) -> pd.DataFrame:
    """B-32 structure, lower-case as code 3 / code 4; the action column only in the constraint tables."""
    if not len(sel):
        cols = RECO_COLUMNS if group["action_column"] else [c for c in RECO_COLUMNS if c != "action"]
        return pd.DataFrame(columns=cols)
    t = pd.DataFrame({"access_method_val": sel[VAL], "item_id": sel["BTL_NAME"].map(item_id), "rank": sel["RANK"],
                      "score": sel["SCORE"], "btl_name": sel["BTL_NAME"], "type": sel.get("TYPE"),
                      "generosity": sel["GENEROSITY"], "service": sel["SERVICE"], "validity": sel["VALIDITY"],
                      "price": sel["PRICE"], "action": sel["ACTION"],
                      "anchor": (sel["CH_ANCHOR"] if "CH_ANCHOR" in sel.columns and sel["CH_ANCHOR"].notna().any()
                                 else sel["ANCHOR"]).round(6),
                      "pass": sel["PASS"], "selected_by": sel["SELECTED_BY"], "score_source": sel["SCORE_SOURCE"]})
    if not group["action_column"]:
        t = t.drop(columns="action")
    return t.reset_index(drop=True)


def results_scoring(contacted: pd.DataFrame) -> pd.DataFrame:
    """B-12 results_scoring_nba: access_method_val, action, proba (= logged propensity), tg_flag."""
    c = contacted[contacted["DECIDED_BY"] != "no_contact_C-05"]
    return pd.DataFrame({"access_method_val": c[VAL].to_numpy(), "action": c["ACTION"].to_numpy(),
                         "proba": (c["PROBA"].to_numpy() if "PROBA" in c.columns else 1.0), "tg_flag": c["GROUP_KEY"].map(lambda k: config.GROUPS[k]["tg_flag"]).to_numpy()})


In [ ]:
%%engine nbx.fsdr.pipelines package
"""Deterministic FSDR pipelines (train, score, recommend). Each step returns a JSON report,
so the agents can call the steps as tools."""
from __future__ import annotations

import json
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

OUTPUT_DIR = "outputs"


def outputs_dir(data_dir: str | Path) -> Path:
    p = Path(data_dir) / OUTPUT_DIR
    p.mkdir(parents=True, exist_ok=True)
    return p


def _clean(v):
    if v is None or (isinstance(v, float) and np.isnan(v)) or v is pd.NA or v is pd.NaT:
        return None
    if isinstance(v, np.integer):
        return int(v)
    if isinstance(v, np.floating):
        return round(float(v), 6)
    if isinstance(v, np.bool_):
        return bool(v)
    if isinstance(v, date):
        return v.isoformat()
    return v


def write_table(df: pd.DataFrame, data_dir: str | Path, name: str) -> Path:
    """Writes an output table as a JSON array of records (lower-case columns, as the production outputs)."""
    path = outputs_dir(data_dir) / f"{name}.json"
    cols = [str(c).lower() for c in df.columns]
    values = [_column_values(df.iloc[:, i]) for i in range(df.shape[1])]
    records = [dict(zip(cols, row)) for row in zip(*values)] if cols else [{} for _ in range(len(df))]
    path.write_text(json.dumps(records, ensure_ascii=False), encoding="utf-8")
    return path


def _column_values(s: pd.Series) -> list:
    """The JSON values of one column, as the per-value _clean of df.to_dict("records") gave them: NaN -> None,
    numpy numbers -> Python numbers (unrounded: to_dict already boxes them), dates -> ISO strings. Number and
    string columns are converted in one go (review #15: _clean ran once per value, 5 M calls at 100 k lines)."""
    k = s.dtype.kind
    if k in "iub" and not pd.api.types.is_extension_array_dtype(s.dtype):
        return s.tolist()
    if k == "f" and not pd.api.types.is_extension_array_dtype(s.dtype):
        return [None if v != v else v for v in s.tolist()]
    if s.dtype == object and pd.api.types.infer_dtype(s, skipna=True) in ("string", "empty"):
        return [v if isinstance(v, str) else None for v in s.tolist()]
    return [_clean(v) for v in s.to_frame().to_dict("list")[s.name if s.name is not None else 0]]


def read_table(data_dir: str | Path, name: str) -> pd.DataFrame:
    path = Path(data_dir) / OUTPUT_DIR / f"{name}.json"
    if not path.exists():
        raise FileNotFoundError(f"{path} not found")
    df = pd.DataFrame(json.loads(path.read_text(encoding="utf-8")))
    df.columns = [c.upper() for c in df.columns]
    return df


def write_report(report: dict, data_dir: str | Path, name: str) -> Path:
    path = outputs_dir(data_dir) / f"{name}.json"
    path.write_text(json.dumps(report, indent=2, ensure_ascii=False, default=_clean), encoding="utf-8")
    return path


In [ ]:
%%engine nbx.fsdr.segmentation package
"""② Segmentation (master prompt §7, §8) and ②b segment actions (§7.1): S-rules, bands, micro-segments,
action_original (code 1's dictionary) and action_agent (the agent dictionary). All vectorised, no LLM here."""


In [ ]:
%%engine nbx.fsdr.segmentation.actions
"""②b The two actions of every line (master prompt §7.1), plus the week-0 history of code 1 (B-02, B-03, B-09, B-10).

action_original   code 1's dict_segm_actions (case-sensitive, last value wins) applied to the line's segment:
                  the segment-engine segment of targeting_dataiku_auto when the line has one this week, otherwise
                  the S-rule segment translated by the crosswalk (config.SEGMENT_CROSSWALK). Deterministic, no LLM.
action_agent      the agent dictionary dict_segm_actions_agent_vN: per S-segment x nationality group, ranked actions.
                  Week 0 it is a copy of the original dictionary (v0); the LLM changes cells from Phase 4 on, and
                  every version is validated here before it is applied (invalid -> the previous cell is kept).
"""
from __future__ import annotations

import json
import re
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd

from nbx.fsdr import config
from nbx.fsdr.features import norm_id, norm_val

ENGINE, CROSSWALK = "segment_engine", "crosswalk"
AT_RISK_SEGMENTS = {s for _, s, _ in config.S_RULES if s.endswith("_At_Risk") and "Not_At_Risk" not in s}
# H-06 for the dictionary: an at-risk segment may not be moved TO an upsell action. Its original action is kept
# allowed (Onboarding_Early_Churn_At_Risk is UPSELL in code 1). CHECK item (plan §11).
UPSELL_ACTIONS = {"UPSELL", "UPSELL DATA", "UPSELL VOICE"}


def is_nursering(segment: pd.Series) -> pd.Series:
    return segment.astype("string").str.lower().str.startswith("nursering").fillna(False)


# ---------------------------------------------------------------------------
# Crosswalk and action_original
# ---------------------------------------------------------------------------
def crosswalk_table() -> pd.DataFrame:
    """One row per S-rule: the original segment name it maps to, both actions, and the CHECK note."""
    rows = []
    for rid, seg, rule_action in config.S_RULES:
        orig, note = config.SEGMENT_CROSSWALK[seg]
        orig_action = config.DICT_SEGM_ACTIONS.get(orig)
        rows.append({"rule_id": rid, "s_segment": seg, "original_segment": orig, "rule_action": rule_action or "excluded",
                     "original_action": orig_action or "excluded", "same_action": (rule_action == orig_action),
                     "check": "CHECK", "note": note})
    return pd.DataFrame(rows)


def week0_targeting(tg: pd.DataFrame | None, fulfil: pd.DataFrame | None, catalog: pd.DataFrame | None,
                    run_date: date, lines: pd.DataFrame) -> pd.DataFrame:
    """Per line in targeting_dataiku_auto: the latest segment (W0_SEGMENT), its original action (B-03; rows without
    a segment or in Nursering get none, B-02) and the B-09 reward: a catalogue BTL fulfilled in D-7 … D-1.
    lines: ACCESS_METHOD_ID -> ACCESS_METHOD_VAL of the base (targeting carries the id, fulfilments the MSISDN)."""
    cols = ["ACCESS_METHOD_ID", "W0_SEGMENT", "W0_DATE", "W0_ACTION", "W0_REWARD", "W0_BTL_REVENUE"]
    if tg is None or not len(tg):
        return pd.DataFrame(columns=cols)
    t = pd.DataFrame({"ACCESS_METHOD_ID": norm_id(tg["ACCESS_METHOD_ID"]),
                      "W0_SEGMENT": tg["SEGMENT"].astype("string").str.strip().astype(object),
                      "W0_DATE": pd.to_datetime(tg["DATE1"], errors="coerce")})
    t = t.dropna(subset=["ACCESS_METHOD_ID"]).sort_values("W0_DATE", kind="stable").drop_duplicates("ACCESS_METHOD_ID", keep="last")
    t["W0_SEGMENT"] = t["W0_SEGMENT"].where(t["W0_SEGMENT"].notna(), None)
    t["W0_ACTION"] = t["W0_SEGMENT"].map(lambda s: config.DICT_SEGM_ACTIONS.get(s) if isinstance(s, str) else None)
    t = t.merge(lines[["ACCESS_METHOD_ID", "ACCESS_METHOD_VAL"]].drop_duplicates("ACCESS_METHOD_ID"),
                on="ACCESS_METHOD_ID", how="left")
    t["W0_REWARD"] = 0
    t["W0_BTL_REVENUE"] = 0.0
    return t[cols].reset_index(drop=True) if fulfil is None else _rewards(t, fulfil, catalog, run_date)[cols]


def _rewards(t: pd.DataFrame, fulfil: pd.DataFrame, catalog: pd.DataFrame | None, run_date: date) -> pd.DataFrame:
    from nbx.fsdr import calendar as cal

    w = cal.campaign_windows(run_date)["reward"]
    f = pd.DataFrame({"ACCESS_METHOD_VAL": norm_val(fulfil["ACCESS_METHOD_VAL"]),
                      "PRODUCTNAME": fulfil["PRODUCTNAME"].astype(str).str.strip(),
                      "DAY": pd.to_datetime(fulfil["FULFILMENTSUCCESSDATE_TIME"], errors="coerce").dt.date})
    f = f.dropna(subset=["ACCESS_METHOD_VAL", "DAY"])
    f = f[(f["DAY"] >= w.start) & (f["DAY"] <= w.end)]
    if catalog is not None:
        price = catalog.set_index("BTL_NAME")["PRICE"]
        f = f[f["PRODUCTNAME"].isin(price.index)]
        f = f.assign(PRICE=f["PRODUCTNAME"].map(price).astype("float64").fillna(0.0))
    else:
        f = f.assign(PRICE=0.0)
    by_line = f.groupby("ACCESS_METHOD_VAL")["PRICE"].agg(["size", "sum"])
    t = t.reset_index(drop=True)
    vals = t["ACCESS_METHOD_VAL"]
    t["W0_REWARD"] = vals.map(by_line["size"]).fillna(0).gt(0).astype("int8")
    t["W0_BTL_REVENUE"] = vals.map(by_line["sum"]).fillna(0.0).round(3)
    return t


def apply_original(seg: pd.DataFrame) -> pd.DataFrame:
    """ORIGINAL_SEGMENT, ACTION_ORIGINAL, ACTION_ORIGINAL_SOURCE from W0_SEGMENT (engine) or the crosswalk.
    An engine segment that the dictionary does not know (spelling) falls back to the crosswalk; an engine Nursering
    segment is kept (the line gets no original action, as in code 1)."""
    w0 = seg["W0_SEGMENT"] if "W0_SEGMENT" in seg.columns else pd.Series(None, index=seg.index, dtype=object)
    known = w0.map(lambda s: isinstance(s, str) and s in config.DICT_SEGM_ACTIONS) | is_nursering(w0)
    cross = seg["SEGMENT"].map(lambda s: config.SEGMENT_CROSSWALK[s][0])
    seg["ORIGINAL_SEGMENT"] = np.where(known, w0, cross).astype(object)
    seg["ACTION_ORIGINAL"] = seg["ORIGINAL_SEGMENT"].map(config.DICT_SEGM_ACTIONS).astype(object)
    seg["ACTION_ORIGINAL"] = seg["ACTION_ORIGINAL"].where(seg["ACTION_ORIGINAL"].notna(), None)
    seg["ACTION_ORIGINAL_SOURCE"] = np.where(known, ENGINE, CROSSWALK).astype(object)
    return seg


# ---------------------------------------------------------------------------
# Agent dictionary (dict_segm_actions_agent_vN)
# ---------------------------------------------------------------------------
def initial_agent_dictionary(run_date: date) -> dict:
    """v0 = the original dictionary through the crosswalk, for every S-segment x nationality group (70 cells)."""
    cells = []
    for _, seg, _ in config.S_RULES:
        orig = config.DICT_SEGM_ACTIONS.get(config.SEGMENT_CROSSWALK[seg][0])
        for nat in config.NATIONALITY_GROUPS:
            cells.append({"segment": seg, "nationality_group": nat, "excluded": orig is None,
                          "actions": [] if orig is None else
                          [{"action": orig, "rank": 1, "score": 1.0, "evidence_ids": ["B-03"],
                            "reason": "week 0: copy of the original dictionary"}]})
    return {"version": 0, "name": f"{config.AGENT_DICTIONARY_PREFIX}0", "created": run_date.isoformat(),
            "parent": None, "made_by": "code (copy of dict_segm_actions)", "cells": cells}


def validate_agent_dictionary(d: dict, previous: dict | None = None) -> tuple[dict, list[dict]]:
    """⑥ check of every cell: known segment and nationality, allowed actions only, excluded segments stay excluded,
    no aggressive upsell in an at-risk segment (H-06). An invalid cell is replaced by the previous version's cell
    (or v0's). Returns the valid dictionary and the problems found."""
    base = {(c["segment"], c["nationality_group"]): c
            for c in (previous or initial_agent_dictionary(date.fromisoformat(d.get("created", "2000-01-01"))))["cells"]}
    problems, cells, seen = [], [], set()
    for c in d.get("cells", []):
        key = (c.get("segment"), c.get("nationality_group"))
        why = []
        if key[0] not in config.S_SEGMENTS or key[1] not in config.NATIONALITY_GROUPS:
            why.append("unknown segment or nationality group")
        acts = [a.get("action") for a in c.get("actions", [])]
        if any(a not in config.ACTIONS for a in acts):
            why.append(f"action not allowed: {[a for a in acts if a not in config.ACTIONS]}")
        original = config.DICT_SEGM_ACTIONS.get(config.SEGMENT_CROSSWALK.get(key[0], (None,))[0])
        excluded = original is None
        if excluded and acts:
            why.append("excluded segment got an action (B-02)")
        if not excluded and not acts:
            why.append("no action for a targetable segment")
        if key[0] in AT_RISK_SEGMENTS and acts[:1] and acts[0] in UPSELL_ACTIONS and acts[0] != original:
            why.append("upsell as top action of an at-risk segment (H-06)")
        if len(acts) != len(set(acts)) or len(acts) > 3:
            why.append("ranked actions must be 1-3 distinct ids")
        if why:
            problems.append({"segment": key[0], "nationality_group": key[1], "problems": why})
            if key in base and key not in seen:
                cells.append(base[key])
                seen.add(key)
        elif key not in seen:
            cells.append(c)
            seen.add(key)
    for key, c in base.items():                                   # a missing cell keeps its previous value
        if key not in seen:
            cells.append(c)
            problems.append({"segment": key[0], "nationality_group": key[1], "problems": ["missing cell (kept previous)"]})
    return {**d, "cells": cells}, problems


def agent_lookup(d: dict) -> pd.DataFrame:
    rows = [{"SEGMENT": c["segment"], "NATIONALITY_GROUP": c["nationality_group"],
             "ACTION_AGENT": (c["actions"][0]["action"] if c["actions"] else None),
             "ACTION_AGENT_RANK2": (c["actions"][1]["action"] if len(c["actions"]) > 1 else None)} for c in d["cells"]]
    return pd.DataFrame(rows)


def apply_agent_dictionary(seg: pd.DataFrame, d: dict) -> pd.DataFrame:
    """ACTION_AGENT, ACTION_AGENT_RANK2, ACTION_AGENT_VERSION of every line, by (S-segment, nationality group)."""
    lk = agent_lookup(d)
    out = seg[["SEGMENT", "NATIONALITY_GROUP"]].merge(lk, on=["SEGMENT", "NATIONALITY_GROUP"], how="left")
    seg["ACTION_AGENT"] = out["ACTION_AGENT"].where(out["ACTION_AGENT"].notna(), None).to_numpy()
    seg["ACTION_AGENT_RANK2"] = out["ACTION_AGENT_RANK2"].where(out["ACTION_AGENT_RANK2"].notna(), None).to_numpy()
    seg["ACTION_AGENT_VERSION"] = d["name"]
    return seg


def dictionary_files(step_dir: Path) -> list[Path]:
    rx = re.compile(re.escape(config.AGENT_DICTIONARY_PREFIX) + r"(\d+)\.json$")
    files = [p for p in Path(step_dir).glob(f"{config.AGENT_DICTIONARY_PREFIX}*.json") if rx.search(p.name)]
    return sorted(files, key=lambda p: int(rx.search(p.name).group(1)))


def previous_agent_dictionary(runs_root: Path, run_date: date) -> tuple[dict | None, Path | None]:
    """The latest agent dictionary of an EARLIER run date (data/runs/<D'>/train/T3_segments/), if any."""
    root = Path(runs_root)
    dates = sorted(p.name for p in root.glob("20??-??-??") if p.is_dir() and p.name < run_date.isoformat())
    for d in reversed(dates):
        files = dictionary_files(root / d / "train" / "T3_segments")
        if files:
            return json.loads(files[-1].read_text(encoding="utf-8")), files[-1]
    return None, None


def action_agreement(original: pd.Series, agent: pd.Series, sent: pd.Series) -> np.ndarray:
    """§7.1 log field: all_same / sent_original / sent_agent / other."""
    o, a, s = (x.astype("string") for x in (original, agent, sent))
    so, sa = (s == o).fillna(False).to_numpy(bool), (s == a).fillna(False).to_numpy(bool)
    return np.select([so & sa, so, sa], ["all_same", "sent_original", "sent_agent"], "other").astype(object)


In [ ]:
%%engine nbx.fsdr.segmentation.bands
"""Bands and micro-segments (master prompt §8). Micro-segment = nationality group x age band x tenure band
(5 x 4 x 6 = 120 cells); value band and usage type are context in customer_360, not splits."""
from __future__ import annotations

import itertools

import numpy as np
import pandas as pd

from nbx.fsdr import config


def _bands(x: pd.Series, bands: list[tuple[str, float, float]], valid: pd.Series) -> np.ndarray:
    v = pd.to_numeric(x, errors="coerce")
    conds = [(valid & (v >= lo) & (v < hi)).fillna(False).to_numpy(dtype=bool) for _, lo, hi in bands]
    return np.select(conds, [b for b, _, _ in bands], "unknown").astype(object)


def age_band(age: pd.Series) -> np.ndarray:
    """young < 40 (incl. < 20), middle 40-59, senior 60+, unknown = missing or outside 15-100."""
    a = pd.to_numeric(age, errors="coerce")
    return _bands(a, config.AGE_BANDS, (a >= config.AGE_MIN) & (a <= config.AGE_MAX))


def young_split(age: pd.Series) -> np.ndarray:
    """Open item 5: young under 25 / 25-39 (tested in Phase 4, not a split yet)."""
    a = pd.to_numeric(age, errors="coerce")
    young = (a >= config.AGE_MIN) & (a < config.AGE_BANDS[0][2])
    return np.where(young & (a < config.YOUNG_SPLIT_AGE), "lt25", np.where(young, "25_39", None)).astype(object)


def tenure_band(days: pd.Series) -> np.ndarray:
    """t0_lt3m < 90 d, t1_3_6m, t2_6_12m, t3_1_2y, t4_gt2y > 730 d, unknown = missing or negative."""
    d = pd.to_numeric(days, errors="coerce")
    return _bands(d, config.TENURE_BANDS, d >= 0)


def micro_segment(nat: pd.Series, age_b, tenure_b) -> pd.Series:
    return (pd.Series(nat, dtype="string").fillna(config.NATIONALITY_OTHERS) + "|" + pd.Series(age_b, index=nat.index, dtype="string")
            + "|" + pd.Series(tenure_b, index=nat.index, dtype="string")).astype(object)


def all_micro_segments() -> list[tuple[str, str, str]]:
    return list(itertools.product(config.NATIONALITY_GROUPS, config.AGE_BAND_NAMES, config.TENURE_BAND_NAMES))


def value_band(arpu: pd.Series, cutoffs: list[float]) -> np.ndarray:
    """zero (ARPU_M1 = 0), then low / mid / high = tertiles of the paying lines over the whole base."""
    a = pd.to_numeric(arpu, errors="coerce").fillna(0.0)
    lo, hi = cutoffs
    return np.select([a <= 0, a <= lo, a <= hi], ["zero", "low", "mid"], "high").astype(object)


def usage_type(voice_user: pd.Series, data_user: pd.Series) -> np.ndarray:
    v, d = voice_user.astype(bool), data_user.astype(bool)
    return np.select([v & d, d, v], ["dual", "data_only", "voice_only"], "none").astype(object)


In [ ]:
%%engine nbx.fsdr.segmentation.rules
"""S-rules (master prompt §7): the current segment of every line, first match wins, thresholds in config.

    seg = apply_s_rules(features)      # adds RULE_ID, SEGMENT, RULE_ACTION, EXCLUDED and the trigger values

B-05: a missing score never triggers "at risk" (nor reconnect, fragility or appetency): the triggers read a score
only where its *_MISSING flag is 0, so the imputed median cannot move a line into a risk segment.
An unknown tenure is not "new": it never matches S-1 … S-4.
"""
from __future__ import annotations

import numpy as np
import pandas as pd

from nbx.fsdr import config

TRIGGERS = ["S_REV_M1", "S_RGS", "S_VOICE_USER", "S_DATA_USER", "S_AT_RISK", "S_RECONNECT_P", "S_FRAGILITY",
            "S_REV_M1_M2_PCT"]


def _observed(df: pd.DataFrame, feature: str) -> pd.Series:
    """The score where it was observed, 0 where it was missing (or absent)."""
    if feature not in df.columns:
        return pd.Series(0.0, index=df.index)
    s = pd.to_numeric(df[feature], errors="coerce").fillna(0.0)
    flag = f"{feature}{config.MISSING_FLAG_SUFFIX}"
    return s.where(df[flag] == 0, 0.0) if flag in df.columns else s


def _col(df: pd.DataFrame, name: str) -> pd.Series:
    return pd.to_numeric(df[name], errors="coerce").fillna(0.0) if name in df.columns else pd.Series(0.0, index=df.index)


def apply_s_rules(df: pd.DataFrame) -> pd.DataFrame:
    """Returns a frame aligned on df with the segment columns and the values that triggered them."""
    c = config
    tenure = pd.to_numeric(df["TENURE_DAYS"], errors="coerce")
    churn, churn_1d = _observed(df, "PROBA_CHURN"), _observed(df, "PROBA_CHURN_1D")
    rev_m1 = _col(df, "ARPU_M1")
    rgs = (rev_m1 > 0) & (_col(df, "ACTIVE_DAYS_COUNT_M1") > 0)
    voice = _col(df, "LOCAL_VOICE_MIN_M1") >= c.S_VOICE_USER_MIN
    data = _col(df, "DATA_VOLUME_MB_M1") >= c.S_DATA_USER_MB
    reconnect = np.maximum(_observed(df, "PROBA_REACT_30"), _observed(df, "PROBA_REACT_15"))
    fragility = np.maximum.reduce([_observed(df, "PROBA_FRAG_DATA"), _observed(df, "PROBA_FRAG_VOICE"),
                                   _observed(df, "PROBA_REV_DECR")])
    rev_change = _col(df, "VAR_REV_M1_M2")
    at_risk = churn >= c.S_CHURN_AT_RISK
    new_45 = (tenure < c.S_TENURE_NURSERING).fillna(False)
    new_90 = (tenure < c.S_TENURE_ONBOARDING).fillna(False)
    conds = [
        new_45 & at_risk, new_45,                                                       # S-1, S-2
        new_90 & at_risk, new_90,                                                       # S-3, S-4
        ~rgs & (reconnect >= c.S_RECONNECT_HIGH), ~rgs & (reconnect >= c.S_RECONNECT_MED), ~rgs,   # S-5 … S-7
        at_risk | (churn_1d >= c.S_CHURN_1D_AT_RISK),                                   # S-8 (RGS from here)
        (fragility >= c.S_FRAGILITY) | (rev_change <= c.S_REV_DROP_PCT),                # S-9
        voice & data,                                                                   # S-10
        data & (_observed(df, "PROBA_APP_VOICE") >= c.S_APPETENCY), data,               # S-11, S-12
        _observed(df, "PROBA_APP_DATA") >= c.S_APPETENCY,                               # S-13 (voice only or none)
    ]
    idx = np.select([np.asarray(x, dtype=bool) for x in conds], list(range(len(conds))), default=len(conds))  # S-14
    rules = c.S_RULES
    out = pd.DataFrame(index=df.index)
    out["RULE_ID"] = np.array([r[0] for r in rules], dtype=object)[idx]
    out["SEGMENT"] = np.array([r[1] for r in rules], dtype=object)[idx]
    out["RULE_ACTION"] = np.array([r[2] for r in rules], dtype=object)[idx]
    out["EXCLUDED"] = out["RULE_ACTION"].isna().astype("int8")
    out["SEGMENT_SOURCE"] = "s_rules"
    out["S_REV_M1"] = rev_m1.round(4)
    out["S_RGS"] = rgs.astype("int8")
    out["S_VOICE_USER"] = voice.astype("int8")
    out["S_DATA_USER"] = data.astype("int8")
    out["S_AT_RISK"] = at_risk.astype("int8")
    out["S_RECONNECT_P"] = np.round(reconnect, 6)
    out["S_FRAGILITY"] = np.round(fragility, 6)
    out["S_REV_M1_M2_PCT"] = rev_change.round(4)
    return out


def segment_family(segment: pd.Series) -> pd.Series:
    """Nursering / Onboarding / Retention / RGS (for the tenure-agreement report)."""
    s = segment.astype("string")
    return pd.Series(np.select([s.str.startswith("Nursering"), s.str.startswith("Onboarding"),
                                s.str.startswith("Retention")], ["Nursering", "Onboarding", "Retention"], "RGS"),
                     index=segment.index)


In [ ]:
%%engine nbx.fsdr.pipelines.training
"""V2.0 training pipeline, stages T2 and T3 (plan §14, §15; master prompt §6 - §8, §18).

    run_t2(D)        T2_features   base_features/part-NN.parquet (B-01 … B-12) for the full base
    run_t3(D)        T3_segments   segments/part-NN.parquet: S-rule segment, bands, micro-segment, week-0 history,
                                   action_original + crosswalk, action_agent (agent dictionary), and the reports
    run_features(D)  T2 then T3 (fsdr-features)

Both stages read T0's Parquet cache and run partition by partition, `workers` partitions in parallel
(FSDR_FEATURE_WORKERS: 2 on the dev laptop, 8 on the P16). Each worker writes its own partition file and returns
small count tables, which the parent sums: nothing of size 1M is ever gathered in one process except the few
columns T2 needs for the B-05 medians and the ARPU tertiles.

Resume: a stage whose _step.json is done with the same inputs (the cache manifest, the previous stage, the rule
constants) is skipped; force=True re-runs it.
"""
from __future__ import annotations

import hashlib
import json
from concurrent.futures import ProcessPoolExecutor
from datetime import date
from pathlib import Path

import pandas as pd

from nbx.fsdr import base_features as bf
from nbx.fsdr import config
from nbx.fsdr.contract import data_agent
from nbx.fsdr.contract.cache import CACHE_MANIFEST, CacheSource, cache_dir
from nbx.fsdr.offer_rules import load_catalog
from nbx.fsdr.segmentation import actions, bands, rules
from nbx.fsdr.steps import Step, fingerprint, read_step, runs_dir, sha256_file

T2, T3 = "T2_features", "T3_segments"
SEGMENT_COLUMNS = ["ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "RULE_ID", "SEGMENT", "RULE_ACTION", "EXCLUDED",
                   "SEGMENT_SOURCE", *rules.TRIGGERS, "NATIONALITY_GROUP", "AGE", "AGE_BAND", "YOUNG_SPLIT",
                   "TENURE_DAYS", "TENURE_BAND", "MICRO_SEGMENT", "ARPU_M1", "VALUE_BAND", "USAGE_TYPE", "REGION",
                   "DV_SEGMENT", "W0_SEGMENT", "W0_DATE", "W0_TARGETED", "W0_ACTION", "W0_REWARD", "W0_BTL_REVENUE",
                   "ORIGINAL_SEGMENT", "ACTION_ORIGINAL", "ACTION_ORIGINAL_SOURCE", "ACTION_AGENT",
                   "ACTION_AGENT_RANK2", "ACTION_AGENT_VERSION"]


class StageBlocked(RuntimeError):
    """A stage cannot start (its previous stage is missing or blocked)."""


def _iso(d: date | str) -> str:
    return d.isoformat() if isinstance(d, date) else str(d)


def _run(fn, jobs: list, workers: int) -> list:
    if workers <= 1 or len(jobs) <= 1:
        return [fn(j) for j in jobs]
    with ProcessPoolExecutor(max_workers=min(workers, len(jobs))) as ex:
        return list(ex.map(fn, jobs))


def rules_version() -> str:
    """Hash of every constant the S-rules, bands and dictionaries depend on (a change re-runs T3)."""
    keys = [k for k in dir(config) if k.startswith(("S_", "AGE_", "TENURE_", "YOUNG_", "VALUE_", "USAGE_", "NATIONALITY_"))]
    keys += ["DICT_SEGM_ACTIONS_SOURCE", "SEGMENT_CROSSWALK", "ACTIONS"]
    blob = json.dumps({k: getattr(config, k) for k in sorted(keys)}, sort_keys=True, default=str)
    return hashlib.sha256(blob.encode()).hexdigest()[:12]


def _t0(run_date: date, runs_root: Path | None) -> dict:
    t0 = data_agent.load_summary(run_date, runs_root)
    if not t0:
        raise StageBlocked(f"no T0_load for {run_date}: run fsdr-load first")
    if t0.get("status") != "done":
        raise StageBlocked(f"T0_load of {run_date} is {t0.get('status')}: fix the data, re-run fsdr-load")
    return t0


def _sum_counts(frames: list[pd.DataFrame], by: list[str], value: list[str] | None = None) -> pd.DataFrame:
    frames = [f for f in frames if f is not None and len(f)]
    if not frames:
        return pd.DataFrame(columns=by + (value or ["lines"]))
    return pd.concat(frames, ignore_index=True).groupby(by, dropna=False, as_index=False)[value or ["lines"]].sum()


# ---------------------------------------------------------------------------
# T2 features
# ---------------------------------------------------------------------------
def _t2_pass1(job: tuple) -> dict:
    cache_root, run_date, part, anchor, out = job
    cache = CacheSource(run_date, Path(cache_root))
    df, stats = bf.partition_features(cache, part, date.fromisoformat(run_date), anchor)
    df.to_parquet(out, index=False)
    return stats


def _t2_pass2(job: tuple) -> pd.DataFrame:
    path, medians = job
    df = pd.read_parquet(path)
    bf.impute(df, medians)
    df.to_parquet(path, index=False)
    return bf.column_stats(df)


def run_t2(run_date: date, runs_root: Path | None = None, cache_root: Path | None = None, workers: int | None = None,
           force: bool = False) -> dict:
    t0 = _t0(run_date, runs_root)
    anchor = bf.anchor_of(t0["pipeline"])
    cache = CacheSource(run_date, cache_root)
    workers = int(workers or config.FEATURE_WORKERS)
    inputs = fingerprint({"cache": sha256_file(cache_dir(run_date, cache_root) / CACHE_MANIFEST), "anchor": anchor,
                          "registry": [m.table for m in config.MODEL_SCORE_REGISTRY], "trend": config.TREND_PAIRS})
    with Step(runs_dir(run_date, runs_root) / "train", T2, inputs=inputs, force=force) as st:
        if st.done:
            return {**st.read_json("summary.json"), "skipped": True}
        parts = list(range(cache.partitions))
        outs = [st.output(f"base_features/part-{k:02d}.parquet") for k in parts]
        stats = _run(_t2_pass1, [(str(cache_dir(run_date, cache_root).parent), _iso(run_date), k, anchor, str(outs[k]))
                                 for k in parts], workers)
        medians = bf.score_medians(outs)                                       # B-05, whole base
        cutoffs = bf.arpu_cutoffs(outs)
        col_stats = _run(_t2_pass2, [(str(o), medians) for o in outs], workers)

        cs = pd.concat(col_stats).groupby(level=0).agg({"count": "sum", "nulls": "sum", "sum": "sum", "min": "min", "max": "max"})
        cs["mean"] = (cs["sum"] / cs["count"].where(cs["count"] > 0)).round(6)
        st.write_csv(cs.drop(columns=["sum"]).reset_index(names="feature"), "feature_summary.csv")
        st.write_csv(pd.read_parquet(outs[0]).head(config.STEP_PREVIEW_ROWS), "base_features_preview.csv")
        vals = pd.concat([pd.read_parquet(o, columns=["ACCESS_METHOD_VAL"]) for o in outs])["ACCESS_METHOD_VAL"]
        lines = int(sum(s["lines"] for s in stats))
        missing = {f: int(sum(s["missing_scores"][f] for s in stats)) for f in config.proba_features()}
        src_counts = {k: _merge_counts([s[k] for s in stats]) for k in ("age_source", "tenure_source")}
        stats_doc = {"medians_by_nationality": medians, "arpu_tertiles": cutoffs, "anchor": anchor,
                     "party_table": bf.party_table(anchor, cache)}
        st.write_json("stats.json", stats_doc)
        summary = {"run_date": _iso(run_date), "stage": T2, "pipeline": t0["pipeline"], "anchor": anchor,
                   "tables": "*_auto (week 0)" if anchor == "T" else "*_score (week n)",
                   "base_rows": int(sum(s["base_rows"] for s in stats)), "lines": lines,
                   "eligible_filters": {k: int(sum(s[k] for s in stats)) for k in
                                        ("null_keys", "excluded_plan", "bad_msisdn", "duplicates_removed")},
                   "msisdn_in_two_partitions": int(vals.duplicated().sum()),
                   "features": int(len(cs)), "missing_scores": missing,
                   "missing_share": {f: round(n / lines, 4) if lines else 0.0 for f, n in missing.items()},
                   **src_counts, "no_nationality": int(sum(s["no_nationality"] for s in stats)),
                   "arpu_tertiles": cutoffs, "partitions": len(parts), "workers": workers, "step_dir": str(st.dir)}
        st.rows(rows_in=summary["base_rows"], rows_out=lines)
        st.note(anchor=anchor, workers=workers)
        st.write_json("summary.json", summary)
        return summary


def _merge_counts(dicts: list[dict]) -> dict:
    out: dict = {}
    for d in dicts:
        for k, v in d.items():
            out[k] = out.get(k, 0) + int(v)
    return out


# ---------------------------------------------------------------------------
# T3 segments and the two actions
# ---------------------------------------------------------------------------
def segment_partition(feat: pd.DataFrame, cache, part: int, run_date: date, cutoffs: list[float],
                      agent_dict: dict, catalog: pd.DataFrame | None) -> pd.DataFrame:
    """Every T3 column of one partition (pure function of the features, the cache partition and the dictionary)."""
    seg = pd.concat([feat[["ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "NATIONALITY_GROUP", "AGE", "TENURE_DAYS",
                           "ARPU_M1", "REGION", "DV_SEGMENT"]], rules.apply_s_rules(feat)], axis=1)
    seg["AGE_BAND"] = bands.age_band(seg["AGE"])
    seg["YOUNG_SPLIT"] = bands.young_split(seg["AGE"])
    seg["TENURE_BAND"] = bands.tenure_band(seg["TENURE_DAYS"])
    seg["MICRO_SEGMENT"] = bands.micro_segment(seg["NATIONALITY_GROUP"], seg["AGE_BAND"], seg["TENURE_BAND"])
    seg["VALUE_BAND"] = bands.value_band(seg["ARPU_M1"], cutoffs)
    seg["USAGE_TYPE"] = bands.usage_type(seg["S_VOICE_USER"], seg["S_DATA_USER"])

    tg = bf._part(cache, "targeting_dataiku_auto", part)
    ful = bf._part(cache, "subs_btl_auto", part)
    w0 = actions.week0_targeting(tg, ful, catalog, run_date, seg)
    seg = seg.merge(w0.drop(columns=["ACCESS_METHOD_VAL"], errors="ignore"), on="ACCESS_METHOD_ID", how="left")
    seg["W0_TARGETED"] = seg["W0_DATE"].notna().astype("int8")
    # numeric first: without targeting_dataiku_auto (a weekly run) the merged columns are empty object columns
    seg["W0_REWARD"] = pd.to_numeric(seg["W0_REWARD"], errors="coerce").fillna(0).astype("int8")
    seg["W0_BTL_REVENUE"] = pd.to_numeric(seg["W0_BTL_REVENUE"], errors="coerce").fillna(0.0)
    seg = actions.apply_original(seg)
    seg = actions.apply_agent_dictionary(seg, agent_dict)
    return seg[SEGMENT_COLUMNS]


def _counts(seg: pd.DataFrame) -> dict[str, pd.DataFrame]:
    def g(by, **agg):
        d = seg.assign(lines=1, **{k: seg[v] for k, v in agg.items()})
        return d.groupby(by, dropna=False, as_index=False)[["lines", *agg]].sum()
    s = seg.assign(W0_SEGMENT=seg["W0_SEGMENT"].fillna("(not targeted)"),
                   ACTION_ORIGINAL=seg["ACTION_ORIGINAL"].fillna("(none)"), ACTION_AGENT=seg["ACTION_AGENT"].fillna("(none)"),
                   RULE_ACTION=seg["RULE_ACTION"].fillna("excluded"), W0_ACTION=seg["W0_ACTION"].fillna("(none)"),
                   YOUNG_SPLIT=seg["YOUNG_SPLIT"].fillna("-"), FAMILY=rules.segment_family(seg["SEGMENT"]))
    seg = s
    return {
        "segments": g(["RULE_ID", "SEGMENT", "RULE_ACTION"], arpu_sum="ARPU_M1", w0_targeted="W0_TARGETED", w0_takers="W0_REWARD"),
        "micro": g(["NATIONALITY_GROUP", "AGE_BAND", "TENURE_BAND"]),
        "agreement": g(["SEGMENT", "W0_SEGMENT"]),
        "tenure": g(["FAMILY", "TENURE_BAND"]),
        "actions": g(["ACTION_ORIGINAL_SOURCE", "ACTION_ORIGINAL", "ACTION_AGENT"]),
        "context": g(["VALUE_BAND", "USAGE_TYPE"]),
        "young": g(["NATIONALITY_GROUP", "YOUNG_SPLIT"], takers="W0_REWARD", targeted="W0_TARGETED"),
        "w0": g(["W0_ACTION"], targeted="W0_TARGETED", takers="W0_REWARD", btl_revenue="W0_BTL_REVENUE"),
    }


def _t3_part(job: tuple) -> dict:
    feat_path, cache_root, run_date, part, cutoffs, agent_dict, out = job
    cache = CacheSource(run_date, Path(cache_root))
    catalog = load_catalog(cache) if cache.has("btl_catalog") else None
    seg = segment_partition(pd.read_parquet(feat_path), cache, part, date.fromisoformat(run_date), cutoffs, agent_dict, catalog)
    seg.to_parquet(out, index=False)
    return _counts(seg)


def run_t3(run_date: date, runs_root: Path | None = None, cache_root: Path | None = None, workers: int | None = None,
           force: bool = False) -> dict:
    _t0(run_date, runs_root)
    train = runs_dir(run_date, runs_root) / "train"
    t2 = read_step(train / T2)
    if t2.get("status") != "done":
        raise StageBlocked(f"T2_features of {run_date} is {t2.get('status', 'missing')}: run fsdr-features")
    workers = int(workers or config.FEATURE_WORKERS)
    root = Path(runs_root or config.FSDR_RUNS_DIR)
    prev_dict, prev_path = actions.previous_agent_dictionary(root, run_date)
    inputs = fingerprint({"t2": t2.get("inputs"), "t2_started": t2.get("started"), "rules": rules_version(),
                          "previous_dictionary": str(prev_path or "")})
    with Step(train, T3, inputs=inputs, force=force) as st:
        if st.done:
            return {**st.read_json("summary.json"), "skipped": True}
        cache = CacheSource(run_date, cache_root)
        cutoffs = json.loads((train / T2 / "stats.json").read_text(encoding="utf-8"))["arpu_tertiles"]
        agent_dict, problems = actions.validate_agent_dictionary(prev_dict or actions.initial_agent_dictionary(run_date), prev_dict)
        for f in actions.dictionary_files(st.dir):          # an agent-2b version of an earlier T3 run of this date:
            if f.name != f"{agent_dict['name']}.json":     # the segments are rewritten with agent_dict, so drop it
                f.unlink()
        st.write_json(f"{agent_dict['name']}.json", agent_dict)
        st.write_json("agent_dictionary_check.json", {"version": agent_dict["name"], "source": str(prev_path or "v0 (week 0)"),
                                                      "problems": problems})
        st.write_json("dict_segm_actions_original.json", {"source_order": config.DICT_SEGM_ACTIONS_SOURCE,
                                                          "effective (last value wins)": config.DICT_SEGM_ACTIONS})
        crosswalk = actions.crosswalk_table()
        st.write_csv(crosswalk, "crosswalk.csv")

        parts = list(range(cache.partitions))
        outs = [st.output(f"segments/part-{k:02d}.parquet") for k in parts]
        jobs = [(str(train / T2 / "base_features" / f"part-{k:02d}.parquet"), str(cache_dir(run_date, cache_root).parent),
                 _iso(run_date), k, cutoffs, agent_dict, str(outs[k])) for k in parts]
        counts = _run(_t3_part, jobs, workers)
        summary = _t3_reports(st, counts, run_date, crosswalk, agent_dict)
        st.write_csv(pd.read_parquet(outs[0]).head(config.STEP_PREVIEW_ROWS), "segments_preview.csv")
        summary.update({"run_date": _iso(run_date), "stage": T3, "rules_version": rules_version(),
                        "agent_dictionary": agent_dict["name"], "agent_dictionary_problems": len(problems),
                        "partitions": len(parts), "workers": workers, "step_dir": str(st.dir)})
        st.rows(rows_in=summary["lines"], rows_out=summary["lines"])
        st.note(rules_version=rules_version(), agent_dictionary=agent_dict["name"])
        st.write_json("summary.json", summary)
        return summary


def current_agent_dictionary(run_date: date, runs_root: Path | None = None) -> dict:
    """The agent dictionary T3 of this run date applied last (the highest version in its folder)."""
    t3 = runs_dir(run_date, runs_root) / "train" / T3
    files = actions.dictionary_files(t3)
    if not files:
        raise StageBlocked(f"T3_segments of {run_date} has no agent dictionary: run fsdr-features first")
    return json.loads(files[-1].read_text(encoding="utf-8"))


def update_agent_dictionary(run_date: date, proposal: dict, runs_root: Path | None = None,
                            made_by: str = "agent 2b") -> dict:
    """Agent 2b (V4.0): a new version of the agent dictionary for this run date. The proposal (cells as in v0) is
    checked by validate_agent_dictionary against the current version (an invalid cell keeps its current value),
    saved as dict_segm_actions_agent_v<N+1>.json in T3_segments/, and applied to every segment partition
    (ACTION_AGENT, ACTION_AGENT_RANK2, ACTION_AGENT_VERSION), so the panel and Target 2 use it. Next week's T3
    starts from this version. Returns the version, the changed cells and the problems."""
    t3 = runs_dir(run_date, runs_root) / "train" / T3
    if read_step(t3).get("status") != "done":
        raise StageBlocked(f"T3_segments of {run_date} is not done: run fsdr-features first")
    current = current_agent_dictionary(run_date, runs_root)
    version = int(current["version"]) + 1
    draft = {**proposal, "version": version, "name": f"{config.AGENT_DICTIONARY_PREFIX}{version}",
             "created": _iso(run_date), "parent": current["name"], "made_by": made_by}
    new, problems = actions.validate_agent_dictionary(draft, current)
    old = {(c["segment"], c["nationality_group"]): [a["action"] for a in c["actions"]] for c in current["cells"]}
    changed = [{"segment": c["segment"], "nationality_group": c["nationality_group"],
                "before": old.get((c["segment"], c["nationality_group"])), "after": [a["action"] for a in c["actions"]],
                "reason": (c["actions"][0].get("reason", "") if c["actions"] else "")}
               for c in new["cells"] if old.get((c["segment"], c["nationality_group"])) != [a["action"] for a in c["actions"]]]
    (t3 / f"{new['name']}.json").write_text(json.dumps(new, indent=2, default=str), encoding="utf-8")
    (t3 / f"agent_dictionary_check_v{version}.json").write_text(
        json.dumps({"version": new["name"], "parent": current["name"], "made_by": made_by, "changed": changed,
                    "problems": problems}, indent=2, default=str), encoding="utf-8")
    lines = 0
    for part in sorted((t3 / "segments").glob("part-*.parquet")):
        seg = actions.apply_agent_dictionary(pd.read_parquet(part), new)
        seg.to_parquet(part, index=False)
        lines += len(seg)
    return {"version": new["name"], "parent": current["name"], "changed_cells": changed, "problems": problems,
            "lines": lines}


def _t3_reports(st: Step, counts: list[dict], run_date: date, crosswalk: pd.DataFrame, agent_dict: dict) -> dict:
    get = lambda k: [c[k] for c in counts]                                        # noqa: E731
    seg = _sum_counts(get("segments"), ["RULE_ID", "SEGMENT", "RULE_ACTION"], ["lines", "arpu_sum", "w0_targeted", "w0_takers"])
    order = {r: i for i, (r, _, _) in enumerate(config.S_RULES)}
    seg = seg.sort_values("RULE_ID", key=lambda s: s.map(order)).reset_index(drop=True)
    lines = int(seg["lines"].sum())
    seg["share"] = (seg["lines"] / max(lines, 1)).round(4)
    seg["avg_arpu_m1"] = (seg["arpu_sum"] / seg["lines"].where(seg["lines"] > 0)).round(3)
    st.write_csv(seg.drop(columns=["arpu_sum"]), "segment_counts.csv")

    micro = _sum_counts(get("micro"), ["NATIONALITY_GROUP", "AGE_BAND", "TENURE_BAND"])
    grid = pd.DataFrame(bands.all_micro_segments(), columns=["NATIONALITY_GROUP", "AGE_BAND", "TENURE_BAND"])
    micro = grid.merge(micro, how="left").fillna({"lines": 0})
    micro["lines"] = micro["lines"].astype(int)
    micro["MICRO_SEGMENT"] = micro["NATIONALITY_GROUP"] + "|" + micro["AGE_BAND"] + "|" + micro["TENURE_BAND"]
    st.write_csv(micro[["MICRO_SEGMENT", "NATIONALITY_GROUP", "AGE_BAND", "TENURE_BAND", "lines"]], "microsegment_counts.csv")

    agree = _sum_counts(get("agreement"), ["SEGMENT", "W0_SEGMENT"])
    cw = {s: config.SEGMENT_CROSSWALK[s][0].lower() for s in config.S_SEGMENTS}
    targeted = agree[agree["W0_SEGMENT"] != "(not targeted)"]
    match = targeted["SEGMENT"].map(cw) == targeted["W0_SEGMENT"].astype(str).str.lower()
    agree["match"] = (agree["SEGMENT"].map(cw) == agree["W0_SEGMENT"].astype(str).str.lower())
    st.write_csv(agree.sort_values(["SEGMENT", "lines"], ascending=[True, False]), "segment_agreement.csv")

    ten = _sum_counts(get("tenure"), ["FAMILY", "TENURE_BAND"])
    st.write_csv(ten.pivot_table(index="FAMILY", columns="TENURE_BAND", values="lines", aggfunc="sum", fill_value=0)
                   .astype(int).reset_index(), "tenure_agreement.csv")
    acts = _sum_counts(get("actions"), ["ACTION_ORIGINAL_SOURCE", "ACTION_ORIGINAL", "ACTION_AGENT"])
    st.write_csv(acts, "action_counts.csv")
    st.write_csv(_sum_counts(get("context"), ["VALUE_BAND", "USAGE_TYPE"]), "value_usage.csv")
    young = _sum_counts(get("young"), ["NATIONALITY_GROUP", "YOUNG_SPLIT"], ["lines", "takers", "targeted"])
    st.write_csv(young[young["YOUNG_SPLIT"] != "-"], "young_split.csv")

    w0 = _sum_counts(get("w0"), ["W0_ACTION"], ["lines", "targeted", "takers", "btl_revenue"])
    acted = w0[w0["W0_ACTION"].isin(config.ACTIONS)]
    n = int(acted["targeted"].sum())
    recomputed = {a: round(float(acted.loc[acted["W0_ACTION"] == a, "targeted"].sum()) / n, 7) if n else None
                  for a in config.ACTIONS}
    take = {a: (round(float(r["takers"]) / r["targeted"], 4) if r["targeted"] else None)
            for a, r in acted.set_index("W0_ACTION").iterrows()}
    priors = {"rule": "B-10", "configured": config.ACTION_PRIOR, "recomputed_week0": recomputed,
              "targeted_with_action": n, "take_rate_week0": take, "order": config.ACTIONS,
              "used": "recomputed_week0" if n else "configured"}
    st.write_json("priors.json", priors)

    excluded = int(seg.loc[seg["RULE_ACTION"] == "excluded", "lines"].sum())
    src = acts.groupby("ACTION_ORIGINAL_SOURCE")["lines"].sum().to_dict()
    same = acts[acts["ACTION_ORIGINAL"] == acts["ACTION_AGENT"]]["lines"].sum()
    non_empty = int((micro["lines"] > 0).sum())
    return {"lines": lines, "excluded_nursering": excluded, "targetable": lines - excluded,
            "segments_non_empty": int((seg["lines"] > 0).sum()),
            "segment_lines": dict(zip(seg["SEGMENT"], seg["lines"].astype(int))),
            "micro_segments_non_empty": non_empty, "micro_segments_total": len(micro),
            "micro_segments_ge_300": int((micro["lines"] >= 300).sum()),
            "engine_targeted_lines": int(targeted["lines"].sum()),
            "engine_vs_s_rules_agreement": round(float(targeted.loc[match, "lines"].sum() / max(targeted["lines"].sum(), 1)), 4),
            "action_original_source": {k: int(v) for k, v in src.items()},
            "action_original_eq_agent": round(float(same / max(lines, 1)), 4),
            "crosswalk_same_action": int(crosswalk["same_action"].sum()), "crosswalk_lines": len(crosswalk),
            "priors_used": priors["used"], "week0_targeted_with_action": n}


def run_features(run_date: date, runs_root: Path | None = None, cache_root: Path | None = None,
                 workers: int | None = None, force: bool = False) -> dict:
    """fsdr-features: T2 then T3 (force re-runs both)."""
    t2 = run_t2(run_date, runs_root, cache_root, workers, force)
    t3 = run_t3(run_date, runs_root, cache_root, workers, force)
    return {"T2_features": t2, "T3_segments": t3}


In [ ]:
%%engine nbx.fsdr.targets
"""Target upload folders (V4.0): the lists of the four groups and the control, chosen outside the engine. They sit
in the scoring folder of the run, one folder per target and one sub-folder per group (a = constraint, b = free):

    data/scoring/<D>/
    ├── target1/a_constraint/target1_a.csv.gz    T1C  production code 3
    ├── target1/b_free/target1_b.csv.gz          T1F  production code 4
    ├── target2/a_constraint/target2_a.csv.gz    T2C  challenger CC-v1
    ├── target2/b_free/target2_b.csv.gz          T2F  challenger CF-v1
    ├── control/control.csv.gz                   never contacted
    └── _targets_manifest.json                   written by the generator (source, sizes, seed)

One row per line, at least `access_method_val` (or `msisdn`); any other column (the sample_cg_tg fields: stratum
`Final`, plan, nationality, …) is kept and carried to the details file. `.csv.gz` in production; `.csv`, `.json`,
`.jsonl`, `.parquet` for tests. When the folder holds lists, P1 uses them instead of drawing the panel.

Checks (read_targets + check_targets), each with a count and examples:
  blocking  a line in two groups; no list at all for a contacted group; no access_method_val column
  dropped   not an 8-digit MSISDN; not in the eligible base; Nursering (B-02); in another campaign this week (H-05)
  warning   a duplicate inside one list; a size off the configured size by more than 2 %
"""
from __future__ import annotations

import hashlib
import json
from pathlib import Path

import pandas as pd

from nbx.fsdr import config, csvstd
from nbx.fsdr.features import norm_val

VAL = "ACCESS_METHOD_VAL"
FORMATS = (csvstd.SUFFIX, ".csv", ".json", ".jsonl", ".parquet")


def targets_dir(input_folder: Path) -> Path:
    return Path(input_folder) / config.TARGETS_FOLDER


def group_file(folder: Path, key: str) -> Path | None:
    stem = targets_dir(folder) / config.TARGET_FILES[key]
    return next((stem.with_name(stem.name + f) for f in FORMATS if stem.with_name(stem.name + f).exists()), None)


def list_files(input_folder: Path) -> list[Path]:
    """The list files present (+ the manifest): what P1 fingerprints (never the whole scoring folder)."""
    files = [group_file(input_folder, k) for k in config.TARGET_FILES]
    man = targets_dir(input_folder) / config.TARGETS_MANIFEST
    return sorted([f for f in files if f is not None] + ([man] if man.exists() else []))


def has_targets(input_folder: Path) -> bool:
    return any(group_file(input_folder, k) for k in config.TARGET_FILES)


def _sha(path: Path) -> str:
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def dummy_lists(input_folder: Path) -> bool:
    """True when the lists present are exactly the ones the generator wrote: its manifest is there and every list
    file still has the checksum it recorded. A list copied over a dummy one (your real list) makes this False, even
    if the old manifest was left behind."""
    m = targets_dir(input_folder) / config.TARGETS_MANIFEST
    try:
        man = json.loads(m.read_text(encoding="utf-8"))
    except (OSError, ValueError):
        return False
    if man.get("source") != "generator":
        return False
    sums = man.get("sha256")
    if sums is None:                # an older manifest, without checksums: cannot prove the lists are dummy ones,
        return False                # so they are treated as real lists (kept; configured sizes)
    for key in config.TARGET_FILES:
        f = group_file(input_folder, key)
        if f is not None and sums.get(key) != _sha(f):
            return False
    return True


def expected_sizes(input_folder: Path) -> dict[str, int]:
    """Sizes to check the lists against: the generator's own sizes for dummy lists, else the configured sizes."""
    if not dummy_lists(input_folder):
        return config.group_sizes()
    man = json.loads((targets_dir(input_folder) / config.TARGETS_MANIFEST).read_text(encoding="utf-8"))
    return {k: int(v) for k, v in man["sizes"].items()}


def _read(path: Path) -> pd.DataFrame:
    fmt = csvstd.file_format(path)
    if fmt in (csvstd.SUFFIX, ".csv"):
        return csvstd.read_csv(path)
    if fmt == ".parquet":
        return pd.read_parquet(path)
    return pd.read_json(path, lines=fmt == ".jsonl", dtype=False)


def read_targets(input_folder: Path) -> tuple[pd.DataFrame, list[dict]]:
    """All the lists -> one frame (ACCESS_METHOD_VAL, GROUP_KEY, + the file's own columns, prefixed T_)."""
    frames, issues = [], []
    keys = list(config.group_sizes())
    for key in keys:
        p = group_file(input_folder, key)
        if p is None:
            issues.append({"level": "blocking", "group": key, "check": "missing_list",
                           "message": f"no list {config.TARGET_FILES[key]}.csv.gz", "count": 0})
            continue
        df = _read(p)
        cols = {c.lower(): c for c in df.columns}
        idc = cols.get("access_method_val") or cols.get("msisdn")
        if idc is None:
            issues.append({"level": "blocking", "group": key, "check": "no_id_column",
                           "message": f"{p.name}: no access_method_val / msisdn column", "count": 0})
            continue
        out = df.rename(columns={c: f"T_{c}" for c in df.columns if c != idc}).rename(columns={idc: VAL})
        out[VAL] = norm_val(out[VAL])
        frames.append(out.assign(GROUP_KEY=key, TARGET_FILE=str(p.relative_to(targets_dir(input_folder)))))
    t = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame(columns=[VAL, "GROUP_KEY"])
    return t, issues


def check_targets(t: pd.DataFrame, segments: pd.DataFrame, busy: set[str] | None = None,
                  sizes: dict[str, int] | None = None) -> tuple[pd.DataFrame, pd.DataFrame, list[dict]]:
    """-> (kept lines, rejected lines with their reason, issues)."""
    issues: list[dict] = []
    sizes = sizes or config.group_sizes()

    def add(level, check, msg, rows):
        if len(rows):
            issues.append({"level": level, "check": check, "message": msg, "count": int(len(rows)),
                           "examples": rows[VAL].astype(str).head(5).tolist(),
                           "groups": rows["GROUP_KEY"].value_counts().to_dict()})

    dup_in = t[t.duplicated([VAL, "GROUP_KEY"])]
    add("warning", "duplicate_in_list", "a line listed twice in one group (kept once)", dup_in)
    t = t.drop_duplicates([VAL, "GROUP_KEY"])
    two = t[t.duplicated(VAL, keep=False)]
    add("blocking", "line_in_two_groups", "a line in two groups (A-30): fix the lists", two)
    rejected = []

    def reject(mask, reason):
        nonlocal t
        rows = t[mask]
        if len(rows):
            rejected.append(rows.assign(REJECT_REASON=reason))
            add("warning", "dropped", f"dropped: {reason}", rows)
            t = t[~mask]

    reject(~t[VAL].astype(str).str.fullmatch(r"\d{8}").fillna(False), "not an 8-digit MSISDN")
    base = segments.set_index(VAL)
    reject(~t[VAL].isin(base.index), "not in the eligible base (D-07)")
    reject(t[VAL].map(base["EXCLUDED"]).eq(1) & t["GROUP_KEY"].ne(config.CONTROL_GROUP), "Nursering segment (B-02)")
    if busy:
        reject(t[VAL].isin(busy) & t["GROUP_KEY"].ne(config.CONTROL_GROUP), "in another campaign this week (H-05)")
    for k, v in sizes.items():
        n = int((t["GROUP_KEY"] == k).sum())
        if v and abs(n - v) > config.GROUP_SIZE_TOLERANCE * v:
            issues.append({"level": "warning", "check": "size", "group": k, "count": n,
                           "message": f"{k}: {n:,} lines kept, configured {v:,} (+-2 %)"})
    rej = pd.concat(rejected, ignore_index=True) if rejected else pd.DataFrame(columns=[VAL, "GROUP_KEY", "REJECT_REASON"])
    return t.reset_index(drop=True), rej, issues


def write_targets(panel: pd.DataFrame, input_folder: Path, fmt: str = "csv.gz", meta: dict | None = None) -> list[Path]:
    """Writes one list per group (dummy targets for tests, or a drawn panel to hand over)."""
    cols = [VAL, "ACCESS_METHOD_ID", "NATIONALITY_GROUP", "SEGMENT", "MICRO_SEGMENT", "VALUE_BAND", "USAGE_TYPE",
            "CURRENT_PLAN", "STRATUM"]
    written, sums = [], {}
    for key, rel in config.TARGET_FILES.items():
        part = panel.loc[panel["GROUP_KEY"] == key, [c for c in cols if c in panel.columns]]
        part = part.rename(columns={VAL: "access_method_val", "STRATUM": "Final"})
        part.columns = [c.lower() if c != "Final" else c for c in part.columns]
        p = targets_dir(input_folder) / f"{rel}.{fmt}"
        p.parent.mkdir(parents=True, exist_ok=True)
        if fmt == "csv.gz":
            csvstd.write_csv(part, p)
        elif fmt == "csv":
            csvstd.write_csv(part, p, compress=False)
        else:
            part.to_json(p, orient="records", lines=fmt == "jsonl")
        written.append(p)
        sums[key] = _sha(p)
    man = {"source": "generator", "sizes": panel["GROUP_KEY"].value_counts().to_dict(), "format": fmt,
           "sha256": sums, **(meta or {})}
    (targets_dir(input_folder) / config.TARGETS_MANIFEST).write_text(json.dumps(man, indent=2), encoding="utf-8")
    return written


In [ ]:
%%engine nbx.fsdr.pipelines.scoring_groups
"""Scoring steps of the four groups (plan §14, update §3 - §5), run after training T2 + T3:

    run_p1_panel(D)    score/P1_panel/    panel (T1C / T1F / T2C / T2F / control), group sizes, balance (A-30 / A-31)
    run_p6_offers(D)   score/P6_offers/   results_scoring_nba, the 4 code-2 tables, the 8 reco tables (FSDR CSV
                                          standard .csv.gz + schema + _outputs_manifest.json), traces, group audit

V4.0, each target scored separately (its own steps, its own output folder):

    run_p6_offers(D, target="target1")   score/P6_offers_target1/   the tables of T1C (a) and T1F (b) only
    run_target_final(D, "target1")       data/output/<D>/target1/   input_flytxt_target1 (on PASS), a_constraint/ and
                                                                    b_free/ files, details, checks, summary
    run_final_file(D)                          data/output/<D>/final/      both targets merged when both PASS; control/
    run_targets(D)                       P1, then P6 + final of each target, then the final file

P6 uses the documented fallbacks of the LLM agents until Phases 5-6 build them (recos.py says which); the rules,
universes, anchors, passes and bands are the final ones.
"""
from __future__ import annotations

import json
from datetime import date, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

from nbx.fsdr import audit_groups, config, csvstd, llm_budget
from nbx.fsdr import targets as targets_mod
from nbx.fsdr.contract.cache import CacheSource
from nbx.fsdr.features import norm_val
from nbx.fsdr.offers.recos import build_group_recos
from nbx.fsdr.panel import assign_panel, balance, check_sizes, eligible_pool, panel_group, stratum
from nbx.fsdr.pipelines.training import T2, T3, StageBlocked
from nbx.fsdr.steps import Step, fingerprint, read_step, runs_dir

P1, P6 = "P1_panel", "P6_offers"
VAL = "ACCESS_METHOD_VAL"
SEG_COLS = ["ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "SEGMENT", "RULE_ACTION", "EXCLUDED", "NATIONALITY_GROUP",
            "AGE_BAND", "TENURE_BAND", "MICRO_SEGMENT", "VALUE_BAND", "USAGE_TYPE", "ACTION_ORIGINAL", "ACTION_AGENT",
            "ACTION_AGENT_RANK2"]


def _parts(folder: Path, columns: list[str] | None = None) -> pd.DataFrame:
    files = sorted(Path(folder).glob("part-*.parquet"))
    return pd.concat([pd.read_parquet(f, columns=columns) for f in files], ignore_index=True) if files else pd.DataFrame()


def base_segments(run_date: date, runs_root: Path | None = None) -> pd.DataFrame:
    train = runs_dir(run_date, runs_root) / "train"
    for st in (T2, T3):
        if read_step(train / st).get("status") != "done":
            raise StageBlocked(f"{st} of {run_date} is not done: run fsdr-features first")
    seg = _parts(train / T3 / "segments", SEG_COLS)
    plan = _parts(train / T2 / "base_features", [VAL, "CURRENT_PLAN"])
    return seg.merge(plan.drop_duplicates(VAL), on=VAL, how="left")


def other_campaigns(cache: CacheSource, run_date: date) -> set[str]:
    """H-05: lines in another campaign this week (targeting_auto START_DATE in D-7 … D)."""
    if not cache.has("targeting_auto"):
        return set()
    t = cache.read("targeting_auto")
    start = pd.to_datetime(t.get("START_DATE"), errors="coerce").dt.date
    lo, hi = run_date - timedelta(days=7), run_date
    return set(norm_val(t.loc[(start >= lo) & (start <= hi), "MSISDN"]).dropna())


def _sample(cache: CacheSource) -> pd.DataFrame | None:
    if not cache.has(config.TARGET_SAMPLE_TABLE):
        return None
    s = cache.read(config.TARGET_SAMPLE_TABLE)
    return pd.DataFrame({VAL: norm_val(s["access_method_val"]), "group": s["group"].astype(str)})


def input_folder(run_date: date, runs_root: Path | None = None) -> Path:
    """The folder of the target lists: the scoring twin T0 loaded (V4.0 data/scoring/<D>/), else the single input
    folder of a V3.0 run (data/input/<D>/), else data/scoring/<D>/."""
    from nbx.fsdr.contract import data_agent

    t0 = data_agent.load_summary(run_date, runs_root)
    src = t0.get("scoring_source") or t0.get("source")
    return Path(src) if src else config.scoring_folder(run_date.isoformat())


def generate_targets(run_date: date, runs_root: Path | None = None, cache_root: Path | None = None,
                     fmt: str = "csv.gz", seed: int = config.GENERATOR_SEED, folder: Path | None = None,
                     overwrite: bool = False) -> dict:
    """Dummy target lists for tests: a stratified draw of the eligible base, written to <input>/targets/.
    Lists that the generator did not write (your real lists) are never replaced, unless overwrite=True."""
    folder = Path(folder or input_folder(run_date, runs_root))
    if not overwrite and real_targets(folder):
        raise StageBlocked(f"{targets_mod.targets_dir(folder)} holds target lists the generator did not write "
                           "(real lists): they are kept. Use them (GENERATE_TARGETS = False, or no --generate), "
                           "or delete them first to write dummy lists")
    seg = base_segments(run_date, runs_root)
    cache = CacheSource(run_date, cache_root)
    busy = other_campaigns(cache, run_date)
    pool = eligible_pool(seg, exclude=busy)
    panel, rep = assign_panel(pool, config.group_sizes(), seed)
    files = targets_mod.write_targets(panel, folder, fmt, {"run_date": run_date.isoformat(), "seed": seed,
                                                            "scale_factor": rep["scale_factor"]})
    return {"folder": str(targets_mod.targets_dir(folder)), "files": [str(f) for f in files], **rep}


def real_targets(folder: Path) -> bool:
    """True when the folder holds lists that are not the generator's dummy lists (see targets.dummy_lists)."""
    return targets_mod.has_targets(folder) and not targets_mod.dummy_lists(folder)


def _panel_from_targets(folder: Path, seg: pd.DataFrame, busy: set[str]) -> tuple[pd.DataFrame, pd.DataFrame, list, dict]:
    t, issues = targets_mod.read_targets(folder)
    kept, rejected, more = targets_mod.check_targets(t, seg, busy, targets_mod.expected_sizes(folder))
    issues += more
    panel = kept.merge(seg, on=VAL, how="left")
    panel = panel.assign(STRATUM=stratum(panel), SOURCE="uploaded",
                         PANEL_GROUP=panel["GROUP_KEY"].map(panel_group),
                         TARGET=panel["GROUP_KEY"].map(lambda k: "control" if k == config.CONTROL_GROUP
                                                       else config.GROUPS[k]["target"]))
    sizes = panel["GROUP_KEY"].value_counts().reindex(list(config.group_sizes()), fill_value=0).astype(int).to_dict()
    rep = {"mode": "uploaded", "pool": int(len(t)), "requested": sizes, "configured": config.group_sizes(),
           "scale_factor": 1.0, "sizes": sizes, "from_sample": 0, "rejected": int(len(rejected))}
    return panel, rejected, issues, rep


def run_p1_panel(run_date: date, runs_root: Path | None = None, cache_root: Path | None = None,
                 use_sample: bool = True, seed: int = config.GENERATOR_SEED, force: bool = False) -> dict:
    """P1: the uploaded target lists when <input>/targets/ holds them, otherwise a stratified random draw."""
    seg = base_segments(run_date, runs_root)
    cache = CacheSource(run_date, cache_root)
    t3 = read_step(runs_dir(run_date, runs_root) / "train" / T3)
    folder = input_folder(run_date, runs_root)
    uploaded = targets_mod.has_targets(folder)
    tfiles = targets_mod.list_files(folder) if uploaded else []
    inputs = fingerprint({"t3": t3.get("started"), "sizes": config.group_sizes(), "seed": seed, "sample": use_sample,
                          "t2_split": config.T2_SPLIT, "targets": fingerprint(tfiles) if tfiles else ""})
    with Step(runs_dir(run_date, runs_root) / "score", P1, inputs=inputs, force=force) as st:
        if st.done:
            return {**st.read_json("summary.json"), "skipped": True}
        busy = other_campaigns(cache, run_date)
        issues: list = []
        if uploaded:
            panel, rejected, issues, rep = _panel_from_targets(folder, seg, busy)
            st.write_csv(rejected, "targets_rejected.csv")
            st.write_json("targets_check.json", issues)
            pool_n = rep["pool"]
        else:
            pool = eligible_pool(seg, exclude=busy)
            panel, rep = assign_panel(pool, config.group_sizes(), seed, _sample(cache) if use_sample else None)
            rep["mode"] = "drawn"
            pool_n = len(pool)
        st.write_table(panel, "panel")
        counts = pd.DataFrame([{"group_key": k, "panel_group": panel_group(k), "requested": v,
                                "assigned": int((panel["GROUP_KEY"] == k).sum())} for k, v in rep["requested"].items()])
        st.write_csv(counts, "group_sizes.csv")
        bal = balance(panel) if len(panel) else pd.DataFrame(columns=["max_gap_points"])
        st.write_csv(bal, "balance.csv")
        blocking = [i for i in issues if i["level"] == "blocking"]
        summary = {"run_date": run_date.isoformat(), "stage": P1, **rep, "eligible_base": int(len(seg)),
                   "excluded_nursering": int((seg["EXCLUDED"] == 1).sum()), "in_other_campaigns": len(busy),
                   "panel": len(panel), "size_problems": check_sizes(panel, rep["requested"]),
                   "target_issues": issues, "status": "blocked" if blocking else "done",
                   "max_balance_gap_points": float(bal["max_gap_points"].max()) if len(bal) else 0.0,
                   "targets_folder": str(targets_mod.targets_dir(folder)) if uploaded else None, "step_dir": str(st.dir)}
        if blocking:
            st.block("; ".join(i["message"] for i in blocking))
        st.rows(rows_in=pool_n, rows_out=len(panel))
        st.write_json("summary.json", summary)
        return summary


def _traces(out: dict, panel: pd.DataFrame, per_group: int = 2) -> pd.DataFrame:
    """Update §12: 2 decisions per group for the SAME type of customer (segment x nationality x value band)."""
    dec, sel = out["_decisions"], out["_selected"]
    if not len(sel):
        return pd.DataFrame()
    typ = dec.assign(T=dec["SEGMENT"].astype(str) + " | " + dec["NATIONALITY_GROUP"].astype(str) + " | "
                     + dec["VALUE_BAND"].astype(str))
    with_offer = typ[typ[VAL].isin(sel[VAL])]
    present = with_offer.groupby("T")["GROUP_KEY"].nunique()
    common = present[present == present.max()].index
    t = with_offer[with_offer["T"].isin(common)]["T"].value_counts().index[0]
    rows = []
    for key in config.GROUPS:
        for v in with_offer.loc[(with_offer["T"] == t) & (with_offer["GROUP_KEY"] == key), VAL].head(per_group):
            d = dec[dec[VAL] == v].iloc[0]
            for r in sel[sel[VAL] == v].itertuples():
                rows.append({"group": key, "customer_type": t, VAL: v, "rule_action_source": config.GROUPS[key]["rule_action"],
                             "action": d["ACTION"], "decided_by": d["DECIDED_BY"], "consistency_retry": d["CONSISTENCY_RETRY"],
                             "validity": r.VALIDITY, "candidates": r.N_CANDIDATES, "btl_name": r.BTL_NAME,
                             "service": r.SERVICE, "price": r.PRICE, "score": r.SCORE,
                             "anchor": getattr(r, "CH_ANCHOR", None) if key in ("t2c", "t2f") else r.ANCHOR,
                             "pass": r.PASS, "selected_by": r.SELECTED_BY, "rules_version": r.RULES_VERSION})
    return pd.DataFrame(rows)


def target_keys(target: str | None) -> list[str]:
    """The group keys of a target ("target1" -> t1c, t1f), all four for None."""
    if target is None:
        return list(config.GROUPS)
    if target not in config.TARGETS:
        raise ValueError(f"target must be one of {list(config.TARGETS)}")
    return [k for k in config.TARGETS[target] if k in config.group_sizes()]


def p6_name(target: str | None) -> str:
    return P6 if target is None else f"{P6}_{target}"


def run_p6_offers(run_date: date, runs_root: Path | None = None, cache_root: Path | None = None,
                  seconds_per_customer: float = 0.36, force: bool = False, target: str | None = None,
                  agents: dict | None = None) -> dict:
    """P6 of the four groups (target None), or of one target only (V4.0: target1 = T1C + T1F, target2 = T2C + T2F).
    agents: the LLM agents ③ NBA / ④ NBO of the GPU notebook (offers/recos.py); None = the documented fallbacks."""
    score = runs_dir(run_date, runs_root) / "score"
    p1 = read_step(score / P1)
    if p1.get("status") != "done":
        raise StageBlocked(f"{P1} of {run_date} is {p1.get('status', 'missing')}: run fsdr-panel (and fix the "
                           f"target lists if it is blocked)")
    keys = target_keys(target)
    inputs = fingerprint({"p1": p1.get("started"), "groups": config.GROUPS, "band": config.CHALLENGER_BAND,
                          "keys": keys, "reco_offers": config.RECO_OFFERS,
                          "agents": (agents or {}).get("label", ""), "agent_groups": sorted((agents or {}).get("groups", ()))})
    with Step(score, p6_name(target), inputs=inputs, force=force) as st:
        if st.done:
            return {**st.read_json("summary.json"), "skipped": True}
        panel = pd.read_parquet(score / P1 / "panel.parquet")
        panel = panel[panel["GROUP_KEY"].isin(keys + [config.CONTROL_GROUP])].reset_index(drop=True)
        cache = CacheSource(run_date, cache_root)
        out = build_group_recos(panel, cache, agents=agents)
        own = {"results_scoring_nba"} | {config.GROUPS[k][t] for k in keys
                                         for t in ("scored_table", "recos_monthly", "recos_weekly")}
        files = []
        for name, df in out.items():
            if not name.startswith("_") and name in own:
                files.append(csvstd.write_csv(df, st.output(f"{name}{csvstd.SUFFIX}")))
                st.outputs.append(csvstd.schema_path(files[-1]).name)
        st.outputs.append(csvstd.write_outputs_manifest(st.dir, files, run_id=f"score_{run_date.isoformat()}"
                                                        + (f"_{target}" if target else ""),
                                                        versions={k: config.GROUPS[k]["rules_version"] for k in keys}).name)
        st.write_table(out["_decisions"], "decisions", csv="none")
        tr = _traces(out, panel)
        st.write_csv(tr, "traces.csv")
        sizes = json.loads((score / P1 / "summary.json").read_text(encoding="utf-8"))["requested"]   # scaled if small
        sizes = {k: v for k, v in sizes.items() if k in keys or k == config.CONTROL_GROUP}
        audit = audit_groups.audit_groups(panel, out["_selected"], out["_recommendable"], sizes)
        st.write_json("audit_groups.json", audit)
        q = llm_budget.apply_budget(llm_budget.llm_queue(out["_decisions"]), seconds_per_customer)
        sel = out["_selected"]
        per_group = []
        for key in keys:
            g = config.GROUPS[key]
            d = out["_decisions"][out["_decisions"]["GROUP_KEY"] == key]
            s = sel[sel["GROUP_KEY"] == key] if len(sel) else sel
            per_group.append({"group": key, "rules": g["rules_version"], "contacted": len(d),
                              "with_offers": int(s[VAL].nunique()) if len(s) else 0,
                              "offers": len(s), "consistency_retry": int(d["CONSISTENCY_RETRY"].sum()),
                              "by_rule": s["SELECTED_BY"].value_counts().to_dict() if len(s) else {},
                              "decided_by": d["DECIDED_BY"].value_counts().to_dict(),
                              "llm_scored_offers": int(s["SCORE_SOURCE"].astype(str).str.startswith("llm:").sum()) if len(s) else 0,
                              "llm_coverage_at_budget": float(llm_budget.coverage(q).get(key, 0.0))})
        st.write_csv(pd.DataFrame(per_group).assign(by_rule=lambda d: d["by_rule"].map(json.dumps),
                                                      decided_by=lambda d: d["decided_by"].map(json.dumps)), "group_report.csv")
        summary = {"run_date": run_date.isoformat(), "stage": p6_name(target), "target": target or "all",
                   "decisions": (f"agents {agents.get('label', '')} on {sorted(agents.get('groups', ()))}, fallbacks elsewhere"
                                 if agents else "fallback (LLM agents: the GPU notebook)"), "reco_offers": config.RECO_OFFERS,
                   "tables": [n for n in out if not n.startswith("_") and n in own], "groups": per_group,
                   "audit_status": audit["status"], "blocked": audit["blocked"],
                   "step_dir": str(st.dir)}
        st.rows(rows_in=len(panel), rows_out=len(sel))
        st.write_json("summary.json", summary)
        return summary


# ---------------------------------------------------------------------------
# P7 final: input_flytxt (final file) + per-target files + details (the "big data" behind every row)
# ---------------------------------------------------------------------------
FINAL = "final"
TARGET_DIRS = {"t1c": "target1", "t1f": "target1", "t2c": "target2", "t2f": "target2", "control": "control"}
RECO_FIELDS = ["btl_name", "service", "generosity", "price", "score", "anchor", "pass", "selected_by"]


def _read_reco(p6: Path, table: str) -> pd.DataFrame:
    p = p6 / f"{table}{csvstd.SUFFIX}"
    df = csvstd.read_csv(p) if p.exists() else pd.DataFrame(columns=["access_method_val", "btl_name", "price"])
    for c in ("price", "score", "anchor", "pass"):
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


def _flat_recos(recos: pd.DataFrame, prefix: str) -> pd.DataFrame:
    """Up to RECOS_PER_VALIDITY offers per customer (2, or 3 with 6 offers), price desc -> reco_<prefix>1_* …"""
    if not len(recos):
        return pd.DataFrame(index=pd.Index([], name=VAL))
    r = recos.assign(_K=recos.groupby("access_method_val")["price"].rank(method="first", ascending=False).astype(int))
    parts = []
    for k in range(1, config.RECOS_PER_VALIDITY + 1):
        top = r[r["_K"] == k].set_index("access_method_val")[[c for c in RECO_FIELDS if c in r.columns]]
        parts.append(top.add_prefix(f"reco_{prefix}{k}_"))
    return pd.concat(parts, axis=1).rename_axis(VAL)


def output_checks(flytxt: pd.DataFrame, control: set[str], catalog_names: set[str], run_date: date) -> list[dict]:
    """A-20 … A-24 on the final file (the full set comes with Phase 7)."""
    offers = pd.unique(flytxt[[f"destination_offer_name_{i}" for i in range(1, 7)]].to_numpy().ravel())
    not_cat = sorted(o for o in offers if o and o not in catalog_names)
    checks = [("A-20", "38 columns in the F-19 order", list(flytxt.columns) == config.INPUT_FLYTXT_COLUMNS),
              ("A-21", "one row per msisdn, 8-digit text", bool(flytxt["msisdn"].is_unique
               and flytxt["msisdn"].astype(str).str.fullmatch(r"\d{8}").all())),
              ("A-21", "no control msisdn in the file", not (set(flytxt["msisdn"]) & control)),
              ("A-22", "date / start_date / end_date / channels / flag_splio",
               bool((flytxt["date"] == run_date.isoformat()).all()
                    and (flytxt["start_date"] == (run_date + timedelta(days=1)).isoformat()).all()
                    and (flytxt["end_date"] == "").all() and (flytxt["flag_splio"] == 1).all())),
              ("A-23", f"every offer in the catalogue ({len(not_cat)} not)", not not_cat),
              ("A-24", "segment is one of the 7 actions", bool(flytxt["segment"].isin(config.ACTIONS).all()))]
    return [{"id": i, "check": what, "status": "PASS" if ok else "FAIL"} for i, what, ok in checks]


def _count_removed(det: pd.DataFrame, rule: str) -> int:
    cols = [f"REMOVED_{i}" for i in range(1, 7) if f"REMOVED_{i}" in det.columns]
    return int(sum(det[c].astype(str).str.startswith(rule).sum() for c in cols))


def _details(run_date: date, runs_root: Path | None, dec: pd.DataFrame, panel: pd.DataFrame, recos: dict,
             slots: pd.DataFrame, anchors: pd.DataFrame, f05: pd.DataFrame) -> pd.DataFrame:
    """One row per panel line: profile, segment, both actions, the action sent, the features, both anchors, the
    recommended offers with their pass / rule, the 6 slots with their reason and what removed them."""
    from nbx.fsdr.segmentation.actions import action_agreement

    train = runs_dir(run_date, runs_root) / "train"
    vals = set(panel[VAL])
    feats = pd.concat([d[d[VAL].isin(vals)] for d in (pd.read_parquet(f) for f in
                       sorted((train / T2 / "base_features").glob("part-*.parquet")))], ignore_index=True)
    feats = feats.drop(columns=[c for c in feats.columns if c in panel.columns and c != VAL]).drop_duplicates(VAL)
    d = panel.merge(dec[[VAL, "ACTION", "DECIDED_BY", "CONSISTENCY_RETRY"]], on=VAL, how="left")
    d = d.rename(columns={"ACTION": "ACTION_SENT"})
    d["RULES_VERSION"] = d["GROUP_KEY"].map(lambda k: config.GROUPS[k]["rules_version"] if k in config.GROUPS else "")
    d["PROPENSITY"] = np.where(d["ACTION_SENT"].notna(), 1.0, np.nan)
    d["ACTION_AGREEMENT"] = np.where(d["ACTION_SENT"].notna(),
                                     action_agreement(d["ACTION_ORIGINAL"], d["ACTION_AGENT"], d["ACTION_SENT"]), None)
    b26 = anchors.reindex(d[VAL]).add_prefix("B26_").reset_index(drop=True)
    f5 = f05.reindex(d[VAL]).reset_index(drop=True)
    d = pd.concat([d.reset_index(drop=True), b26, f5], axis=1)
    for prefix, table in (("m", "monthly"), ("w", "weekly")):
        d = d.merge(_flat_recos(recos[table], prefix), left_on=VAL, right_index=True, how="left")
    s = slots.rename(columns={f"S{i}": f"SLOT_{i}" for i in range(1, 7)})
    s = s[[c for c in s.columns if c.startswith(("SLOT_", "REASON_", "REMOVED_", "IN_INPUT", "DROP_")) or c == VAL]]
    d = d.merge(s, on=VAL, how="left")
    d["IN_INPUT_FLYTXT"] = d["IN_INPUT_FLYTXT"].astype("boolean").fillna(False).astype(bool)
    d["DROP_RULE"] = np.select([d["GROUP_KEY"] == config.CONTROL_GROUP, d["ACTION_SENT"].isna(), ~d["IN_INPUT_FLYTXT"]],
                               ["control (H-01)", "no decision", "F-04 no offer in slots 1-4"], "")
    return d.merge(feats, on=VAL, how="left")


def _final_core(run_date: date, runs_root: Path | None, cache_root: Path | None, d6: Path, keys: list[str]) -> dict:
    """Code 5 on the P6 tables of some groups: input_flytxt, slots, output checks, the P6 audit and the details."""
    from nbx.fsdr.final_output import build_input_flytxt, f05_anchors
    from nbx.fsdr.offer_rules import arpu_anchors, load_catalog

    run = runs_dir(run_date, runs_root)
    cache = CacheSource(run_date, cache_root)
    panel = pd.read_parquet(run / "score" / P1 / "panel.parquet")
    panel = panel[panel["GROUP_KEY"].isin(keys + [config.CONTROL_GROUP])].reset_index(drop=True)
    dec = pd.read_parquet(d6 / "decisions.parquet")
    scored = csvstd.read_csv(d6 / f"results_scoring_nba{csvstd.SUFFIX}")
    groups = [k for k in keys if (panel["GROUP_KEY"] == k).any()]
    recos = {t: pd.concat([_read_reco(d6, config.GROUPS[k][f"recos_{t}"]).assign(GROUP_KEY=k) for k in groups],
                          ignore_index=True) if groups else _read_reco(d6, "_none_") for t in ("monthly", "weekly")}
    flytxt, slots = build_input_flytxt(recos["monthly"], recos["weekly"], scored, cache, run_date)       # F-01
    catalog = load_catalog(cache)
    control = set(panel.loc[panel["GROUP_KEY"] == config.CONTROL_GROUP, VAL])
    checks = output_checks(flytxt, control, set(catalog["BTL_NAME"]), run_date)
    audit6 = json.loads((d6 / "audit_groups.json").read_text(encoding="utf-8"))
    ok = all(c["status"] == "PASS" for c in checks) and not audit6["blocked"]
    details = _details(run_date, runs_root, dec, panel, recos, slots, arpu_anchors(cache),
                       f05_anchors(cache, panel[VAL]))
    return {"panel": panel, "groups": groups, "flytxt": flytxt, "checks": checks, "audit6": audit6, "ok": ok,
            "details": details, "group_of": panel.set_index(VAL)["GROUP_KEY"]}


def _group_report(key: str, det: pd.DataFrame, part: pd.DataFrame) -> dict:
    return {"group": key, "rules": config.GROUPS[key]["rules_version"], "lines": len(det),
            "in_input_flytxt": len(part),
            "dropped_F04": int((det["DROP_RULE"] == "F-04 no offer in slots 1-4").sum()),
            "favourites": int(det[["REASON_5", "REASON_6"]].eq("favourite").sum().sum()),
            "forced_F08": int(det["REASON_6"].astype(str).str.startswith("forced").sum()),
            "cleared_F09": _count_removed(det, "F-09"), "cleared_F11": _count_removed(det, "F-11"),
            "cleared_F12": _count_removed(det, "F-12"),
            "offers_per_line": round(float(part[[f"destination_offer_name_{i}" for i in range(1, 7)]].ne("").sum(axis=1)
                                           .mean()), 3) if len(part) else 0.0}


def _blocked_text(what: str, checks: list[dict], audit6: dict) -> str:
    return (f"{what} NOT written (H-08). Failed checks:\n"
            + "\n".join(f"- {c['id']} {c['check']}" for c in checks if c["status"] != "PASS")
            + ("\n- P6 group audit FAIL (audit_groups.json)" if audit6["blocked"] else ""))


def run_p7_final(run_date: date, runs_root: Path | None = None, cache_root: Path | None = None,
                 force: bool = False) -> dict:
    """V3.0 final of the four groups together: data/runs/<D>/final/ (all_groups/, target1/, target2/, control/)."""
    run = runs_dir(run_date, runs_root)
    p6 = read_step(run / "score" / P6)
    if p6.get("status") != "done":
        raise StageBlocked(f"{P6} of {run_date} is {p6.get('status', 'missing')}: run fsdr-offers first")
    inputs = fingerprint({"p6": p6.get("started"), "prefix": config.OPPORTUNITY_PREFIX, "forced": config.FORCED_BUNDLES,
                          "reco_offers": config.RECO_OFFERS})
    with Step(run, FINAL, inputs=inputs, force=force) as st:
        if st.done:
            return {**st.read_json("summary.json"), "skipped": True}
        c = _final_core(run_date, runs_root, cache_root, run / "score" / P6, list(config.GROUPS))
        flytxt, details, group_of, checks, audit6, ok = (c["flytxt"], c["details"], c["group_of"], c["checks"],
                                                         c["audit6"], c["ok"])
        files = []
        if (st.dir / "all_groups" / "BLOCKED.txt").exists():
            (st.dir / "all_groups" / "BLOCKED.txt").unlink()
        if ok:                                                                    # H-08: final file only on PASS
            files.append(csvstd.write_csv(flytxt, st.output(f"all_groups/input_flytxt{csvstd.SUFFIX}")))
            if config.OUTPUT_COMPRESSION == "none":
                files.append(csvstd.write_csv(flytxt, st.output("all_groups/input_flytxt.csv"), compress=False))
        else:
            for stale in (st.dir / "all_groups").glob("input_flytxt*"):              # never leave an older final file
                stale.unlink()
            st.output("all_groups/BLOCKED.txt").write_text(_blocked_text("input_flytxt", checks, audit6), encoding="utf-8")
        per_target = []
        for key in c["groups"] + [config.CONTROL_GROUP]:
            folder = TARGET_DIRS[key]
            det = details[details["GROUP_KEY"] == key]
            files.append(csvstd.write_csv(det, st.output(f"{folder}/details_{key}{csvstd.SUFFIX}")))
            if key == config.CONTROL_GROUP:
                files.append(csvstd.write_csv(det[[VAL, "GROUP_KEY", "STRATUM", "SEGMENT", "MICRO_SEGMENT"]],
                                              st.output(f"control/control_{run_date.isoformat()}{csvstd.SUFFIX}")))
                per_target.append({"group": key, "rules": "never contacted", "lines": len(det), "in_input_flytxt": 0})
                continue
            part = flytxt[flytxt["msisdn"].map(group_of) == key]
            files.append(csvstd.write_csv(part, st.output(f"{folder}/input_flytxt_{key}{csvstd.SUFFIX}")))
            per_target.append(_group_report(key, det, part))
        for tgt, keys in (("target1", ["t1c", "t1f"]), ("target2", ["t2c", "t2f"])):
            part = flytxt[flytxt["msisdn"].map(group_of).isin(keys)]
            files.append(csvstd.write_csv(part, st.output(f"{tgt}/input_flytxt_{tgt}{csvstd.SUFFIX}")))
        st.write_table(details, "details_all", csv="none")
        st.write_csv(pd.DataFrame(per_target), "final_rules_report.csv")
        st.output(csvstd.OUTPUTS_MANIFEST)
        csvstd.write_outputs_manifest(st.dir, files, run_id=f"final_{run_date.isoformat()}",
                                      versions={k: g["rules_version"] for k, g in config.GROUPS.items()})
        summary = {"run_date": run_date.isoformat(), "stage": FINAL, "ready": ok,
                   "input_flytxt_rows": len(flytxt), "output_checks": checks, "p6_group_audit": audit6["status"],
                   "per_target": per_target, "details_rows": len(details), "details_columns": int(details.shape[1]),
                   "reco_offers": config.RECO_OFFERS, "step_dir": str(st.dir)}
        if not ok:
            st.block("final file blocked: see all_groups/BLOCKED.txt")
        st.rows(rows_in=len(c["panel"]), rows_out=len(flytxt))
        st.write_json("summary.json", summary)
        return summary


# ---------------------------------------------------------------------------
# V4.0: each target scored separately into data/output/<D>/<target>/, then one final file of both targets
# ---------------------------------------------------------------------------
FINAL_FILE, CONTROL_DIR = "final", "control"


def run_target_final(run_date: date, target: str, runs_root: Path | None = None, cache_root: Path | None = None,
                     out_root: Path | None = None, force: bool = False) -> dict:
    """Code 5 on ONE target -> data/output/<D>/<target>/:

        input_flytxt_<target>.csv.gz        the target's final file (only on PASS, else BLOCKED.txt)
        a_constraint/input_flytxt_<t?c>.csv.gz   details_<t?c>.csv.gz     group a (constraint universe)
        b_free/input_flytxt_<t?f>.csv.gz         details_<t?f>.csv.gz     group b (free universe)
        details_<target>.parquet            every line of the target with every column (the big file to validate)
        final_rules_report.csv  output_checks.json  audit_groups.json  summary.json  _outputs_manifest.json
    """
    keys = target_keys(target)
    run = runs_dir(run_date, runs_root)
    d6 = run / "score" / p6_name(target)
    p6 = read_step(d6)
    if p6.get("status") != "done":
        raise StageBlocked(f"{p6_name(target)} of {run_date} is {p6.get('status', 'missing')}: "
                           f"run fsdr-score-target --target {target} first")
    inputs = fingerprint({"p6": p6.get("started"), "prefix": config.OPPORTUNITY_PREFIX, "forced": config.FORCED_BUNDLES,
                          "reco_offers": config.RECO_OFFERS})
    with Step(config.output_folder(run_date.isoformat(), out_root), target, inputs=inputs, force=force) as st:
        if st.done:
            return {**st.read_json("summary.json"), "skipped": True}
        c = _final_core(run_date, runs_root, cache_root, d6, keys)
        flytxt, details, group_of = c["flytxt"], c["details"], c["group_of"]
        for stale in [*st.dir.glob(f"input_flytxt_{target}*"), st.dir / "BLOCKED.txt"]:   # never an older final file
            if stale.exists():
                stale.unlink()
        files = []
        if c["ok"]:
            files.append(csvstd.write_csv(flytxt, st.output(f"input_flytxt_{target}{csvstd.SUFFIX}")))
        else:
            st.output("BLOCKED.txt").write_text(_blocked_text(f"input_flytxt_{target}", c["checks"], c["audit6"]),
                                                encoding="utf-8")
        per_group = []
        for key in c["groups"]:
            sub = config.GROUP_SUBFOLDER[key]
            det = details[details["GROUP_KEY"] == key]
            part = flytxt[flytxt["msisdn"].map(group_of) == key]
            files.append(csvstd.write_csv(part, st.output(f"{sub}/input_flytxt_{key}{csvstd.SUFFIX}")))
            files.append(csvstd.write_csv(det, st.output(f"{sub}/details_{key}{csvstd.SUFFIX}")))
            per_group.append({**_group_report(key, det, part), "folder": sub})
        st.write_table(details[details["GROUP_KEY"] != config.CONTROL_GROUP], f"details_{target}", csv="none")
        st.write_csv(pd.DataFrame(per_group), "final_rules_report.csv")
        st.write_json("output_checks.json", c["checks"])
        st.write_json("audit_groups.json", c["audit6"])
        st.output(csvstd.OUTPUTS_MANIFEST)
        csvstd.write_outputs_manifest(st.dir, files, run_id=f"final_{run_date.isoformat()}_{target}",
                                      versions={k: config.GROUPS[k]["rules_version"] for k in keys})
        summary = {"run_date": run_date.isoformat(), "stage": f"final_{target}", "target": target, "groups": keys,
                   "ready": c["ok"], "input_flytxt_rows": len(flytxt), "output_checks": c["checks"],
                   "p6_group_audit": c["audit6"]["status"], "per_group": per_group, "reco_offers": config.RECO_OFFERS,
                   "details_rows": int((details["GROUP_KEY"] != config.CONTROL_GROUP).sum()),
                   "details_columns": int(details.shape[1]), "step_dir": str(st.dir)}
        if not c["ok"]:
            st.block(f"{target} blocked: see {target}/BLOCKED.txt")
        st.rows(rows_in=int((c["panel"]["GROUP_KEY"] != config.CONTROL_GROUP).sum()), rows_out=len(flytxt))
        st.write_json("summary.json", summary)
        return summary


def write_control(run_date: date, runs_root: Path | None = None, out_root: Path | None = None) -> dict:
    """data/output/<D>/control/control_<D>.csv.gz: the control lines (never contacted, H-01) and their strata."""
    panel = pd.read_parquet(runs_dir(run_date, runs_root) / "score" / P1 / "panel.parquet")
    ctrl = panel[panel["GROUP_KEY"] == config.CONTROL_GROUP]
    if not len(ctrl):                                   # V4.0 default: no control group (FSDR_CONTROL_SIZE=0)
        return {"file": None, "lines": 0}
    cols = [VAL, "GROUP_KEY", "STRATUM", "SEGMENT", "MICRO_SEGMENT", "NATIONALITY_GROUP", "VALUE_BAND", "USAGE_TYPE"]
    cols += [c for c in ctrl.columns if c.startswith("T_")]
    path = config.output_folder(run_date.isoformat(), out_root) / CONTROL_DIR / f"control_{run_date.isoformat()}{csvstd.SUFFIX}"
    path.parent.mkdir(parents=True, exist_ok=True)
    csvstd.write_csv(ctrl[[c for c in cols if c in ctrl.columns]], path)
    return {"file": str(path), "lines": len(ctrl)}


def run_final_file(run_date: date, out_root: Path | None = None, force: bool = True) -> dict:
    """data/output/<D>/final/input_flytxt.csv.gz: the target files merged, only when EVERY target passed (H-08) and no
    MSISDN is in two targets (A-30); else final/BLOCKED.txt and no file."""
    out = config.output_folder(run_date.isoformat(), out_root)
    states = {t: read_step(out / t) for t in config.TARGETS}
    inputs = fingerprint({t: s.get("started") for t, s in states.items()})
    with Step(out, FINAL_FILE, inputs=inputs, force=force) as st:
        if st.done:
            return {**st.read_json("summary.json"), "skipped": True}
        for stale in [*st.dir.glob("input_flytxt*"), st.dir / "BLOCKED.txt"]:
            if stale.exists():
                stale.unlink()
        reasons, parts, rows = [], [], {}
        for t, s in states.items():
            f = out / t / f"input_flytxt_{t}{csvstd.SUFFIX}"
            if s.get("status") != "done" or not f.exists():
                reasons.append(f"{t}: {s.get('status', 'not scored')} (see {t}/BLOCKED.txt or run it)")
                continue
            parts.append(csvstd.read_csv(f))
            rows[t] = len(parts[-1])
        flytxt = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=config.INPUT_FLYTXT_COLUMNS)
        twice = int(flytxt["msisdn"].duplicated().sum()) if len(flytxt) else 0
        if twice:
            reasons.append(f"A-30: {twice} MSISDN in two targets")
        ok = not reasons
        files = []
        if ok:
            flytxt = flytxt[config.INPUT_FLYTXT_COLUMNS]
            files.append(csvstd.write_csv(flytxt, st.output(f"input_flytxt{csvstd.SUFFIX}")))
            if config.OUTPUT_COMPRESSION == "none":
                files.append(csvstd.write_csv(flytxt, st.output("input_flytxt.csv"), compress=False))
            st.output(csvstd.OUTPUTS_MANIFEST)
            csvstd.write_outputs_manifest(st.dir, files, run_id=f"final_{run_date.isoformat()}",
                                          versions={k: g["rules_version"] for k, g in config.GROUPS.items()})
        else:
            st.output("BLOCKED.txt").write_text("input_flytxt NOT written (H-08):\n" + "\n".join(f"- {r}" for r in reasons),
                                                encoding="utf-8")
            st.block("; ".join(reasons))
        summary = {"run_date": run_date.isoformat(), "stage": FINAL_FILE, "ready": ok, "rows_per_target": rows,
                   "input_flytxt_rows": len(flytxt) if ok else 0, "reasons": reasons, "step_dir": str(st.dir)}
        st.rows(rows_out=summary["input_flytxt_rows"])
        st.write_json("summary.json", summary)
        return summary


def run_targets(run_date: date, runs_root: Path | None = None, cache_root: Path | None = None,
                out_root: Path | None = None, targets: list[str] | None = None, force: bool = True) -> dict:
    """V4.0 scoring in one go: P1 panel, then each target on its own (P6 + final), the control file, the final file."""
    p1 = run_p1_panel(run_date, runs_root, cache_root, force=force)
    res = {"P1_panel": p1}
    if p1["status"] != "done":
        return res
    for t in targets or list(config.TARGETS):
        res[f"P6_{t}"] = run_p6_offers(run_date, runs_root, cache_root, force=force, target=t)
        res[t] = run_target_final(run_date, t, runs_root, cache_root, out_root, force=force)
    res["control"] = write_control(run_date, runs_root, out_root)
    res["final"] = run_final_file(run_date, out_root)
    return res


In [ ]:
%%engine nbx.fsdr.synth package
"""Dummy-data world, scenarios and generator (same JSON files as the real data)."""


In [ ]:
%%engine nbx.fsdr.synth.scenarios
"""Use-case scenarios for the dummy data (combinable, e.g. "churn_wave,data_growth").

The behavioural scenarios change the hidden world (synth/world.py reads
World.scenarios): drifts ramp up over the last 4 weeks before the world's
as_of date, so a single generated date already shows them in the W1..W4 trends,
and the weekly simulation keeps them going. `data_quality_issues` instead
damages the generated tables after they are built, so validation has real
problems to catch.
"""
from __future__ import annotations

from dataclasses import dataclass

import numpy as np
import pandas as pd

from nbx.fsdr import config


@dataclass(frozen=True)
class Scenario:
    name: str
    description: str
    expected_signal: str


SCENARIOS: dict[str, Scenario] = {s.name: s for s in [
    Scenario("baseline", "Normal weekly behaviour.", "-"),
    Scenario("churn_wave", "Churn risk rises over 4 weeks for lines in North Batinah (generate --weeks 4).",
             "PROBA_CHURN and REV_DECR up and usage down in those regions; RETAIN pays off there."),
    Scenario("data_growth", "Voice-only customers with data appetite gradually start using data.",
             "DATA_VOLUME_MB W1 > W4 for them, PROBA_APP_DATA up; UPSELL / CROSS SELL DATA grow."),
    Scenario("price_sensitive", "Price sensitivity x2.5: acceptance drops fast above the ARPU anchor.",
             "Take rate falls steeply with (price - anchor)."),
    Scenario("new_offers", "3 new BTL offers (name ends with _NEW) in the catalogue, with no purchase history (cold start).",
             "Offers in btl_catalog absent from sub_btl_3months; ranker must fall back."),
    Scenario("campaign_fatigue", "Acceptance falls with the number of contacts in the last 4 weeks.",
             "Take rate decreases with NB_CONTACTS_LAST_4W."),
    Scenario("data_quality_issues", "Duplicate MSISDNs, null keys, stacked churn duplicates, keys as strings.",
             "Validation reports each issue."),
]}


def parse(value: str | list[str] | None) -> list[str]:
    """'churn_wave,data_growth' -> ['churn_wave', 'data_growth'] (validated, order kept, duplicates dropped)."""
    if value is None:
        return ["baseline"]
    items = value.split(",") if isinstance(value, str) else list(value)
    out = []
    for s in (x.strip() for x in items):
        if not s:
            continue
        if s not in config.SCENARIOS:
            raise ValueError(f"Unknown scenario '{s}'. Valid: {', '.join(config.SCENARIOS)}")
        if s not in out:
            out.append(s)
    return out or ["baseline"]


# ---------------------------------------------------------------------------
# data_quality_issues: applied to the raw (object) frames just before writing
# ---------------------------------------------------------------------------
QUALITY_SHARE = 0.03          # share of rows damaged per issue (above the 1 % blocking threshold)


def _rows(rng: np.random.Generator, df: pd.DataFrame, share: float = QUALITY_SHARE) -> np.ndarray:
    k = max(1, int(len(df) * share))
    return rng.choice(len(df), size=min(k, len(df)), replace=False)


def inject_quality_issues(tables: dict[str, pd.DataFrame], seed: int) -> list[dict]:
    """Damages the tables in place; returns what was injected (written to the manifest)."""
    rng = np.random.default_rng(np.random.SeedSequence([seed, 99]))
    log: list[dict] = []

    # 1. duplicate MSISDNs in base_prepaid (same line, another ACCESS_METHOD_ID)
    base = tables["base_prepaid"]
    dup = base.iloc[_rows(rng, base)].copy()
    dup["ACCESS_METHOD_ID"] = [int(v) + 900_000_000 for v in dup["ACCESS_METHOD_ID"]]
    tables["base_prepaid"] = pd.concat([base, dup], ignore_index=True)
    log.append({"issue": "duplicate_msisdn", "table": "base_prepaid", "rows": len(dup)})

    # 2. null keys
    for table, col in [("base_prepaid", "PARTY_ID"), ("activities_w1_auto", "ACCESS_METHOD_ID"),
                       ("base_prepaid", "ACCESS_METHOD_VAL")]:
        df = tables[table]
        idx = _rows(rng, df)
        df.loc[df.index[idx], col] = None
        log.append({"issue": "null_key", "table": table, "column": col, "rows": len(idx)})

    # 3. stacked duplicates in the global churn model (same line scored twice, different probabilities)
    churn_t = config.MODEL_SCORE_REGISTRY[0].table
    ch = tables[churn_t]
    stacked = ch.iloc[_rows(rng, ch, 0.05)].copy()
    stacked["proba_1"] = [round(float(rng.uniform(0, 1)), 6) for _ in range(len(stacked))]
    stacked["proba_0"] = [round(1 - p, 6) for p in stacked["proba_1"]]
    tables[churn_t] = pd.concat([ch, stacked], ignore_index=True)
    log.append({"issue": "stacked_duplicates", "table": churn_t, "rows": len(stacked)})

    # 4. integer keys delivered as strings
    for table, col in [("subs_amt_auto", "ACCESS_METHOD_ID"), (churn_t, "access_method_val")]:
        df = tables[table]
        idx = _rows(rng, df)
        df.loc[df.index[idx], col] = [None if v is None else str(v) for v in df.loc[df.index[idx], col]]
        log.append({"issue": "key_as_string", "table": table, "column": col, "rows": len(idx)})

    # 5. customer_profile disagreeing with party_* / the activation date, out of range, negative (V2.0)
    prof = tables.get(config.PROFILE_TABLE)
    if prof is not None and len(prof) >= 10:
        idx = _rows(rng, prof, 0.04)
        q = len(idx) // 4
        ages, tenures = prof["age"].copy(), prof["tenure"].copy()
        for i in idx[:q]:
            ages.iat[i] = int(ages.iat[i]) + 7                              # > 2 years from party_*.AGE
        for i in idx[q:2 * q]:
            tenures.iat[i] = int(tenures.iat[i]) + 90                       # > 30 days from the activation date
        for i in idx[2 * q:3 * q]:
            ages.iat[i] = 7                                                 # outside 15-100
        for i in idx[3 * q:]:
            tenures.iat[i] = -5                                             # negative
        prof["age"], prof["tenure"] = ages, tenures
        log.append({"issue": "profile_disagreement", "table": config.PROFILE_TABLE, "rows": len(idx),
                    "age_off": q, "tenure_off": q, "age_out_of_range": q, "negative_tenure": len(idx) - 3 * q})
    return log


In [ ]:
%%engine nbx.fsdr.synth.world
"""Hidden customer world for the dummy data and the closed-loop simulation.

Every customer has a hidden state (churn risk, data / voice appetite, price
sensitivity, preferred validity, spend level, usage type). From it the world
produces:
  - a daily behaviour history (calls, minutes, MB, PAYG, subscriptions) --
    the generator aggregates it with the business-rule windows;
  - the hidden signals that drive the 11 model scores (scores are noisy
    readings of the truth, not pure noise);
  - a hidden response function P(accept | customer, action, offer, price).

Daily values come from a random stream keyed on (seed, calendar day, chunk),
so two generation dates of the same world agree on every shared day: W2 of
date D equals W1 of date D-7 by construction.

Nothing here is visible to the pipelines; they only see the JSON tables.
"""
from __future__ import annotations

from dataclasses import dataclass, field
from datetime import date, timedelta

import numpy as np
import pandas as pd

from nbx.fsdr import config

USAGE_TYPES = np.array(["voice_only", "data_only", "dual", "inactive"])
NATIONALITIES = ["Oman", "India", "Bangladesh", "Pakistan", "Egypt", "Philippines", "Sri Lanka", "Nepal",
                 "Jordan", "Sudan", "Yemen", "Syria", "Indonesia", "United Kingdom", "Tanzania"]
NATIONALITY_P = [0.45, 0.22, 0.12, 0.09, 0.025, 0.02, 0.015, 0.012, 0.01, 0.008, 0.008, 0.006, 0.005, 0.004, 0.007]
REGULAR_PLANS = ["FSDR Prepaid Basic", "FSDR Prepaid Plus", "FSDR Prepaid Smart", "FSDR Hala Flex", "FSDR Visitor SIM",
                 "FSDR Shababiah", "FSDR Prepaid Data"]
REGULAR_PLAN_P = [0.30, 0.22, 0.16, 0.12, 0.05, 0.08, 0.07]
EXCLUDED_PLAN_SHARE = 0.02
DV_SEGMENTS = ["DV_SMARTPHONE_HIGH", "DV_SMARTPHONE_MID", "DV_SMARTPHONE_LOW", "DV_FEATURE_PHONE", "DV_ROUTER"]
REGIONS = ["Muscat", "Dhofar", "Musandam", "Al Buraimi", "Ad Dakhiliyah", "North Batinah", "South Batinah",
           "South Sharqiyah", "North Sharqiyah", "Ad Dhahirah", "Al Wusta"]          # the 11 governorates
WAVE_REGIONS = {"North Batinah"}                    # churn_wave hits this region
GENEROSITY_BONUS = {"L0": 0.0, "L1": 0.35, "L2": 0.7}
ORGANIC_WEEKLY_BTL_RATE = 0.03                      # BTL bought without contact (control arm baseline)
RESPONSE_BASE_LOGIT = -5.6                          # tuned: ~10 % take rate per targeted customer (4 offers)


def _sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def _logit(p):
    p = np.clip(p, 1e-4, 1 - 1e-4)
    return np.log(p / (1 - p))


def _unique_ids(rng: np.random.Generator, n: int, start: int, max_step: int) -> np.ndarray:
    """n unique integers >= start in random order (cheap, no huge permutation)."""
    ids = start + np.cumsum(rng.integers(1, max_step, n))
    rng.shuffle(ids)
    return ids


def msisdns(rng: np.random.Generator, n: int) -> np.ndarray:
    """n unique 8-digit MSISDNs > 69999999 (BR-02) at any size: the mean step shrinks above ~590k lines, so the
    1M / 2M modes stay below 100000000 (smaller runs draw exactly as before)."""
    return _unique_ids(rng, n, 70_000_000, max(2, min(100, 59_000_000 // max(n, 1)))).astype(str)


# ---------------------------------------------------------------------------
# Offer catalogue
# ---------------------------------------------------------------------------
_PRICE_LADDER = {  # (service, validity) -> {generosity: [prices]}; generosity monotonic with price
    ("DATA", 30): {"L0": [2.0, 3.0, 4.0], "L1": [5.0, 6.0, 7.0], "L2": [8.0, 10.0, 12.0]},
    ("VOICE", 30): {"L0": [1.5, 2.5, 3.5], "L1": [4.0, 5.0, 6.0], "L2": [7.0, 8.5, 10.0]},
    ("COMBO", 30): {"L0": [3.0, 4.5, 6.0], "L1": [7.0, 8.5, 10.0], "L2": [11.0, 13.0, 15.0]},
    ("DATA", 7): {"L0": [0.5, 0.8, 1.0], "L1": [1.2, 1.5, 1.8], "L2": [2.0, 2.5, 3.0]},
    ("VOICE", 7): {"L0": [0.5, 0.7, 0.9], "L1": [1.0, 1.3, 1.6], "L2": [1.8, 2.2, 2.6]},
    ("COMBO", 7): {"L0": [0.8, 1.2, 1.5], "L1": [1.8, 2.2, 2.6], "L2": [3.0, 3.5, 4.0]},
    # add-on families for the batch flow (roaming data, international minutes)
    ("ROAMING", 30): {"L0": [5.0], "L1": [9.0], "L2": [14.0]},
    ("ROAMING", 7): {"L0": [2.0], "L1": [3.5], "L2": [5.0]},
    ("INTERNATIONAL", 30): {"L0": [2.0], "L1": [4.0], "L2": [7.0]},
    ("INTERNATIONAL", 7): {"L0": [0.7], "L1": [1.3], "L2": [2.2]},
}
_GB_PER_OMR = {"L0": 0.8, "L1": 1.2, "L2": 1.8}
_MIN_PER_OMR = {"L0": 25, "L1": 35, "L2": 50}
_GB_SHARE = {"DATA": 1.0, "COMBO": 0.6, "ROAMING": 0.25}           # services that carry data
_MIN_SHARE = {"VOICE": 1.0, "COMBO": 0.5, "INTERNATIONAL": 0.4}     # services that carry minutes


def build_catalog(new_offers: bool = False) -> pd.DataFrame:
    """~54 BTL offers (+3 'new' ones, never bought yet, for the new_offers scenario). Typed frame;
    the generator writes it with the production string types."""
    rows = []
    for (service, validity), ladder in _PRICE_LADDER.items():
        for gen, prices in ladder.items():
            for price in prices:
                rows.append(_offer_row(service, validity, gen, price, is_new=False))
    if new_offers:
        for service, validity, gen, price in [("DATA", 30, "L2", 9.0), ("COMBO", 30, "L2", 12.0), ("COMBO", 7, "L1", 2.4)]:
            rows.append(_offer_row(service, validity, gen, price, is_new=True))
    df = pd.DataFrame(rows).drop_duplicates("btl_name").reset_index(drop=True)
    return df


def _offer_row(service: str, validity: int, gen: str, price: float, is_new: bool) -> dict:
    gb = round(price * _GB_PER_OMR[gen] * _GB_SHARE[service], 1) if service in _GB_SHARE else 0.0
    mins = int(price * _MIN_PER_OMR[gen] * _MIN_SHARE[service]) if service in _MIN_SHARE else 0
    parts = [f"BTL_{service}_{gen}_{validity}D"]
    if gb:
        parts.append(f"{gb:g}GB")
    if mins:
        parts.append(f"{mins}MIN")
    if is_new:
        parts.append("NEW")
    price_data = price if service == "DATA" else (round(price * 0.7, 3) if service == "COMBO" else 0.0)
    return {"btl_name": "_".join(parts), "type": f"BTL_{service}", "generosity": gen, "gb": gb, "min": mins,
            "price": price, "price_data": price_data, "price_voice": round(price - price_data, 3),
            "service": service, "validity": validity, "is_new": is_new}


# ---------------------------------------------------------------------------
# World
# ---------------------------------------------------------------------------
@dataclass
class World:
    seed: int
    scenarios: list[str]
    as_of: date                       # scenario drifts end here (ramp over the previous 4 weeks)
    c: pd.DataFrame                   # one row per customer: ids, demographics, hidden state
    catalog: pd.DataFrame
    params: dict = field(default_factory=dict)

    @property
    def n(self) -> int:
        return len(self.c)

    def has(self, scenario: str) -> bool:
        return scenario in self.scenarios

    # -- creation ------------------------------------------------------------
    @classmethod
    def create(cls, n: int, seed: int = config.GENERATOR_SEED, scenarios: list[str] | None = None,
               as_of: date | None = None, arms: list[str] | None = None) -> "World":
        scenarios = [s for s in (scenarios or ["baseline"]) if s]
        unknown = set(scenarios) - set(config.SCENARIOS)
        if unknown:
            raise ValueError(f"Unknown scenario(s) {sorted(unknown)}; valid: {config.SCENARIOS}")
        if not 1 <= n <= config.GENERATOR_MAX_N:
            raise ValueError(f"n must be between 1 and {config.GENERATOR_MAX_N}")
        as_of = as_of or date.today()
        rng = np.random.default_rng(np.random.SeedSequence([seed, 1]))

        nat = rng.choice(NATIONALITIES, n, p=np.array(NATIONALITY_P) / sum(NATIONALITY_P))
        omani = nat == "Oman"
        u = rng.random(n)
        tenure = np.where(u < 0.06, rng.integers(1, 45, n),
                          np.where(u < 0.11, rng.integers(45, 91, n),
                                   np.clip(90 + rng.lognormal(np.log(700), 0.9, n), 91, 3650))).astype(int)
        usage_p = np.where(omani[:, None], [[0.22, 0.20, 0.53, 0.05]], [[0.38, 0.10, 0.47, 0.05]])
        usage = USAGE_TYPES[(usage_p.cumsum(axis=1) < rng.random(n)[:, None]).sum(axis=1).clip(0, 3)]
        plan = rng.choice(REGULAR_PLANS, n, p=REGULAR_PLAN_P)
        excl = rng.random(n) < EXCLUDED_PLAN_SHARE
        plan = np.where(excl, rng.choice(config.EXCLUDED_PLANS, n), plan)
        spend = np.clip(rng.lognormal(np.log(4.5), 0.75, n), 0.3, 60.0)          # OMR / month, heavy-tailed
        churn = rng.beta(1.5, 6.0, n)
        churn = np.clip(churn + np.where(tenure < 60, 0.18, 0.0) + np.where(usage == "inactive", 0.35, 0.0), 0, 0.97)
        data_app = rng.beta(2.0, 4.0, n)
        data_app = np.where((usage == "voice_only") & (rng.random(n) < 0.45), np.clip(data_app + 0.4, 0, 1), data_app)
        voice_app = rng.beta(2.0, 4.0, n)
        voice_app = np.where((usage == "data_only") & (rng.random(n) < 0.45), np.clip(voice_app + 0.4, 0, 1), voice_app)
        price_sens = rng.beta(2.0, 3.0, n) * 2.0
        if "price_sensitive" in scenarios:
            price_sens = price_sens * 2.5
        churned = rng.random(n) < churn * 0.35                                   # some lines stop being active
        churn_day = np.where(churned, rng.integers(-40, 25, n), 10_000)          # offset (days) from as_of

        region = rng.choice(REGIONS, n)
        c = pd.DataFrame({
            "access_method_id": _unique_ids(rng, n, 1_000_000, 60),
            "msisdn": msisdns(rng, n),
            "account_id": _unique_ids(rng, n, 30_000_000, 50),
            "party_id": _unique_ids(rng, n, 5_000_000, 50),
            "subscription_id": (4_000_000_000 + _unique_ids(rng, n, 0, 40)).astype(str),
            "tenure": tenure,
            "nationality": nat,
            "gender": np.where(omani, rng.choice(["M", "F"], n), rng.choice(["M", "F"], n, p=[0.8, 0.2])),
            "age": np.clip(rng.normal(34, 10, n), 18, 75).astype(int),
            "plan": plan,
            "usage_type": usage,
            "region": region,
            "wilayat": [f"WILAYAT_{REGIONS.index(r) + 1:02d}{k}" for r, k in zip(region, rng.integers(1, 6, n))],
            "area": [f"AREA_{x:04d}" for x in rng.integers(1, 400, n)],
            "dv_segment": rng.choice(DV_SEGMENTS, n, p=[0.25, 0.35, 0.22, 0.12, 0.06]),
            "spend": spend,
            "churn_risk": churn,
            "data_appetite": data_app,
            "voice_appetite": voice_app,
            "price_sensitivity": price_sens,
            "preferred_validity": np.where(rng.random(n) < 0.55, 7, 30),
            "onnet_share": rng.uniform(0.4, 0.8, n),
            "intl_share": np.where(omani, rng.uniform(0.0, 0.04, n), rng.uniform(0.05, 0.3, n)),
            "payg_share": rng.uniform(0.1, 0.6, n),
            "churn_day": churn_day,
            "contacts_last_4w": rng.poisson(0.8, n),
            # travels abroad (drives roaming purchases): more often expats and high spenders
            "traveller": rng.random(n) < np.clip(np.where(omani, 0.05, 0.12) + 0.02 * np.clip(spend / 10, 0, 2), 0, 0.5),
            "wave": np.isin(region, list(WAVE_REGIONS)) if "churn_wave" in scenarios else np.zeros(n, bool),
            "growth": ((usage == "voice_only") & (data_app > 0.5)) if "data_growth" in scenarios else np.zeros(n, bool),
        })
        arms = [a for a in config.ARMS if a in (arms or config.ARMS)]
        if not arms:
            raise ValueError(f"arms must be a subset of {config.ARMS}")
        params = {"fatigue": 0.35 if "campaign_fatigue" in scenarios else 0.05, "arms": arms}
        return cls(seed, scenarios, as_of, c, build_catalog("new_offers" in scenarios), params)

    # -- time-varying state ----------------------------------------------------
    def ramp(self, day: date) -> float:
        """0 before the scenario drift starts, 1 at as_of (linear over SCENARIO_RAMP_DAYS)."""
        start = self.as_of - timedelta(days=config.SCENARIO_RAMP_DAYS)
        return float(np.clip((day - start).days / config.SCENARIO_RAMP_DAYS, 0.0, 1.0))

    def churn_risk_at(self, day: date, idx=slice(None)) -> np.ndarray:
        base = self.c["churn_risk"].to_numpy()[idx]
        wave = self.c["wave"].to_numpy()[idx]
        return np.clip(base + wave * 0.55 * self.ramp(day) * (1 - base), 0, 0.99)

    def data_appetite_at(self, day: date, idx=slice(None)) -> np.ndarray:
        base = self.c["data_appetite"].to_numpy()[idx]
        return np.clip(base + self.c["growth"].to_numpy()[idx] * 0.3 * self.ramp(day), 0, 1)

    # -- daily behaviour -------------------------------------------------------
    def daily(self, day: date, idx: slice, chunk_id: int) -> dict[str, np.ndarray]:
        """One day of behaviour for customers idx. Deterministic in (seed, day, chunk_id)."""
        rng = np.random.default_rng(np.random.SeedSequence([self.seed, 2, day.toordinal(), chunk_id]))
        c = self.c.iloc[idx]
        m = len(c)
        usage = c["usage_type"].to_numpy()
        spend = c["spend"].to_numpy()
        churn_eff = self.churn_risk_at(day, idx)
        off = (day - self.as_of).days
        gone = off >= c["churn_day"].to_numpy()
        base_act = np.select([usage == "dual", usage == "voice_only", usage == "data_only"], [0.88, 0.82, 0.82], 0.08)
        p_act = np.where(gone, 0.03, base_act * (1.0 - 0.45 * churn_eff))
        active = rng.random(m) < p_act

        voice_user = usage != "data_only"
        growth_on = c["growth"].to_numpy() & (rng.random(m) < 0.9 * self.ramp(day))
        data_user = (usage == "data_only") | (usage == "dual") | growth_on | ((usage == "inactive") & (rng.random(m) < 0.3))

        level = np.sqrt(spend / 4.5)
        calls = rng.poisson(4.0 * level) * active * voice_user
        mins_total = np.where(calls > 0, rng.gamma(np.maximum(1.5 * calls, 1e-9), 1.6), 0.0)
        intl = mins_total * c["intl_share"].to_numpy()
        local_min = mins_total - intl
        local_calls = np.rint(calls * (1 - c["intl_share"].to_numpy())).astype(int)
        onnet = c["onnet_share"].to_numpy()
        onnet_calls = np.rint(local_calls * onnet).astype(int)

        mb_per_session = 35.0 * level * (1.0 + 0.8 * self.data_appetite_at(day, idx))
        sessions = rng.poisson(np.where(growth_on & (usage == "voice_only"), 2.0 + 4.0 * self.ramp(day), 6.0) * level) * active * data_user
        mb = np.where(sessions > 0, rng.gamma(np.maximum(0.8 * sessions, 1e-9), mb_per_session), 0.0)

        payg = c["payg_share"].to_numpy()
        pg_voice = local_min * 0.012 * payg + intl * 0.08
        pg_data = mb * 0.0015 * payg
        pg_other = np.where(active & (rng.random(m) < 0.1), 0.05, 0.0)
        pg_neg = np.where(active & (rng.random(m) < 0.005), -rng.uniform(0.05, 0.5, m), 0.0)

        p_sub = np.clip(spend * (1 - 0.5 * payg) / 3.2 / 30.0, 0, 0.9) * np.where(gone, 0.05, 1.0)
        buys = rng.random(m) < p_sub
        amt = np.where(buys, np.maximum(0.3, np.round(spend * rng.uniform(0.15, 0.5, m) * 2) / 2), 0.0)
        cls_u = rng.random(m)
        p_data_cls = np.where(data_user, 0.6, 0.1)
        is_data = buys & (cls_u < p_data_cls)
        is_voice = buys & ~is_data & (cls_u < p_data_cls + 0.3)

        return {
            "voice_calls": calls.astype(float),
            "voice_min_rounded": mins_total * 1.07,
            "local_calls": local_calls.astype(float),
            "local_min": local_min,
            "onnet_calls": onnet_calls.astype(float),
            "onnet_min": local_min * onnet,
            "offnet_calls": (local_calls - onnet_calls).astype(float),
            "offnet_min": local_min * (1 - onnet),
            "data_sessions": sessions.astype(float),
            "data_mb_rounded": mb * 1.03,
            "data_mb": mb,
            "pg_voice": pg_voice,
            "pg_data": pg_data,
            "pg_pos": pg_voice + pg_data + pg_other,                 # positive charges only (BR-33 _tg)
            "pg_all": pg_voice + pg_data + pg_other + pg_neg,        # every charge (activities)
            "sub_amt": amt,                                          # SUBSCRIPTION_AMT (BR-19 _tg)
            "sub_tot_amt": amt * 1.05,                               # SUBSCRIPTION_TOT_AMT incl. tax (BR-19 NBA)
            "sub_amt_data": np.where(is_data, amt, 0.0),
            "sub_amt_voice": np.where(is_voice, amt, 0.0),
            "sub_count": buys.astype(float),
            "voice_day": (calls > 0).astype(float),
            "data_day": (sessions > 0).astype(float),
            "any_day": ((calls > 0) | (sessions > 0) | (np.abs(pg_other + pg_neg) > 0)).astype(float),
        }

    # -- hidden signals behind the model scores --------------------------------
    def signal(self, name: str, idx=slice(None), day: date | None = None) -> np.ndarray:
        """True propensity in [0, 1] that model `name` tries to estimate on `day` (default as_of): the scores of
        week D read the world at D, so a scenario drift shows up week after week."""
        day = day or self.as_of
        c = self.c.iloc[idx]
        churn = self.churn_risk_at(day, idx)
        gone = (day - self.as_of).days >= c["churn_day"].to_numpy()
        tenure = c["tenure"].to_numpy()
        usage = c["usage_type"].to_numpy()
        if name == "churn":
            return np.clip(churn + 0.25 * gone + np.where(tenure < 60, 0.08, 0.0), 0, 0.99)
        if name == "churn_1d":
            return np.clip(0.8 * churn * ~gone + 0.05, 0, 0.99)
        if name in ("reactivation_30", "reactivation_15"):
            k = 1.0 if name == "reactivation_30" else 0.75
            low = gone | (usage == "inactive")
            return np.clip(k * np.where(low, 0.75 * (1 - c["churn_risk"].to_numpy()), 0.08), 0, 0.99)
        if name == "data_appetite":
            return self.data_appetite_at(day, idx)
        if name == "voice_appetite":
            return c["voice_appetite"].to_numpy()
        if name == "revenue_decrease":
            return np.clip(0.6 * churn + 0.3 * c["wave"].to_numpy() * self.ramp(day) + 0.05, 0, 0.99)
        if name == "fragility_data":
            return np.clip(np.where(usage == "voice_only", 0.05, 0.5 * churn + 0.3 * (1 - c["data_appetite"].to_numpy())), 0, 0.99)
        if name == "fragility_voice":
            return np.clip(np.where(usage == "data_only", 0.05, 0.5 * churn + 0.3 * (1 - c["voice_appetite"].to_numpy())), 0, 0.99)
        # unknown signal (a model added through config only): a noisy mix of the main signals
        return np.clip(0.5 * churn + 0.5 * self.data_appetite_at(day, idx), 0, 0.99)

    def score(self, name: str, rng: np.random.Generator, idx=slice(None), day: date | None = None) -> np.ndarray:
        """Noisy model score = sigmoid(logit(signal) + noise)."""
        s = self.signal(name, idx, day)
        return _sigmoid(_logit(s) + rng.normal(0.0, 0.6, len(s)))

    # -- hidden response function ---------------------------------------------
    def accept_probability(self, idx: np.ndarray, action: np.ndarray, service: np.ndarray, generosity: np.ndarray,
                           price: np.ndarray, validity: np.ndarray, day: date | None = None) -> np.ndarray:
        """P(accept offer | customer, action, offer, price vs spend) for aligned arrays."""
        day = day or self.as_of
        c = self.c.iloc[idx]
        churn = self.churn_risk_at(day, idx)
        da = self.data_appetite_at(day, idx)
        va = c["voice_appetite"].to_numpy()
        usage = c["usage_type"].to_numpy()
        voice_only, data_only, dual = usage == "voice_only", usage == "data_only", usage == "dual"
        spend = c["spend"].to_numpy()

        fit = np.select(
            [action == "RETAIN", action == "UPSELL DATA", action == "CROSS SELL DATA", action == "UPSELL VOICE",
             action == "CROSS SELL VOICE", action == "UPSELL", action == "MAINTAIN"],
            [2.5 * churn, 2.0 * da * voice_only, 1.5 * da * ~data_only * (da > 0.3), 2.0 * va * data_only,
             1.5 * va * ~voice_only * (va > 0.3), 1.2 * np.clip(spend / 10, 0, 1), 0.8 * (1 - churn) * dual],
            0.0)
        service_fit = np.select([service == "DATA", service == "VOICE"], [da, va], 0.5 * (da + va))
        gen = np.vectorize(GENEROSITY_BONUS.get)(generosity) if len(generosity) else np.zeros(0)
        anchor = np.where(validity == 7, spend / 4.3, spend)
        gap = np.maximum(0.0, price - anchor) / np.maximum(anchor, 0.5)
        logit = (RESPONSE_BASE_LOGIT + 1.6 * fit + 1.2 * service_fit + gen
                 - 1.2 * c["price_sensitivity"].to_numpy() * gap
                 + 0.3 * (validity == c["preferred_validity"].to_numpy())
                 - self.params["fatigue"] * c["contacts_last_4w"].to_numpy())
        return _sigmoid(logit)

    def organic_probability(self, idx: np.ndarray) -> np.ndarray:
        """Weekly chance of buying a BTL with no contact (control arm baseline)."""
        spend = self.c["spend"].to_numpy()[idx]
        return np.clip(ORGANIC_WEEKLY_BTL_RATE + 0.02 * np.clip(spend / 10, 0, 1), 0, 0.2)


In [ ]:
%%engine nbx.fsdr.synth.generator
"""Dummy-data generator: writes exactly the files of the real data contract.

Every usage / revenue table is aggregated from the world's daily history with
the windows of nbx/fsdr/calendar.py (BR §4), so the values are consistent
with each other and with the run date: activities M1 = W1 + W2 + W3 + W4,
Sub_Amt_m1 >= the four weeks inside it, and W2 of D = W1 of D-7.

Last week's campaign is simulated too: targeting (D-8..D-2) by the segment
policy, offers chosen like production's constraint arm (all-L2), acceptances
drawn from the hidden response function, fulfilments in D-7..D-1 plus organic
purchases and non-BTL noise, and 270 days of purchase history for the ranker.

    run = generate_run(World.create(5000, seed=42, scenarios=["baseline"], as_of=d), d)
    write_run(run, Path("data/input") / d.isoformat())                    # 5 folders, JSON
    write_run(run, Path("data/input") / d.isoformat(), fmt="parquet")     # 1M mode: ~10x smaller on disk

V2.0 (plan §13): the files go to the 5 input folders (usage / models /
subscription / catalog / others) unless layout="flat" (V1.0), in JSON, JSON
Lines, CSV or Parquet. customer_profile (age, tenure in days) agrees with
party_*.AGE and the activation date, except where the data_quality_issues
scenario breaks it on purpose.
"""
from __future__ import annotations

import json
import time
from collections import defaultdict
from dataclasses import dataclass
from datetime import date, datetime, timedelta
from pathlib import Path

import numpy as np
import pandas as pd

from nbx.fsdr import calendar as cal
from nbx.fsdr import config, schemas
from nbx.fsdr.calendar import MONTHS, WEEKS
from nbx.fsdr.synth import scenarios as scen
from nbx.fsdr.synth.world import GENEROSITY_BONUS, World

NON_BTL_PRODUCTS = ["DAILY_DATA_PASS_1GB", "WEEKLY_SOCIAL_PASS", "INTL_CALLING_PACK_5", "HALA_FLEX_RENEWAL"]
NON_BTL_SHARE = 0.06


@dataclass
class GeneratedRun:
    run_date: date
    anchor_date: date
    tables: dict[str, pd.DataFrame]       # raw frames: python values, schema column order, None = null
    manifest: dict
    world: World


# ---------------------------------------------------------------------------
# Raw conversion (typed frame -> JSON-ready python values)
# ---------------------------------------------------------------------------
def to_raw(df: pd.DataFrame, table: str) -> pd.DataFrame:
    """Typed frame -> object frame holding exactly what the JSON file will hold."""
    schema = schemas.TABLES[table]
    n = len(df)
    cols = {}
    for f in schema.fields:
        if f.name not in df.columns:
            cols[f.name] = np.full(n, None, dtype=object)
            continue
        s = df[f.name]
        mask = s.isna().to_numpy()
        if f.raw_type in schemas.INT_TYPES:
            vals = pd.to_numeric(s, errors="coerce").fillna(0).round().astype("int64").astype(object)
        elif f.raw_type == "double":
            vals = pd.to_numeric(s, errors="coerce").fillna(0.0).astype(float).round(3).astype(object)
        else:
            vals = s.astype(str).astype(object)
        arr = vals.to_numpy(dtype=object)
        arr[mask] = None
        cols[f.name] = arr
    return pd.DataFrame(cols, columns=schema.field_names)


def _fmt_num(x: float) -> str:
    return f"{x:g}"


# ---------------------------------------------------------------------------
# Usage / revenue tables from the daily history
# ---------------------------------------------------------------------------
def _usage_tables(world: World, run_date: date) -> dict[str, pd.DataFrame]:
    t_anchor = cal.anchor_date(run_date)
    start = cal.usage_history_start(run_date)
    days = [start + timedelta(days=i) for i in range(config.HISTORY_DAYS_USAGE)]
    parts: dict[str, list[pd.DataFrame]] = defaultdict(list)

    def ix(d: date) -> int:
        i = (d - start).days
        if i < 0 or i >= len(days):
            raise ValueError(f"window day {d} outside the generated history {start}..{days[-1]}")
        return i

    for chunk_id, lo in enumerate(range(0, world.n, config.GENERATOR_CHUNK)):
        sl = slice(lo, min(lo + config.GENERATOR_CHUNK, world.n))
        daily = [world.daily(d, sl, chunk_id) for d in days]
        m = sl.stop - sl.start
        cum = {k: np.vstack([np.zeros((1, m)), np.cumsum(np.vstack([x[k] for x in daily]), axis=0)]) for k in daily[0]}

        def win(k: str, w: cal.Window) -> np.ndarray:
            return cum[k][ix(w.end) + 1] - cum[k][ix(w.start)]

        def span(k: str, a: date, b: date) -> np.ndarray:
            return cum[k][ix(min(b, days[-1])) + 1] - cum[k][ix(max(a, start))]

        c = world.c.iloc[sl]
        val, amid, acc = c["msisdn"].to_numpy(), c["access_method_id"].to_numpy(), c["account_id"].to_numpy()

        # activities_* (BR 4.2): one table per period and anchor
        for anchor_name, anchor in (("T", t_anchor), ("D", run_date)):
            for p, w in cal.activity_windows(anchor).items():
                cols = {
                    "ACCESS_METHOD_VAL": val, "ACCESS_METHOD_ID": amid, "ACCOUNT_ID": acc,
                    f"Call_Count_{p}": win("voice_calls", w), f"Data_Session_Count_{p}": win("data_sessions", w),
                    f"Voice_Traffic_min_{p}": win("voice_min_rounded", w), f"Data_Volume_MB_{p}": win("data_mb_rounded", w),
                    f"PG_Total_Amount_{p}": win("pg_all", w), f"PG_Voice_Amount_{p}": win("pg_voice", w),
                    f"PG_Data_Amount_{p}": win("pg_data", w), f"Onnet_Call_Count_{p}": win("onnet_calls", w),
                    f"Onnet_Voice_min_{p}": win("onnet_min", w), f"Offnet_Call_Count_{p}": win("offnet_calls", w),
                    f"Offnet_Voice_min_{p}": win("offnet_min", w), f"Local_Call_Count_{p}": win("local_calls", w),
                    f"Local_Voice_min_{p}": win("local_min", w), f"Active_Days_Count_{p}": win("any_day", w),
                    f"Active_Days_Voice_Count_{p}": win("voice_day", w), f"Active_Days_Data_Count_{p}": win("data_day", w),
                }
                present = (cols[f"Active_Days_Count_{p}"] > 0) | (np.abs(cols[f"PG_Total_Amount_{p}"]) > 0)
                parts[schemas.activity_table_name(p, anchor_name)].append(pd.DataFrame(cols)[present])

        # subs_amt_* (BR 4.1, SUBSCRIPTION_TOT_AMT): rows for lines with any subscription
        has_sub = span("sub_tot_amt", start, run_date) > 0
        for table, anchor in (("subs_amt_auto", t_anchor), ("subs_amt_score", run_date)):
            cols = {"ACCESS_METHOD_VAL": val, "ACCESS_METHOD_ID": amid, "ACCOUNT_ID": acc.astype(float)}
            cols.update({f"Sub_Amt_{p.lower()}": win("sub_tot_amt", w) for p, w in cal.subs_windows(anchor).items()})
            parts[table].append(pd.DataFrame(cols)[has_sub])

        # paygo_activity_auto (T, positive charges only)
        cols = {"ACCESS_METHOD_ID": amid, "ACCOUNT_ID": acc, "ACCESS_METHOD_VAL": val}
        cols.update({f"PG_Total_Amount_{p.lower()}": win("pg_pos", w) for p, w in cal.payg_windows(t_anchor).items()})
        present = span("pg_pos", start, t_anchor) > 0
        parts["paygo_activity_auto"].append(pd.DataFrame(cols)[present])

        # subscriptions_tg (D, SUBSCRIPTION_AMT, BR-17/18/19)
        cols = {"ACCESS_METHOD_VAL": val, "ACCESS_METHOD_ID": amid, "ACCOUNT_ID": acc.astype(float)}
        for p, w in cal.subscriptions_tg_windows(run_date).items():
            sfx = p.lower() if p in WEEKS else p
            if p in MONTHS:
                cols[f"Sub_Count_{p}"] = win("sub_count", w)
            cols[f"Total_Sub_Amt_{sfx}"] = win("sub_amt", w)
            cols[f"Data_Sub_Amt_{sfx}"] = win("sub_amt_data", w)
            cols[f"Voice_Sub_Amt_{sfx}"] = win("sub_amt_voice", w)
        present = span("sub_amt", run_date - timedelta(days=95), run_date - timedelta(days=1)) > 0
        parts["subscriptions_tg"].append(pd.DataFrame(cols)[present])

        # usage_data_voice_payg_tg (D, actual durations, positive PAYG, BR-33/34)
        cols = {"ACCESS_METHOD_ID": amid, "ACCOUNT_ID": acc, "ACCESS_METHOD_VAL": val}
        for p, w in cal.usage_tg_volume_windows(run_date).items():
            cols[f"local_Voice_min_{p}"] = win("local_min", w)
            cols[f"data_volume_mb_{p}"] = win("data_mb", w)
        cols.update({f"PG_Total_Amount_{p.lower()}": win("pg_pos", w) for p, w in cal.payg_windows(run_date).items()})
        present = span("any_day", run_date - timedelta(days=95), run_date - timedelta(days=2)) > 0
        parts["usage_data_voice_payg_tg"].append(pd.DataFrame(cols)[present])

    return {t: pd.concat(v, ignore_index=True) for t, v in parts.items()}


# ---------------------------------------------------------------------------
# Base and reference tables
# ---------------------------------------------------------------------------
def _catalog_strings(cat: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "btl_name": cat["btl_name"], "type": cat["type"], "generosity": cat["generosity"],
        "gb": cat["gb"].map(_fmt_num), "min": cat["min"].map(_fmt_num), "price": cat["price"].map(_fmt_num),
        "Price data": cat["price_data"].map(_fmt_num), "Price voice": cat["price_voice"].map(_fmt_num),
        "service": cat["service"], "validity": cat["validity"].map(str),
    })


def _offer_text(o) -> tuple[str, str]:
    bits_en, bits_ar = [], []
    if o["gb"]:
        bits_en.append(f"{o['gb']:g}GB data")
        bits_ar.append(f"{o['gb']:g} جيجابايت")
    if o["min"]:
        bits_en.append(f"{int(o['min'])} local minutes")
        bits_ar.append(f"{int(o['min'])} دقيقة محلية")
    en = f"{config.OPERATOR_NAME}: enjoy {' + '.join(bits_en)} for {o['price']:g} {config.CURRENCY}, valid {o['validity']} days. Reply YES to activate."
    ar = f"{config.OPERATOR_NAME}: استمتع بـ {' + '.join(bits_ar)} مقابل {o['price']:g} ر.ع لمدة {o['validity']} يوم. أرسل نعم للتفعيل."
    return en, ar


def _conflicts(cat: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for (_, _), g in cat.sort_values("price").groupby(["service", "validity"]):
        g = g.reset_index(drop=True)
        for i in range(len(g) - 1):
            a, b = g.iloc[i], g.iloc[i + 1]
            if b["price"] - a["price"] > 1.5:
                continue
            rows.append({
                "Offer_A": a["btl_name"], "Gen_Level_A": a["generosity"], "Price_A": _fmt_num(a["price"]),
                "Data_A": _fmt_num(a["gb"]), "Min_A": _fmt_num(a["min"]), "Validity_A": str(a["validity"]),
                "Offer_Type_A": a["service"],
                "Offer_B": b["btl_name"], "Gen_Level_B": b["generosity"], "Price_B": _fmt_num(b["price"]),
                "Data_B": _fmt_num(b["gb"]), "Min_B": _fmt_num(b["min"]), "Validity_B": str(b["validity"]),
                "Offer_Type_B": b["service"],
                "Price_Difference": _fmt_num(round(b["price"] - a["price"], 3)),
                "Data_Difference": _fmt_num(round(b["gb"] - a["gb"], 3)),
                "SAME_VALIDITY": "Y", "SAME_OFFER_TYPE": "Y",
            })
    return pd.DataFrame(rows)


def _base_tables(world: World, run_date: date, rng: np.random.Generator) -> dict[str, pd.DataFrame]:
    c = world.c
    act = [(run_date - timedelta(days=int(t))).isoformat() for t in c["tenure"]]
    month, year = cal.location_month(run_date)
    party = pd.DataFrame({"party_id": c["party_id"], "AGE": c["age"], "GENDER": c["gender"], "NATIONALITY": c["nationality"],
                          "nationality_identifier": np.where(c["nationality"] == "Oman", "OMANI", "EXPAT")})
    loc = pd.DataFrame({"Access_Method_id": c["access_method_id"], "Calling_Month": month, "Calling_Year": year,
                        "REGION": c["region"], "WILAYAT": c["wilayat"], "AREA": c["area"]})
    extra = loc.sample(frac=0.03, random_state=int(rng.integers(1 << 30))).copy()     # BR-14: several locations
    extra["AREA"] = [f"AREA_{x:04d}" for x in rng.integers(400, 800, len(extra))]
    cat = world.catalog
    scripts = [_offer_text(o) for _, o in cat.iterrows()]
    dv = c.sample(frac=0.92, random_state=int(rng.integers(1 << 30)))
    return {
        "base_prepaid": pd.DataFrame({
            "UBSCRIPTION_ID": c["subscription_id"], "ACCESS_METHOD_VAL": c["msisdn"], "ACCESS_METHOD_ID": c["access_method_id"],
            "ACCOUNT_ID": c["account_id"], "PARTY_ID": c["party_id"], "access_method_subs_start_dt": act,
            "CURRENT_PLAN": c["plan"]}),
        "consumer_aact_account": pd.DataFrame({"ACCOUNT_ID": c["account_id"], "PARTY_ID": c["party_id"]}),
        "access_method_id_auto": pd.DataFrame({"ACCESS_METHOD_ID": c["access_method_id"], "MSISDN": c["msisdn"],
                                               "ACTIVATION_DATE": act}),
        "party_auto": party,
        "party_scoring": party.copy(),
        "location_auto": pd.concat([loc, extra], ignore_index=True),
        "btl_catalog": _catalog_strings(cat),
        "BTL_script_english_arabic_ok": pd.DataFrame({
            "Sl No.": [str(i + 1) for i in range(len(cat))], "destination_offer_name": cat["btl_name"],
            "English Script": [s[0] for s in scripts], "Arabic Script": [s[1] for s in scripts]}),
        "conflicting_offers_h": _conflicts(cat),
        "DEVICEBEHAVIORSEGMENTATION.dv_segment": pd.DataFrame({"ACCESS_METHOD_ID": dv["access_method_id"],
                                                                "DV_SEGMENT": dv["dv_segment"]}),
    }


# ---------------------------------------------------------------------------
# Model scores (one shared structure)
# ---------------------------------------------------------------------------
def _life_range(t: np.ndarray) -> np.ndarray:
    return np.select([t <= 30, t <= 90, t <= 180, t <= 365, t <= 730], ["0-30", "31-90", "91-180", "181-365", "1Y-2Y"], "2Y+")


def generate_model_score(world: World, m: config.ModelScore, rng: np.random.Generator, day: date | None = None) -> pd.DataFrame:
    """Any model file in MODEL_SCORE_SCHEMA: driven by the matching hidden signal + noise, partial coverage."""
    t = world.c["tenure"].to_numpy()
    mask = np.ones(world.n, bool) if m.tenure_range is None else (t >= m.tenure_range[0]) & (t <= m.tenure_range[1])
    cand = np.flatnonzero(mask)
    k = int(len(cand) * rng.uniform(*m.coverage))
    idx = np.sort(rng.choice(cand, size=k, replace=False)) if k else np.array([], dtype=int)
    p1 = np.round(world.score(m.signal or m.feature.lower(), rng, idx, day), 6) if k else np.array([])
    c = world.c.iloc[idx]
    return pd.DataFrame({
        "subscription_id": c["subscription_id"].to_numpy(), "access_method_val": c["msisdn"].astype("int64").to_numpy(),
        "access_method_id": c["access_method_id"].to_numpy(), "tenure": c["tenure"].to_numpy(),
        "life_in_nw_range": _life_range(c["tenure"].to_numpy()), "proba_0": np.round(1 - p1, 6), "proba_1": p1,
        "prediction": np.where(p1 >= 0.5, "1", "0"),
    })


# ---------------------------------------------------------------------------
# Segments, last week's campaign, purchase history
# ---------------------------------------------------------------------------
def _segments(world: World, rng: np.random.Generator) -> np.ndarray:
    c = world.c
    churn = world.score("churn", rng)
    react = world.score("reactivation_30", rng)
    da, va = world.score("data_appetite", rng), world.score("voice_appetite", rng)
    frag = np.maximum(world.score("fragility_data", rng), world.score("fragility_voice", rng))
    tenure, usage = c["tenure"].to_numpy(), c["usage_type"].to_numpy()
    low = (c["churn_day"].to_numpy() <= 0) | (usage == "inactive")
    seg = np.select(
        [(tenure < config.NURSERING_MAX_TENURE_DAYS) & (churn >= 0.5), tenure < config.NURSERING_MAX_TENURE_DAYS,
         (tenure < config.ONBOARDING_MAX_TENURE_DAYS) & (churn >= 0.5), tenure < config.ONBOARDING_MAX_TENURE_DAYS,
         low & (react < 0.3), low & (react < 0.6), low,
         churn >= 0.6, frag >= 0.55, usage == "dual",
         (usage == "data_only") & (va >= 0.4), usage == "data_only", da >= 0.4],
        ["Nursering_Early_Churn_At_Risk", "Nursering_Early_Churn_Not_At_Risk",
         "Onboarding_Early_Churn_At_Risk", "Onboarding_Early_Churn_Not_At_Risk",
         "Retention_Low_Probability_To_Reconnect", "Retention_Med_Probability_To_Reconnect",
         "Retention_High_Probability_To_Reconnect",
         "Retention_RGS_At_Risk", "Retention_RGS_Fragile", "RGS_Not_At_Risk_DualPlay_Upgrade",
         "RGS_Not_At_Risk_Data_Only_Voice_Appetency", "RGS_Not_At_Risk_Data_Only_No_Voice_Appetency",
         "RGS_Not_At_Risk_Voice_Only_Data_Appetency"],
        "RGS_Not_At_Risk_Voice_Only_No_Data_Appetency").astype(object)
    # production emits both spellings ("RGS_..." and "Rgs_..."): mix them
    mixed = (rng.random(len(seg)) < 0.5) & np.array(["RGS" in s for s in seg])
    seg[mixed] = [s.replace("RGS", "Rgs").replace("DualPlay", "Dualplay") for s in seg[mixed]]
    return seg


def _preference_weights(world: World, idx: np.ndarray, cat: pd.DataFrame) -> np.ndarray:
    c = world.c.iloc[idx]
    da, va = c["data_appetite"].to_numpy()[:, None], c["voice_appetite"].to_numpy()[:, None]
    svc = cat["service"].to_numpy()[None, :]
    fit = np.where(svc == "DATA", da, np.where(svc == "VOICE", va, 0.5 * (da + va)))
    gen = np.vectorize(GENEROSITY_BONUS.get)(cat["generosity"].to_numpy())[None, :]
    validity = cat["validity"].to_numpy()[None, :]
    spend = c["spend"].to_numpy()[:, None]
    anchor = np.where(validity == 7, spend / 4.3, spend)
    gap = np.maximum(0, cat["price"].to_numpy()[None, :] - anchor) / np.maximum(anchor, 0.5)
    w = np.exp(1.4 * fit + 0.5 * gen - 0.8 * c["price_sensitivity"].to_numpy()[:, None] * gap)
    # add-on families: roaming is bought by travellers, international packs by people who call abroad
    travel = np.where(c["traveller"].to_numpy(), 1.0, 0.01)[:, None]
    intl = np.clip(c["intl_share"].to_numpy() * 6, 0.01, 2.0)[:, None]
    w = w * np.where(svc == "ROAMING", travel, 1.0) * np.where(svc == "INTERNATIONAL", intl, 1.0)
    w = w * (~cat["is_new"].to_numpy())[None, :]                              # new offers: no history (cold start)
    return w.astype(np.float32)


def _sample_offers(rng: np.random.Generator, weights: np.ndarray) -> np.ndarray:
    cdf = np.cumsum(weights / weights.sum(axis=1, keepdims=True), axis=1)
    return np.minimum((cdf < rng.random(len(weights))[:, None]).sum(axis=1), weights.shape[1] - 1)


def _legacy_offers(world: World, idx: np.ndarray, actions: np.ndarray) -> list[np.ndarray]:
    """Offers last week's campaign sent: production constraint arm with all-L2 (BR-66 active), the
    cheapest family offer above the customer's spend, per validity -> up to 4 destination offers."""
    cat = world.catalog[~world.catalog["is_new"]]
    spend = world.c["spend"].to_numpy()[idx]
    slots = [np.full(len(idx), None, dtype=object) for _ in range(4)]
    for action, (svc_a, svc_b) in config.ACTION_FAMILIES.items():
        sel = np.flatnonzero(actions == action)
        if not len(sel):
            continue
        for v_i, validity in enumerate((30, 7)):
            anchor = spend[sel] / (4.3 if validity == 7 else 1.0)
            for s_i, svc in enumerate((svc_a, svc_b)):
                fam = cat[(cat["service"] == svc) & (cat["validity"] == validity) & (cat["generosity"] == "L2")].sort_values("price")
                if fam.empty:
                    continue
                prices = fam["price"].to_numpy()
                pos = np.minimum(np.searchsorted(prices, anchor, side="right"), len(prices) - 1)
                slots[v_i * 2 + s_i][sel] = fam["btl_name"].to_numpy()[pos]
    return slots


def _ts(rng: np.random.Generator, days: list[date]) -> list[str]:
    secs = rng.integers(0, 86_400, len(days))
    return [(datetime.combine(d, datetime.min.time()) + timedelta(seconds=int(s))).isoformat() for d, s in zip(days, secs)]


def _our_campaign(world: World, campaign: pd.DataFrame, segments: np.ndarray):
    """Last week's campaign as logged by the app (campaign_log rows with offers): who, which action, which
    offers, which day -> the same arrays as the simulated legacy campaign."""
    pos = {m: i for i, m in enumerate(world.c["msisdn"].astype(str))}
    sent = campaign[campaign["offers"].map(lambda o: isinstance(o, list) and len(o) > 0)]
    sent = sent[sent["msisdn"].astype(str).isin(pos)].drop_duplicates("msisdn")
    targeted = np.array([pos[str(m)] for m in sent["msisdn"]], dtype=int)
    actions = sent["action"].astype(str).to_numpy(dtype=object)
    date1 = [date.fromisoformat(str(d)) for d in sent["run_date"]]
    slots = [np.full(len(sent), None, dtype=object) for _ in range(4)]
    for r, offers in enumerate(sent["offers"]):
        for i, o in enumerate(offers[:4]):
            slots[i][r] = o["code"]
    return targeted, segments[targeted].copy(), actions, date1, slots


def _campaign_tables(world: World, run_date: date, segments: np.ndarray, rng: np.random.Generator,
                     campaign: pd.DataFrame | None = None) -> dict[str, pd.DataFrame]:
    """Last week's campaign (targeting, responses, organic purchases, history). campaign = the app's own
    log of last week (closed-loop simulation); None = a simulated production campaign."""
    c = world.c
    n = world.n
    cat = world.catalog.set_index("btl_name")
    scripts = {name: _offer_text(row) for name, row in world.catalog.set_index("btl_name", drop=False).iterrows()}
    if campaign is not None:
        targeted, seg, actions, date1, slots = _our_campaign(world, campaign, segments)
    else:
        targeted = np.flatnonzero(rng.random(n) < config.TARGETED_SHARE)
        seg = segments[targeted].copy()
        seg[rng.random(len(seg)) < 0.01] = None                              # BR-37 rows without segment
        lower = np.array([s.lower() if s else "" for s in seg])
        actions = np.array([config.SEGMENT_TO_ACTION.get(s, "UPSELL" if s else "") for s in lower], dtype=object)
        d1_off = rng.integers(config.TARGETING_WINDOW[0], config.TARGETING_WINDOW[1] + 1, len(targeted))
        date1 = [run_date + timedelta(days=int(o)) for o in d1_off]
        slots = _legacy_offers(world, targeted, actions)

    targeting = pd.DataFrame({"date1": [d.isoformat() for d in date1], "access_method_id": c["access_method_id"].to_numpy()[targeted],
                              "account_id": c["account_id"].to_numpy()[targeted], "segment": seg})
    for i in range(6):
        targeting[f"destination_offer_name_{i + 1}"] = slots[i] if i < 4 else None

    # acceptances among the targeted (hidden response function)
    probs = np.zeros((len(targeted), 4))
    for i, s in enumerate(slots):
        ok = np.array([x is not None for x in s])
        if ok.any():
            o = cat.loc[s[ok]]
            probs[ok, i] = world.accept_probability(targeted[ok], actions[ok].astype(str), o["service"].to_numpy(),
                                                    o["generosity"].to_numpy(), o["price"].to_numpy(),
                                                    o["validity"].to_numpy(), day=run_date)
    p_any = 1 - np.prod(1 - probs, axis=1)
    takers = np.flatnonzero(rng.random(len(targeted)) < p_any)
    fulfil_rows = []
    btl_names = world.catalog.loc[~world.catalog["is_new"], "btl_name"].to_numpy()
    for t in takers:
        p = probs[t] / probs[t].sum()
        product = slots[int(rng.choice(4, p=p))][t]
        if rng.random() < 0.05:                                              # took another BTL than the one offered
            product = rng.choice(btl_names)
        first = max(date1[t], run_date + timedelta(days=config.REWARD_WINDOW[0]))
        span = (run_date + timedelta(days=config.REWARD_WINDOW[1]) - first).days
        fulfil_rows.append((targeted[t], product, first + timedelta(days=int(rng.integers(0, span + 1)))))

    # organic BTL purchases and non-BTL noise, D-7..D-1: every line that did not take a campaign offer -- being
    # contacted must not remove a customer's own buying (else contacted arms fall below the control)
    not_takers = np.setdiff1d(np.arange(n), targeted[takers])
    organic = not_takers[rng.random(len(not_takers)) < world.organic_probability(not_takers)]
    if len(organic):
        picks = _sample_offers(rng, _preference_weights(world, organic, world.catalog))
        for i, j in zip(organic, picks):
            fulfil_rows.append((i, world.catalog["btl_name"].iat[j], run_date - timedelta(days=int(rng.integers(1, 8)))))
    for i in np.flatnonzero(rng.random(n) < NON_BTL_SHARE):
        fulfil_rows.append((i, rng.choice(NON_BTL_PRODUCTS), run_date - timedelta(days=int(rng.integers(1, 8)))))

    f_idx = np.array([r[0] for r in fulfil_rows], dtype=int)
    fulfil = pd.DataFrame({"access_method_val": c["msisdn"].to_numpy()[f_idx], "PRODUCTNAME": [r[1] for r in fulfil_rows],
                           "FULFILMENTSUCCESSDATE_TIME": _ts(rng, [r[2] for r in fulfil_rows]), "sub_flag": 1, "nb_sub": 1})
    fulfil = fulfil.groupby(["access_method_val", "PRODUCTNAME", "FULFILMENTSUCCESSDATE_TIME"], as_index=False).agg(
        sub_flag=("sub_flag", "max"), nb_sub=("nb_sub", "sum"))

    # arbitration table (BR-22): a subset of the targeted, start_date in D-7..D
    arb = np.flatnonzero(rng.random(len(targeted)) < 0.35)
    start = [run_date + timedelta(days=int(o)) for o in rng.integers(config.ARBITRATION_WINDOW[0], config.ARBITRATION_WINDOW[1] + 1, len(arb))]
    arb_df = pd.DataFrame({"DATE1": [d.isoformat() for d in start], "START_DATE": [d.isoformat() for d in start],
                           "END_DATE": [(d + timedelta(days=6)).isoformat() for d in start],
                           "MSISDN": c["msisdn"].to_numpy()[targeted[arb]]})
    for i in range(1, 7):
        names = slots[i - 1][arb] if i <= 4 else np.full(len(arb), None, dtype=object)
        arb_df[f"CHANNEL_{i}"] = np.where([x is not None for x in names], "SMS", None)
        arb_df[f"OPPORTUNITY_NAME_{i}"] = [f"OPP_{a.replace(' ', '_')}" if x is not None else None for a, x in zip(actions[arb], names)]
        arb_df[f"DESTINATION_OFFER_NAME_{i}"] = names
        arb_df[f"OFFER_SCRIPT_EN_{i}"] = [scripts[x][0] if x is not None else None for x in names]
        arb_df[f"OFFER_SCRIPT_AR_{i}"] = [scripts[x][1] if x is not None else None for x in names]
    arb_df["FLAG_SPLIO"] = "N"
    arb_df["SEGMENT"] = seg[arb]
    arb_df["FILENAME"] = [f"arbitration_{d.isoformat()}.csv" for d in start]
    arb_df["ETL_CREATE_DT"] = [d.isoformat() for d in start]
    arb_df["JOB_ID"] = 10_000 + (run_date.toordinal() % 10_000)
    arb_df["SEGMENT_BTL_TAKER"] = rng.choice(["Y", "N"], len(arb))
    arb_df["SEGMENT_BTL_TAKER_UPSELL"] = rng.choice(["Y", "N"], len(arb))

    # 270-day purchase history (BR-24) for the ranker: preference-driven BTL purchases + last week's
    usage = c["usage_type"].to_numpy()
    lam = 3.0 * (0.4 + np.clip(c["spend"].to_numpy() / 8, 0, 1.2)) * np.where(usage == "inactive", 0.3, 1.0)
    counts = rng.poisson(lam)
    hist_rows = []
    for lo in range(0, n, config.GENERATOR_CHUNK):
        sel = np.arange(lo, min(lo + config.GENERATOR_CHUNK, n))
        ev = np.repeat(sel, counts[sel])
        if len(ev):
            w = _preference_weights(world, sel, world.catalog)
            picks = _sample_offers(rng, w[ev - lo])
            hist_rows.append(pd.DataFrame({"i": ev, "PRODUCTNAME": world.catalog["btl_name"].to_numpy()[picks]}))
    hist = pd.concat(hist_rows + [pd.DataFrame({"i": f_idx, "PRODUCTNAME": [r[1] for r in fulfil_rows]})], ignore_index=True)
    hist["access_method_val"] = c["msisdn"].to_numpy()[hist["i"].to_numpy()]
    history = hist.groupby(["access_method_val", "PRODUCTNAME"], as_index=False).size().rename(columns={"size": "nb_sub"})
    history["sub_flag"] = 1
    return {"targeting_dataiku_auto": targeting, "targeting_auto": arb_df, "subs_btl_auto": fulfil,
            "sub_btl_3months": history[["access_method_val", "PRODUCTNAME", "sub_flag", "nb_sub"]]}


# ---------------------------------------------------------------------------
# Test sample (BR-59..62): control + 3 arms, stratified
# ---------------------------------------------------------------------------
def _bucket(x: np.ndarray, edges: list[float], labels: list[str]) -> np.ndarray:
    return np.array(labels, dtype=object)[np.searchsorted(edges, x, side="left")]


def _arm_pattern(length: int, offset: int = 0, arms: list[str] | None = None) -> list[str]:
    """Interleaved arm sequence respecting ARM_SHARES over the chosen arms (stride scheduling). A random
    offset per stratum keeps small strata from all starting with the same arm."""
    shares = {a: s for a, s in config.ARM_SHARES.items() if a in (arms or config.ARMS)}
    total = sum(shares.values())
    marks = []
    for arm, share in ((a, s / total) for a, s in shares.items()):
        k = max(1, round(share * 100))
        marks += [((i + 0.5) / k, arm) for i in range(k)]
    seq = [a for _, a in sorted(marks)]
    return [seq[(offset + i) % len(seq)] for i in range(length)]


def _sample_table(world: World, usage: dict[str, pd.DataFrame], rng: np.random.Generator) -> pd.DataFrame:
    c = world.c
    eligible = c[~c["plan"].isin(config.EXCLUDED_PLANS)]
    k = int(len(eligible) * config.TARGET_SAMPLE_SHARE)
    s = eligible.sample(n=k, random_state=int(rng.integers(1 << 30))).copy()
    sub = usage["subscriptions_tg"].set_index("ACCESS_METHOD_VAL")
    use = usage["usage_data_voice_payg_tg"].set_index("ACCESS_METHOD_VAL")

    def col(frame, name):
        return frame[name].reindex(s["msisdn"]).fillna(0.0).to_numpy()

    sub_m1, payg_m1 = col(sub, "Total_Sub_Amt_M1"), col(use, "PG_Total_Amount_m1")
    total = sub_m1 + payg_m1
    total_3m = sum(col(sub, f"Total_Sub_Amt_M{i}") + col(use, f"PG_Total_Amount_m{i}") for i in (1, 2, 3))
    voice, data = col(use, "local_Voice_min_p1"), col(use, "data_volume_mb_p1")
    nat_group = np.where(s["nationality"].isin(config.NATIONALITY_KEEP), s["nationality"], config.NATIONALITY_OTHERS)
    rev_band = _bucket(total_3m, [0, 5, 15, 40], ["0", "0-5", "5-15", "15-40", "40+"])
    voice_g = _bucket(voice, [0, 100, 500], ["0", "1-100", "100-500", "500+"])
    data_g = _bucket(data, [0, 1000, 10000], ["0", "1-1000", "1000-10000", "10000+"])
    strata = np.array(["|".join(map(str, t)) for t in zip(nat_group, rev_band, voice_g, data_g, s["plan"])], dtype=object)

    group = np.empty(len(s), dtype=object)
    counts = np.empty(len(s), dtype=object)
    for st in pd.unique(strata):
        pos = np.flatnonzero(strata == st)
        rng.shuffle(pos)
        group[pos] = _arm_pattern(len(pos), int(rng.integers(0, 100)), world.params.get("arms"))
        counts[pos] = str(len(pos))

    def f3(x):
        return [f"{v:.3f}" for v in x]

    return pd.DataFrame({
        "access_method_val": s["msisdn"].to_numpy(), "access_method_id": s["access_method_id"].astype(str).to_numpy(),
        "account_id": s["account_id"].astype(str).to_numpy(), "tenure": s["tenure"].astype(str).to_numpy(),
        "activation_date": [(world.as_of - timedelta(days=int(t))).isoformat() for t in s["tenure"]],
        "plan_name": s["plan"].to_numpy(), "payg_revenue": f3(payg_m1), "subscription_revenue": f3(sub_m1),
        "total_revenue": f3(total), "total_revenue_3m": f3(total_3m), "voice_traffic": f3(voice), "data_volume": f3(data),
        "nationality": s["nationality"].to_numpy(), "region": s["region"].to_numpy(), "ARPU": f3(total),
        "ARPU_3m": f3(total_3m / 3), "voice_trf_group": voice_g, "data_trf_group": data_g,
        "Final": np.where(group == "cg", "CG", "TG"), "channel": "SMS", "group": group, "strata": strata, "counts": counts,
    })


# ---------------------------------------------------------------------------
# Orchestration
# ---------------------------------------------------------------------------
def _profile_table(world: World) -> pd.DataFrame:
    """customer_profile: msisdn, age, tenure. Same customers, ages and tenures as the other tables."""
    c = world.c
    tenure = c["tenure"].to_numpy()
    if config.PROFILE_TENURE_UNIT == "months":
        tenure = tenure // 30
    return pd.DataFrame({"msisdn": c["msisdn"].to_numpy(), "age": c["age"].to_numpy(), "tenure": tenure})


def _consent_table(world: World, run_date: date) -> pd.DataFrame:
    """V1.0: ~3 % do-not-contact, ~2 % marketing opt-out, fixed per customer. Own random stream, so every other
    table is identical to the one generated without consent data (same world for V0 / V1.0 comparisons)."""
    rng = np.random.default_rng(np.random.SeedSequence([world.seed, 9]))
    n = world.n
    return pd.DataFrame({"access_method_val": world.c["msisdn"].to_numpy(), "dnd": (rng.random(n) < 0.03).astype(int),
                         "marketing_opt_in": (rng.random(n) >= 0.02).astype(int),
                         "updated_at": [(run_date - timedelta(days=int(d))).isoformat() for d in rng.integers(1, 400, n)]})


def generate_run(world: World, run_date: date, campaign: pd.DataFrame | None = None) -> GeneratedRun:
    """Builds every table of the contract for run date D from the world. campaign: the app's logged campaign
    of D-7 (closed-loop simulation); its responses become this run's fulfilments."""
    t0 = time.time()
    rng = np.random.default_rng(np.random.SeedSequence([world.seed, 5, run_date.toordinal()]))
    typed: dict[str, pd.DataFrame] = {}
    typed.update(_base_tables(world, run_date, rng))
    usage = _usage_tables(world, run_date)
    typed.update(usage)
    for k, m in enumerate(config.MODEL_SCORE_REGISTRY):
        typed[m.table] = generate_model_score(world, m, np.random.default_rng(np.random.SeedSequence([world.seed, 3, k])), run_date)
    typed.update(_campaign_tables(world, run_date, _segments(world, rng), rng, campaign))
    typed[config.TARGET_SAMPLE_TABLE] = _sample_table(world, usage, rng)
    typed[config.CONSENT_TABLE] = _consent_table(world, run_date)
    typed[config.PROFILE_TABLE] = _profile_table(world)

    missing = set(schemas.TABLES) - set(typed)
    if missing:
        raise RuntimeError(f"generator did not build: {sorted(missing)}")
    tables = {name: to_raw(typed[name], name) for name in schemas.TABLES}
    issues = scen.inject_quality_issues(tables, world.seed) if world.has("data_quality_issues") else []

    manifest = {
        "run_date": run_date.isoformat(),
        "anchor_date": cal.anchor_date(run_date).isoformat(),
        "source": "generator",
        "operator": config.OPERATOR_NAME,
        "schema_version": config.SCHEMA_VERSION,
        "generator": {"n": world.n, "seed": world.seed, "scenarios": world.scenarios, "world_as_of": world.as_of.isoformat(),
                      "catalog_size": len(world.catalog), "new_offers": int(world.catalog["is_new"].sum())},
        "row_counts": {name: len(df) for name, df in tables.items()},
        "quality_issues_injected": issues,
        "windows": cal.all_windows(run_date),
        "generated_at": datetime.now().isoformat(timespec="seconds"),
        "generation_seconds": round(time.time() - t0, 2),
    }
    return GeneratedRun(run_date, cal.anchor_date(run_date), tables, manifest, world)


WRITE_FORMATS = ("csv.gz", "json", "jsonl", "csv", "parquet")   # csv.gz = production (V3.0)
LAYOUTS = ("split", "folders", "flat")         # split = V4.0 data/training/<D> + data/scoring/<D>


def _arrow_frame(df: pd.DataFrame, table: str) -> pd.DataFrame:
    """Raw frame -> typed columns for Parquet: Int64 / float64 / string by raw type. A column holding a value of
    the wrong type (data_quality_issues) is written as text, so the reader sees the same defect as in JSON."""
    types = schemas.TABLES[table].types()
    out = {}
    for c in df.columns:
        rt = types.get(c, "string")
        try:
            if rt in schemas.INT_TYPES:
                out[c] = pd.array(df[c].tolist(), dtype="Int64")
            elif rt == "double":
                out[c] = pd.array(df[c].tolist(), dtype="Float64")
            else:
                raise TypeError
        except (TypeError, ValueError):
            out[c] = pd.array([None if v is None else str(v) for v in df[c].tolist()], dtype="string")
    return pd.DataFrame(out, columns=df.columns)


def write_table(df: pd.DataFrame, table: str, path: Path, fmt: str) -> Path:
    """One raw frame -> one file in the given format."""
    if fmt == "csv.gz":
        from nbx.fsdr import csvstd

        csvstd.write_csv(df, path)                      # FSDR CSV standard + <table>.schema.json
    elif fmt == "parquet":
        _arrow_frame(df, table).to_parquet(path, index=False)
    elif fmt == "csv":
        df.to_csv(path, index=False, encoding="utf-8")
    else:
        records = df.to_dict("records")
        with path.open("w", encoding="utf-8") as fh:
            if fmt == "json":
                json.dump(records, fh, ensure_ascii=False, allow_nan=False)
            else:
                for r in records:
                    fh.write(json.dumps(r, ensure_ascii=False, allow_nan=False) + "\n")
    return path


def write_run(run: GeneratedRun, out_dir: Path, fmt: str = "json", layout: str = "folders") -> Path:
    """One file per table + _manifest.json. layout "folders": <out_dir>/<input folder>/<table>.<fmt> (plan §13);
    "flat": every file in out_dir (V1.0)."""
    if fmt not in WRITE_FORMATS:
        raise ValueError(f"fmt must be one of {WRITE_FORMATS}")
    if layout not in LAYOUTS:
        raise ValueError(f"layout must be one of {LAYOUTS}")
    out_dir = Path(out_dir)
    if layout == "split":
        return _write_split(run, out_dir, fmt)
    out_dir.mkdir(parents=True, exist_ok=True)
    for name, df in run.tables.items():
        folder = out_dir / schemas.TABLES[name].folder if layout == "folders" else out_dir
        folder.mkdir(exist_ok=True)
        write_table(df, name, folder / f"{name}.{fmt}", fmt)
    manifest = {**run.manifest, "format": fmt, "layout": layout,
                "folders": ({t: schemas.TABLES[t].folder for t in run.tables} if layout == "folders" else {})}
    (out_dir / config.MANIFEST_FILE).write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    return out_dir


def split_twins(training_dir: Path) -> tuple[Path, Path]:
    """data/training/<D> -> (data/training/<D>, data/scoring/<D>). Any other folder name: <folder>/training/<D> is
    not guessed, the caller passes the training folder of the V4.0 layout."""
    training_dir = Path(training_dir)
    if training_dir.parent.name != config.TRAINING_FOLDER:
        raise ValueError(f"split layout: write to <data>/{config.TRAINING_FOLDER}/<D>, got {training_dir}")
    return training_dir, training_dir.parent.parent / config.SCORING_FOLDER / training_dir.name


def _write_split(run: GeneratedRun, training_dir: Path, fmt: str) -> Path:
    """V4.0: the training tables under data/training/<D>/<folder>/, the scoring tables under data/scoring/<D>/<folder>/
    (config.SCORING_TABLES; btl_catalog in both), the same 5 folders on both sides, one _manifest.json in each."""
    twins = dict(zip((config.TRAINING_FOLDER, config.SCORING_FOLDER), split_twins(training_dir)))
    for root in twins.values():
        for f in config.INPUT_FOLDERS:
            (root / f).mkdir(parents=True, exist_ok=True)
    placed: dict[str, list[str]] = {k: [] for k in twins}
    for name, df in run.tables.items():
        sides = list(twins) if name in config.BOTH_TABLES else [config.table_side(name)]
        for side in sides:
            folder = twins[side] / (schemas.TABLES[name].folder if name in schemas.TABLES else "others")
            write_table(df, name, folder / f"{name}.{fmt}", fmt)
            placed[side].append(name)
    for side, root in twins.items():
        manifest = {**run.manifest, "format": fmt, "layout": "split", "side": side,
                    "twin": str(twins[config.SCORING_FOLDER if side == config.TRAINING_FOLDER else config.TRAINING_FOLDER]),
                    "folders": {t: schemas.TABLES[t].folder if t in schemas.TABLES else "others" for t in placed[side]}}
        (root / config.MANIFEST_FILE).write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")
    return training_dir


def generate(n: int = config.GENERATOR_DEFAULT_N, run_date: date | None = None, scenarios=None,
             seed: int = config.GENERATOR_SEED, out_dir: Path | None = None, fmt: str = "json",
             layout: str = "folders") -> GeneratedRun:
    """Convenience entry point (CLI / API): new world as of D, generate, optionally write."""
    run_date = run_date or date.today()
    world = World.create(n, seed=seed, scenarios=scen.parse(scenarios), as_of=run_date)
    run = generate_run(world, run_date)
    if out_dir is not None:
        write_run(run, out_dir, fmt, layout)
    return run


def generate_weeks(n: int, first_date: date, weeks: int, scenarios=None, seed: int = config.GENERATOR_SEED,
                   out_root: Path | None = None, fmt: str = "json", arms: list[str] | None = None,
                   layout: str = "folders") -> list[GeneratedRun]:
    """Consecutive weekly runs D, D+7, ... from ONE world anchored on the last run date, so the shared days
    agree (W2 of D+7 = W1 of D) and scenario drifts grow week after week. Writes <out_root>/<date>/."""
    if weeks < 1:
        raise ValueError("weeks must be >= 1")
    dates = [first_date + timedelta(days=7 * k) for k in range(weeks)]
    world = World.create(n, seed=seed, scenarios=scen.parse(scenarios), as_of=dates[-1], arms=arms)
    runs = [generate_run(world, d) for d in dates]
    if out_root is not None:
        for run in runs:
            write_run(run, Path(out_root) / run.run_date.isoformat(), fmt, layout)
    return runs


## Part B · The steps

The engine is loaded. Each step below calls it and shows what it wrote.

In [ ]:
step("Part B")
# --- contract: relax what no step of the flux reads (notebook 05 step 2 "NOT READ"), before any data check ---------
if RELAX_UNUSED_CONTRACT:
    _sch = use("nbx.fsdr", "schemas")
    _F = _sch.Field
    _sch.MODEL_SCORE_FIELDS[:] = [f for f in _sch.MODEL_SCORE_FIELDS if f.name not in ("subscription_id", "tenure", "life_in_nw_range")]
    _sch.refresh()                                           # the model files: access_method_val / _id, proba_0 / 1, prediction
    _bp = _sch.TABLES["base_prepaid"]
    _bp.fields = [f for f in _bp.fields if f.name not in ("UBSCRIPTION_ID", "SUBSCRIPTION_ID")]   # either spelling
    for _t, _c in (("access_method_id_auto", "MSISDN"), ("subs_amt_auto", "ACCOUNT_ID"), ("subs_amt_score", "ACCOUNT_ID")):
        _s = _sch.TABLES[_t]
        _s.fields = [_F(f.name, f.raw_type, False) if f.name == _c else f for f in _s.fields]
    print("contract relaxed (not read by the flux): model files without subscription_id / tenure / life_in_nw_range; "
          "base_prepaid.(S)UBSCRIPTION_ID; empty access_method_id_auto.MSISDN and subs_amt_*.ACCOUNT_ID allowed")
if TARGETING_AUTO_OPTIONAL:
    for _p in ("train_week0", "train_weekly", "score"):
        if "targeting_auto" in use("nbx.fsdr", "config").PIPELINE_TABLES[_p]["required"]:
            use("nbx.fsdr", "config").PIPELINE_TABLES[_p]["required"].remove("targeting_auto")
            use("nbx.fsdr", "config").PIPELINE_TABLES[_p]["optional"].append("targeting_auto")
    print("targeting_auto optional: empty = nobody excluded as 'in another campaign' (H-05)")
if REV_DECREASE_OPTIONAL:
    _cfg = use("nbx.fsdr", "config")
    _rd = "REV_DECREASE_204_SCORING_TD.predictions"
    for _p in ("train_week0", "train_weekly"):
        if _rd in _cfg.PIPELINE_TABLES[_p]["required"]:
            _cfg.PIPELINE_TABLES[_p]["required"].remove(_rd)
            _cfg.PIPELINE_TABLES[_p]["optional"].append(_rd)
    print(f"{_rd} optional: when absent, PROBA_REV_DECR is missing (flag 1) for every line and is not used by S-9 "
          "(fragility = FRAG_DATA / FRAG_VOICE)")

# --- paths and helpers (do not edit) ---------------------------------------------
import json, os, sys, time
from datetime import date
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

config, csvstd, steps = use("nbx.fsdr", "config", "csvstd", "steps")

pd.set_option("display.max_columns", 80, "display.width", 220, "display.max_colwidth", 80)
D = date.fromisoformat(RUN_DATE)
WEEK = config.FSDR_WEEKLY_DIR / RUN_DATE           # data/weekly/<D> (data/weekly_sample/<D> in TEST_10K)
RUN, CACHE, OUTPUT = config.FSDR_RUNS_DIR / RUN_DATE, config.FSDR_CACHE_DIR / RUN_DATE, config.output_folder(RUN_DATE)


def show(obj, n=PREVIEW_ROWS, title=None):
    """A table (DataFrame, list of dicts, dict or CSV / Parquet / csv.gz path), first n rows."""
    if isinstance(obj, (str, Path)):
        p = Path(obj)
        title = title or (str(p.relative_to(ROOT)) if p.is_relative_to(ROOT) else str(p))
        if not p.exists() or p.stat().st_size == 0:
            print(f"{title}: empty or not written")
            return
        try:
            obj = (csvstd.read_csv(p) if p.name.endswith(".csv.gz") else pd.read_parquet(p) if p.suffix == ".parquet"
                   else pd.read_csv(p) if p.suffix == ".csv" else json.loads(p.read_text(encoding="utf-8")))
        except pd.errors.EmptyDataError:
            print(f"{title}: no rows")
            return
    if isinstance(obj, dict):
        obj = pd.DataFrame([{"key": k, "value": v if not isinstance(v, (dict, list)) else json.dumps(v, default=str)[:150]}
                            for k, v in obj.items()])
    if isinstance(obj, list):
        obj = pd.DataFrame(obj)
    if title:
        display(Markdown(f"**{title}** · {len(obj):,} rows x {obj.shape[1]} columns"))
    display(obj.head(n))


def tree(folder, depth=3, skip_schemas=True):
    """The files of a folder, as a tree with sizes."""
    folder = Path(folder)
    if not folder.exists():
        print(f"{folder}: does not exist yet")
        return
    print(folder)
    for p in sorted(folder.rglob("*")):
        rel = p.relative_to(folder)
        if len(rel.parts) > depth or (skip_schemas and p.name.endswith(".schema.json")):
            continue
        pad = "    " * (len(rel.parts) - 1)
        print(f"{pad}{p.name}{'/' if p.is_dir() else f'  ({p.stat().st_size / 1e3:,.1f} kB)'}")


print(f"run date : {RUN_DATE}")
for name, p in (("weekly", WEEK), ("runs", RUN), ("cache", CACHE), ("output", OUTPUT)):
    print(f"{name:8s}: {p}  ({'found' if p.exists() else 'not yet'})")
print(f"offers per line: {config.RECO_OFFERS}  (FSDR_RECO_OFFERS)")
print(f"data logic: {'NBA (train *_auto at T, score *_score at D)' if SCORE_VIEW else 'week 0 (*_auto only)'}"
      + (" | TEST_10K" if TEST_10K else ""))

In [ ]:
# --- the LLM client: Ollama on this machine, JSON replies checked against a schema (do not edit) -------------
import hashlib, threading, urllib.error, urllib.request
from concurrent.futures import ThreadPoolExecutor
from urllib.parse import urlparse

assert urlparse(OLLAMA_URL).hostname in ("127.0.0.1", "localhost", "::1"), "local only: OLLAMA_URL must be this machine"
AGENTS_DIR = RUN / "agents"
AGENTS_DIR.mkdir(parents=True, exist_ok=True)
LLM_CACHE = AGENTS_DIR / "llm_cache.jsonl"       # every answer, keyed by (model, prompt): a re-run costs nothing
_cache, _lock = {}, threading.Lock()
if LLM_CACHE.exists():
    for line in LLM_CACHE.read_text(encoding="utf-8").splitlines():
        try:
            row = json.loads(line)
            _cache[row["key"]] = row["answer"]
        except (ValueError, KeyError):
            pass
STATS = {}                                       # per agent: calls, cached, errors, seconds, tokens


def _http(path, body=None, timeout=600):
    req = urllib.request.Request(OLLAMA_URL + path, data=None if body is None else json.dumps(body).encode("utf-8"),
                                 headers={"Content-Type": "application/json"})
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read().decode("utf-8"))


def _stat(agent):
    return STATS.setdefault(agent, {"calls": 0, "cached": 0, "errors": 0, "seconds": 0.0, "prompt_tokens": 0,
                                    "output_tokens": 0})


def ask(agent, system, user, schema, num_predict=256, use_cache=True):
    """One JSON answer of the local LLM, or None (the caller then uses the rule fallback). Temperature 0."""
    key = hashlib.sha256(json.dumps([MODEL, system, user, schema], sort_keys=True).encode()).hexdigest()
    st = _stat(agent)
    with _lock:
        if use_cache and key in _cache:
            st["cached"] += 1
            return _cache[key]
    body = {"model": MODEL, "stream": False, "format": schema, "keep_alive": "30m",
            "options": {"temperature": 0, "seed": SEED, "num_ctx": 4096, "num_predict": num_predict},
            "messages": [{"role": "system", "content": system + " Reply with compact JSON only."},
                         {"role": "user", "content": user}]}
    if THINK_OFF:
        body["think"] = False
    t = time.time()
    try:
        r = _http("/api/chat", body)
        answer = json.loads(r["message"]["content"])
    except (OSError, ValueError, KeyError, urllib.error.URLError) as e:
        with _lock:
            st["errors"] += 1
            st["last_error"] = str(e)[:200]
        return None
    with _lock:
        st["calls"] += 1
        st["seconds"] += time.time() - t
        st["prompt_tokens"] += r.get("prompt_eval_count", 0)
        st["output_tokens"] += r.get("eval_count", 0)
        if use_cache:
            _cache[key] = answer
            with LLM_CACHE.open("a", encoding="utf-8") as f:
                f.write(json.dumps({"key": key, "agent": agent, "answer": answer}) + "\n")
    return answer


def ask_many(agent, jobs):
    """jobs: (system, user, schema, num_predict[, use_cache]) tuples, sent PARALLEL at a time. Answers in order."""
    t = time.time()
    with ThreadPoolExecutor(max_workers=max(1, PARALLEL)) as ex:
        out = list(ex.map(lambda j: ask(agent, *j), jobs))
    _stat(agent)["wall_seconds"] = _stat(agent).get("wall_seconds", 0.0) + time.time() - t
    return out


def ints(n, lo=0, hi=100):
    """Schema: a list of exactly n integers in [lo, hi] (compact: one number per option)."""
    return {"type": "array", "items": {"type": "integer", "minimum": lo, "maximum": hi}, "minItems": n, "maxItems": n}


def fmt(x, nd=2):
    return "-" if x is None or (isinstance(x, float) and x != x) else (f"{x:.{nd}f}" if isinstance(x, float) else str(x))


print("LLM client ready (answers cached in", LLM_CACHE.relative_to(ROOT), ")")

## The LLM does not stop: retries, Ollama restart, cache, progress, Windows awake

- **Retries:** each request is retried `LLM_RETRIES` (6) times on a timeout, a connection error or an invalid JSON
  answer (one that does not match the schema), with a back-off of 2, 4, 8, 16, 32, 60 s. A retry changes the seed, so
  an invalid answer is not repeated.
- **Ollama restart:** when Ollama does not answer, `ollama serve` is started and the request waits for it (up to 60 s).
- **Cache:** every answer is appended to the cache file at once (`agents/llm_cache.jsonl`, and the shared
  `data/real25k/runs/_llm_cache.jsonl` from step 5b): a re-run resumes where it stopped.
- **Progress:** every 100 answers, one line with the speed and the ETA, printed and written to `agents/llm_progress.log`.
- **Windows awake:** sleep is blocked while the notebook runs (released by the final report).

In [ ]:
step("The LLM does not stop")
import ctypes, http.client, random as _rnd, shutil as _sh, subprocess as _sp, sys
from concurrent.futures import ThreadPoolExecutor, as_completed

LLM_LOG = AGENTS_DIR / "llm_progress.log"
_RESTART = threading.Lock()


def llm_log(msg):
    line = f"{datetime.now():%Y-%m-%d %H:%M:%S} {msg}"
    print(line, flush=True)
    with LLM_LOG.open("a", encoding="utf-8") as f:
        f.write(line + "\n")


def _valid(ans, schema):
    """The answer has the schema's shape: required keys, array lengths, enums (else it is retried)."""
    if schema.get("type") == "object":
        return isinstance(ans, dict) and all(k in ans for k in schema.get("required", [])) and all(
            _valid(ans[k], s) for k, s in schema.get("properties", {}).items() if k in ans)
    if schema.get("type") == "array":
        return (isinstance(ans, list) and schema.get("minItems", 0) <= len(ans) <= schema.get("maxItems", 10**9)
                and all(_valid(a, schema.get("items", {})) for a in ans))
    if "enum" in schema:
        return ans in schema["enum"]
    if schema.get("type") == "integer":
        return isinstance(ans, (int, float)) and schema.get("minimum", -1e18) <= ans <= schema.get("maximum", 1e18)
    return True


def ollama_alive():
    try:
        _http("/api/version", timeout=5)
        return True
    except Exception:
        return False


def restart_ollama():
    with _RESTART:
        if ollama_alive():
            return True
        exe = _sh.which("ollama")
        if not exe:
            llm_log("Ollama does not answer and the ollama command is not found: start the Ollama app")
            return False
        llm_log("Ollama does not answer: starting 'ollama serve'")
        _sp.Popen([exe, "serve"], stdout=_sp.DEVNULL, stderr=_sp.DEVNULL,
                  creationflags=getattr(_sp, "CREATE_NO_WINDOW", 0))
        for _ in range(30):
            time.sleep(2)
            if ollama_alive():
                llm_log("Ollama is back")
                return True
        llm_log("Ollama did not come back within 60 s")
        return False


def ask(agent, system, user, schema, num_predict=256, use_cache=True):
    """One JSON answer of the local LLM, retried up to LLM_RETRIES times; None only after every retry failed."""
    key = hashlib.sha256(json.dumps([MODEL, system, user, schema], sort_keys=True).encode()).hexdigest()
    st = _stat(agent)
    with _lock:
        st.setdefault("retried", 0)
        if use_cache and key in _cache:
            st["cached"] += 1
            return _cache[key]
    last = ""
    for attempt in range(LLM_RETRIES + 1):
        body = {"model": MODEL, "stream": False, "format": schema, "keep_alive": "60m",
                "options": {"temperature": 0, "seed": SEED + attempt, "num_ctx": 4096, "num_predict": num_predict},
                "messages": [{"role": "system", "content": system + " Reply with compact JSON only."},
                             {"role": "user", "content": user}]}
        if THINK_OFF:
            body["think"] = False
        t = time.time()
        try:
            r = _http("/api/chat", body, timeout=LLM_TIMEOUT_S)
            answer = json.loads(r["message"]["content"])
            if not _valid(answer, schema):
                raise ValueError("the answer does not match the schema")
            with _lock:
                st["calls"] += 1
                st["seconds"] += time.time() - t
                st["prompt_tokens"] += r.get("prompt_eval_count", 0)
                st["output_tokens"] += r.get("eval_count", 0)
                if use_cache:
                    _cache[key] = answer
                    with LLM_CACHE.open("a", encoding="utf-8") as f:       # at once: a re-run resumes from here
                        f.write(json.dumps({"key": key, "agent": agent, "answer": answer}) + "\n")
            return answer
        except (OSError, ValueError, KeyError, TypeError, http.client.HTTPException, urllib.error.URLError) as e:
            last = f"{type(e).__name__}: {str(e)[:160]}"
            if attempt == LLM_RETRIES:
                break
            with _lock:
                st["retried"] += 1
            if isinstance(e, (OSError, urllib.error.URLError, http.client.HTTPException)) and not ollama_alive():
                restart_ollama()
            time.sleep(min(60, 2 ** (attempt + 1)) * (0.8 + 0.4 * _rnd.random()))
    with _lock:
        st["errors"] += 1
        st["last_error"] = last
    llm_log(f"[llm] {agent}: an answer FAILED after {LLM_RETRIES} retries ({last})")
    return None


def ask_many(agent, jobs):
    """jobs: (system, user, schema, num_predict[, use_cache]); PARALLEL at a time; answers in order; progress every 100."""
    t, n, done = time.time(), len(jobs), 0
    out = [None] * n
    with ThreadPoolExecutor(max_workers=max(1, PARALLEL)) as ex:
        futs = {ex.submit(ask, agent, *j): i for i, j in enumerate(jobs)}
        for f in as_completed(futs):
            out[futs[f]] = f.result()
            done += 1
            if n >= 100 and (done % 100 == 0 or done == n):
                el = time.time() - t
                st = STATS.get(agent, {})
                llm_log(f"[llm] {agent}: {done:,}/{n:,} answers | {done / max(el, 1e-9) * 60:,.0f}/min | "
                        f"ETA {_mm((n - done) * el / max(done, 1))} | cached {st.get('cached', 0):,} | "
                        f"retried {st.get('retried', 0):,} | failed {st.get('errors', 0):,}")
    _stat(agent)["wall_seconds"] = _stat(agent).get("wall_seconds", 0.0) + time.time() - t
    return out


_AWAKE = 0x80000000 | 0x00000001                                   # ES_CONTINUOUS | ES_SYSTEM_REQUIRED
if sys.platform == "win32":
    ctypes.windll.kernel32.SetThreadExecutionState(_AWAKE)
    print("Windows kept awake until the final report")
print(f"LLM client: {LLM_RETRIES} retries, {LLM_TIMEOUT_S} s per request, progress in {LLM_LOG.relative_to(ROOT)}")

## G1 · The GPU and Ollama

This cell only reads. It checks the GPU (`nvidia-smi`), that Ollama answers on this machine, which models you have,
picks `MODEL`, loads it, and shows **how much of the model sits in GPU memory** (it must be 100 %: a model that
does not fit runs partly on the CPU and is much slower).

In [ ]:
step("G1")
import shutil, subprocess

smi = shutil.which("nvidia-smi")
if smi:
    q = subprocess.run([smi, "--query-gpu=name,memory.total,memory.used,utilization.gpu,driver_version",
                        "--format=csv,noheader"], capture_output=True, text=True, timeout=20).stdout.strip()
    print("GPU    :", q or "nvidia-smi gave no answer")
else:
    print("GPU    : nvidia-smi not found (no NVIDIA driver?): Ollama would run on the CPU, much slower")

try:
    print("Ollama :", _http("/api/version", timeout=5)["version"], "at", OLLAMA_URL)
except OSError as e:
    raise SystemExit(f"Ollama does not answer at {OLLAMA_URL}: start the Ollama app (or `ollama serve`), then run "
                     f"this cell again ({e})")
installed = [m["name"] for m in _http("/api/tags", timeout=10).get("models", [])]
print("models :", ", ".join(installed) or "none")
PREFERRED = ["qwen3:14b", "qwen2.5:14b", "qwen3:8b", "qwen2.5:7b-instruct-q8_0", "llama3.1:8b", "llama3.1:latest", "qwen2.5:7b",
             "mistral:7b", "qwen3:4b", "qwen3:1.7b"]
if MODEL == "auto":
    MODEL = next((m for m in PREFERRED if m in installed), installed[0] if installed else None)
if MODEL in ("qwen2.5:3b", "qwen3:1.7b", "qwen3:4b"):
    print(f"NOTE: {MODEL} is a small model (weaker answers): pull qwen2.5:7b or set MODEL")
if MODEL not in installed:
    raise SystemExit(f"model {MODEL!r} is not installed: run `ollama pull {MODEL}` (or set MODEL), then this cell again")
THINK_OFF = MODEL.startswith("qwen3")              # qwen3 "thinks" by default: off, the answers are short JSON
print("MODEL  :", MODEL)

t = time.time()
_http("/api/generate", {"model": MODEL, "prompt": "ok", "stream": False, "keep_alive": "30m",
                        "options": {"num_predict": 1}, **({"think": False} if THINK_OFF else {})})
print(f"loaded in {time.time() - t:.1f} s")
for m in _http("/api/ps", timeout=10).get("models", []):
    if m["name"] == MODEL:
        share = m.get("size_vram", 0) / max(m.get("size", 1), 1)
        print(f"in GPU memory: {share:.0%} of {m['size'] / 1e9:.1f} GB"
              + ("" if share > 0.99 else "  <- not all on the GPU: choose a smaller model, or close other GPU programs"))

## Pre-flight check

Before the long run: Ollama answers, `MODEL` is installed, the model is **100 % in GPU memory**, and
`OLLAMA_NUM_PARALLEL` (seen by this process) equals `PARALLEL`. With `PREFLIGHT_STRICT = True` a failed check stops
the notebook here, with what to fix.

In [ ]:
step("Pre-flight check")
_pf = []
if not ollama_alive() and not restart_ollama():
    _pf.append("Ollama does not answer on " + OLLAMA_URL + ": open the Ollama app (or ollama serve)")
else:
    _tags = [m["name"] for m in _http("/api/tags", timeout=30).get("models", [])]
    if MODEL not in _tags:
        _pf.append(f"model {MODEL} is not installed (ollama pull {MODEL})")
    _ps = next((m for m in _http("/api/ps", timeout=30).get("models", []) if m["name"] == MODEL), None)
    _gpu = None if _ps is None else _ps.get("size_vram", 0) / max(_ps.get("size", 1), 1)
    if _gpu is None or _gpu < 0.99:
        _pf.append(f"the model is {'not loaded' if _gpu is None else f'only {_gpu:.0%} in GPU memory'}: choose a smaller "
                   "MODEL or close the other GPU programs, then run from G1 again")
_np = os.environ.get("OLLAMA_NUM_PARALLEL")
if _np is None or str(_np).strip() != str(PARALLEL):
    _pf.append(f"OLLAMA_NUM_PARALLEL = {_np} in this process, PARALLEL = {PARALLEL}: run setx OLLAMA_NUM_PARALLEL {PARALLEL}, "
               "then restart Ollama AND Jupyter (or set PARALLEL to the server's value)")
print("pre-flight:", "OK" if not _pf else "FAILED")
for _x in _pf:
    print(" -", _x)
if _pf and PREFLIGHT_STRICT:
    raise SystemExit("pre-flight check failed (PREFLIGHT_STRICT = True): " + " | ".join(_pf))

## G2 · Speed test and the time estimate

Ten customers' worth of agent 3 (NBA) and agent 4 (NBO) prompts, `PARALLEL` at a time, on your GPU. From the measured
time it estimates this run and a production week (50k T2C + 50k T2F). The agents only need GPU time for the
customers of Target 2; everything else takes seconds to minutes on the CPU.

In [ ]:
step("G2")
ACTIONS = list(config.ACTIONS)
_sys = "You are a telecom marketing agent. Score each option 0-100 for this prepaid customer."
# the same shape and length as the real prompts of steps 6 and 7, so the estimate holds
_facts = ("segment Retention_RGS_Fragile, value band mid, usage both, tenure 812 days, revenue M1/M2/M3 4.20/5.10/5.60, "
          "data 2310 MB, voice 143 min, active days 27, churn 0.4{i}, data appetency 0.31, voice appetency 0.12, "
          "last week: targeted with UPSELL, no offer taken")
_nba = [(_sys, f"Customer: {_facts.format(i=i)}. Cell policy: UPSELL. Allowed actions, in order: {ACTIONS}. "
               "Return one score per allowed action, in that order, and a short reason.",
         {"type": "object", "properties": {"scores": ints(len(ACTIONS)), "reason": {"type": "string", "maxLength": 100}},
          "required": ["scores", "reason"]}, 120, False) for i in range(10)]
_offers = "; ".join(f"{k}: DATA L2, {30 if k < NBO_SHORTLIST else 7} days, price {1.5 + k / 2:.2f}, {2 + k:.1f} GB, - min, "
                    f"rule score 0.{50 + k}" for k in range(2 * NBO_SHORTLIST))
_nbo = [(_sys, f"Customer: {_facts.format(i=i)}. Action: UPSELL. Price anchor 4.{i}0. Offers: {_offers}. "
               f"Return one score per offer, in order.",
         {"type": "object", "properties": {"scores": ints(2 * NBO_SHORTLIST)}, "required": ["scores"]}, 160, False)
        for i in range(10)]
t = time.time(); ok_nba = ask_many("G2 speed test", _nba); s_nba = (time.time() - t) / len(_nba)
t = time.time(); ok_nbo = ask_many("G2 speed test", _nbo); s_nbo = (time.time() - t) / len(_nbo)
st = STATS["G2 speed test"]
tok_s = st["output_tokens"] / max(st["seconds"], 1e-9) * min(PARALLEL, len(_nba))
SEC_T2C, SEC_T2F = s_nba + s_nbo, s_nbo                 # per customer: T2C = NBA + NBO, T2F = NBO only
print(f"{MODEL}, PARALLEL {PARALLEL}: {s_nba:.2f} s per NBA call, {s_nbo:.2f} s per NBO call (wall, parallel) | "
      f"~{tok_s:.0f} output tokens/s | {sum(a is None for a in ok_nba + ok_nbo)} failed")
failed = sum(a is None for a in ok_nba + ok_nbo)
if failed == len(ok_nba + ok_nbo):
    raise SystemExit(f"every test call failed ({st.get('last_error')}): the LLM server is not working. Check the server "
                     "(its terminal or log), restart it, then run G1 and this cell again")
if failed:
    print(f"WARNING: {failed} test calls failed ({st.get('last_error')}): lower PARALLEL, or check the server")


def estimate(n_t2c, n_t2f):
    return n_t2c * SEC_T2C + n_t2f * SEC_T2F


print("estimated GPU time of agents 3 + 4 (the other agents add about 1-3 minutes):")
print(f"production week (50k T2C + 50k T2F): about {estimate(50_000, 50_000) / 3600:.1f} h with {MODEL} at PARALLEL {PARALLEL}")
print(f"per 1,000 Target-2 customers (half T2C, half T2F): about {estimate(500, 500) / 60:.1f} min")
if TEST_10K and TIME_BUDGET_MIN != float("inf"):
    print(f"TEST_10K: {TIME_BUDGET_MIN} min of budget = about {int(TIME_BUDGET_MIN * 60 / max(SEC_T2C, 1e-9)):,} T2C customers")

## The groups to score (`SCORE_GROUPS`)

Only the lists of `SCORE_GROUPS` are read: `["t2c"]` scores group a (`target2_a`), `["t2f"]` group b (`target2_b`),
both = the two lists. The other list may be absent from the week folder. Every step below (the quality gate of 4b,
P1, the agents, P6, the final file, the 360 view) then works on the chosen groups only.

In [ ]:
step("The groups to score")
_tg = use("nbx.fsdr", "targets")
_tg.read_targets = getattr(_tg, "_read_targets_orig", _tg.read_targets)
_tg._read_targets_orig = _tg.read_targets


def _read_chosen(folder):
    """The engine's read_targets, keeping only the lists of SCORE_GROUPS (and their issues)."""
    t, issues = _tg._read_targets_orig(folder)
    keep = t["GROUP_KEY"].isin(SCORE_GROUPS + [config.CONTROL_GROUP])
    return t[keep].reset_index(drop=True), [i for i in issues if i.get("group") in SCORE_GROUPS + [config.CONTROL_GROUP]]


_tg.read_targets = _read_chosen
print(f"target lists read: {SCORE_GROUPS} (sizes {config.group_sizes()})")

# --- the dates of the real run: T = the campaign day, purchases from T to D (before any engine step) -------------------
from datetime import date as _date
_cal = use("nbx.fsdr", "calendar")
_T = _date.fromisoformat(CAMPAIGN_DATE)
_cal.anchor_date = lambda d: _T                                  # training (*_auto) anchored on T
_n = (D - _T).days
config.REWARD_WINDOW = (-_n, 0)                                   # takers: purchases from T to D
config.TARGETING_WINDOW = (-_n, -_n)                              # last week's campaign: the day T
_ends = {**{f"activities_*_auto {p}": w.end for p, w in _cal.activity_windows(_T).items()},
         **{f"subs_amt_auto {p}": w.end for p, w in _cal.subs_windows(_T).items()},
         **{f"paygo_activity_auto {p}": w.end for p, w in _cal.payg_windows(_T).items()}}
_bad = {k: v for k, v in _ends.items() if (v >= _T if STRICT_AUTO_WINDOW else v > _T)}
if _bad:
    raise SystemExit(f"*_auto windows reach T = {_T}: {_bad} -- the training data must describe the customer before the campaign")
print(f"engine dates: D = {D}, T = {_T}; takers counted {_cal.campaign_windows(D)['reward'].start} .. "
      f"{_cal.campaign_windows(D)['reward'].end} ({_n + 1} days); the *_auto windows end on or before {max(_ends.values())}")

## Step 1 · The weekly folder (written by notebook 02)

`data/weekly/<D>/` (or `data/weekly_sample/<D>/` with `TEST_10K = True`): the 5 sub-folders with every table under its
contract name, the target lists `target2/a_constraint/target2_a.csv.gz` and `target2/b_free/target2_b.csv.gz`, and the
`_manifest.json` with D and T. Notebook 02 already renamed, checked the dates and removed the duplicates with the
NBA rule. **Both** the training tables (`*_auto`, `party_auto`) and the scoring tables (`*_score`, `party_scoring`)
are needed.

In [ ]:
step("Step 1")
generator = use("nbx.fsdr.synth", "generator")
JsonSource = use("nbx.fsdr.datasource", "JsonSource")

if GENERATE:
    man = WEEK / config.MANIFEST_FILE
    if WEEK.exists() and any(WEEK.iterdir()) and (not man.exists() or json.loads(man.read_text())["source"] != "generator"):
        raise SystemExit(f"{WEEK} holds real data: set GENERATE = False")
    run = generator.generate(N_LINES, D, ["baseline"], config.GENERATOR_SEED, WEEK, "csv.gz", "folders")
    print(f"dummy data: {len(run.tables)} tables for {N_LINES:,} lines")
if not GENERATE and not (WEEK.exists() and any(p.is_file() for p in WEEK.rglob("*.csv*"))):
    raise SystemExit(f"no exports in {WEEK}: run notebook 02 (prepare the week) with RUN_DATE = {RUN_DATE!r}"
                     + (" and TEST_10K = True" if TEST_10K else ""))
src = JsonSource(WEEK)
found = pd.DataFrame([{"folder": src.path(t).parent.name, "table": t} for t in src.tables()])
print(f"{len(found)} tables in {WEEK}")
man = json.loads((WEEK / config.MANIFEST_FILE).read_text(encoding="utf-8")) if (WEEK / config.MANIFEST_FILE).exists() else {}
if man.get("run_date") and man["run_date"] != RUN_DATE:
    raise SystemExit(f"the folder was prepared for {man['run_date']}, not {RUN_DATE}: same RUN_DATE in 02 and here")
print("prepared by notebook 02:", {k: man.get(k) for k in ("run_date", "anchor_date", "source")},
      "| duplicates removed:", sum(t["duplicates removed"] for t in man.get("dedup", {}).get("tables", [])))
SCORE_NEEDED = ["party_scoring", "subs_amt_score"] + [f"activities_{p}_score" for p in ("w1", "w2", "w3", "w4", "m1", "m2", "m3")]
miss_score = [t for t in SCORE_NEEDED if not src.has(t)]
if SCORE_VIEW and miss_score:
    print(f"scoring tables missing: {miss_score} -> step 4b uses their *_auto twin (anchor T) for those, and says so")
display(found.groupby("folder")["table"].apply(lambda s: ", ".join(sorted(s))).to_frame("tables"))

## Step 2 · Agent 1 (①) data agent

**The rules decide:** T0 checks every table against the data contract (columns, types, keys, ranges, windows) and
the completeness of the week; a *blocking* issue stops the run. It then writes the Parquet cache.
**The LLM explains:** it reads the report and writes, in plain words, whether the data is usable and what to fix.
It cannot unblock anything. The last part checks **which week** last week's campaign (`targeting_dataiku_auto`) is
from.

In [ ]:
step("Step 2")
if TEST_10K and "targeting_auto" in config.PIPELINE_TABLES["train_week0"]["required"]:
    config.PIPELINE_TABLES["train_week0"]["required"].remove("targeting_auto")    # TEST ONLY: may be empty (H-05 only)
    config.PIPELINE_TABLES["train_week0"]["optional"].append("targeting_auto")
data_agent = use("nbx.fsdr.contract", "data_agent")

t0 = data_agent.run_t0(WEEK, run_date=D, force=FORCE)
print(f"T0 {t0['status'].upper()} -- pipeline {t0['pipeline']}, {t0['tables']} tables, {t0['rows_in']:,} rows, "
      f"validation {t0['validation']} {t0['issue_counts']}")
rep = json.loads((RUN / "train" / "T0_load" / "validation_report.json").read_text(encoding="utf-8"))
top = [f"{i['level']}: {i['table']} {i.get('column') or ''} {i['message']}" for i in rep["issues"]
       if "not needed by the" not in i["message"]][:15]
a1 = ask("Agent 1 data", "You are the data agent of a telecom campaign engine. Explain a data validation report to a "
         "business user in plain words. Only report real problems.",
         f"Run {RUN_DATE}, status {t0['status']}, pipeline {t0['pipeline']}, {t0['tables']} tables, {t0['rows_in']} rows. "
         f"Issues to explain: {top or 'none'}. Training uses the *_auto history (anchor T) and scoring the *_score "
         "tables (anchor D). Say if the data is usable, the most important points, and what to fix first (nothing if "
         "there is no issue).",
         {"type": "object", "properties": {"usable": {"type": "boolean"}, "summary": {"type": "string", "maxLength": 500},
                                           "fix_first": {"type": "array", "items": {"type": "string", "maxLength": 160}, "maxItems": 3}},
          "required": ["usable", "summary", "fix_first"]}, 300)
display(Markdown(f"**Agent 1 (LLM):** {a1['summary'] if a1 else '(no LLM answer: the rule text above stands)'}"))
if a1 and a1["fix_first"]:
    display(Markdown("**Fix first:** " + "; ".join(a1["fix_first"])))
if t0["status"] != "done":
    raise SystemExit("T0 is BLOCKED by the rules: fix the data (see validation_report.md, or notebook 05), then run again")

# --- which week does last week's campaign (targeting_dataiku_auto) come from? ------------------------------------
_tcache = use("nbx.fsdr.contract.cache", "CacheSource")(D)
_win = use("nbx.fsdr", "calendar").campaign_windows(D)
if _tcache.has("targeting_dataiku_auto"):
    import pyarrow.parquet as _pq
    _d1 = []
    for _f in sorted((_tcache.dir / "targeting_dataiku_auto").glob("*.parquet")):
        _c = next((c for c in _pq.read_schema(_f).names if c.upper() == "DATE1"), None)
        if _c:
            _d1.append(pd.to_datetime(pd.read_parquet(_f, columns=[_c])[_c], errors="coerce").dt.date)
    _d1 = pd.concat(_d1) if _d1 else pd.Series(dtype=object)
    _in = ((_d1 >= _win["targeting"].start) & (_d1 <= _win["targeting"].end)).mean() if len(_d1) else 0.0
    print(f"last campaign (targeting_dataiku_auto): {len(_d1):,} rows, date1 {_d1.min()} .. {_d1.max()}; "
          f"{_in:.0%} inside the expected window {_win['targeting'].start} .. {_win['targeting'].end}; "
          f"takers counted {_win['reward'].start} .. {_win['reward'].end}")
    if _in < 0.5:
        print("WARNING: last week's targeting is from another week than the one expected for D. Agent 2b will learn "
              "from those customers with the takers of the reward window above: the rewards are then measured 1 week "
              "late (fewer takers, a weaker cell policy). The run goes on; for a real week, ask Dataiku for the "
              "targeting of the week before D, or set RUN_DATE so that its targeting window holds date1.")
else:
    print("no targeting_dataiku_auto: no history, agent 2b keeps its prior (the segment's own action)")

## Step 3 · Agent 2 (②) segmentation (training, `*_auto`, anchor T)

**The rules decide (CPU):** T2 builds the features of every line of the base from the `*_auto` history; T3 applies
the S-rules (first match wins) and gives every line its segment, bands, micro-segment and the two actions.
**The LLM explains** each segment in one line for the business (one call per segment).

In [ ]:
step("Step 3")
training = use("nbx.fsdr.pipelines", "training")

t2 = training.run_t2(D, workers=1, force=FORCE)
t3 = training.run_t3(D, workers=1, force=FORCE)
print(f"T2 {t2['lines']:,} lines, {t2['features']} features (anchor {t2['anchor']}) | T3 {t3['lines']:,} lines, "
      f"{t3['targetable']:,} targetable, {t3['excluded_nursering']:,} Nursering excluded")
seg = pd.read_csv(RUN / "train" / "T3_segments" / "segment_counts.csv")
jobs = [("You are the segmentation agent of a telecom campaign engine.",
         f"Segment {r.SEGMENT} (rule {r.RULE_ID}, action {r.RULE_ACTION}): {r.lines} lines ({r.share:.1%} of the base), "
         f"average ARPU {r.avg_arpu_m1}/month, {r.w0_targeted} targeted last week, {r.w0_takers} took an offer. "
         "In one sentence: who these customers are and what to watch.",
         {"type": "object", "properties": {"why": {"type": "string", "maxLength": 200}}, "required": ["why"]}, 90)
        for r in seg.itertuples()]
seg["agent_2_explains"] = [(a or {}).get("why", "-") for a in ask_many("Agent 2 segmentation", jobs)]
show(seg[["RULE_ID", "SEGMENT", "RULE_ACTION", "lines", "share", "avg_arpu_m1", "agent_2_explains"]], n=20)

## Step 4 · Agent 2b (②b) the agent dictionary (learnt on the training lines)

For every **segment × nationality** cell the LLM proposes the action(s) of Target 2, from the evidence of the cell:
size, value, risk and fragility shares, and **last week's reward rate per action** (who took an offer after being
targeted). The **check** (`validate_agent_dictionary`) then refuses an unknown action, an action for an excluded
segment, and an upsell as the top action of an at-risk segment (H-06); a refused cell keeps its current action. The
new version `dict_segm_actions_agent_v<N>.json` is applied to every line and next week starts from it.

In [ ]:
step("Step 4")
AT_RISK_SEGMENTS, UPSELL_ACTIONS = use("nbx.fsdr.segmentation.actions", "AT_RISK_SEGMENTS", "UPSELL_ACTIONS")

segs = pd.concat([pd.read_parquet(p) for p in sorted((RUN / "train" / "T3_segments" / "segments").glob("*.parquet"))])
if TEST_10K:
    segs = segs[segs["ACCESS_METHOD_VAL"].isin(TRAIN_VALS)]       # TEST_10K: agent 2b learns from the 10k training lines
    print(f"agent 2b evidence: {len(segs):,} training lines")
current = training.current_agent_dictionary(D)
cells, jobs = [], []
for c in current["cells"]:
    lines = segs[(segs["SEGMENT"] == c["segment"]) & (segs["NATIONALITY_GROUP"] == c["nationality_group"])]
    if c["excluded"] or not len(lines):
        continue
    orig = c["actions"][0]["action"] if c["actions"] else None
    allowed = [a for a in ACTIONS if not (c["segment"] in AT_RISK_SEGMENTS and a in UPSELL_ACTIONS and a != orig)]
    tg = lines[lines["W0_TARGETED"] == 1]
    rew = tg.groupby("W0_ACTION")["W0_REWARD"].agg(["mean", "size"]) if len(tg) else pd.DataFrame()
    reward = "; ".join(f"{a}: {m:.0%} of {n}" for a, (m, n) in rew.iterrows()) or "no targeting last week"
    cells.append((c, allowed))
    jobs.append(("You are the action-dictionary agent of a telecom campaign engine. Choose the best marketing action "
                 "for a whole cell of prepaid customers, from the evidence.",
                 f"Cell: segment {c['segment']}, nationality group {c['nationality_group']}, {len(lines)} lines. "
                 f"Average ARPU {lines['ARPU_M1'].mean():.2f}/month; at risk {lines['S_AT_RISK'].mean():.0%}; fragile "
                 f"{lines['S_FRAGILITY'].mean():.0%}; data users {lines['S_DATA_USER'].mean():.0%}; voice users "
                 f"{lines['S_VOICE_USER'].mean():.0%}. Current action: {[a['action'] for a in c['actions']]}. "
                 f"Last week's reward rate by action: {reward}. Allowed actions: {allowed}. Rank 1 to 2 actions.",
                 {"type": "object", "properties": {"actions": {"type": "array", "items": {"type": "string", "enum": allowed},
                                                               "minItems": 1, "maxItems": 2},
                                                   "reason": {"type": "string", "maxLength": 160}},
                  "required": ["actions", "reason"]}, 120))
answers = ask_many("Agent 2b dictionary", jobs)
proposal = {"cells": [dict(c, actions=[{"action": a, "rank": i + 1, "score": round(1 - i / 4, 2), "evidence_ids": ["W0", "T3"],
                                        "reason": ans["reason"]} for i, a in enumerate(dict.fromkeys(ans["actions"]))])
                      for (c, _), ans in zip(cells, answers) if ans and ans.get("actions")]}
r2b = training.update_agent_dictionary(D, proposal, made_by=f"agent 2b ({MODEL})")
new_top = [c for c in r2b["changed_cells"] if (c["before"] or [None])[0] != (c["after"] or [None])[0]]
refused = [p for p in r2b["problems"] if "missing cell" not in p["problems"][0]]
print(f"{r2b['version']} (parent {r2b['parent']}): {len(jobs)} cells asked, {len(proposal['cells'])} answered; "
      f"{len(new_top)} with a NEW top action (what Target 2 sends), {len(r2b['changed_cells']) - len(new_top)} only a new "
      f"second choice; {len(refused)} refused by the check (they keep the current action)")
show([{**c, "before": c["before"][0] if c["before"] else None, "after": c["after"][0]} for c in new_top]
     or [{"segment": "-", "after": "no top action changed"}], n=20, title="cells with a new top action")
show(refused or [{"problems": "none"}], n=10, title="refused by the check (H-06, unknown action, excluded segment)")

## Step 4b · The scoring view: the targets described on `*_score` (anchor D, NBA logic)

Training (T2 / T3, agent 2b) used the `*_auto` tables, anchored on the last Thursday T, because the reward of the last
campaign has to be explained by what the customer looked like **before** it. To decide **this** week, the customers
are described with the freshest data, the `*_score` tables and `party_scoring`, anchored on the run date D, exactly
as the NBA recipe scores its target list.

This step re-runs the **same** feature code (T2) and segment code (T3) with anchor D into `T2_features_score` and
`T3_segments_score`, with the training medians (missing scores) and ARPU bands, and the agent dictionary agent 2b just
wrote. P1, agents 3 / 4, P6 and the final file then read this view. The training steps are not touched.

**Quality gate:** a `*_score` table that is missing, or covers less than `SCORE_MIN_COVERAGE` of the target lines its
`*_auto` twin covers, is replaced by the twin for that period only (listed `FALLBACK`); a byte-identical copy is flagged.

It also writes the two NBA tables with the **NBA variable names** (`proba_churn`, `rev_w1_zero`, `log_var_rev_w1_w2`,
`voice_min_w1_zero`, `log_data_volume_w1_w2`, …, same formulas as `calcul_evol`) to `data/runs/<D>/nba/`:
`nba_train_context` (targeted lines on `*_auto`, with `action`, `reward`, `cost`) and `nba_score_context` (the target
lines on `*_score`, with `tg_flag`).

In [ ]:
step("Step 4b")
import numpy as np
import pyarrow.parquet as pq

bf = use("nbx.fsdr", "base_features")
targets = use("nbx.fsdr", "targets")
scoring_groups = use("nbx.fsdr.pipelines", "scoring_groups")
CacheSource = use("nbx.fsdr.contract.cache", "CacheSource")
load_catalog = use("nbx.fsdr.offer_rules", "load_catalog")
Step, fingerprint = use("nbx.fsdr.steps", "Step", "fingerprint")
cal = use("nbx.fsdr", "calendar")
schemas_mod = use("nbx.fsdr", "schemas")

T2_TRAIN, T3_TRAIN = training.T2, training.T3                       # T2_features / T3_segments  (*_auto, anchor T)
T2_SCORE, T3_SCORE = "T2_features_score", "T3_segments_score"      # the same steps on *_score + party_scoring, anchor D
train_dir = RUN / "train"
cache = CacheSource(D)
print(f"D = {D} (scoring, *_score)   T = {cal.anchor_date(D)} (training, *_auto)")

TWIN = {f"activities_{p}_score": schemas_mod.activity_table_name(p, "T") for p in ("w1", "w2", "w3", "w4", "m1", "m2", "m3")}
TWIN.update({"subs_amt_score": "subs_amt_auto", "party_scoring": "party_auto"})
TG_VALS = set(targets.read_targets(scoring_groups.input_folder(D))[0]["ACCESS_METHOD_VAL"].astype(str))   # the lines we score


def key_values(table):
    """The MSISDNs (or PARTY_IDs for the party tables) of a cached table, None if absent."""
    if not cache.has(table):
        return None, None
    vals, kind = [], None
    for f in sorted((cache.dir / table).glob("*.parquet")):
        names = pq.read_schema(f).names
        col = next((c for c in names if c.upper() == "ACCESS_METHOD_VAL"), None)
        kind = "msisdn" if col else "party"
        col = col or next((c for c in names if c.upper() == "PARTY_ID"), None)
        if col is None:
            return None, None
        vals.append(pd.read_parquet(f, columns=[col])[col].astype(str).str.strip())
    return (set(pd.concat(vals)) if vals else set()), kind


def table_bytes(table):
    return sum(f.stat().st_size for f in (cache.dir / table).glob("*.parquet")) if cache.has(table) else 0


REMAP, gate = {}, []
if SCORE_VIEW:
    for sc, au in TWIN.items():
        v_s, kind = key_values(sc)
        v_a, kind_a = key_values(au)
        kind = kind or kind_a
        if v_s is None and v_a is None:
            raise SystemExit(f"neither {sc} nor {au} is in the cache: export them")
        if kind == "msisdn":                                   # coverage of the target lines, the ones that matter here
            n_s, n_a = (None if v_s is None else len(v_s & TG_VALS)), len((v_a or set()) & TG_VALS)
        else:
            n_s, n_a = (None if v_s is None else len(v_s)), (None if v_a is None else len(v_a))
        cov = None if v_s is None or not n_a else n_s / n_a
        use_twin = v_s is None or (cov is not None and cov < SCORE_MIN_COVERAGE)
        copy = (kind == "msisdn" and v_s is not None and v_a is not None and v_s == v_a      # party tables may not change
                and table_bytes(sc) == table_bytes(au) and D != cal.anchor_date(D))
        if use_twin:
            REMAP[sc] = au
        gate.append({"scoring table": sc, "key": "target lines" if kind == "msisdn" else "parties", "in it": n_s,
                     "in twin (*_auto)": n_a, "coverage": None if cov is None else round(cov, 3),
                     "used": au + "  <- FALLBACK" if use_twin else sc + ("  (same file as *_auto: copy?)" if copy else "")})
    show(pd.DataFrame(gate), n=20, title=f"*_score quality gate on the {len(TG_VALS):,} target lines "
                                         f"(fallback to *_auto under {SCORE_MIN_COVERAGE:.0%} coverage)")
    if REMAP:
        print(f"WARNING: {len(REMAP)} scoring table(s) replaced by their *_auto twin: {sorted(REMAP)}. Those periods "
              "describe the customer at T, not D: ask Dataiku for complete *_score exports")
    if any("copy?" in str(g["used"]) for g in gate):
        print("WARNING: a *_score table is byte-identical to its *_auto twin although D != T: the run uses it, but it "
              "is the T period, not the D period")
# every read of a remapped *_score table goes to its twin (base_features reads tables through _part / _reference)
bf._part = getattr(bf, "_part_orig", bf._part)
bf._reference = getattr(bf, "_reference_orig", bf._reference)
bf._part_orig, bf._reference_orig = bf._part, bf._reference
bf._part = lambda c, table, part: bf._part_orig(c, REMAP.get(table, table), part)
bf._reference = lambda c, table: bf._reference_orig(c, REMAP.get(table, table))

if SCORE_VIEW:
    stats = json.loads((train_dir / T2_TRAIN / "stats.json").read_text(encoding="utf-8"))
    medians, cutoffs = stats["medians_by_nationality"], stats["arpu_tertiles"]   # learnt on *_auto: same scale as training
    agent_dict = training.current_agent_dictionary(D)                           # the version agent 2b just wrote
    catalog = load_catalog(cache) if cache.has("btl_catalog") else None
    inputs = fingerprint({"cache": str(cache.dir), "dict": agent_dict["name"], "medians": medians, "cutoffs": cutoffs})
    with Step(train_dir, T2_SCORE, inputs=inputs, force=True) as s2, Step(train_dir, T3_SCORE, inputs=inputs, force=True) as s3:
        n = 0
        for part in range(cache.partitions):
            feat, st = bf.partition_features(cache, part, D, "D")          # *_score + party_scoring (B-01 ... B-12)
            bf.impute(feat, medians)                                    # missing score -> training median + flag
            feat.to_parquet(s2.output(f"base_features/part-{part:02d}.parquet"), index=False)
            seg_p = training.segment_partition(feat, cache, part, D, cutoffs, agent_dict, catalog)
            seg_p.to_parquet(s3.output(f"segments/part-{part:02d}.parquet"), index=False)
            n += len(seg_p)
        s2.rows(rows_out=n), s3.rows(rows_out=n)
        s2.note(anchor="D", party_table=REMAP.get(bf.party_table("D", cache), bf.party_table("D", cache)), tables="*_score",
                fallback_to_auto=sorted(REMAP))
        s3.note(anchor="D", agent_dictionary=agent_dict["name"])
    # P1 / P6 / the final details now read the scoring view (the training steps are left untouched)
    scoring_groups.T2, scoring_groups.T3 = T2_SCORE, T3_SCORE
    segs_score = pd.concat([pd.read_parquet(p) for p in sorted((train_dir / T3_SCORE / "segments").glob("*.parquet"))])
    a = segs[["ACCESS_METHOD_VAL", "SEGMENT"]].merge(segs_score[["ACCESS_METHOD_VAL", "SEGMENT"]], on="ACCESS_METHOD_VAL",
                                                      suffixes=("_AUTO", "_SCORE"))
    moved = a[a["SEGMENT_AUTO"] != a["SEGMENT_SCORE"]]
    print(f"scoring view: {n:,} lines on *_score (anchor D) vs training on *_auto (anchor T); "
          f"{len(moved):,} lines ({len(moved) / max(len(a), 1):.1%}) changed segment between T and D; "
          f"agent dictionary {agent_dict['name']}")
    if len(moved):
        show(pd.crosstab(moved["SEGMENT_AUTO"], moved["SEGMENT_SCORE"]).stack().rename("lines").reset_index()
             .query("lines > 0").sort_values("lines", ascending=False), n=10, title="segment moves T -> D (largest)")
else:
    scoring_groups.T2, scoring_groups.T3 = T2_TRAIN, T3_TRAIN
    segs_score = pd.concat([pd.read_parquet(p) for p in sorted((train_dir / T3_TRAIN / "segments").glob("*.parquet"))])
    print("SCORE_VIEW = False: targets are described with the *_auto tables (anchor T)")
FEAT_DIR = RUN / "train" / scoring_groups.T2 / "base_features"         # used by agents 3 / 4 (facts of each customer)

# --- the NBA tables, with the NBA variable names (target_infos_vf / tg_all_infos_red of the Dataiku recipe) ------
NBA_PROBAS = ["PROBA_CHURN", "PROBA_REACT_30", "PROBA_REACT_15", "PROBA_APP_DATA", "PROBA_APP_VOICE", "PROBA_CHURN_1D",
              "PROBA_REV_DECR", "PROBA_FRAG_DATA", "PROBA_FRAG_VOICE"]
# calcul_evol(current, previous): exactly the NBA pairs and names (current = the later period index, previous = W1 ...)
NBA_EVOL = [("REV_W2", "REV_W1", "VAR_REV_W1_W2", "REV_W1_ZERO", "LOG_VAR_REV_W1_W2"),
            ("REV_W3", "REV_W2", "VAR_REV_W2_W3", "REV_W2_ZERO", "LOG_VAR_REV_W2_W3"),
            ("REV_W4", "REV_W3", "VAR_REV_W3_W4", "REV_W3_ZERO", "LOG_VAR_REV_W3_W4"),
            ("REV_M2", "REV_M1", "VAR_REV_M1_M2", "REV_M1_ZERO", "LOG_VAR_REV_M1_M2"),
            ("REV_M3", "REV_M2", "VAR_REV_M2_M3", "REV_M2_ZERO", "LOG_VAR_REV_M2_M3"),
            ("LOCAL_VOICE_MIN_W2", "LOCAL_VOICE_MIN_W1", "VAR_VOICE_W1_W2", "VOICE_MIN_W1_ZERO", "LOG_VOICE_MIN_W1_W2"),
            ("LOCAL_VOICE_MIN_W3", "LOCAL_VOICE_MIN_W2", "VAR_VOICE_W2_W3", "VOICE_MIN_W2_ZERO", "LOG_VOICE_MIN_W2_W3"),
            ("LOCAL_VOICE_MIN_W4", "LOCAL_VOICE_MIN_W3", "VAR_VOICE_W3_W4", "VOICE_MIN_W3_ZERO", "LOG_VOICE_MIN_W3_W4"),
            ("LOCAL_VOICE_MIN_M2", "LOCAL_VOICE_MIN_M1", "VAR_VOICE_M1_M2", "VOICE_MIN_M1_ZERO", "LOG_VOICE_MIN_M1_M2"),
            ("DATA_VOLUME_MB_W2", "DATA_VOLUME_MB_W1", "VAR_DATA_W1_W2", "DATA_VOLUME_MB_W1_ZERO", "LOG_DATA_VOLUME_W1_W2"),
            ("DATA_VOLUME_MB_W3", "DATA_VOLUME_MB_W2", "VAR_DATA_W2_W3", "DATA_VOLUME_MB_W2_ZERO", "LOG_DATA_VOLUME_W2_W3"),
            ("DATA_VOLUME_MB_W4", "DATA_VOLUME_MB_W3", "VAR_DATA_W3_W4", "DATA_VOLUME_MB_W3_ZERO", "LOG_DATA_VOLUME_W3_W4"),
            ("DATA_VOLUME_MB_M2", "DATA_VOLUME_MB_M1", "VAR_DATA_M1_M2", "DATA_VOLUME_MB_M1_ZERO", "LOG_DATA_VOLUME_M1_M2")]
NBA_RAW = [f"{k}_{p}" for k in ("REV", "LOCAL_VOICE_MIN", "DATA_VOLUME_MB") for p in cal.PERIODS]


def nba_table(t2_step: str, vals=None) -> pd.DataFrame:
    """One row per line: the NBA context with the NBA names (lower case, as the VW recipe). Missing model score ->
    uniform(0, 0.02) drawn PER ROW (BR-47; the recipe drew one value for all rows)."""
    cols = ["ACCESS_METHOD_VAL", "ACCESS_METHOD_ID", "NATIONALITY_GROUP", *NBA_PROBAS,
            *[f"{p}{config.MISSING_FLAG_SUFFIX}" for p in NBA_PROBAS], *NBA_RAW]
    df = pd.concat([pd.read_parquet(p, columns=cols) for p in sorted((train_dir / t2_step / "base_features").glob("*.parquet"))])
    if vals is not None:
        df = df[df["ACCESS_METHOD_VAL"].isin(vals)]
    rng = np.random.default_rng(SEED)
    for p in NBA_PROBAS:
        miss = df[f"{p}{config.MISSING_FLAG_SUFFIX}"] == 1
        df.loc[miss, p] = rng.uniform(0, 0.02, int(miss.sum()))
    for cur, prev, var, zero, logv in NBA_EVOL:
        c, q = df[cur].fillna(0.0), df[prev].fillna(0.0)
        df[zero] = (q == 0).astype(int)
        df[var] = np.where(q == 0, 0.0, (c - q) / q.where(q != 0, 1.0) * 100)
        df[logv] = np.log1p(c.clip(lower=0)) - np.log1p(q.clip(lower=0))
    df = df.drop(columns=[f"{p}{config.MISSING_FLAG_SUFFIX}" for p in NBA_PROBAS])
    df.columns = [c.lower() for c in df.columns]
    return df.drop_duplicates("access_method_val")


NBA_DIR = RUN / "nba"
NBA_DIR.mkdir(parents=True, exist_ok=True)
# training table = target_infos_vf: lines targeted last week (no Nursering), action of the segment, reward = BTL taker
lab = pd.concat([pd.read_parquet(p, columns=["ACCESS_METHOD_VAL", "W0_SEGMENT", "W0_TARGETED", "W0_ACTION", "W0_REWARD"])
                 for p in sorted((train_dir / T3_TRAIN / "segments").glob("*.parquet"))])
lab = lab[(lab["W0_TARGETED"] == 1) & lab["W0_ACTION"].notna()]
if TEST_10K:
    lab = lab[lab["ACCESS_METHOD_VAL"].isin(TRAIN_VALS)]
nba_train = nba_table(T2_TRAIN, set(lab["ACCESS_METHOD_VAL"])).merge(
    lab.rename(columns={"ACCESS_METHOD_VAL": "access_method_val", "W0_SEGMENT": "segment", "W0_ACTION": "action",
                        "W0_REWARD": "reward"})[["access_method_val", "segment", "action", "reward"]], on="access_method_val")
nba_train["cost"] = np.where(nba_train["reward"] == 1, -1, 0)
# scoring table = tg_all_infos_red: the target lines, described with the scoring view
tg, _ = targets.read_targets(scoring_groups.input_folder(D))
tg = tg[["ACCESS_METHOD_VAL", "GROUP_KEY"]].assign(TG_FLAG=lambda d: d["GROUP_KEY"].map(
    {k: g["tg_flag"] for k, g in config.GROUPS.items()}))
nba_score = nba_table(scoring_groups.T2, set(tg["ACCESS_METHOD_VAL"])).merge(
    tg.rename(columns=str.lower), on="access_method_val")
nba_train.to_parquet(NBA_DIR / "nba_train_context.parquet", index=False)
nba_score.to_parquet(NBA_DIR / "nba_score_context.parquet", index=False)
print(f"NBA tables (NBA names): training {len(nba_train):,} targeted lines on *_auto, "
      f"{int(nba_train['reward'].sum()):,} takers ({nba_train['reward'].mean() if len(nba_train) else 0:.1%}) | scoring "
      f"{len(nba_score):,} target lines on {'*_score' if SCORE_VIEW else '*_auto'} -> {NBA_DIR}")
show(nba_score, n=3, title="nba_score_context (first rows)")

## Step 5 · Target lists and the panel

The lists are read from the weekly folder (`target2/a_constraint/target2_a.csv.gz`, `target2/b_free/target2_b.csv.gz`;
with `TEST_10K` only `SCORE_GROUP`'s list has lines). P1 joins each listed line with its **scoring view** (step 4b):
segment, actions (the agent dictionary of step 4), bands.

In [ ]:
step("Step 5")
folder = scoring_groups.input_folder(D)
if GENERATE_TARGETS and scoring_groups.real_targets(folder):
    print("the folder holds YOUR lists: they are kept and used")
elif GENERATE_TARGETS:
    print("dummy lists:", scoring_groups.generate_targets(D)["sizes"])
p1 = scoring_groups.run_p1_panel(D, force=True)
panel = pd.read_parquet(RUN / "score" / "P1_panel" / "panel.parquet")
print(f"P1 {p1['status'].upper()} -- {len(panel):,} lines, sizes {p1['sizes']}, lists are "
      f"{'DUMMY' if targets.dummy_lists(folder) else 'YOURS'}")

## Step 5b · The customer profiles (FAST_MODE), before any LLM call

Each Target-2 customer's facts are rounded into bands, and the customers with the same rounded facts make one
**profile**. A profile is sent once, and its answer is used for every customer in it.

- **Bands:** revenue, data and voice use quantile bands over the target customers (shown as their value range, for
  example `revenue M1 2.10-3.40`). The probabilities use fixed bands (`0.2-0.4`), tenure and active days use fixed
  ranges, and the revenue trend is `down` / `flat` / `up` (M1 against the mean of M2 and M3, ±10 %).
- **Levels:** `fine` (5 bands, every fact), `medium` (3 bands, every fact), `coarse` (3 bands; leaves out data,
  voice, active days and tenure) and `minimal` (as coarse, also without the appetencies; last week is only
  "targeted, took an offer" / "targeted, no offer taken" / "not targeted").
- **Unit of work:** a profile × group (T2C: × the cell policy). It costs one NBA answer plus one NBO answer for T2C,
  and one NBO answer for T2F.
- **Budget:** profiles are taken in priority order (tier of the segment, then the biggest profiles first, with the
  groups interleaved) until `TIME_BUDGET_MIN`. Customers of the profiles past it get the cell policy.

In [ ]:
step("Step 5b")
llm_budget = use("nbx.fsdr", "llm_budget")

VAL = "ACCESS_METHOD_VAL"
feat = pd.concat([pd.read_parquet(p, columns=[VAL, "PROBA_CHURN", "PROBA_APP_DATA", "PROBA_APP_VOICE", "PROBA_FRAG_DATA",
                                              "REV_M1", "REV_M2", "REV_M3", "DATA_VOLUME_MB_M1", "LOCAL_VOICE_MIN_M1",
                                              "ACTIVE_DAYS_COUNT_M1", "TENURE_DAYS"])
                  for p in sorted(FEAT_DIR.glob("*.parquet"))])          # scoring view (D) or training (T)
ctx = panel.merge(feat, on=VAL, how="left").merge(segs_score[[VAL, "W0_TARGETED", "W0_ACTION", "W0_REWARD", "S_AT_RISK"]],
                                                  on=VAL, how="left").drop_duplicates(VAL).set_index(VAL)
t2 = panel[panel["TARGET"] == AGENT_TARGET].drop_duplicates(VAL).assign(DECIDED_BY="-")
queue = llm_budget.llm_queue(t2)[VAL].tolist()                     # priority tier first, groups interleaved
T = ctx.loc[t2[VAL]]


def facts_exact(v):
    """The exact facts of one customer: the notebooks2 prompt (PROFILE_LEVEL "off" and the quality check)."""
    r = ctx.loc[v]
    return (f"segment {r['SEGMENT']}, value band {r['VALUE_BAND']}, usage {r['USAGE_TYPE']}, tenure {fmt(r['TENURE_DAYS'], 0)} days, "
            f"revenue M1/M2/M3 {fmt(r['REV_M1'])}/{fmt(r['REV_M2'])}/{fmt(r['REV_M3'])}, data {fmt(r['DATA_VOLUME_MB_M1'], 0)} MB, "
            f"voice {fmt(r['LOCAL_VOICE_MIN_M1'], 0)} min, active days {fmt(r['ACTIVE_DAYS_COUNT_M1'], 0)}, churn {fmt(r['PROBA_CHURN'])}, "
            f"data appetency {fmt(r['PROBA_APP_DATA'])}, voice appetency {fmt(r['PROBA_APP_VOICE'])}, "
            f"last week: {'targeted with ' + str(r['W0_ACTION']) + (', took an offer' if r['W0_REWARD'] == 1 else ', no offer taken') if r['W0_TARGETED'] == 1 else 'not targeted'}")


def _num(col):
    return pd.to_numeric(T[col], errors="coerce") if col in T.columns else pd.Series(np.nan, index=T.index)


def _qband(x, k, nd=2):
    """k quantile bands of x over the target customers, written as their range ('-' = missing)."""
    out = pd.Series("-", index=x.index, dtype=object)
    ok = x.notna()
    if not ok.any():
        return out
    e = np.unique(np.quantile(x[ok], np.linspace(0, 1, k + 1)))
    if len(e) < 2:
        out[ok] = fmt(float(e[0]), nd)
        return out
    j = np.clip(np.searchsorted(e, x[ok].to_numpy(), side="right") - 1, 0, len(e) - 2)
    labels = [f"{fmt(float(e[i]), nd)}-{fmt(float(e[i + 1]), nd)}" for i in range(len(e) - 1)]
    out[ok] = [labels[i] for i in j]
    return out


def _fband(x, edges, names):
    """Fixed bands: edges[i] <= x < edges[i + 1] -> names[i] ('-' = missing)."""
    out = pd.Series("-", index=x.index, dtype=object)
    ok = x.notna()
    j = np.clip(np.searchsorted(np.asarray(edges[1:-1], float), x[ok].to_numpy(), side="right"), 0, len(names) - 1)
    out[ok] = [names[i] for i in j]
    return out


def _pband(x, k):
    e = np.linspace(0, 1, k + 1)
    return _fband(x.clip(0, 1), e, [f"{e[i]:.1f}-{e[i + 1]:.1f}" for i in range(k)])


_m1, _prev = _num("REV_M1"), pd.concat([_num("REV_M2"), _num("REV_M3")], axis=1).mean(axis=1)
_ratio = _m1 / _prev.where(_prev > 0)
TREND = pd.Series(np.select([_ratio < 0.9, _ratio > 1.1, _ratio.notna()], ["down", "up", "flat"], "-"), index=T.index)
TREND[_ratio.isna() & (_m1 > 0)] = "new revenue"
_lw = np.where(_num("W0_TARGETED") == 1, "targeted with " + T["W0_ACTION"].astype(str)
               + np.where(_num("W0_REWARD") == 1, ", took an offer", ", no offer taken"), "not targeted")
LAST_WEEK = pd.Series(_lw, index=T.index)
LAST_WEEK_SHORT = pd.Series(np.where(_num("W0_TARGETED") == 1, np.where(_num("W0_REWARD") == 1, "targeted, took an offer",
                                                                         "targeted, no offer taken"), "not targeted"), index=T.index)
LEVELS = {   # bands per fact (0 / None = the fact is left out of the prompt)
    "fine":    {"rev": 5, "data": 5, "voice": 5, "proba": 5, "apps": True, "lw": "full",
                "days": ([0, 8, 16, 24, 1e9], ["0-7", "8-15", "16-23", "24+"]),
                "tenure": ([0, 90, 365, 1095, 1e9], ["under 3 months", "3-12 months", "1-3 years", "over 3 years"])},
    "medium":  {"rev": 3, "data": 3, "voice": 3, "proba": 3, "apps": True, "lw": "full",
                "days": ([0, 11, 21, 1e9], ["0-10", "11-20", "21+"]),
                "tenure": ([0, 180, 730, 1e9], ["under 6 months", "6-24 months", "over 2 years"])},
    "coarse":  {"rev": 3, "data": 0, "voice": 0, "proba": 3, "apps": True, "lw": "full", "days": None, "tenure": None},
    "minimal": {"rev": 3, "data": 0, "voice": 0, "proba": 3, "apps": False, "lw": "short", "days": None, "tenure": None},
}
LEVEL_ORDER = list(LEVELS)                             # finest first


def profile_facts(level):
    """The rounded facts of every target customer at this level (a Series indexed by the line)."""
    if level == "off":
        return pd.Series([facts_exact(v) for v in T.index], index=T.index)
    L = LEVELS[level]
    s = ("segment " + T["SEGMENT"].astype(str) + ", value band " + T["VALUE_BAND"].astype(str) + ", usage "
         + T["USAGE_TYPE"].astype(str) + ", revenue M1 " + _qband(_m1, L["rev"]) + " (trend " + TREND + ")")
    if L["data"]:
        s = s + ", data " + _qband(_num("DATA_VOLUME_MB_M1"), L["data"], 0) + " MB"
    if L["voice"]:
        s = s + ", voice " + _qband(_num("LOCAL_VOICE_MIN_M1"), L["voice"], 0) + " min"
    if L["days"]:
        s = s + ", active days " + _fband(_num("ACTIVE_DAYS_COUNT_M1"), *L["days"])
    if L["tenure"]:
        s = s + ", tenure " + _fband(_num("TENURE_DAYS"), *L["tenure"])
    s = s + ", churn " + _pband(_num("PROBA_CHURN"), L["proba"])
    if L["apps"]:
        s = s + ", data appetency " + _pband(_num("PROBA_APP_DATA"), L["proba"]) + ", voice appetency " + _pband(_num("PROBA_APP_VOICE"), L["proba"])
    return s + ", last week: " + (LAST_WEEK if L["lw"] == "full" else LAST_WEEK_SHORT)


def units(level):
    """One row per target customer: its facts at this level and its unit of work (profile x group x cell policy)."""
    u = pd.DataFrame({"FACTS": profile_facts(level), "GROUP_KEY": T["GROUP_KEY"], "CELL": T["ACTION_AGENT"].astype(str),
                      "SEGMENT": T["SEGMENT"].astype(str)}, index=T.index)
    u["ACT"] = np.where(u["GROUP_KEY"] == "t2c", u["CELL"], "any")
    u["UNIT"] = u.groupby(["FACTS", "GROUP_KEY", "ACT"], sort=False).ngroup()
    return u


def cost(u):
    """Estimated GPU seconds of the units: T2C = NBA + NBO (the epsilon draw adds a few NBO answers), T2F = NBO."""
    n_c = u.loc[u["GROUP_KEY"] == "t2c", "UNIT"].nunique()
    n_f = u.loc[u["GROUP_KEY"] != "t2c", "UNIT"].nunique()
    return n_c, n_f, n_c * (s_nba + s_nbo * (1 + EPSILON)) + n_f * s_nbo


n_cust = len(T)
n_t2c = int((T["GROUP_KEY"] == "t2c").sum())
rows, UNITS = [], {}
for lv in LEVEL_ORDER:
    UNITS[lv] = units(lv)
    n_c, n_f, sec = cost(UNITS[lv])
    rows.append({"level": lv, "customers": n_cust, "T2C profiles": n_c, "T2F profiles": n_f,
                 "customers per profile": round(n_cust / max(n_c + n_f, 1), 1), "estimated GPU min": round(sec / 60, 1)})
_off = n_t2c * (s_nba + s_nbo) + (n_cust - n_t2c) * s_nbo
rows.append({"level": "off", "customers": n_cust, "T2C profiles": n_t2c, "T2F profiles": n_cust - n_t2c,
             "customers per profile": 1.0, "estimated GPU min": round(_off / 60, 1)})
LEVEL_TABLE = pd.DataFrame(rows)
show(LEVEL_TABLE, n=4, title=f"profiles of the {n_cust:,} Target-2 customers ({MODEL}, PARALLEL {PARALLEL}, speed of G2)")

if PROFILE_LEVEL == "auto":
    LEVEL = next((r["level"] for r in rows[:len(LEVEL_ORDER)] if r["estimated GPU min"] <= TIME_BUDGET_MIN), LEVEL_ORDER[-1])
else:
    assert PROFILE_LEVEL in LEVEL_ORDER + ["off"], f"PROFILE_LEVEL {PROFILE_LEVEL!r}"
    LEVEL = PROFILE_LEVEL
PROF = UNITS[LEVEL] if LEVEL in UNITS else units("off")

# --- the budget: units in priority order until TIME_BUDGET_MIN (or MAX_LLM_LINES customers) -------------------------
_rank = pd.Series(np.arange(len(queue)), index=queue)
U = PROF.assign(RANK=_rank.reindex(PROF.index).to_numpy()).groupby("UNIT").agg(
    n=("RANK", "size"), rank=("RANK", "min"), group=("GROUP_KEY", "first"), segment=("SEGMENT", "first"))
U["tier"] = U["segment"].map(llm_budget.TIER).fillna(len(llm_budget.PRIORITY_TIERS) + 1).astype(int)
U = U.sort_values(["tier", "n", "rank"], ascending=[True, False, True], kind="mergesort")
U["_k"] = U.groupby("group").cumcount()
U = U.sort_values(["_k", "group"], kind="mergesort")
U["sec"] = np.where(U["group"] == "t2c", s_nba + s_nbo * (1 + EPSILON), s_nbo)
keep = U["sec"].cumsum() <= TIME_BUDGET_MIN * 60
if MAX_LLM_LINES is not None:
    keep &= U["n"].cumsum() <= MAX_LLM_LINES
LLM_UNITS = set(U.index[keep])
LLM_LINES = set(PROF.index[PROF["UNIT"].isin(LLM_UNITS)])
FACTS = PROF["FACTS"]
print(f"PROFILE_LEVEL {PROFILE_LEVEL} -> {LEVEL}: {PROF['UNIT'].nunique():,} units of work for {n_cust:,} customers; "
      f"{len(LLM_UNITS):,} units = {len(LLM_LINES):,} customers fit {TIME_BUDGET_MIN} min "
      f"(about {U.loc[keep, 'sec'].sum() / 60:.1f} min); the rest ({n_cust - len(LLM_LINES):,}): cell policy")
show(PROF.loc[PROF["UNIT"].isin(LLM_UNITS)].groupby("UNIT").agg(customers=("FACTS", "size"), group=("GROUP_KEY", "first"),
                                                                 facts=("FACTS", "first"))
     .sort_values("customers", ascending=False), n=5, title="the biggest profiles (no MSISDN in a prompt)")

# --- the answers are kept across weeks: the same model + prompt = the same answer (temperature 0) -----------------
if SHARED_LLM_CACHE:
    _shared = config.FSDR_RUNS_DIR / "_llm_cache.jsonl"
    if _shared.exists():
        _n0 = len(_cache)
        for line in _shared.read_text(encoding="utf-8").splitlines():
            try:
                _row = json.loads(line)
                _cache.setdefault(_row["key"], _row["answer"])
            except (ValueError, KeyError):
                pass
        print(f"shared LLM cache: {len(_cache) - _n0:,} answers of earlier weeks loaded from {_shared.name}")
    LLM_CACHE = _shared                                # the new answers of agents 3 / 4 go there too

## Step 6 · Agent 3 (③) NBA: one answer per T2C profile, the action drawn per customer

For each **T2C profile** of the budget, the LLM scores the allowed actions from the profile's facts. No MSISDN is sent.
- **Allowed:** the 7 actions, minus an upsell for an at-risk segment (H-06). The segment and the cell policy are part
  of the profile, so the allowed list is the same for every customer in it.
- **Choice, per customer (`ACTION_POLICY`):**
  - `"greedy_rule"` (default): the agent's **best** action when its score leads the second one by `CLEAR_MARGIN`
    points or more; when the agent hesitates, the segment's **rule action** (BR-39, the old campaign's logic).
    Then, for a share ε of the customers, an action drawn uniformly (exploration): p(chosen) = 1 − ε + ε/k.
  - `"greedy"`: always the agent's best action, ε exploration. `"proportional"`: the previous draw
    p = (1 − ε)·score/Σscores + ε/k (with close scores most customers got another action than the best).
  - `DECIDED_BY` = `llm_nba` (the agent's best), `segment_rule` (the agent hesitated), `explore` (the ε draw).
    The **propensity** p of the chosen action is logged, so the uplift stays measurable.
- **Fallback:** a customer past the budget, or with an invalid answer, keeps the cell policy of agent 2b.
- **Quality check:** `FAST_CHECK_N` customers are also asked with their **exact** facts (the notebooks2 prompt), and
  the cell prints how often both give the same top action.

In [ ]:
step("Step 6")
def allowed_of(seg, cell):
    return [a for a in ACTIONS if not (seg in AT_RISK_SEGMENTS and a in UPSELL_ACTIONS and a != cell)]


SYS3 = ("You are the next-best-action agent of a telecom campaign engine. Score each allowed action 0-100 for this "
        "prepaid customer: the chance it makes the customer stay and spend more.")


def nba_job(facts_text, cell, allowed):
    return (SYS3, f"Customer: {facts_text}. Cell policy: {cell}. Allowed actions, in order: {allowed}. "
                  "Return one score per allowed action, in that order, and a short reason.",
            {"type": "object", "properties": {"scores": ints(len(allowed)), "reason": {"type": "string", "maxLength": 100}},
             "required": ["scores", "reason"]}, 120)


sel = PROF[PROF.index.isin(LLM_LINES) & (PROF["GROUP_KEY"] == "t2c")]
first = sel.groupby("UNIT", sort=False).head(1)
keys = [(u, allowed_of(s, c)) for u, s, c in zip(first["UNIT"], first["SEGMENT"], first["CELL"])]
answers = ask_many("Agent 3 NBA", [nba_job(f, c, al) for f, c, (_, al) in zip(first["FACTS"], first["CELL"], keys)])
ANS3 = {u: (al, ans) for (u, al), ans in zip(keys, answers)}


assert ACTION_POLICY in ("greedy_rule", "greedy", "proportional"), f"ACTION_POLICY {ACTION_POLICY!r}"
GAPS = []                                            # per customer: best score - second score (points of 100)


def rule_action(seg):
    """BR-39: the segment's production action, the logic of the old campaign."""
    k = str(seg).strip().lower()
    return config.SEGMENT_TO_ACTION.get(k) or config.SEGMENT_TO_ACTION.get("rgs_not_at_risk_" + k)


def choose_action(s, allowed, seg, v):
    """(index of the chosen action, the propensities p, DECIDED_BY) for one customer, own seed."""
    n = len(allowed)
    rng = np.random.default_rng([SEED, int(v) % 1_000_000_007])
    top = int(np.argmax(s))
    if ACTION_POLICY == "proportional":
        p = (1 - EPSILON) * (s / s.sum() if s.sum() > 0 else np.full(n, 1 / n)) + EPSILON / n
        p = p / p.sum()
        k = int(rng.choice(n, p=p))
        return k, p, "llm_nba" if k == top else "llm_nba_sampled"
    gap = float(np.sort(s)[-1] - np.sort(s)[-2]) if n > 1 else 100.0
    GAPS.append(gap)
    rule = rule_action(seg)
    if ACTION_POLICY == "greedy_rule" and gap < CLEAR_MARGIN and rule in allowed and allowed[top] != rule:
        best, how = allowed.index(rule), "segment_rule"
    else:
        best, how = top, "llm_nba"
    p = np.full(n, EPSILON / n)
    p[best] += 1 - EPSILON
    k = int(rng.choice(n, p=p))
    return k, p, how if k == best else "explore"


def ok3(ans, allowed):
    return bool(ans) and len(ans.get("scores", [])) == len(allowed)


rows = []
for v, unit, cell, seg in zip(sel.index, sel["UNIT"], sel["CELL"], sel["SEGMENT"]):
    allowed, ans = ANS3[unit]
    if not ok3(ans, allowed):
        rows.append({VAL: v, "ACTION": cell, "PROBA": 1.0, "DECIDED_BY": "fallback_cell_policy", "PROFILE": unit,
                     "REASON": "invalid answer"})
        continue
    s = np.clip(np.asarray(ans["scores"], float), 0, None)
    k, p, how = choose_action(s, allowed, seg, v)
    rows.append({VAL: v, "ACTION": allowed[k], "PROBA": round(float(p[k]), 6), "PROFILE": unit,
                 "DECIDED_BY": how, "REASON": ans.get("reason", "")})
NBA = pd.DataFrame(rows, columns=[VAL, "ACTION", "PROBA", "DECIDED_BY", "PROFILE", "REASON"])
NBA.drop(columns="REASON").to_csv(AGENTS_DIR / "agent3_nba.csv", index=False)
if len(NBA):
    _dec = NBA["DECIDED_BY"].value_counts()
    print(f"ACTION_POLICY {ACTION_POLICY} (EPSILON {EPSILON}, CLEAR_MARGIN {CLEAR_MARGIN}): "
          + ", ".join(f"{k} {n:,} ({n / len(NBA):.0%})" for k, n in _dec.items()))
    if GAPS:
        _g = np.asarray(GAPS)
        print(f"best - second score (points of 100): median {np.median(_g):.0f}; the agent hesitates (< CLEAR_MARGIN) "
              f"for {(_g < CLEAR_MARGIN).mean():.0%} of the customers")
    _rule = NBA.merge(sel[["SEGMENT"]], left_on=VAL, right_index=True)
    print(f"same action as the segment's rule (BR-39, the old campaign's logic): "
          f"{(_rule['ACTION'] == _rule['SEGMENT'].map(rule_action)).mean():.0%} of the customers")
if len(NBA):
    cmp = NBA.merge(ctx[["ACTION_AGENT"]], left_on=VAL, right_index=True)
    print(f"agent 3: {len(NBA):,} T2C customers from {len(ANS3):,} profile answers "
          f"({len(NBA) / max(len(ANS3), 1):.1f} customers per answer); same action as the cell policy for "
          f"{(cmp['ACTION'] == cmp['ACTION_AGENT']).mean():.0%}; {NBA['DECIDED_BY'].value_counts().to_dict()}")
    display(pd.crosstab(cmp["ACTION_AGENT"], cmp["ACTION"]).rename_axis(index="cell policy (2b)", columns="agent 3"))
    show(NBA, n=5, title="agent 3 decisions (first rows)")

# --- quality check: the profile's answer against the customer's exact facts ---------------------------------------
CHECK3 = {}
if FAST_CHECK_N and LEVEL != "off" and len(sel):
    smp = sel.sample(min(FAST_CHECK_N, len(sel)), random_state=SEED)
    exact = ask_many("Agent 3 check (exact facts)", [nba_job(facts_exact(v), c, ANS3[u][0])
                                                      for v, c, u in zip(smp.index, smp["CELL"], smp["UNIT"])])
    same, diff, n_ok = 0, [], 0
    for u, ex in zip(smp["UNIT"], exact):
        allowed, pa = ANS3[u]
        if ok3(ex, allowed) and ok3(pa, allowed):
            n_ok += 1
            a, b = np.asarray(pa["scores"], float), np.asarray(ex["scores"], float)
            same += int(np.argmax(a) == np.argmax(b))
            diff.append(float(np.abs(a - b).mean()))
    CHECK3 = {"customers": int(len(smp)), "compared": n_ok, "same_top_action": round(same / max(n_ok, 1), 3),
              "mean_score_gap": round(float(np.mean(diff)), 1) if diff else None}
    print(f"quality check ({LEVEL}): the profile and the exact facts give the same top action for "
          f"{CHECK3['same_top_action']:.0%} of {n_ok} customers (mean score gap {CHECK3['mean_score_gap']} points of 100)")
    if n_ok and CHECK3["same_top_action"] < 0.7:
        print("WARNING: the profiles lose too much of the customer's facts: set PROFILE_LEVEL to a finer level "
              "(or raise TIME_BUDGET_MIN so that 'auto' picks one), then run again from step 5b")

# --- guard: did the LLM really decide? -------------------------------------------------------------------------
if len(NBA):
    _fb = (NBA["DECIDED_BY"] == "fallback_cell_policy").mean()
    print(f"agent 3 by the LLM: {1 - _fb:.0%} of {len(NBA):,} T2C customers sent to it ({_fb:.0%} fallback)")
    if _fb > MAX_FALLBACK_SHARE:
        _msg = (f"agent 3: {_fb:.0%} of the answers failed or were invalid ({STATS.get('Agent 3 NBA', {}).get('last_error')}). "
                "Check the LLM server, lower PARALLEL, or pick a stronger MODEL; then run from G1 again")
        if STOP_ON_FALLBACK:
            raise SystemExit(_msg)
        print("WARNING:", _msg)
elif LLM_LINES:
    print("agent 3: no T2C customer in the LLM queue (only T2F lines?): agent 4 still scores the offers")

## Step 7 · Agent 4 (④) NBO: one answer per (profile, action), with the challenger rules (Agent 6B)

The **rules first**: for each Target 2 customer the universe rules (CC-v1 / CF-v1: families, catalogue, the
customer's relative price band) give the allowed offers, and the best `NBO_SHORTLIST` per validity (by the rule
score) are short-listed.

- **One answer per (profile, action):** the customers of a profile with the same action (T2F: "any") are grouped.
  The LLM scores the offers that are most often on their short lists, `NBO_SHORTLIST` per validity. Every customer of
  the group gets those scores for the offers the customer is allowed; the other allowed offers rank after them (rule score × 0.01).
- **Why not per short list:** each customer's short list and its order depend on his own rule scores, so almost
  no two customers share one. Grouping by short list saves next to nothing (measured: 1.2 customers per answer).
- **Price:** the rules have already kept only offers in the customer's price band, so in profile mode the prompt gives
  no per-customer price anchor (the revenue band of the profile stands for it).
- The passes **still choose** the offers per customer (price band, one per family, EXT-6 for slots 5 / 6).
  Customers past the budget keep the rule score. `PROFILE_LEVEL = "off"` = the notebooks2 prompt, per customer.
- **Coverage** (printed): the share of the customers' own short-listed offers that the group's answer scored.

In [ ]:
step("Step 7")
NBA_ACTION = NBA.set_index(VAL)["ACTION"] if len(NBA) else pd.Series(dtype=object)
EXACT = LEVEL == "off"
SYS4 = ("You are the next-best-offer agent of a telecom campaign engine. Score each offer 0-100: the chance "
        "this prepaid customer buys it and it raises his spend without overpricing.")
NBO_PROFILES = {}


def _offer_text(rows, with_score):
    return "; ".join(f"{i}: {r.SERVICE} {r.GENEROSITY}, {int(r.VALIDITY)} days, price {r.PRICE:.2f}, "
                     f"{fmt(getattr(r, 'GB', None), 1)} GB, {fmt(getattr(r, 'MIN', None), 0)} min"
                     + (f", rule score {r.SCORE:.2f}" if with_score else "") for i, r in enumerate(rows.itertuples()))


def _job(user, n):
    return (SYS4, user, {"type": "object", "properties": {"scores": ints(n)}, "required": ["scores"]}, 40 + 5 * n)


def nbo(group_key, cands):
    """Agent 4, called by P6 with the candidates the rules allowed (both validities): a score per row, NaN = rule score."""
    out = pd.Series(np.nan, index=cands.index)
    mine = cands[cands[VAL].isin(LLM_LINES)]
    acc = NBO_PROFILES.setdefault(group_key, {"customers": 0, "answers": 0, "own_short": 0, "covered": 0})
    if not len(mine):
        return out
    short = mine.sort_values("SCORE", ascending=False, kind="mergesort").groupby([VAL, "VALIDITY"], group_keys=False).head(NBO_SHORTLIST)
    act = {v: (NBA_ACTION.get(v, ctx.loc[v, "ACTION_AGENT"]) if group_key == "t2c" else "any (free universe)")
           for v in short[VAL].unique()}
    if EXACT:                                       # notebooks2: one prompt per customer, with his anchor and rule scores
        work, jobs = [], []
        for v, c in short.groupby(VAL, sort=False):
            anchor = fmt(c["ANCHOR"].iloc[0]) if "ANCHOR" in c.columns else "-"
            work.append((mine.index[mine[VAL] == v], c.index))
            jobs.append(_job(f"Customer: {facts_exact(v)}. Action: {act[v]}. Price anchor {anchor}. Offers: "
                             f"{_offer_text(c, True)}. Return one score per offer, in order.", len(c)))
        for (all_idx, short_idx), ans in zip(work, ask_many("Agent 4 NBO", jobs)):
            if ans and len(ans.get("scores", [])) == len(short_idx):
                out.loc[all_idx] = cands.loc[all_idx, "SCORE"] * 0.01
                out.loc[short_idx] = np.asarray(ans["scores"], float) / 100.0
        acc["customers"] += len(work); acc["answers"] += len(work)
        acc["own_short"] += len(short); acc["covered"] += len(short)
        return out
    # profile mode: one answer per (profile, action) over the offers most short-listed by its customers
    short = short.assign(_K=short[VAL].map(lambda v: (FACTS[v], act[v])))
    keys, jobs, lists = [], [], []
    for k, s in short.groupby("_K", sort=False):
        top = (s.groupby(["VALIDITY", "BTL_NAME"]).agg(n=(VAL, "size"), sc=("SCORE", "mean")).reset_index()
                .sort_values(["VALIDITY", "n", "sc"], ascending=[True, False, False], kind="mergesort")
                .groupby("VALIDITY").head(NBO_SHORTLIST))
        rows = s.drop_duplicates(["VALIDITY", "BTL_NAME"]).set_index(["VALIDITY", "BTL_NAME"]).loc[
            list(zip(top["VALIDITY"], top["BTL_NAME"]))].reset_index()
        keys.append(k); lists.append(rows[["VALIDITY", "BTL_NAME"]])
        jobs.append(_job(f"Customer: {k[0]}. Action: {k[1]}. Offers (all in the customer's price band, best first by the "
                         f"rules): {_offer_text(rows, False)}. Return one score per offer, in order.", len(rows)))
    scored = []
    for k, lst, ans in zip(keys, lists, ask_many("Agent 4 NBO", jobs)):
        if ans and len(ans.get("scores", [])) == len(lst):
            scored.append(lst.assign(_K=[k] * len(lst), LLM=np.asarray(ans["scores"], float) / 100.0))
    m = mine.assign(_K=mine[VAL].map(lambda v: (FACTS[v], act.get(v))))
    if scored:
        sc = pd.concat(scored, ignore_index=True)
        ok = m["_K"].isin(set(sc["_K"]))
        out.loc[m.index[ok]] = cands.loc[m.index[ok], "SCORE"] * 0.01          # allowed, not scored: after the LLM's
        hit = m.reset_index().merge(sc, on=["_K", "VALIDITY", "BTL_NAME"], how="inner").set_index("index")
        out.loc[hit.index] = hit["LLM"]
        own = short.reset_index().merge(sc, on=["_K", "VALIDITY", "BTL_NAME"], how="left")
        acc["own_short"] += len(own); acc["covered"] += int(own["LLM"].notna().sum())
    acc["customers"] += short[VAL].nunique(); acc["answers"] += len(jobs)
    return out


CAND_STORE = []


def _nbo_rec(group_key, cands):
    """Agent 4 (nbo above), and the scored candidates kept for the top-up of step 7b (the short list first)."""
    out = nbo(group_key, cands)
    CAND_STORE.append(pd.DataFrame({VAL: cands[VAL].astype(str).to_numpy(), "BTL_NAME": cands["BTL_NAME"].astype(str).to_numpy(),
                                    "VALIDITY": cands["VALIDITY"].to_numpy(), "PRICE": cands["PRICE"].to_numpy(),
                                    "RULE_SCORE": cands["SCORE"].to_numpy(),
                                    "SCORE_TOPUP": out.where(out.notna(), cands["SCORE"] * 0.01).to_numpy()}))
    return out


AGENTS = {"actions": NBA.drop(columns=["PROFILE", "REASON"]), "nbo": _nbo_rec, "groups": set(config.TARGETS[AGENT_TARGET]),
          "label": MODEL}
p6_t2 = scoring_groups.run_p6_offers(D, force=True, target=AGENT_TARGET, agents=AGENTS)
P6 = RUN / "score" / f"P6_offers_{AGENT_TARGET}"
print(f"P6 {AGENT_TARGET}: group audit {p6_t2['audit_status']} -- {p6_t2['decisions']}")
for _g, _s in NBO_PROFILES.items():
    print(f"agent 4, {_g}: {_s['customers']:,} customer requests (P6 batches) from {_s['answers']:,} answers "
          f"({_s['customers'] / max(_s['answers'], 1):.1f} per answer); the answer scored "
          f"{_s['covered'] / max(_s['own_short'], 1):.0%} of the customers' own short-listed offers")
show(pd.read_csv(P6 / "group_report.csv")[["group", "rules", "contacted", "with_offers", "llm_scored_offers", "decided_by"]],
     title="who decided, per group")
_st4 = STATS.get("Agent 4 NBO", {})
_n4 = _st4.get("calls", 0) + _st4.get("cached", 0) + _st4.get("errors", 0)
if _n4:
    _fb4 = _st4.get("errors", 0) / _n4
    print(f"agent 4 by the LLM: {1 - _fb4:.0%} of {_n4:,} NBO answers ({_fb4:.0%} failed -> rule scores)")
    if _fb4 > MAX_FALLBACK_SHARE:
        _msg = f"agent 4: {_fb4:.0%} of the calls failed ({_st4.get('last_error')})"
        if STOP_ON_FALLBACK:
            raise SystemExit(_msg)
        print("WARNING:", _msg)

## Step 7b · 6 offers for every line: the top-up with the engine's own rules

Code 5 (step 8) can leave a slot empty, for these reasons (step 8b counts them on your data):
- **slots 1–4:** the challenger price passes found no offer of that type in the price band (1.0–1.5× the anchor,
  widened once to 2.0×), and the fallback passes found none either;
- **slots 5–6:** no third offer of the validity above the anchor (EXT-6), and no favourite or forced bundle;
- **F-09 / F-11 / F-12:** an offer was cleared as a duplicate (same type and price, or same offer) or a conflict;
- **F-04:** a line with nothing in slots 1–4 is dropped from the file.

The top-up fills each empty slot (monthly slots 1, 2, 5 with a 30-day offer and weekly slots 3, 4, 6 with a 7-day offer
first, any validity after) from, in order:
1. **the short list**: the best `NBO_SHORTLIST` candidates per validity agent 4 scored for the line;
2. **the same action's offers**: the line's other allowed candidates (its universe: CC-v1 for T2C, CF-v1 for T2F);
3. **the favourites**: the line's own purchases of the last 270 days (code 5 F-06);
4. **the segment's other allowed actions**: the second choice of the cell first, then the others by their historical
   share; an at-risk segment gets no upsell (H-06); T2F keeps its free universe (L2 data and combo only).

Inside each source an offer priced at or above the line's usual spend (F-05) comes first. Every added offer is in the
catalogue, has both scripts (English and Arabic), and is not the same offer, nor the same type and price, nor in conflict
with another slot. `TOPUP_DROPPED = True` also tops up the lines F-04 drops, so they stay in the file. The auditor adds
**A-25: every row of the final file has 6 distinct, valid offers** (blocking).

In [ ]:
step("Step 7b")
fo = use("nbx.fsdr", "final_output")
uni = use("nbx.fsdr.offers", "universe")
SLOTS6 = range(1, 7)
MONTHLY_SLOT = {1, 2, 5}
_CAT = load_catalog(cache)
INFO = _CAT.assign(BTL_NAME=_CAT["BTL_NAME"].astype(str).str.strip()).drop_duplicates("BTL_NAME").set_index("BTL_NAME")
_scr = cache.read("BTL_script_english_arabic_ok") if cache.has("BTL_script_english_arabic_ok") else pd.DataFrame()
_scr.columns = [str(c).upper() for c in _scr.columns]
SCRIPTED = set() if _scr.empty else set(_scr.loc[(_scr["ENGLISH SCRIPT"].fillna("").astype(str).str.strip() != "")
                                                 & (_scr["ARABIC SCRIPT"].fillna("").astype(str).str.strip() != ""),
                                                 "DESTINATION_OFFER_NAME"].astype(str).str.strip())
VALID_OFFERS = set(INFO.index) & SCRIPTED
KEYTP = (INFO["TYPE"].astype(str) + "|" + INFO["PRICE"].astype(str)).to_dict()           # F-09: same type + price
VALIDITY_OF = pd.to_numeric(INFO["VALIDITY"], errors="coerce").to_dict()
SERVICE_OF = INFO["SERVICE"].astype(str).str.upper().to_dict()
PRICE_OF = pd.to_numeric(INFO["PRICE"], errors="coerce").to_dict()
CONFLICT = set(fo._conflict_pairs(cache).index)                                           # F-12 pairs "a|b"
_bought = uni.purchases(cache)
_bought = _bought[_bought["BTL_NAME"].isin(INFO.index)]
POP = _bought.groupby("BTL_NAME")["NB_SUB"].sum()                                         # E-O1: buyers in the base
REC = uni.recommendable(_CAT, _bought)
LINE = panel.drop_duplicates(VAL).set_index(VAL)
PRIOR_ORDER = sorted(config.ACTIONS, key=lambda a: -config.ACTION_PRIOR.get(a, 0))
_UNI = {}


def _ordered(names):
    names = pd.Index(pd.unique(pd.Series(list(names), dtype=object)))
    return sorted(names, key=lambda o: (-float(POP.get(o, 0)), -float(PRICE_OF.get(o, 0) or 0)))


def universe_offers(action, free_only):
    """The recommendable offers of an action's universe, both validities, by buyers then price."""
    k = (action, free_only)
    if k not in _UNI:
        names = [n for v in (30, 7) for n in uni.constraint_universe(REC, action, v)["BTL_NAME"].astype(str)]
        if free_only:
            names = [n for n in names if SERVICE_OF.get(n) in config.FREE_UNIVERSE_SERVICES
                     and str(INFO.at[n, "GENEROSITY"]) == config.FREE_UNIVERSE_GENEROSITY]
        _UNI[k] = _ordered(names)
    return _UNI[k]


def _cand_lists():
    """Per line: (short list, the action's other offers) from the candidates agent 4 scored (step 7)."""
    if not CAND_STORE:
        return {}, {}
    c = pd.concat(CAND_STORE, ignore_index=True).drop_duplicates([VAL, "BTL_NAME"], keep="last")
    c["IN_SHORT"] = c.groupby([VAL, "VALIDITY"])["RULE_SCORE"].rank(method="first", ascending=False) <= NBO_SHORTLIST
    c = c.sort_values([VAL, "SCORE_TOPUP", "PRICE"], ascending=[True, False, False], kind="mergesort")
    short = c[c["IN_SHORT"]].groupby(VAL, sort=False)["BTL_NAME"].agg(list).to_dict()
    rest = c[~c["IN_SHORT"]].groupby(VAL, sort=False)["BTL_NAME"].agg(list).to_dict()
    return short, rest


TOPUP_STATS = {"before": {}, "after": {}, "why_missing": {}, "added_by_source": {}, "brought_back_F04": 0,
               "still_short": 0, "rows": 0}
_CANDS = {}


def _bump(d, k, n=1):
    d[k] = d.get(k, 0) + n


def _anchor(s, v, o):
    per = "WEEKLY" if VALIDITY_OF.get(o) == 7 else "MONTHLY"
    col = f"F05_DATA_{per}" if SERVICE_OF.get(o) == "DATA" else f"F05_{per}"
    a = s.at[v, col] if col in s.columns else np.nan
    return np.nan if a is None else float(a)


def topup(s, scored, f04=False):
    """Fills the empty slots of s (index = line, S1..S6, REASON_i, REMOVED_i, F05_* anchors) in place."""
    if not _CANDS:
        _CANDS["short"], _CANDS["rest"] = _cand_lists()
    short, rest = _CANDS["short"], _CANDS["rest"]
    act = scored.assign(access_method_val=scored["access_method_val"].astype(str)).drop_duplicates(
        "access_method_val").set_index("access_method_val")["action"]
    fav = fo.favourites(cache, s.index, _CAT)
    fav = fav.assign(BTL_NAME=fav["BTL_NAME"].astype(str)).groupby(VAL, sort=False)["BTL_NAME"].agg(list).to_dict()
    cols = [f"S{i}" for i in SLOTS6]
    for i in SLOTS6:                         # an offer without both scripts (or not in the catalogue) is replaced (A-25)
        c = f"S{i}"
        bad = s[c].notna() & ~s[c].astype(str).str.strip().isin(VALID_OFFERS)
        if bad.any():
            if f"REMOVED_{i}" not in s.columns:
                s[f"REMOVED_{i}"] = None
            s.loc[bad, f"REMOVED_{i}"] = "INVALID: " + s.loc[bad, c].astype(str)
            s.loc[bad, c] = None
            if f"REASON_{i}" in s.columns:
                s.loc[bad, f"REASON_{i}"] = None
            _bump(TOPUP_STATS, "invalid_replaced", int(bad.sum()))
    n_before = s[cols].notna().sum(axis=1)
    for k, n in n_before.value_counts().items():
        _bump(TOPUP_STATS["before"], int(k), int(n))
    for i in SLOTS6:                                                        # why each slot is empty
        empty = s[f"S{i}"].isna()
        if f04:
            _bump(TOPUP_STATS["why_missing"], "F-04: no offer in slots 1-4 (line dropped by code 5)", int(empty.sum()))
            continue
        rem = s.get(f"REMOVED_{i}", pd.Series(None, index=s.index))
        by_rule = rem[empty & rem.notna()].astype(str).str.split(":").str[0]
        for r, n in by_rule.value_counts().items():
            _bump(TOPUP_STATS["why_missing"], "an offer without both scripts or not in the catalogue: replaced" if r == "INVALID"
                  else f"{r}: cleared by code 5", int(n))
        n_none = int((empty & rem.isna()).sum())
        if n_none:
            _bump(TOPUP_STATS["why_missing"], "slots 1-4: no offer in the price band (passes)" if i <= 4 else
                  "slots 5-6: no 3rd offer above the anchor (EXT-6), no favourite / forced bundle", n_none)
    for v in s.index[n_before < 6]:
        names = [x for x in (s.at[v, c] for c in cols) if isinstance(x, str) and x]
        keys = {KEYTP.get(x) for x in names}
        line = LINE.loc[v] if v in LINE.index else None
        group = str(line["GROUP_KEY"]) if line is not None else "t2c"
        action = str(act.get(v, line["ACTION_AGENT"] if line is not None else ""))
        segment = str(line["SEGMENT"]) if line is not None else ""
        rank2 = line.get("ACTION_AGENT_RANK2") if line is not None else None
        others = [a for a in ([rank2] if isinstance(rank2, str) else []) + PRIOR_ORDER if a and a != action]
        others = [a for a in dict.fromkeys(others) if not (segment in AT_RISK_SEGMENTS and a in UPSELL_ACTIONS
                                                            and a != (line["ACTION_AGENT"] if line is not None else None))]
        sources = [("short list", short.get(v, [])), ("action offers", rest.get(v, [])), ("favourite", fav.get(v, []))]
        sources += [(f"other action {a}", universe_offers(a, group == "t2f")) for a in others]
        for i in [i for i in SLOTS6 if not isinstance(s.at[v, f"S{i}"], str) or not s.at[v, f"S{i}"]]:
            want, pick = (30 if i in MONTHLY_SLOT else 7), None
            for relax in (False, True):
                for src_name, offers in sources:
                    second = None
                    for o in offers:
                        if o not in VALID_OFFERS or o in names or KEYTP.get(o) in keys:
                            continue
                        if not relax and VALIDITY_OF.get(o) != want:
                            continue
                        if any(f"{o}|{x}" in CONFLICT for x in names):
                            continue
                        a = _anchor(s, v, o)
                        if np.isnan(a) or (PRICE_OF.get(o) or 0) >= a:
                            pick = (o, src_name)
                            break
                        second = second or (o, src_name)
                    pick = pick or second
                    if pick:
                        break
                if pick:
                    break
            if pick:
                s.at[v, f"S{i}"], s.at[v, f"REASON_{i}"] = pick[0], f"top-up: {pick[1]}"
                names.append(pick[0])
                keys.add(KEYTP.get(pick[0]))
                _bump(TOPUP_STATS["added_by_source"], pick[1].split(" ")[0] if pick[1].startswith("other") else pick[1])
    n_after = s[cols].notna().sum(axis=1)
    for k, n in n_after.value_counts().items():
        _bump(TOPUP_STATS["after"], int(k), int(n))
    TOPUP_STATS["rows"] += len(s)
    TOPUP_STATS["still_short"] += int((n_after < 6).sum())
    if f04:
        TOPUP_STATS["brought_back_F04"] += int((n_after > 0).sum())


# --- code 5 with the top-up: inside build_input_flytxt (after F-09 / F-11 / F-12), and the F-04 lines after it --------
_format_orig = getattr(fo, "_format_orig", fo._format)
fo._format_orig = _format_orig
_build_orig = getattr(fo, "_build_orig", fo.build_input_flytxt)
fo._build_orig = _build_orig


def _format_topup(s, scored, src, catalog, run_date):
    topup(s, scored)
    return _format_orig(s, scored, src, catalog, run_date)


def _build_topup(monthly, weekly, scored, src, run_date):
    flytxt, slots = _build_orig(monthly, weekly, scored, src, run_date)
    if TOPUP_DROPPED:
        dropped = slots.loc[~slots["IN_INPUT_FLYTXT"].astype(bool), VAL].astype(str)
        if len(dropped):
            s2 = pd.DataFrame(index=pd.Index(dropped.unique(), name=VAL))
            for i in SLOTS6:
                s2[f"S{i}"], s2[f"REASON_{i}"], s2[f"REMOVED_{i}"] = None, None, None
            s2 = s2.join(fo.f05_anchors(src, pd.Series(s2.index)))
            topup(s2, scored, f04=True)
            s2 = s2[s2[[f"S{i}" for i in SLOTS6]].notna().any(axis=1)]
            if len(s2):
                flytxt = pd.concat([flytxt, _format_orig(s2, scored, src, load_catalog(src), run_date)],
                                   ignore_index=True).drop_duplicates("msisdn")
                slots = pd.concat([slots[~slots[VAL].astype(str).isin(s2.index)], s2.reset_index().assign(IN_INPUT_FLYTXT=True)],
                                  ignore_index=True)
    return flytxt, slots


fo._format = _format_topup
fo.build_input_flytxt = _build_topup

# --- the auditor: A-25, every row of the final file has 6 distinct, valid offers (blocking) ----------------------------
_oc_orig = getattr(scoring_groups, "_oc_orig", scoring_groups.output_checks)
scoring_groups._oc_orig = _oc_orig


def _checks_6(flytxt, control, catalog_names, run_date):
    out = _oc_orig(flytxt, control, catalog_names, run_date)
    names = flytxt[[f"destination_offer_name_{i}" for i in SLOTS6]].astype(str).apply(lambda c: c.str.strip())
    scripts = flytxt[[f"offer_script_{l}_{i}" for l in ("en", "ar") for i in SLOTS6]].astype(str).apply(lambda c: c.str.strip())
    ok = ((names != "").all(axis=1) & (names.nunique(axis=1) == 6) & names.isin(VALID_OFFERS).all(axis=1)
          & (scripts != "").all(axis=1))
    out.append({"id": "A-25", "check": f"6 distinct, valid offers (catalogue + both scripts) in every row "
                                       f"({int((~ok).sum()):,} of {len(flytxt):,} rows not)",
                "status": "PASS" if len(flytxt) and bool(ok.all()) else "FAIL"})
    return out


scoring_groups.output_checks = _checks_6
print(f"top-up ready: {len(VALID_OFFERS):,} valid offers (catalogue + both scripts) of {len(INFO):,}, {len(CONFLICT) // 2:,} "
      f"conflict pairs, {len(REC):,} recommendable; A-25 added to the auditor")

## Step 8 · Agent 7 (⑦) code 5: the final file of Target 2

Code 5 of production (F-01 … F-19): 38 columns, favourites and forced bundles in empty slots, conflicts and
scripts. Only on PASS is `input_flytxt_target2.csv.gz` written.

In [ ]:
step("Step 8")
fin_t2 = scoring_groups.run_target_final(D, AGENT_TARGET, force=True)
print(f"{AGENT_TARGET}: final {'READY' if fin_t2['ready'] else 'BLOCKED'}, {fin_t2['input_flytxt_rows']:,} rows")
det2 = pd.read_parquet(OUTPUT / AGENT_TARGET / f"details_{AGENT_TARGET}.parquet")
show(det2[[c for c in ("GROUP_KEY", "SEGMENT", "ACTION_AGENT", "ACTION_SENT", "DECIDED_BY", "SLOT_1", "REASON_1", "SLOT_2")
           if c in det2.columns]], n=5, title="Target 2 details (first rows)")

## Step 8b · The top-up in figures: why lines had fewer than 6 offers, and what was added

In [ ]:
step("Step 8b")
_dist = pd.DataFrame({"lines before": pd.Series(TOPUP_STATS["before"]), "lines after": pd.Series(TOPUP_STATS["after"])})
_dist = _dist.reindex(range(0, 7)).fillna(0).astype(int).rename_axis("offers per line")
show(_dist.reset_index(), n=7, title="offers per line, before and after the top-up")
show(pd.Series(TOPUP_STATS["why_missing"], name="empty slots").sort_values(ascending=False).rename_axis("why the slot was empty")
     .reset_index(), n=10, title="why slots were empty before the top-up")
show(pd.Series(TOPUP_STATS["added_by_source"], name="offers added").rename_axis("source").reset_index(), n=10,
     title="offers added, by source")
_b = sum(k * n for k, n in TOPUP_STATS["before"].items()) / max(sum(TOPUP_STATS["before"].values()), 1)
_a = sum(k * n for k, n in TOPUP_STATS["after"].items()) / max(sum(TOPUP_STATS["after"].values()), 1)
print(f"offers per line: {_b:.2f} before -> {_a:.2f} after | F-04 lines brought back: {TOPUP_STATS['brought_back_F04']:,} | "
      f"lines still under 6: {TOPUP_STATS['still_short']:,} | offers without scripts replaced: {TOPUP_STATS.get('invalid_replaced', 0):,}")
if TOPUP_STATS["still_short"]:
    print("WARNING: some lines could not reach 6 offers (no valid offer left in any source): A-25 blocks the final file. "
          "Check the catalogue scripts (BTL_script_english_arabic_ok) and the conflict pairs")

## Step 9 · Agent 9 (⑨) auditor

**The rules block:** the group audit (A-12 universe, A-12b catalogue, A-17 rules version, A-21 no control line,
A-30 / A-31) and the file checks (A-20 … A-24) of Target 2. Any FAIL and the final file is not written.
**The LLM explains** the result for the business; it cannot change it.

In [ ]:
step("Step 9")
checks = {t: {"audit": json.loads((OUTPUT / t / "audit_groups.json").read_text(encoding="utf-8"))["status"],
              "file_checks": [f"{c['id']} {c['status']}" for c in json.loads((OUTPUT / t / "summary.json").read_text(encoding="utf-8"))["output_checks"]],
              "ready": json.loads((OUTPUT / t / "summary.json").read_text(encoding="utf-8"))["ready"]} for t in config.TARGETS}
a9 = ask("Agent 9 auditor", "You are the auditor agent of a telecom campaign engine. Explain audit results to the business.",
         f"Audit of {RUN_DATE}: {json.dumps(checks)}. Say whether the target files may be handed over, and why.",
         {"type": "object", "properties": {"verdict": {"type": "string", "enum": ["READY", "BLOCKED"]},
                                           "explanation": {"type": "string", "maxLength": 400}},
          "required": ["verdict", "explanation"]}, 200)
rule_verdict = "READY" if all(c["ready"] for c in checks.values()) else "BLOCKED"
show([{"target": t, **c, "file_checks": ", ".join(c["file_checks"])} for t, c in checks.items()], title="the rules")
display(Markdown(f"**Verdict of the rules: {rule_verdict}.** Agent 9 (LLM): {a9['explanation'] if a9 else '-'}"
                 + ("" if not a9 or a9["verdict"] == rule_verdict else "  *(the LLM disagrees: the rules win)*")))

## Step 10 · Agent 8 (⑧) strategy: group a vs group b, agents vs cell policy

The LLM compares what was sent to group **a** (T2C) and group **b** (T2F), and, inside Target 2, the customers the
agents decided one by one (agents 3 + 4) with those who got the cell policy (past the budget). A proposal only;
nothing changes by itself. The real comparison is the reward at D+7 (who bought).

In [ ]:
step("Step 10")
def profile(det):
    sent = det[det["IN_INPUT_FLYTXT"]] if "IN_INPUT_FLYTXT" in det.columns else det
    act = sent["ACTION_SENT"] if "ACTION_SENT" in sent.columns else sent.get("ACTION_AGENT")
    n = sum(sent[f"SLOT_{i}"].notna() & sent[f"SLOT_{i}"].astype(str).ne("") for i in range(1, 7) if f"SLOT_{i}" in sent.columns)
    return {"lines": int(len(det)), "in_file": int(len(sent)), "offers_per_line": round(float(np.mean(n)), 2) if len(sent) else 0,
            "actions": act.value_counts(normalize=True).round(3).to_dict() if act is not None and len(sent) else {}}


cmp = {"group a (T2C)": profile(det2[det2["GROUP_KEY"] == "t2c"]), "group b (T2F)": profile(det2[det2["GROUP_KEY"] == "t2f"]),
       "decided by the agents": profile(det2[det2["ACCESS_METHOD_VAL"].isin(LLM_LINES)]),
       "cell policy (past the budget)": profile(det2[~det2["ACCESS_METHOD_VAL"].isin(LLM_LINES)])}
a8 = ask("Agent 8 strategy", "You are the strategy agent of a telecom campaign engine.",
         f"This week's Target 2: {json.dumps(cmp)}. Agent 3 kept the cell policy for "
         f"{(NBA['ACTION'] == NBA[VAL].map(ctx['ACTION_AGENT'])).mean() if len(NBA) else 0:.0%} of its customers. "
         "Compare group a and group b, and the agents with the cell policy, and give 3 things to watch or change next "
         "week (the reward of this week is measured at D+7: who bought, group a against group b).",
         {"type": "object", "properties": {"comparison": {"type": "string", "maxLength": 500},
                                           "next_week": {"type": "array", "items": {"type": "string", "maxLength": 160}, "maxItems": 3}},
          "required": ["comparison", "next_week"]}, 300)
show(pd.DataFrame(cmp).T.astype(str), title="group a vs group b, agents vs cell policy")
if a8:
    display(Markdown(f"**Agent 8 (LLM):** {a8['comparison']}\n\n" + "\n".join(f"- {x}" for x in a8["next_week"])))

## Step 11 · The final file

`final/input_flytxt.csv.gz` holds Target 2 (groups a and b), only if it passed. The engine only **saves** files; it
sends nothing: you check the file and hand it over yourself.

In [ ]:
step("Step 11")
ctrl = scoring_groups.write_control(D)
fin = scoring_groups.run_final_file(D)
print(f"final file {'READY' if fin['ready'] else 'BLOCKED'} -- {fin['input_flytxt_rows']:,} rows {fin['rows_per_target']}"
      + (f" -- {fin['reasons']}" if fin["reasons"] else ""))
tree(OUTPUT, depth=1)

## Step 12 · The 360 view of every scored target line (for your checks)

Two files, written next to the final file in `data/output/<D>/target2/` (Parquet + `.csv.gz` each):
- `target2_scored_360`: **the scored MSISDNs only** (the lines of the final file), with all the data below, the flags,
  `MSISDN_FLYTXT` and two checks against `input_flytxt` (`CHK_NOT_IN_FLYTXT`, `CHK_SLOTS_DIFFER_FROM_FLYTXT`: both must be 0);
- `target2_360`: **every target line**, scored or not (`DROP_RULE` says why a line is not in the file).

One row per line, in this order (for the scored view, see the two flytxt checks just after the `CHK_*` columns):


| Columns | What |
|---|---|
| `ACCESS_METHOD_VAL`, `GROUP_KEY`, `IN_INPUT_FLYTXT`, `DROP_RULE` | the line, its group (`t2c` / `t2f`), whether it is in the final file, and why not |
| `SEGMENT_AT_T`, `SEGMENT`, `SEGMENT_MOVED_T_TO_D`, `RULE_ID`, `VALUE_BAND`, `USAGE_TYPE`, `NATIONALITY_GROUP` | the segment at training (T) and at scoring (D), and the bands |
| `ACTION_ORIGINAL`, `ACTION_AGENT`, `ACTION_SENT`, `DECIDED_BY` | the production action, the cell policy, the action sent, and who decided |
| `AGENT3_ACTION`, `AGENT3_SCORES`, `AGENT3_REASON`, `AGENT3_DECIDED_BY`, `PROPENSITY_LOGGED` | agent 3: its score per allowed action, its reason, and the real logged propensity |
| `SENT_TO_LLM`, `PROFILE_LEVEL`, `PROFILE_ID`, `PROFILE_FACTS` | FAST_MODE: whether the LLM decided the line, and the profile it was decided with (the prompt text, no MSISDN) |
| `SLOT_1` … `SLOT_6`, `REASON_i`, `REMOVED_i`, `SLOT_i_PRICE`, `SLOT_i_ANCHOR`, `SLOT_i_VS_ANCHOR` | the 6 offers of the file (= `destination_offer_name_i`), why each is there, the price, the anchor of the price band for that service and validity (combo = total spend, data = the customer's own data share × total, voice = the rest; mean + std), and price ÷ anchor |
| `CHK_*`, `N_OFFERS` | the check flags (below) |
| `reco_m*` / `reco_w*`, `B26_*`, `F05_*` | the recommended offers with score, anchor, pass and rule; the two anchors |
| W0 columns and every T2 feature | last week (targeted, action, reward), revenue / voice / data per period, model scores, … |

**The check flags:**
- `CHK_OFFERS_BELOW_ANCHOR`: the number of slots priced **below** the customer's usual spend (a cannibalization
  risk). The challenger band starts at 1.0 × the anchor, so the challenger offers are never below it. It can be > 0
  for favourites (code 5 adds them without a price floor: F-07 is off, as in production) and for the fallback passes.
- `CHK_FAVOURITE_BELOW_ANCHOR`: of which favourites.
- `CHK_FALLBACK_PASS_BELOW_ANCHOR`: of which offers of the **fallback passes** (production code 3 / 4, B-28 … B-30,
  used when the challenger band finds nothing). They price against the production anchor with a fixed 90 / 10
  data / voice split, so a voice or data offer can sit under the customer's own split. `SLOT_i_SELECTED_BY` says
  which rule picked each offer. Anything else below the anchor is printed as a WARNING (expected 0).
- `CHK_UPSELL_AT_RISK_SEGMENT`: an upsell sent to an at-risk segment (H-06). It must be 0.
- `CHK_UPSELL_CHURN_GE_060`: an upsell sent to a customer with churn ≥ 0.60 (for information).

The cell prints the summary per group. The file holds MSISDNs: read it with pandas, here or in another notebook on
this laptop, and do not open it in Excel.

In [ ]:
step("Step 12")
det = pd.read_parquet(OUTPUT / AGENT_TARGET / f"details_{AGENT_TARGET}.parquet")
v360 = det.copy()


def _c(name):
    return pd.to_numeric(v360[name], errors="coerce") if name in v360.columns else pd.Series(np.nan, index=v360.index)


# --- agent 3: the real decision, its scores, reason and logged propensity ------------------------------------------
a3 = NBA.set_index(VAL) if len(NBA) else pd.DataFrame(columns=["ACTION", "PROBA", "DECIDED_BY", "PROFILE", "REASON"])


def _scores(v):
    u = PROF.at[v, "UNIT"] if v in PROF.index else None
    if u is None or u not in ANS3:
        return None
    allowed, ans = ANS3[u]
    return "; ".join(f"{a}: {int(s)}" for a, s in zip(allowed, ans["scores"])) if ok3(ans, allowed) else None


v360["AGENT3_ACTION"] = v360[VAL].map(a3["ACTION"])
v360["AGENT3_SCORES"] = [(_scores(v) if v in a3.index else None) for v in v360[VAL]]
v360["AGENT3_REASON"] = v360[VAL].map(a3["REASON"])
v360["AGENT3_DECIDED_BY"] = v360[VAL].map(a3["DECIDED_BY"])
v360["PROPENSITY_LOGGED"] = v360[VAL].map(a3["PROBA"]).where(v360[VAL].isin(a3.index), v360.get("PROPENSITY"))
v360["SENT_TO_LLM"] = v360[VAL].isin(LLM_LINES)
v360["PROFILE_LEVEL"] = LEVEL
v360["PROFILE_ID"] = v360[VAL].map(PROF["UNIT"])
v360["PROFILE_FACTS"] = v360[VAL].map(FACTS)

# --- the segment at T (training, *_auto) against the segment at D (scoring view) -----------------------------------
_t = pd.concat([pd.read_parquet(p, columns=[VAL, "SEGMENT"]) for p in sorted((train_dir / T3_TRAIN / "segments").glob("*.parquet"))])
_t = _t[_t[VAL].isin(set(v360[VAL]))].drop_duplicates(VAL).set_index(VAL)["SEGMENT"]
v360["SEGMENT_AT_T"] = v360[VAL].map(_t)
v360["SEGMENT_MOVED_T_TO_D"] = v360["SEGMENT_AT_T"].notna() & (v360["SEGMENT_AT_T"] != v360["SEGMENT"])

# --- each slot: its price against the customer's usual spend (F-05 anchor: monthly for 30 days, weekly for 7 days) --
_cat = load_catalog(cache).drop_duplicates("BTL_NAME").set_index("BTL_NAME")
_share = use("nbx.fsdr.offers", "anchors").data_share(cache)                      # the customer's own data share
DATA_SHARE = v360[VAL].map(_share).fillna(config.DEFAULT_DATA_SHARE).astype(float).to_numpy()
_rec = [pd.read_csv(f, dtype=str, keep_default_na=False, usecols=lambda c: c in ("access_method_val", "btl_name", "selected_by"))
        for f in sorted(P6.glob("recos_*"))]                                   # which rule selected each recommended offer
_rec = pd.concat(_rec, ignore_index=True) if _rec else pd.DataFrame(columns=["access_method_val", "btl_name", "selected_by"])
_sel = _rec.assign(_K=_rec["access_method_val"].str.strip() + "|" + _rec["btl_name"].str.strip()).drop_duplicates("_K").set_index("_K")["selected_by"]
fb_below = pd.Series(0, index=v360.index)
tu_below = pd.Series(0, index=v360.index)
below = pd.Series(0, index=v360.index)
fav_below = pd.Series(0, index=v360.index)
n_off = pd.Series(0, index=v360.index)
for i in range(1, 7):
    if f"SLOT_{i}" not in v360.columns:
        continue
    s = v360[f"SLOT_{i}"].where(v360[f"SLOT_{i}"].astype(str).str.strip().ne("") & v360[f"SLOT_{i}"].notna())
    price = pd.to_numeric(s.map(_cat["PRICE"]), errors="coerce")
    per = np.where(pd.to_numeric(s.map(_cat["VALIDITY"]), errors="coerce").astype(float).eq(7).to_numpy(), "WEEKLY", "MONTHLY")
    svc = s.map(_cat["SERVICE"]).astype(str).str.upper()
    # the anchor of the challenger price band (CC-03 / CF-03): total spend (mean + std) for COMBO, the customer's own
    # data share x total for DATA, (1 - share) x total for VOICE (no spend history: 90 / 10)
    total = _c("B26_AVG_REV_WEEKLY").where(per == "WEEKLY", _c("B26_AVG_REV_MONTHLY"))
    anc = total * np.select([svc == "DATA", svc == "VOICE"], [DATA_SHARE, 1 - DATA_SHARE], 1.0)
    anc = anc.where(s.notna() & (total > 0))
    v360[f"SLOT_{i}_PRICE"] = price
    v360[f"SLOT_{i}_ANCHOR"] = anc.round(3)
    v360[f"SLOT_{i}_VS_ANCHOR"] = (price / anc.where(anc > 0)).round(2)
    sel_by = (v360[VAL].astype(str) + "|" + s.fillna("").astype(str)).map(_sel).where(s.notna())
    v360[f"SLOT_{i}_SELECTED_BY"] = sel_by
    b = (price.astype(float) < anc.astype(float)).fillna(False).astype(bool)
    reason = v360[f"REASON_{i}"] if f"REASON_{i}" in v360.columns else pd.Series("", index=v360.index)
    below += b.astype(int)
    fav_below += (b & reason.astype(str).eq("favourite")).astype(int)
    fb_below += (b & sel_by.astype(str).str.startswith("fallback")).astype(int)
    tu_below += (b & reason.astype(str).str.startswith("top-up")).astype(int)
    n_off += s.notna().astype(int)
act = v360["ACTION_SENT"].astype(str)
v360["N_OFFERS"] = n_off
v360["CHK_OFFERS_BELOW_ANCHOR"] = below
v360["CHK_FAVOURITE_BELOW_ANCHOR"] = fav_below
v360["CHK_FALLBACK_PASS_BELOW_ANCHOR"] = fb_below
v360["CHK_TOPUP_BELOW_ANCHOR"] = tu_below
v360["CHK_UPSELL_AT_RISK_SEGMENT"] = act.isin(UPSELL_ACTIONS) & v360["SEGMENT"].isin(AT_RISK_SEGMENTS)
v360["CHK_UPSELL_CHURN_GE_060"] = act.isin(UPSELL_ACTIONS) & (_c("PROBA_CHURN") >= 0.6)

first = [VAL, "GROUP_KEY", "IN_INPUT_FLYTXT", "DROP_RULE", "SEGMENT_AT_T", "SEGMENT", "SEGMENT_MOVED_T_TO_D", "RULE_ID",
         "VALUE_BAND", "USAGE_TYPE", "NATIONALITY_GROUP", "ACTION_ORIGINAL", "ACTION_AGENT", "ACTION_SENT", "DECIDED_BY",
         "AGENT3_ACTION", "AGENT3_SCORES", "AGENT3_REASON", "AGENT3_DECIDED_BY", "PROPENSITY_LOGGED", "SENT_TO_LLM",
         "PROFILE_LEVEL", "PROFILE_ID", "PROFILE_FACTS",
         *[c for i in range(1, 7) for c in (f"SLOT_{i}", f"REASON_{i}", f"SLOT_{i}_SELECTED_BY", f"REMOVED_{i}", f"SLOT_{i}_PRICE",
                                         f"SLOT_{i}_ANCHOR", f"SLOT_{i}_VS_ANCHOR")],
         "N_OFFERS", "CHK_OFFERS_BELOW_ANCHOR", "CHK_FAVOURITE_BELOW_ANCHOR", "CHK_FALLBACK_PASS_BELOW_ANCHOR",
         "CHK_TOPUP_BELOW_ANCHOR",
         "CHK_UPSELL_AT_RISK_SEGMENT",
         "CHK_UPSELL_CHURN_GE_060"]
first = [c for c in first if c in v360.columns]
v360 = v360[first + [c for c in v360.columns if c not in first]]
OUT360 = OUTPUT / AGENT_TARGET
v360.to_parquet(OUT360 / f"{AGENT_TARGET}_360.parquet", index=False)
v360.to_csv(OUT360 / f"{AGENT_TARGET}_360.csv.gz", index=False, compression="gzip")
print(f"360 view: {len(v360):,} lines x {v360.shape[1]} columns -> {OUT360 / (AGENT_TARGET + '_360.parquet')} (+ .csv.gz)")

# --- the scored MSISDNs only (the lines of the final file), with all the data, the flags and the flytxt check -------
scored = v360[v360["IN_INPUT_FLYTXT"]].copy()
_fp = OUT360 / f"input_flytxt_{AGENT_TARGET}.csv.gz"
if _fp.exists():
    _fly = pd.read_csv(_fp, dtype=str, keep_default_na=False)
    _fly = _fly.assign(_V=_fly["msisdn"].str.strip().str[-8:]).drop_duplicates("_V").set_index("_V")
    scored.insert(1, "MSISDN_FLYTXT", scored[VAL].map(_fly["msisdn"]))
    _diff = pd.Series(False, index=scored.index)
    for i in range(1, 7):
        a = scored[f"SLOT_{i}"].fillna("").astype(str).str.strip() if f"SLOT_{i}" in scored.columns else ""
        b = scored[VAL].map(_fly[f"destination_offer_name_{i}"]).fillna("").astype(str).str.strip()
        _diff |= a != b
    scored["CHK_NOT_IN_FLYTXT"] = scored["MSISDN_FLYTXT"].isna()
    scored["CHK_SLOTS_DIFFER_FROM_FLYTXT"] = _diff & ~scored["CHK_NOT_IN_FLYTXT"]
    _fc = ["CHK_NOT_IN_FLYTXT", "CHK_SLOTS_DIFFER_FROM_FLYTXT"]
    _head = [VAL, "MSISDN_FLYTXT"] + [c for c in first if c != VAL] + _fc
    scored = scored[_head + [c for c in scored.columns if c not in _head]]
    print(f"flytxt check: {len(_fly):,} lines in {_fp.name}; {int(scored['CHK_NOT_IN_FLYTXT'].sum()):,} scored lines "
          f"not found in it, {int(scored['CHK_SLOTS_DIFFER_FROM_FLYTXT'].sum()):,} with other offers (both must be 0)")
    _extra = len(set(_fly.index) - set(scored[VAL]))
    if _extra:
        print(f"WARNING: {_extra:,} flytxt lines are not in the scored view")
else:
    print(f"no {_fp.name}: the final file was not written (see steps 8 / 9), the scored view has no flytxt check")
scored.to_parquet(OUT360 / f"{AGENT_TARGET}_scored_360.parquet", index=False)
scored.to_csv(OUT360 / f"{AGENT_TARGET}_scored_360.csv.gz", index=False, compression="gzip")
print(f"scored view: {len(scored):,} scored MSISDNs x {scored.shape[1]} columns -> "
      f"{OUT360 / (AGENT_TARGET + '_scored_360.parquet')} (+ .csv.gz)")

summary = v360.groupby("GROUP_KEY").agg(
    lines=(VAL, "size"), in_final_file=("IN_INPUT_FLYTXT", "sum"), sent_to_llm=("SENT_TO_LLM", "sum"),
    offers_per_line=("N_OFFERS", "mean"), segment_moved_T_to_D=("SEGMENT_MOVED_T_TO_D", "sum"),
    lines_offer_below_anchor=("CHK_OFFERS_BELOW_ANCHOR", lambda s: int((s > 0).sum())),
    of_which_favourite=("CHK_FAVOURITE_BELOW_ANCHOR", lambda s: int((s > 0).sum())),
    of_which_fallback_pass=("CHK_FALLBACK_PASS_BELOW_ANCHOR", lambda s: int((s > 0).sum())),
    of_which_topup=("CHK_TOPUP_BELOW_ANCHOR", lambda s: int((s > 0).sum())),
    upsell_at_risk_segment=("CHK_UPSELL_AT_RISK_SEGMENT", "sum"), upsell_churn_ge_060=("CHK_UPSELL_CHURN_GE_060", "sum"))
show(summary.round(2).reset_index(), n=5, title="checks per group")
if int(v360["CHK_UPSELL_AT_RISK_SEGMENT"].sum()):
    print("WARNING: upsells sent to an at-risk segment (H-06): look at those lines in the 360 view")
_fb = int((v360["CHK_FALLBACK_PASS_BELOW_ANCHOR"] > 0).sum())
if _fb:
    print(f"note: {_fb:,} lines got an offer from the FALLBACK passes (production code 3 / 4, B-28 ... B-30) under the band "
          "anchor: those passes use the production anchor (fixed 90 / 10 data / voice split), not the customer's own "
          "split; filter CHK_FALLBACK_PASS_BELOW_ANCHOR > 0 and read SLOT_i_SELECTED_BY")
_bad = int(((v360["CHK_OFFERS_BELOW_ANCHOR"] - v360["CHK_FAVOURITE_BELOW_ANCHOR"] - v360["CHK_FALLBACK_PASS_BELOW_ANCHOR"] - v360["CHK_TOPUP_BELOW_ANCHOR"]) > 0).sum())
print(f"{'WARNING' if _bad else 'OK'}: {_bad:,} lines with another offer below the band anchor (forced bundle or a pass): "
      + ("filter CHK_OFFERS_BELOW_ANCHOR > 0 and read REASON_i / SLOT_i_SELECTED_BY" if _bad else "expected 0"))
show(v360[first].head(3), n=3, title="the first columns (first rows)")

## Time per agent on your GPU

`seconds` is the GPU time of the calls (summed over the parallel calls), `wall` the time you waited. Cached answers
(a re-run of the same week) cost nothing. The last line extrapolates agents 3 + 4 to a production week.

In [ ]:
step("Time per agent on your GPU")
tt = pd.DataFrame([{"agent": a, **{k: v for k, v in s.items() if k != "last_error"}} for a, s in STATS.items()])
tt = tt.assign(wall=tt.get("wall_seconds", pd.Series(0.0, index=tt.index)).fillna(0).round(1),
               seconds=tt["seconds"].round(1)).drop(columns=[c for c in ("wall_seconds",) if c in tt.columns])
show(tt, n=20, title=f"LLM calls per agent ({MODEL}, PARALLEL {PARALLEL})")
n2c = sum(1 for v in LLM_LINES if ctx.loc[v, "GROUP_KEY"] == "t2c")
measured = sum(STATS.get(a, {}).get("wall_seconds", 0.0) for a in ("Agent 3 NBA", "Agent 4 NBO"))
fresh = sum(STATS.get(a, {}).get("calls", 0) for a in ("Agent 3 NBA", "Agent 4 NBO"))
errors = sum(STATS.get(a, {}).get("errors", 0) for a in ("Agent 3 NBA", "Agent 4 NBO"))
if errors:
    print(f"agents 3 + 4: {errors} calls FAILED (those customers got the cell policy / rule score): "
          f"{STATS.get('Agent 4 NBO', STATS.get('Agent 3 NBA', {})).get('last_error')}")
if LLM_LINES and fresh == 0 and not errors:
    print("agents 3 + 4: every answer came from the cache (a re-run of this week): see step G2 for the estimate")
elif LLM_LINES and measured > 0:
    per_line = measured / len(LLM_LINES)
    print(f"agents 3 + 4: {measured / 60:.1f} min for {len(LLM_LINES):,} customers ({n2c:,} T2C) = {per_line:.2f} s per customer; "
          f"a production week of 100,000 Target-2 customers (50k a + 50k b): about {100_000 * per_line / 3600:.1f} h")
json.dump({"model": MODEL, "parallel": PARALLEL, "stats": STATS, "llm_lines": len(LLM_LINES)},
          (AGENTS_DIR / "agent_times.json").open("w", encoding="utf-8"), indent=2, default=str)

## FAST_MODE in figures

How many customers each LLM answer covered, and the GPU time against the per-customer estimate (`off` in step 5b).
Saved to `data/runs/<D>/agents/fast_mode.json`.

In [ ]:
step("FAST_MODE in figures")
_a = {a: STATS.get(a, {}) for a in ("Agent 3 NBA", "Agent 4 NBO")}
answers34 = sum(s.get("calls", 0) + s.get("cached", 0) + s.get("errors", 0) for s in _a.values())
wall34 = sum(s.get("wall_seconds", 0.0) for s in _a.values())
_c4 = sum(g["customers"] for g in NBO_PROFILES.values())
_n4 = sum(g["answers"] for g in NBO_PROFILES.values())
per3 = round(len(NBA) / max(len(ANS3), 1), 2)
per4 = round(_c4 / max(_n4, 1), 2)
FAST = {"profile_level": LEVEL, "profile_level_setting": PROFILE_LEVEL, "target_customers": n_cust,
        "customers_sent_to_llm": len(LLM_LINES), "units_sent": len(LLM_UNITS), "answers_agents_3_4": answers34,
        "agent3_customers_per_answer": per3, "agent4_customers_per_answer": per4,
        "calls_saved_share": round(1 - answers34 / max(len(NBA) + _c4, 1), 3),
        "from_cache_agents_3_4": sum(s.get("cached", 0) for s in _a.values()), "gpu_wall_minutes_agents_3_4": round(wall34 / 60, 1),
        "estimate_minutes_per_customer_mode": float(LEVEL_TABLE.loc[LEVEL_TABLE["level"] == "off", "estimated GPU min"].iloc[0]),
        "levels": LEVEL_TABLE.to_dict("records"), "nbo_profiles": NBO_PROFILES, "quality_check_agent3": CHECK3}
(AGENTS_DIR / "fast_mode.json").write_text(json.dumps(FAST, indent=2, default=str), encoding="utf-8")
print(f"FAST_MODE {LEVEL}: {len(LLM_LINES):,} of {n_cust:,} customers decided by {answers34:,} LLM answers "
      f"(agent 3: {per3} customers per answer, agent 4: {per4}; {FAST['calls_saved_share']:.0%} of the per-customer calls saved; "
      f"{FAST['from_cache_agents_3_4']:,} from the cache) in "
      f"{FAST['gpu_wall_minutes_agents_3_4']} min; one call per customer would take about "
      f"{FAST['estimate_minutes_per_customer_mode']} min")

## G3 · Report: was this an LLM + GPU run?

Reads the GPU (`nvidia-smi`), the share of the model in GPU memory (Ollama only), and counts who decided: the LLM, the
cache of a previous run, or the rule fallback. Written to `data/runs/<D>/agents/llm_gpu_report.json`.

In [ ]:
step("G3")
import shutil, subprocess

gpu = {}
if shutil.which("nvidia-smi"):
    _q = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.used,memory.total,utilization.gpu",
                         "--format=csv,noheader,nounits"], capture_output=True, text=True, timeout=20).stdout.strip()
    if _q:
        _n, _mu, _mt, _ut = [x.strip() for x in _q.splitlines()[0].split(",")]
        gpu = {"gpu": _n, "memory_used_mb": int(float(_mu)), "memory_total_mb": int(float(_mt)), "utilization_now_pct": int(float(_ut))}
SERVER = "ollama" if "OLLAMA_URL" in globals() else "vllm"
_ps = None
if SERVER == "ollama":                                 # vLLM: the model always runs on the GPU (it cannot run on the CPU)
    try:
        _ps = next((m for m in _http("/api/ps", timeout=10).get("models", []) if m["name"] == MODEL), None)
    except OSError:
        _ps = None
gpu["model_in_gpu_memory"] = None if _ps is None else round(_ps.get("size_vram", 0) / max(_ps.get("size", 1), 1), 3)

agents = {a: {k: v for k, v in s.items() if k != "last_error"} for a, s in STATS.items() if a not in ("G2 speed test", "G1 first request")}
fresh = sum(s.get("calls", 0) for s in agents.values())
cached = sum(s.get("cached", 0) for s in agents.values())
failed = sum(s.get("errors", 0) for s in agents.values())
decided = NBA["DECIDED_BY"].value_counts().to_dict() if len(NBA) else {}
grp = pd.read_csv(P6 / "group_report.csv") if (P6 / "group_report.csv").exists() else pd.DataFrame()
report = {
    "run_date": RUN_DATE, "test_10k": TEST_10K, "score_view": SCORE_VIEW, "server": SERVER, "model": MODEL, "parallel": PARALLEL,
    "gpu": gpu,
    "llm_answers": {"fresh": fresh, "from_cache": cached, "failed": failed,
                    "gpu_seconds": round(sum(s.get("seconds", 0.0) for s in agents.values()), 1)},
    "target_customers": int((panel["TARGET"] == AGENT_TARGET).sum()), "sent_to_llm": len(LLM_LINES),
    "agent3_decided_by": decided,
    "agent4_llm_scored_offers": int(grp["llm_scored_offers"].sum()) if "llm_scored_offers" in grp.columns else None,
    "score_tables_replaced_by_auto": sorted(REMAP),
    "notebook_minutes": round((time.time() - NB_START) / 60, 1),
}
(AGENTS_DIR / "llm_gpu_report.json").write_text(json.dumps(report, indent=2, default=str), encoding="utf-8")

on_gpu = bool(gpu.get("gpu")) and (SERVER == "vllm" or (gpu.get("model_in_gpu_memory") or 0) > 0.99)
llm_ok = fresh + cached > 0 and failed <= MAX_FALLBACK_SHARE * max(fresh + cached + failed, 1)
print(f"{SERVER} | model {MODEL} | GPU {gpu.get('gpu', 'not found')} | model in GPU memory "
      f"{'-' if gpu.get('model_in_gpu_memory') is None else format(gpu['model_in_gpu_memory'], '.0%')}")
print(f"LLM answers: {fresh:,} fresh, {cached:,} from cache, {failed:,} failed | "
      f"{len(LLM_LINES):,} of {report['target_customers']:,} target customers sent to agents 3 + 4 | "
      f"agent 3 {decided} | notebook {report['notebook_minutes']} min")
print("VERDICT:", "LLM + GPU run" if (llm_ok and on_gpu) else
      ("LLM run, but the model is NOT fully on the GPU (slow): smaller MODEL or close other GPU programs" if gpu.get("gpu")
       else "LLM run, but nvidia-smi was not found: check the NVIDIA driver") if llm_ok else
      "NOT an LLM run: most decisions are the rule fallback (see the warnings of agents 3 / 4)")
if REMAP:
    print(f"note: {len(REMAP)} *_score table(s) were replaced by *_auto (step 4b): {sorted(REMAP)}")

In [ ]:
_rep = json.loads((AGENTS_DIR / "llm_gpu_report.json").read_text(encoding="utf-8"))
_rep["fast_mode"] = FAST
(AGENTS_DIR / "llm_gpu_report.json").write_text(json.dumps(_rep, indent=2, default=str), encoding="utf-8")
print(f"FAST_MODE added to the report: level {LEVEL}, {FAST['calls_saved_share']:.0%} of the per-customer calls saved"
      + (f", quality check {CHECK3['same_top_action']:.0%} same top action" if CHECK3 else ""))

## Final report (real run)

The dates used, the takers of last week's campaign, the propensities of agent 3, the offers per line, the LLM answers
and the speed. Written to `data/real25k/output/<D>/real_report_<RUN_SIZE>.json`. It also releases the "keep awake" of Windows.

In [ ]:
step("Final report")
_seg = pd.concat([pd.read_parquet(p, columns=[VAL, "W0_TARGETED", "W0_REWARD"])
                  for p in sorted((train_dir / T3_TRAIN / "segments").glob("*.parquet"))])
_tr = _seg[_seg[VAL].isin(TRAIN_VALS)]
_tg = int((_tr["W0_TARGETED"] == 1).sum())
_tk = int(((_tr["W0_TARGETED"] == 1) & (_tr["W0_REWARD"] == 1)).sum())
_spread = []
for _u, (_al, _ans) in ANS3.items():
    if ok3(_ans, _al):
        _s = np.clip(np.asarray(_ans["scores"], float), 0, None)
        _p = (1 - EPSILON) * (_s / _s.sum() if _s.sum() > 0 else np.full(len(_s), 1 / len(_s))) + EPSILON / len(_s)
        _spread.append(float(_p.max() - _p.min()))
_flat = float(np.mean(np.asarray(_spread) < 0.05)) if _spread else None
_ag = {a: s for a, s in STATS.items() if a not in ("G2 speed test", "G1 first request")}
_llm = {k: int(sum(s.get(k, 0) for s in _ag.values())) for k in ("calls", "cached", "retried", "errors")}
_no_llm = n_cust - len(LLM_LINES)
_fb3 = int((NBA["DECIDED_BY"] == "fallback_cell_policy").sum()) if len(NBA) else 0
_wall34 = sum(STATS.get(a, {}).get("wall_seconds", 0.0) for a in ("Agent 3 NBA", "Agent 4 NBO"))
_b = sum(k * n for k, n in TOPUP_STATS["before"].items()) / max(sum(TOPUP_STATS["before"].values()), 1)
_a = sum(k * n for k, n in TOPUP_STATS["after"].items()) / max(sum(TOPUP_STATS["after"].values()), 1)
REPORT = {
    "sizes": {"RUN_SIZE": RUN_SIZE, "TRAIN_LINES": TRAIN_LINES, "score_groups": list(SCORE_GROUPS)},
    "dates": {"D_last_purchase": RUN_DATE, "T_campaign": CAMPAIGN_DATE, "purchase_window_days": PURCHASE_DAYS},
    "training": {"lines": len(TRAIN_VALS), "targeted_last_week": _tg, "takers": _tk,
                 "taker_rate": round(_tk / _tg, 4) if _tg else None},
    "agent3_propensity": {"customers": int(len(NBA)),
                          **({k: round(float(v), 4) for k, v in NBA["PROBA"].describe()[["min", "25%", "50%", "75%", "max"]].items()}
                             if len(NBA) else {}),
                          "profiles_almost_flat_share": None if _flat is None else round(_flat, 3)},
    "offers_per_line": {"before_topup": round(_b, 2), "after_topup": round(_a, 2),
                        "brought_back_F04": TOPUP_STATS["brought_back_F04"], "still_under_6": TOPUP_STATS["still_short"]},
    "llm_answers": {"fresh": _llm["calls"], "cached": _llm["cached"], "retried": _llm["retried"], "failed": _llm["errors"]},
    "cell_policy": {"customers_not_sent_to_llm": int(_no_llm), "agent3_fallbacks": _fb3,
                    "share": round((_no_llm + _fb3) / max(n_cust, 1), 4)},
    "speed": {"customers": int(len(LLM_LINES)), "agents_3_4_minutes": round(_wall34 / 60, 1),
              "customers_per_minute": round(len(LLM_LINES) / (_wall34 / 60), 1) if _llm["calls"] and _wall34 > 30 else None,
              "notebook_minutes": round((time.time() - NB_START) / 60, 1)},
    "final_file": {"ready": bool(fin.get("ready")), "rows": int(fin.get("input_flytxt_rows", 0))},
}
(OUTPUT / f"real_report_{RUN_SIZE}.json").write_text(json.dumps(REPORT, indent=2, default=str), encoding="utf-8")
for _k, _v in REPORT.items():
    print(f"{_k:18s} {_v}")
if _flat is not None and _flat > 0.5:
    print(f"WARNING: agent 3 propensities are almost flat for {_flat:.0%} of the profiles (all actions scored alike): "
          "the LLM barely separates the actions -- try a stronger MODEL or a finer PROFILE_LEVEL")
if REPORT["cell_policy"]["share"] > 0:
    print(f"WARNING: {REPORT['cell_policy']['share']:.2%} of the customers got the cell policy (must be 0): see the failed answers in "
          f"{LLM_LOG.relative_to(ROOT)}")
if _llm["errors"]:
    print(f"WARNING: {_llm['errors']:,} LLM answers failed after {LLM_RETRIES} retries")
if sys.platform == "win32":
    ctypes.windll.kernel32.SetThreadExecutionState(0x80000000)
    print("Windows may sleep again")

## Proof: nothing came from the project folder

Every engine module was defined by a cell of this notebook, and no engine module uses vowpalwabbit (it may be preloaded by the environment).

In [ ]:
step("Proof")
# --- proof: everything ran from this notebook; vowpalwabbit may be preloaded by the environment, never used --------
import types as _types


def _from_vw(v):
    try:
        name = v.__name__ if isinstance(v, _types.ModuleType) else (getattr(v, "__module__", "") or "")
    except Exception:
        return False
    return str(name).split(".")[0] == "vowpalwabbit"


mods = sorted(m for m in sys.modules if m == "nbx" or m.startswith("nbx."))
from_disk = [m for m in mods if not getattr(sys.modules[m], "__fsdr_notebook__", False)]
vw_users = [m for m in mods if any(_from_vw(v) for v in list(vars(sys.modules[m]).values()))]
print(f"{len(mods)} engine modules, all defined in this notebook: {not from_disk}")
print("vowpalwabbit:", "not loaded" if "vowpalwabbit" not in sys.modules else
      ("preloaded by the environment (a .pth file), used by no engine module" if not vw_users else f"USED by {vw_users}"))
assert not from_disk, f"loaded from the project folder: {from_disk}"
assert not vw_users, f"an engine module uses vowpalwabbit: {vw_users}"
print("api / gradio loaded:", any(m in sys.modules for m in ("gradio", "fastapi", "uvicorn")))

## Time per step

The time of every step of this run against its estimate. The next run of the same `RUN_SIZE` uses these times as its estimates (`data/runs/_step_times.json`, step names and seconds only).

In [ ]:
step("Time per step")
step_report()